# Atlas Electoral de Costa Rica · Edición pública 38
## Elecciones de 1953–2026

**Autoría:** Juan Ignacio Garbanzo Fallas · Ciencia Política, Economía y Ciencia de Datos.  
**Proyecto:** Práctica Profesional · IFED–TSE.

Este notebook integra las fuentes, realiza los controles y genera la aplicación pública interactiva. Debe ejecutarse de principio a fin desde la raíz del proyecto, conservando sus bases y recursos compatibles.

| Producto | Ubicación |
|---|---|
| Aplicación principal | `Atlas_Electoral_Costa_Rica_1953_2026.html`, en la raíz |
| Datos derivados y descargas | `salidas/datos/` y archivos auxiliares en `salidas/` |

Las etapas de construcción y sus recursos utilizan nombres funcionales estables. Las fuentes originales no se sobrescriben.


## 1. Entorno, rutas y presentación

La carpeta de las bases puede cambiarse en `DATA_DIR`. No se utilizan rutas de la computadora del autor ni servicios externos para generar el HTML.

In [1]:
from pathlib import Path
import sys, json, hashlib, zipfile, importlib.metadata
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, FileLink

# Fuentes incorporadas sin carpetas de proyecto adicionales.
from pathlib import Path
import base64, hashlib, io, json, os, tempfile, types, zipfile

_ATLAS_BUNDLE_B64 = '...'
_ATLAS_BUNDLE_SHA256 = 'c167dc0d344e0bcdd1d2ed9148ba8ca0e3c3703c86b05886fdced61323f86e99'
PROJECT_DIR = Path.cwd().resolve()
_bundle = base64.b64decode(_ATLAS_BUNDLE_B64)
if hashlib.sha256(_bundle).hexdigest() != _ATLAS_BUNDLE_SHA256:
    raise RuntimeError('Los recursos incorporados no superaron la comprobación SHA-256.')
_created, _preserved = [], []
with zipfile.ZipFile(io.BytesIO(_bundle), 'r') as _zip:
    for _item in _zip.infolist():
        _relative = Path(_item.filename)
        if _relative.is_absolute() or '..' in _relative.parts:
            raise RuntimeError('Ruta no permitida dentro de los recursos incorporados.')
        _destination = PROJECT_DIR / _relative
        if _destination.exists():
            # Un archivo existente pertenece a la persona usuaria. No sustituir.
            _preserved.append(str(_relative))
            continue
        _destination.parent.mkdir(parents=True, exist_ok=True)
        _tmp_name = None
        try:
            with tempfile.NamedTemporaryFile('wb', dir=_destination.parent,
                    prefix='.atlas_recurso_', delete=False) as _out:
                _tmp_name = Path(_out.name)
                with _zip.open(_item) as _src:
                    import shutil
                    shutil.copyfileobj(_src, _out)
            os.replace(_tmp_name, _destination)
            _created.append(str(_relative))
        finally:
            if _tmp_name is not None and _tmp_name.exists():
                _tmp_name.unlink()
_manifest = PROJECT_DIR/'recursos'/'atlas_recursos.json'
_manifest.write_text(json.dumps({'version':'38',
    'bundle_sha256':_ATLAS_BUNDLE_SHA256,
    'archivos_incorporados':_created,
    'archivos_preexistentes_respetados':_preserved},
    indent=2,ensure_ascii=False),encoding='utf-8')
print('Atlas · carpeta de trabajo:',PROJECT_DIR)
print('Recursos nuevos:',len(_created),'· recursos previos conservados:',len(_preserved))
print('El HTML, los datos derivados y las figuras se guardarán en salidas/.')
os.chdir(PROJECT_DIR)
DATA_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / "salidas"
for folder in (OUTPUT_DIR, OUTPUT_DIR / "datos"):
    folder.mkdir(parents=True, exist_ok=True)

# Cargar el código de ESTE paquete, incluso si Jupyter tenía una versión anterior.
RESOURCE_DIR = PROJECT_DIR / "recursos"
sys.path.insert(0, str(RESOURCE_DIR))
for module_name in ("atlas_enhancements", "atlas_geography", "atlas_outcomes", "atlas_pipeline"):
    module_path = RESOURCE_DIR / (module_name + ".py")
    module = types.ModuleType(module_name)
    module.__file__ = str(module_path)
    sys.modules[module_name] = module
    exec(compile(module_path.read_text(encoding="utf-8"), str(module_path), "exec"), module.__dict__)
from atlas_pipeline import build_dataset, render_app, FILES, YEARS, PROVINCES

palette_path = PROJECT_DIR / "recursos" / "colores_partidos.json"
PARTY_COLORS = {r["partido"]: r["color"] for r in json.loads(palette_path.read_text(encoding="utf-8")) if r.get("color")}
PROVINCE_COLORS = {"San José":"#0072B2", "Alajuela":"#D55E00", "Cartago":"#009E73", "Heredia":"#8B55A2", "Guanacaste":"#B18A00", "Puntarenas":"#CB5D92", "Limón":"#3A8F9A"}
SEX_COLORS = {"Mujer":"#933B8F", "Hombre":"#167D8D"}
COLORS = ["#007f7a", "#3276c2", "#c29049", "#a55579", "#6266a4", "#5b8f42"]
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = COLORS
# El destino primario del notebook es el HTML. Evitar renderizar repetidamente
# decenas de Plotly dentro del cuaderno reduce memoria y tiempo de ejecución.
# Para previsualizar las figuras de EDA dentro de Jupyter, cambie a True.
NOTEBOOK_INLINE_PLOTS = False
_plotly_loaded = False

def show(fig, title=None):
    """Salida Plotly sin CDN, con estilo y descarga PNG."""
    global _plotly_loaded
    if not NOTEBOOK_INLINE_PLOTS: return
    for trace in fig.data:
        color = {**PARTY_COLORS, **PROVINCE_COLORS, **SEX_COLORS,
                 "Legislativa":"#126CB5", "Presidencial":"#D05A22"}.get(trace.name)
        if color:
            if hasattr(trace, "marker"): trace.marker.color = color
            if hasattr(trace, "line"): trace.line.color = color
        if getattr(trace, "type", None) == "scatter" and trace.name == "Presidencial":
            trace.line.dash = "dash"
            trace.marker.symbol = "diamond"
    fig.update_layout(font=dict(family="Arial", color="#172c3c"),
                      title=title or fig.layout.title.text,
                      margin=dict(l=65, r=25, t=60, b=60), height=460,
                      legend=dict(orientation="h", y=-0.22))
    display(HTML(fig.to_html(full_html=False, include_plotlyjs=not _plotly_loaded,
        config={"displaylogo": False, "responsive": True,
                "toImageButtonOptions": {"format": "png", "scale": 3}})))
    _plotly_loaded = True

print("Entorno:", {p: importlib.metadata.version(p) for p in ("pandas", "numpy", "openpyxl", "plotly")})


Atlas · carpeta de trabajo: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica
Recursos nuevos: 0 · recursos previos conservados: 77
El HTML, los datos derivados y las figuras se guardarán en salidas/.
Entorno: {'pandas': '2.3.3', 'numpy': '2.0.2', 'openpyxl': '3.1.5', 'plotly': '6.9.0'}


In [2]:
# Nombres de las siete bases: admitir nombres limpios y sufijos históricos sin tocar los archivos.
from pathlib import Path
import re, unicodedata

def _nombre_atlas(nombre):
    texto = ''.join(ch for ch in unicodedata.normalize('NFD', str(nombre).lower()) if unicodedata.category(ch) != 'Mn')
    return re.sub(r'[^a-z0-9]+', '_', texto).strip('_')

def resolver_bases_atlas(carpeta, archivos):
    carpeta = Path(carpeta)
    patrones = {
        'leg': [r'base.*electoral.*diput', r'resultados?.*legislat'],
        'pres': [r'base.*electoral.*presiden', r'resultados?.*presiden'],
        'cand': [r'candidat.*diput', r'base.*candidat.*legislat'],
        'formula': [r'candidat.*presiden', r'formula.*presiden'],
        'electas': [r'diputad.*elect', r'diputacion.*elect', r'diputaciones.*elect'],
        'diseno': [r'disen.*papelet', r'papelet.*diput'],
        'cancelados': [r'partid.*cancelad'],
    }
    disponibles = sorted(p for p in carpeta.iterdir() if p.is_file() and p.suffix.lower() in ('.xlsx', '.xlsm'))
    candidatos = {p: _nombre_atlas(p.stem) for p in disponibles}
    seleccion = {}
    for rol, anteriores in archivos.items():
        anterior = carpeta / anteriores
        if anterior.is_file():
            seleccion[rol] = anterior.name
            continue
        opciones = [p for p, nombre in candidatos.items() if any(re.search(pat, nombre) for pat in patrones[rol])]
        # Excluir otras familias cuando se comparten palabras.
        if rol == 'leg': opciones = [p for p in opciones if 'candidat' not in candidatos[p] and 'electas' not in candidatos[p]]
        if rol == 'pres': opciones = [p for p in opciones if 'candidat' not in candidatos[p]]
        if rol == 'cand': opciones = [p for p in opciones if 'presiden' not in candidatos[p] and 'electas' not in candidatos[p]]
        if rol == 'electas': opciones = [p for p in opciones if 'candidat' not in candidatos[p]]
        if rol == 'diseno': opciones = [p for p in opciones if 'presiden' not in candidatos[p]]
        if len(opciones) == 1:
            seleccion[rol] = opciones[0].name
        elif not opciones:
            raise FileNotFoundError(f'No se encontró la base «{rol}» en {carpeta}. Archivos Excel encontrados: {[x.name for x in disponibles]}')
        else:
            raise RuntimeError(f'Hay varias bases compatibles con «{rol}»: {[x.name for x in opciones]}. Dejá una sola versión para evitar usar datos equivocados.')
    assert len(set(seleccion.values())) == len(archivos), 'Se intentó asignar el mismo Excel a dos bases distintas.'
    archivos.update(seleccion)  # Mismo diccionario importado desde atlas_pipeline.
    print('Siete fuentes localizadas (sin cambiar el nombre de tus Excel):')
    for rol, nombre in archivos.items(): print(f'  {rol:12} → {nombre}')

resolver_bases_atlas(DATA_DIR, FILES)


Siete fuentes localizadas (sin cambiar el nombre de tus Excel):
  leg          → Base_Electoral_Diputaciones_1953-2026.xlsx
  pres         → Base_Electoral_Presidencial_1953-2026.xlsx
  cand         → Base_Final_Candidaturas_Diputaciones_1953-2026.xlsx
  formula      → Base_Final_Candidaturas_Presidenciales_1953-2026.xlsx
  electas      → Base_Final_Diputaciones_Electas_1953-2026.xlsx
  diseno       → Base_Final_Diseno_Papeletas_Diputaciones_1953-2026.xlsx
  cancelados   → Base_partidos_cancelados.xlsx


In [3]:
# Integración compatible e idempotente para el Atlas Electoral
# Sustituye los Integración s // basados en coincidencias literales.
# Mantiene el diseño y los datos, mejora exportación PNG/SVG de personas,
# marca hitos del TSE sobre la serie de género y añade dos tasas junto a los flujos.
from pathlib import Path
import shutil

asset = PROJECT_DIR / 'recursos' / 'atlas_final.js'
if not asset.is_file():
    raise FileNotFoundError(f'No existe {asset}. Ajuste PROJECT_PATH en Entorno: el notebook debe ejecutarse desde la raíz del Atlas.')
raw = asset.read_text(encoding='utf-8')
required = ['views.outcomes=', 'Composición por sexo a través del tiempo', 'transiciones_resultados']
missing = [marker for marker in required if marker not in raw]
if missing:
    raise RuntimeError('El archivo atlas_final.js no parece pertenecer a una versión compatible del Atlas Electoral (V2.0): '+', '.join(missing)+'; encontrado en '+str(asset))
PATCH_MARKER = 'ATLAS_V5_COMPATIBLE_PATCH'
if PATCH_MARKER not in raw:
    extension = 'async function atlasV5ExportNode(id, format=\'png\', filename=\'Atlas_Exportacion\'){\n // ATLAS_VECTOR_EXPORT_V5: elementos SVG nativos, sin HTML/CSS incrustado.\n try {\n  const root=document.getElementById(id);\n  if(!root)throw new Error(\'No se encontró la sección de personas seleccionada.\');\n  const cardNodes=Array.from(root.querySelectorAll(\'.person-result\'));\n  if(!cardNodes.length)throw new Error(\'Esta selección no contiene personas que se puedan exportar.\');\n  const compact=s=>String(s??\'\').replace(/\\s+/g,\' \').trim();\n  const readCard=el=>{\n   const name=compact(el.querySelector(\'.person-link\')?.textContent||el.querySelector(\':scope > span\')?.textContent||\'Persona sin nombre\');\n   const badges=Array.from(el.querySelectorAll(\'.outcome-badge\')).map(n=>compact(n.textContent)).filter(Boolean);\n   const pairs=Array.from(el.querySelectorAll(\'.career-pair\')).map(n=>compact(n.textContent)).filter(Boolean);\n   const extras=Array.from(el.querySelectorAll(\'.mini,.route\')).filter(n=>!n.closest(\'.career-pair\')).map(n=>compact(n.textContent)).filter(Boolean);\n   return {name,badges,details:[...pairs,...extras]};\n  };\n  const people=cardNodes.map(readCard);\n  const heading=compact(root.querySelector(\'h3\')?.textContent||\'Personas de la selección\');\n  const scope=compact(root.querySelector(\'.card-head .mini\')?.textContent||\'\');\n  const methodology=compact(root.querySelector(\':scope > .chart-note\')?.textContent||\'\');\n  const cols=people.length>36?3:people.length>5?2:1;\n  const width=cols===3?2100:cols===2?1560:1120;\n  const margin=55, gap=22, cardWidth=(width-2*margin-gap*(cols-1))/cols;\n  const wrap=(value,max)=>{\n   const words=compact(value).split(\' \').filter(Boolean),out=[];let line=\'\';\n   for(const word of words){\n    if(word.length>max){if(line){out.push(line);line=\'\';}for(let i=0;i<word.length;i+=max)out.push(word.slice(i,i+max));continue;}\n    if((line?line.length+1:0)+word.length>max){out.push(line);line=word;}else line+=(line?\' \':\'\')+word;\n   }\n   if(line)out.push(line);\n   return out.length?out:[\'\'];\n  };\n  const maxTitle=cols===3?43:cols===2?59:100;\n  const maxDetail=cols===3?48:cols===2?65:105;\n  const prepared=people.map(p=>({\n   name:wrap(p.name,maxTitle),\n   badges:p.badges.flatMap(s=>wrap(s,maxDetail)),\n   details:p.details.flatMap(s=>wrap(s,maxDetail)),\n  }));\n  const cardHeight=p=>24+p.name.length*29+(p.badges.length?8+p.badges.length*23:0)+(p.details.length?9+p.details.length*24:0)+22;\n  const headScope=wrap(scope,Math.max(55,Math.floor((width-2*margin)/11)));\n  const headHeight=184+Math.max(0,headScope.length-1)*25;\n  const placement=[];let nextY=headHeight;\n  for(let i=0;i<prepared.length;i+=cols){\n   const slice=prepared.slice(i,i+cols);\n   const rh=Math.max(...slice.map(cardHeight));\n   slice.forEach((p,j)=>placement.push({person:p,x:margin+j*(cardWidth+gap),y:nextY,height:rh}));\n   nextY+=rh+gap;\n  }\n  const noteLines=methodology?wrap(methodology,Math.floor((width-2*margin)/10)):[];\n  const totalHeight=Math.ceil(nextY+46+noteLines.length*24+60);\n  const ns=\'http://www.w3.org/2000/svg\';\n  const svg=document.createElementNS(ns,\'svg\');\n  svg.setAttribute(\'xmlns\',ns);svg.setAttribute(\'width\',String(width));\n  svg.setAttribute(\'height\',String(totalHeight));svg.setAttribute(\'viewBox\',`0 0 ${width} ${totalHeight}`);\n  svg.setAttribute(\'role\',\'img\');svg.setAttribute(\'aria-label\',heading);\n  const draw=(tag,attrs,content,parent=svg)=>{\n   const e=document.createElementNS(ns,tag);\n   for(const [key,value] of Object.entries(attrs||{}))e.setAttribute(key,String(value));\n   if(content!==null&&content!==undefined)e.textContent=String(content);\n   parent.appendChild(e);return e;\n  };\n  const tx=(x,y,value,size=18,color=\'#274251\',weight=400)=>draw(\'text\',{x,y,fill:color,\'font-family\':\'Arial, Helvetica, sans-serif\',\'font-size\':size,\'font-weight\':weight},value);\n  draw(\'rect\',{x:0,y:0,width,height:totalHeight,fill:\'#f5f8fa\'});\n  draw(\'rect\',{x:0,y:0,width,height:10,fill:\'#137f7a\'});\n  tx(margin,51,\'ATLAS ELECTORAL DE COSTA RICA · 1953–2026\',20,\'#167a78\',700);\n  tx(margin,94,heading,29,\'#173143\',700);\n  headScope.forEach((s,i)=>tx(margin,130+i*25,s,17,\'#466474\',400));\n  tx(margin,headHeight-20,`PERSONAS EN ESTA PÁGINA: ${people.length} · Filtros indicados en el encabezado`,17,\'#3f6170\',700);\n  draw(\'line\',{x1:margin,y1:headHeight-7,x2:width-margin,y2:headHeight-7,stroke:\'#bfd4d6\',\'stroke-width\':2});\n  placement.forEach(({person:p,x,y,height})=>{\n   draw(\'rect\',{x,y,width:cardWidth,height,rx:11,fill:\'#ffffff\',stroke:\'#cfdee5\',\'stroke-width\':1.5});\n   draw(\'rect\',{x,y,width:7,height,rx:3,fill:\'#137f7a\'});\n   let baseline=y+36;\n   p.name.forEach(line=>{tx(x+24,baseline,line,21,\'#153a4c\',700);baseline+=29;});\n   if(p.badges.length){baseline+=5;p.badges.forEach(line=>{tx(x+24,baseline,line,16,\'#0c746b\',700);baseline+=23;});}\n   if(p.details.length){baseline+=6;p.details.forEach(line=>{tx(x+24,baseline,line,16,\'#415c69\',400);baseline+=24;});}\n  });\n  const footY=nextY+17;\n  draw(\'line\',{x1:margin,y1:footY-16,x2:width-margin,y2:footY-16,stroke:\'#d4e1e7\',\'stroke-width\':1.5});\n  noteLines.forEach((line,i)=>tx(margin,footY+i*24,line,15,\'#526874\'));\n  tx(margin,totalHeight-27,\'Fuente: Atlas electoral de Costa Rica · Consulta filtrada · Resultados descriptivos del corpus\',15,\'#536a77\');\n  const serialized=new XMLSerializer().serializeToString(svg);\n  const vectorBlob=new Blob([serialized],{type:\'image/svg+xml;charset=utf-8\'});\n  const safeName=(filename||\'Atlas_Personas\').replace(/[^\\p{L}\\p{N}_-]+/gu,\'_\');\n  const saveBlob=(blob,ext)=>{\n   const u=URL.createObjectURL(blob),a=document.createElement(\'a\');\n   a.href=u;a.download=`${safeName}.${ext}`;document.body.appendChild(a);a.click();a.remove();\n   setTimeout(()=>URL.revokeObjectURL(u),2000);\n  };\n  if(format===\'svg\'){saveBlob(vectorBlob,\'svg\');return;}\n  if(format!==\'png\')throw new Error(\'Formato de exportación desconocido.\');\n  const imageUrl=URL.createObjectURL(vectorBlob);\n  try {\n   const bitmap=new Image();\n   await new Promise((resolve,reject)=>{\n    bitmap.onload=resolve;\n    bitmap.onerror=()=>reject(new Error(\'El navegador no pudo cargar la representación vectorial de los nombres.\'));\n    bitmap.src=imageUrl;\n   });\n   const factor=Math.max(.6,Math.min(2,15000/totalHeight,Math.sqrt(45000000/(width*totalHeight))));\n   const canvas=document.createElement(\'canvas\');\n   canvas.width=Math.max(1,Math.round(width*factor));\n   canvas.height=Math.max(1,Math.round(totalHeight*factor));\n   const ctx=canvas.getContext(\'2d\');\n   if(!ctx)throw new Error(\'No se pudo inicializar el lienzo de imagen.\');\n   ctx.fillStyle=\'#ffffff\';ctx.fillRect(0,0,canvas.width,canvas.height);\n   ctx.drawImage(bitmap,0,0,canvas.width,canvas.height);\n   const raster=await new Promise((resolve,reject)=>canvas.toBlob(b=>b?resolve(b):reject(new Error(\'El navegador no pudo convertir el gráfico en PNG.\')),\'image/png\'));\n   saveBlob(raster,\'png\');\n  } finally {URL.revokeObjectURL(imageUrl);}\n }catch(error){console.error(\'Atlas · fallo de exportación:\',error);alert(\'No se pudo exportar la imagen: \'+(error?.message||error));}\n}\n\n/* ATLAS_V5_COMPATIBLE_PATCH\n * Ampliación no destructiva de la última capa de la interfaz.\n * No depende del texto exacto de atlas_final.js; se puede aplicar a clones V2.0\n * o a copias parcialmente parcheadas. No sustituye la aplicación existente.\n */\n(function atlasV5Enhance(){\n  const stamp=\'ATLAS_V5_COMPATIBLE_PATCH\';\n  if (typeof views===\'undefined\' || typeof views.outcomes!==\'function\' || typeof plot!==\'function\') {\n    throw new Error(\'El paquete no expone las funciones esperadas del Atlas. Compruebe que recursos/atlas_final.js corresponda a la versión 2.0.\');\n  }\n  const milestones=[\n    {year:1998,label:\'Cuota 40 %\',detail:\'Ley 7653 (1996): cuota mínima de candidaturas\'},\n    {year:2002,label:\'Puestos elegibles\',detail:\'TSE 1863-1999: aplicación en posiciones elegibles\'},\n    {year:2014,label:\'Paridad vertical\',detail:\'Código Electoral de 2009: paridad y alternancia\'},\n    {year:2018,label:\'Paridad horizontal\',detail:\'TSE 3603-E8-2016: encabezamientos provinciales\'}\n  ];\n  // Amplía opciones del gráfico EXISTENTE sin modificar barras, datos ni resultados.\n  const previousPlot=plot;\n  plot=function(titleText,traces,rows,opts={}){\n    if(titleText===\'Composición por sexo a través del tiempo\'){\n      const years=new Set(traces.flatMap(t=>Array.isArray(t.x)?t.x.map(Number):[]));\n      const active=milestones.filter(m=>years.has(m.year));\n      const configured=Array.isArray(opts.layout?.shapes)&&opts.layout.shapes.some(s=>active.some(m=>Number(s.x0)===m.year && s.type===\'line\'));\n      if(active.length && !configured){\n        const shapes=active.map(m=>({type:\'line\',xref:\'x\',yref:\'paper\',x0:m.year,x1:m.year,y0:0,y1:1,line:{color:\'#b28744\',width:1.75,dash:\'dot\'}}));\n        const annotations=active.map((m,i)=>({xref:\'x\',yref:\'paper\',x:m.year,y:i%2?1.18:1.09,text:`${m.year} · ${m.label}`,showarrow:false,textangle:-30,font:{size:10,color:\'#755c32\'},bgcolor:\'rgba(255,250,241,.97)\',bordercolor:\'rgba(178,135,68,.36)\',borderpad:3}));\n        opts={...opts,height:Math.max(opts.height||420,515),layout:{...(opts.layout||{}),shapes:[...(opts.layout?.shapes||[]),...shapes],annotations:[...(opts.layout?.annotations||[]),...annotations],margin:{...(opts.layout?.margin||{}),t:145}},note:[opts.note,\'Líneas doradas: hitos en la elección de aplicación (1998: cuota 40 %; 2002: puestos elegibles; 2014: paridad vertical; 2018: paridad horizontal). Normas: Ley 7653/1996; TSE 1863-1999; Código Electoral 2009; TSE 3603-E8-2016. La coincidencia temporal no demuestra causalidad.\'].filter(Boolean).join(\' \')};\n      }\n    }\n    if(titleText===\'Evolución del indicador territorial\' && (local.rmetric||\'indice_representacion\')===\'indice_representacion\'){\n      const hasLine=opts.layout?.shapes?.some(s=>s.yref===\'y\'&&Number(s.y0)===1&&Number(s.y1)===1);\n      if(!hasLine){\n        opts={...opts,layout:{...(opts.layout||{}),shapes:[...(opts.layout?.shapes||[]),{type:\'line\',xref:\'paper\',yref:\'y\',x0:0,x1:1,y0:1,y1:1,line:{color:\'#b45353\',width:2,dash:\'dot\'}}]},note:[opts.note,\'La línea roja y = 1 expresa igualdad descriptiva entre peso en escaños y peso en padrón, no la regla constitucional de distribución por población.\'].filter(Boolean).join(\' \')};\n      }\n    }\n    return previousPlot(titleText,traces,rows,opts);\n  };\n\n  // Los controles se montan sobre el bloque VISIBLE, cualquiera que sea su\n  // implementación interna. Esto evita modificar versiones antiguas de views.outcomes.\n  function ensurePersonExports(){\n    const roots=Array.from(document.querySelectorAll(\'#content .card\'));\n    const wanted=new Map([\n      [\'Personas del flujo seleccionado\',\'Atlas_ExitoElectoral_FlujoSeleccionado\'],\n      [\'Quiénes forman este puente de carrera\',\'Atlas_ExitoElectoral_PuenteCarrera\'],\n      [\'Personas del antecedente seleccionado\',\'Atlas_ExitoElectoral_AntecedenteSeleccionado\'],\n      [\'Personas en el antecedente seleccionado\',\'Atlas_ExitoElectoral_AntecedenteSeleccionado\']\n    ]);\n    roots.forEach(root=>{\n      const heading=root.querySelector(\'h3\');\n      if(!heading)return;\n      const name=heading.textContent.trim();\n      if(!wanted.has(name) || root.dataset.atlasV5Export===\'yes\')return;\n      root.dataset.atlasV5Export=\'yes\';\n      if(!root.id)root.id=\'atlas_v5_people_\'+Array.from(root.parentNode.children).indexOf(root)+\'_\'+Math.floor(Math.random()*100000);\n      // Reemplazar solamente los controles exportadores del bloque de personas.\n      // Los botones de las figuras Plotly y de las tablas no se modifican.\n      let head=heading.closest(\'.card-head\');\n      if(!head){head=document.createElement(\'div\');head.className=\'card-head\';heading.parentNode.insertBefore(head,heading);head.appendChild(heading);}\n      head.querySelectorAll(\'.exports\').forEach(el=>el.remove());\n      const tools=document.createElement(\'div\');tools.className=\'exports\';\n      for(const kind of [\'PNG\',\'SVG\']){\n        const b=document.createElement(\'button\');b.type=\'button\';b.textContent=kind;\n        b.title=`Descargar los nombres y trayectorias de esta página como ${kind}`;\n        b.addEventListener(\'click\',()=>atlasV5ExportNode(root.id,kind.toLowerCase(),wanted.get(name)));\n        tools.appendChild(b);\n      }\n      head.appendChild(tools);\n      const mini=root.querySelector(\'.card-head .mini\');\n      if(!mini){\n        const scope=document.createElement(\'span\');scope.className=\'mini\';\n        scope.textContent=`${state.from}–${state.to} · ${state.province} · cargo: ${local.ocargo||\'Todos\'} · partido: ${local.oparty||\'Todos\'} · sexo: ${local.osex||\'Todos\'}`;\n        heading.insertAdjacentElement(\'afterend\',scope);\n      }\n    });\n  }\n\n  function addTransitionRates(){\n    const titles=Array.from(document.querySelectorAll(\'#content .card h3\')).map(x=>x.textContent.trim());\n    if(titles.includes(\'Tasa de elección posterior según continuidad de partido\'))return;\n    const rows=typeof atlasFilteredOutcomes===\'function\'?atlasFilteredOutcomes():[];\n    const ids=new Set(rows.map(r=>r.id_persona));\n    let transitions=(D.transiciones_resultados||[]).filter(r=>r.anio>=state.from&&r.anio<=state.to&&ids.has(r.id_persona));\n    if(state.province!==\'Nacional\')transitions=transitions.filter(r=>r.provincia_origen===state.province||r.provincia_destino===state.province);\n    add(\'<div class="extension-heading">LECTURA PORCENTUAL DE LOS FLUJOS · PARTIDO Y PROVINCIA</div>\');\n    add(\'<div class="clarity-box"><b>Complemento, no sustitución.</b> Los diagramas de flujo muestran personas y transiciones. Las dos tasas siguientes se calculan con el número de transiciones como denominador; una persona puede aportar varias. Son asociaciones descriptivas, no pruebas causales.</div>\');\n    const context=`${state.from}–${state.to} · ${state.province} · Cargo: ${local.ocargo||\'Todos\'} · Sexo: ${local.osex||\'Todos\'} · Partido: ${labelText(local.oparty||\'Todos\')}`;\n    const one=(which,title)=>{\n      const a=which===\'provincia\'?transitions.filter(r=>r.provincia_origen!==\'Nacional\'&&r.provincia_destino!==\'Nacional\'):transitions;\n      const key=which===\'provincia\'?\'cambia_provincia\':\'cambia_partido\';\n      const groups=[[`Mantiene ${which}`,false],[`Cambia ${which}`,true]];\n      const stats=groups.map(([grupo,change])=>{\n        const g=a.filter(r=>Boolean(r[key])===change),won=g.filter(r=>Boolean(r.electa_destino)).length;\n        return {grupo,transiciones:g.length,electas_despues:won,no_electas_despues:g.length-won,tasa:g.length?100*won/g.length:null,personas:uniq(g.map(r=>r.id_persona)).length};\n      });\n      const validRows=stats.filter(x=>x.transiciones>0);\n      if(!validRows.length){note(`No hay transiciones para ${title} con los filtros actuales.`);return;}\n      plot(title,[{type:\'bar\',orientation:\'h\',x:validRows.map(r=>r.tasa),y:validRows.map(r=>r.grupo),text:validRows.map(r=>`${num(r.tasa,1)} % · ${r.electas_despues}/${r.transiciones}`),textposition:\'outside\',marker:{color:validRows.map(r=>r.grupo.startsWith(\'Mantiene\')?\'#168278\':\'#637cac\')},customdata:validRows.map(r=>[r.electas_despues,r.transiciones,r.personas]),hovertemplate:\'%{y}<br>Electas: %{customdata[0]}/%{customdata[1]} transiciones · %{customdata[2]} personas<br>%{x:.2f} %<extra></extra>\'}],validRows,{height:310,xTitle:\'Transiciones con elección posterior / transiciones del grupo (%)\',subtitle:context,note:\'N = transiciones registradas entre elecciones distintas. El gráfico complementa el Sankey; mantener el partido o la provincia no implica causalidad ni una recomendación individual.\'});\n      table(stats,[\'grupo\',\'transiciones\',\'electas_despues\',\'no_electas_despues\',\'tasa\',\'personas\'],`Auditoría · ${title}`);\n    };\n    one(\'partido\',\'Tasa de elección posterior según continuidad de partido\');\n    one(\'provincia\',\'Tasa de elección posterior según continuidad de provincia\');\n  }\n  const previousOutcomes=views.outcomes;\n  views.outcomes=function(...args){\n    const result=previousOutcomes.apply(this,args);\n    ensurePersonExports();\n    addTransitionRates();\n    return result;\n  };\n})();\n'
    asset.write_text(raw+'\n'+extension+'\n', encoding='utf-8')
    print('aplicada en', asset)
else:
    print('ya estaba aplicada: no se ha duplicado ningún gráfico ni control.')
print('Versión compatible · El HTML se generará al ejecutar la celda 20 del notebook.')


ya estaba aplicada: no se ha duplicado ningún gráfico ni control.
Versión compatible · El HTML se generará al ejecutar la celda 20 del notebook.


In [4]:
# Claridad visual para el artículo
# Mejora legibilidad del módulo de género y las tasas de continuidad
# sin reemplazar la lógica del Atlas. Es idempotente: si ya fue aplicado,
# no duplica cambios.
from pathlib import Path
import shutil

asset = PROJECT_DIR / 'recursos' / 'atlas_final.js'
if not asset.is_file():
    raise FileNotFoundError(f'No existe {asset}. Ajuste PROJECT_PATH en Entorno: el notebook debe ejecutarse desde la raíz del Atlas.')
raw = asset.read_text(encoding='utf-8')
required = ['views.genero=', 'views.outcomes=', 'Composición por sexo a través del tiempo']
missing = [marker for marker in required if marker not in raw]
if missing:
    raise RuntimeError('El archivo atlas_final.js no parece pertenecer a una versión compatible del Atlas: '+', '.join(missing)+'; encontrado en '+str(asset))
PATCH_MARKER = 'ATLAS_V6_READABILITY_PATCH'
if PATCH_MARKER not in raw:
    extension = r'''
/* ATLAS_V6_READABILITY_PATCH
 * Mejoras de legibilidad para el artículo: género y éxito electoral.
 */
(function atlasV6ReadabilityPatch(){
  const stamp='ATLAS_V6_READABILITY_PATCH';
  if(typeof views==='undefined' || typeof plot!=='function') return;
  const milestones=[
    {year:1998,label:'1998 · Cuota 40 %',detail:'Ley 7653 (1996): cuota mínima de candidaturas'},
    {year:2002,label:'2002 · Puestos elegibles',detail:'TSE 1863-1999: aplicación en posiciones elegibles'},
    {year:2014,label:'2014 · Paridad vertical',detail:'Código Electoral de 2009: paridad y alternancia'},
    {year:2018,label:'2018 · Paridad horizontal',detail:'TSE 3603-E8-2016: encabezamientos provinciales'}
  ];
  const previousPlot=plot;
  plot=function(titleText,traces,rows,opts={}){
    let title=titleText;
    let nextTraces=Array.isArray(traces)?traces.map(t=>({...t})):traces;
    let nextOpts={...(opts||{})};
    const sexTitleMatch=String(titleText||'').startsWith('Composición por sexo');
    if(sexTitleMatch){
      const pop=(typeof local!=='undefined' && local.sexpop)?local.sexpop:'';
      const party=(typeof local!=='undefined' && local.sexparty)?labelText(local.sexparty):'Todos';
      if(titleText==='Composición por sexo a través del tiempo' && pop) title=`Composición por sexo a través del tiempo · ${pop}`;
      if(Array.isArray(nextTraces)){
        nextTraces=nextTraces.map(t=>({
          ...t,
          textfont:{size:10,color:'#ffffff'},
          insidetextanchor:'middle',
          cliponaxis:false
        }));
      }
      if(String(title).includes('a través del tiempo')){
        const years=new Set((Array.isArray(nextTraces)?nextTraces:[]).flatMap(t=>Array.isArray(t.x)?t.x.map(Number):[]));
        const active=milestones.filter(m=>years.has(m.year));
        const shapes=active.map(m=>({type:'line',xref:'x',yref:'paper',x0:m.year,x1:m.year,y0:0,y1:1,line:{color:'#b28744',width:1.75,dash:'dot'}}));
        const yRows=[1.08,1.14,1.08,1.14];
        const annotations=active.map((m,i)=>({xref:'x',yref:'paper',x:m.year,y:yRows[i%yRows.length],text:m.label,showarrow:false,textangle:0,xanchor:'center',font:{size:10,color:'#755c32'},bgcolor:'rgba(255,250,241,.98)',bordercolor:'rgba(178,135,68,.38)',borderpad:3}));
        nextOpts={
          ...nextOpts,
          height:Math.max(nextOpts.height||420,560),
          subtitle:[nextOpts.subtitle, `${state.from}–${state.to} · ${state.province} · Población: ${pop||'No especificada'} · Partido: ${party}`].filter(Boolean).join(' · '),
          layout:{
            ...(nextOpts.layout||{}),
            barmode:'stack',
            yaxis:{...((nextOpts.layout||{}).yaxis||{}), range:[0,100]},
            shapes:[...(((nextOpts.layout||{}).shapes)||[]), ...shapes],
            annotations:[...(((nextOpts.layout||{}).annotations)||[]), ...annotations],
            margin:{...(((nextOpts.layout||{}).margin)||{}), t:170}
          },
          note:[nextOpts.note,'Líneas doradas: hitos de la normativa electoral sobre participación de mujeres. Su coincidencia con cambios en la serie es descriptiva; no demuestra causalidad por sí sola.'].filter(Boolean).join(' ')
        };
      } else {
        nextOpts={...nextOpts, subtitle:[nextOpts.subtitle, `${state.from}–${state.to} · ${state.province} · Población: ${pop||'No especificada'} · Partido: ${party}`].filter(Boolean).join(' · ')};
      }
    }
    if(titleText==='Tasa de elección posterior según continuidad de partido'){
      title='Tasa de elección posterior según continuidad de partido · candidaturas posteriores';
      nextTraces=(Array.isArray(nextTraces)?nextTraces:[nextTraces]).map(t=>({...t,textposition:'outside',cliponaxis:false}));
      nextOpts={...nextOpts,xTitle:'Porcentaje de transiciones del grupo que terminan en una candidatura electa (%)',subtitle:[nextOpts.subtitle,'Lectura: de todas las transiciones observadas en cada grupo, ¿qué porcentaje terminó en una candidatura electa posterior?'].filter(Boolean).join(' · '),note:[nextOpts.note,'N del rótulo = candidaturas electas posteriores / transiciones observadas dentro del grupo.'].filter(Boolean).join(' '),height:340};
    }
    if(titleText==='Tasa de elección posterior según continuidad de provincia'){
      title='Tasa de elección posterior según continuidad de provincia · candidaturas posteriores';
      nextTraces=(Array.isArray(nextTraces)?nextTraces:[nextTraces]).map(t=>({...t,textposition:'outside',cliponaxis:false}));
      nextOpts={...nextOpts,xTitle:'Porcentaje de transiciones del grupo que terminan en una candidatura electa (%)',subtitle:[nextOpts.subtitle,'Lectura: de todas las transiciones observadas en cada grupo, ¿qué porcentaje terminó en una candidatura electa posterior?'].filter(Boolean).join(' · '),note:[nextOpts.note,'N del rótulo = candidaturas electas posteriores / transiciones observadas dentro del grupo.'].filter(Boolean).join(' '),height:340};
    }
    return previousPlot(title,nextTraces,rows,nextOpts);
  };

  const previousGenero=views.genero;
  views.genero=function(...args){
    const result=previousGenero.apply(this,args);
    const pop=local.sexpop||'Diputación propietaria';
    let base=[], outcomeApplicable=true;
    if(pop==='Diputación propietaria')base=period(D.candidaturas).filter(r=>r.tipo==='Propietaria');
    else if(pop==='Suplencia legislativa'){base=period(D.candidaturas).filter(r=>r.tipo==='Suplencia'); outcomeApplicable=false;}
    else if(pop==='Diputación electa'){base=period(D.electas); outcomeApplicable=false;}
    else base=period(D.formulas,false).filter(r=>r.cargo===pop);
    const parties=['Todos',...uniq(base.map(r=>r.partido).filter(Boolean)).sort()];
    const party=(parties.includes(local.sexparty)?local.sexparty:'Todos');
    base=base.filter(r=>party==='Todos'||r.partido===party);
    const makeComp=(rows,title,noteText)=>{
      const ys=yearsVisible();
      const comp=ys.flatMap(anio=>{const g=rows.filter(r=>r.anio===anio), den=g.filter(r=>['Mujer','Hombre'].includes(r.sexo)).length; return ['Mujer','Hombre'].map(sexo=>({anio,sexo,registros:g.filter(r=>r.sexo===sexo).length,total_con_sexo:den,porcentaje:pct(g.filter(r=>r.sexo===sexo).length,den)}));});
      const visible=ys.filter(y=>comp.some(r=>r.anio===y&&r.total_con_sexo>0));
      if(!visible.length){note(`No hay registros suficientes para “${title}” con los filtros actuales.`); return;}
      plot(title,['Mujer','Hombre'].map(sexo=>({type:'bar',name:sexo,x:visible,y:visible.map(y=>comp.find(r=>r.anio===y&&r.sexo===sexo)?.porcentaje??0),customdata:visible.map(y=>comp.find(r=>r.anio===y&&r.sexo===sexo)?.registros??0),marker:{color:sexColors[sexo]},text:visible.map(y=>{const z=comp.find(r=>r.anio===y&&r.sexo===sexo); return `${num(z?.porcentaje??0,1)}% · N=${z?.registros??0}`;}),hovertemplate:'%{x}<br>'+sexo+': %{customdata} registros · %{y:.1f}%<extra></extra>'})),comp,{height:420,yTitle:'Composición (%)',layout:{barmode:'stack',yaxis:{range:[0,100]}},note:noteText});
    };
    if(outcomeApplicable){
      add('<div class="atlas17-subhead">Lectura anual de electas y no electas por sexo</div>');
      makeComp(base.filter(r=>Boolean(r.electa)),'Composición por sexo entre candidaturas electas','Cada columna suma 100% entre Mujer y Hombre solo dentro de las candidaturas electas de la población seleccionada.');
      makeComp(base.filter(r=>!Boolean(r.electa)),'Composición por sexo entre candidaturas no electas','Cada columna suma 100% entre Mujer y Hombre solo dentro de las candidaturas no electas de la población seleccionada.');
    }
    return result;
  };
})();
'''
    asset.write_text(raw + '\n' + extension + '\n', encoding='utf-8')
    print('✅ Parche de claridad aplicado en', asset)
else:
    print('ℹ️ El parche ya estaba aplicado; no se realizaron cambios.')



ℹ️ El parche ya estaba aplicado; no se realizaron cambios.


### Visualizaciones de votos y escaños en el HTML

El notebook integra directamente la vista **«Votos y escaños»**, después de **«Padrón y escaños»**, en la aplicación HTML del Atlas. No necesita instalar archivos `.js` adicionales.

La participación del voto válido correspondiente a partidos con y sin diputaciones aparece en **un gráfico de tallos y puntos (lollipop), centrado en el porcentaje sin diputaciones, cuyo complemento es el voto con diputaciones**, no en barras ni en cantidades absolutas. Se mantienen las comparaciones entre votos y escaños del líder legislativo y del partido presidencial electo. Las tablas pueden exportarse en CSV y los gráficos en PNG/SVG.

También corrige la primera carga de «Sexo y representación», agrega hitos institucionales a las series legislativas de candidaturas electas y no electas, y reconoce nombres depurados de las ocho bases Excel.


In [5]:
# INTEGRACIÓN DE REPRESENTACIÓN EN LA APLICACIÓN HTML
# Todo el JavaScript nuevo está incrustado en ESTE notebook; no se requieren
# ningún archivo JavaScript adicional. Se conserva la plantilla del Atlas.
from pathlib import Path
import re

ATLAS_V7_INLINE_JS = r'''
/* ATLAS_V7_REPRESENTACION_HTML
   Este módulo transforma las mismas tablas oficiales embebidas en el HTML.
   No depende del notebook, de imágenes estáticas, de internet ni de un servidor.
   Las tres series son nacionales; los controles Desde, Hasta y Elección de
   detalle son efectivos. La clasificación por partido con escaños se refiere
   a obtener al menos una diputación en el conjunto del país en ese año.
*/
(function(){
  'use strict';
  const sColor='#c56c4f',rColor='#137d79',sColor2='#b58a45';
  const alias={'LIBERACION NACIONAL':'PLN','UNION NACIONAL':'UN','UNIFICACION NACIONAL':'PUN','UNIDAD':'UNIDAD','UNIDAD SOCIAL CRISTIANA':'PUSC','ACCION CIUDADANA':'PAC','PROGRESO SOCIAL DEMOCRATICO':'PPSD','PUEBLO SOBERANO':'PPSO'};
  const short=p=>alias[p]||String(p).slice(0,14);
  let _cache=null;
  function makeRecords(){
    if(_cache)return _cache;
    const votes=D.votos.filter(r=>r.provincia==='Nacional'&&r.arena==='Legislativa'&&Number(r.ronda)===1);
    const seats=D.escanos.filter(r=>r.provincia==='Nacional');
    const gov=D.presidencia_asamblea;
    const byVote=new Map(),bySeat=new Map(),byGov=new Map();
    const key=(year,party)=>year+'|'+party;
    votes.forEach(r=>{const k=key(r.anio,r.partido);if(byVote.has(k))throw Error('Votos duplicados: '+k);byVote.set(k,r)});
    seats.forEach(r=>{const k=key(r.anio,r.partido);if(bySeat.has(k))throw Error('Escaños duplicados: '+k);bySeat.set(k,r)});
    gov.forEach(r=>byGov.set(r.anio,r));
    const years=[...new Set(votes.map(r=>r.anio))].sort((a,b)=>a-b);
    const totals=[];const winners=[];const presidential=[];
    for(const year of years){
      const v= votes.filter(r=>r.anio===year);
      const total=v.reduce((s,r)=>s+Number(r.votos||0),0);
      const withSeats=v.reduce((s,r)=>s+(Number(bySeat.get(key(year,r.partido))?.escanos||0)>0?Number(r.votos||0):0),0);
      const withoutSeats=total-withSeats;
      const seatsYear=seats.filter(r=>r.anio===year);
      const chamber=seatsYear.reduce((s,r)=>s+Number(r.escanos||0),0);
      if(!total||!chamber)throw Error('Falta padrón válido o magnitud en '+year);
      const row={anio:year,votos_validos:total,votos_con_escanos:withSeats,votos_sin_escanos:withoutSeats,pct_con_escanos:100*withSeats/total,pct_sin_escanos:100*withoutSeats/total};
      totals.push(row);
      const leader=[...v].sort((a,b)=>Number(b.votos)-Number(a.votos)||a.partido.localeCompare(b.partido,'es'))[0];
      const leaderSeats=Number(bySeat.get(key(year,leader.partido))?.escanos||0);
      winners.push({anio:year,partido:leader.partido,abreviatura:short(leader.partido),votos:Number(leader.votos),votos_pct:100*leader.votos/total,escanos:leaderSeats,total_escanos:chamber,escanos_pct:100*leaderSeats/chamber,brecha_pp:100*leaderSeats/chamber-100*leader.votos/total});
      const g=byGov.get(year);
      if(!g)throw Error('No hay Presidencia electa para '+year);
      const gv=byVote.get(key(year,g.partido));
      if(!gv)throw Error('No hay voto legislativo del partido presidencial electo '+year+'/'+g.partido);
      const gs=Number(bySeat.get(key(year,g.partido))?.escanos||0);
      if(Math.abs(gs-Number(g.escanos))>.001)throw Error('Inconsistencia de bancada para '+year);
      presidential.push({anio:year,presidencia:g.nombre,partido:g.partido,abreviatura:short(g.partido),votos:Number(gv.votos),votos_pct:100*gv.votos/total,escanos:gs,total_escanos:chamber,escanos_pct:100*gs/chamber,brecha_pp:100*gs/chamber-100*gv.votos/total,partido_lider_legislativo:leader.partido,coincide_lider_legislativo:g.partido===leader.partido});
      const original=D.metricas.find(m=>m.anio===year&&m.provincia==='Nacional'&&m.arena==='Legislativa'&&m.ronda===1);
      if(!original||Math.abs(Number(original.validos)-total)>.001||Math.abs(Number(original.voto_sin_escanos_pct)-row.pct_sin_escanos)>1e-6)throw Error('La serie de votos sin escaño no coincide con la validación del Atlas para '+year);
    }
    _cache={totals,winners,presidential};return _cache;
  }
  function shown(rows){return rows.filter(r=>r.anio>=state.from&&r.anio<=state.to)}
  function tracesForShares(rows){
    // Una sola variable independiente: la cuota destinada a partidos sin diputaciones.
    // La cuota con diputaciones es su complemento 100 - cuota sin diputaciones.
    // Gráfico de tallos y puntos (lollipop), NO barras apiladas ni dos líneas redundantes.
    const xs=[],ys=[];
    rows.forEach(r=>{xs.push(r.anio,r.anio,null);ys.push(0,r.pct_sin_escanos,null)});
    return [
      {type:'scatter',mode:'lines',showlegend:false,hoverinfo:'skip',
       x:xs,y:ys,line:{color:'rgba(197,108,79,0.43)',width:2.4}},
      {type:'scatter',mode:'markers',showlegend:false,name:'Partidos sin diputaciones',
       x:rows.map(r=>r.anio),y:rows.map(r=>r.pct_sin_escanos),
       customdata:rows.map(r=>[r.pct_con_escanos]),
       marker:{color:sColor,size:12,symbol:'circle',line:{color:'#ffffff',width:1.5}},
       hovertemplate:'Elección %{x}<br><b>Sin diputaciones: %{y:.2f} %</b><br>Con diputaciones: %{customdata[0]:.2f} %<extra></extra>'}
    ];
  }
  function compareTraces(rows){
    const years=rows.map(r=>r.anio), parties=rows.map(r=>r.partido);
    return [
      {type:'bar',name:'Votos legislativos válidos (%)',x:years,y:rows.map(r=>r.votos_pct),customdata:parties,marker:{color:rColor},hovertemplate:'%{x} · %{customdata}<br>Votos legislativos: %{y:.2f}%<extra></extra>'},
      {type:'bar',name:'Escaños en la Asamblea (%)',x:years,y:rows.map(r=>r.escanos_pct),customdata:parties,marker:{color:sColor2},hovertemplate:'%{x} · %{customdata}<br>Escaños: %{y:.2f}%<extra></extra>'}
    ]
  }
  function integrity(){
    const {totals,winners,presidential}=makeRecords();
    if(totals.length!==19||winners.length!==19||presidential.length!==19)throw Error('La serie debe cubrir las 19 elecciones.');
    return true;
  }
  views.representacion_voto=function(){
    const {totals,winners,presidential}=makeRecords();
    const total=shown(totals),leaders=shown(winners),gov=shown(presidential);
    const focus=total.find(r=>r.anio===state.focus),flead=leaders.find(r=>r.anio===state.focus),fgov=gov.find(r=>r.anio===state.focus);
    title('Votos y escaños','Evolución de la proporción del voto válido obtenido por partidos con y sin diputaciones y comparación de votos y escaños de las principales fuerzas.');
    note('<b>Ámbito nacional.</b> Estas comparaciones corresponden a la Asamblea Legislativa completa. Los filtros <b>Desde, Hasta y Elección de detalle</b> funcionan. El selector de provincia no cambia esta vista, porque el criterio es conseguir al menos una diputación en el país. «Sin escaños» significa sin representación partidaria directa, no ausencia de representación jurídica del electorado.');
    if(!total.length){add('<div class="empty">Sin observaciones para este intervalo.</div>');return}
    const f=focus||total[total.length-1],l=flead||leaders[leaders.length-1],g=fgov||gov[gov.length-1];
    kpis([
      ['Voto sin escaños · '+f.anio,num(f.pct_sin_escanos,2)+' %','Porcentaje de los votos válidos'],
      ['Voto con escaños · '+f.anio,num(f.pct_con_escanos,2)+' %','Porcentaje de los votos válidos'],
      ['Líder legislativo · '+l.anio,num(l.escanos,0)+' / '+num(l.total_escanos,0),esc(labelText(l.partido))+' · diputaciones'],
      ['Partido presidencial · '+g.anio,num(g.escanos_pct,2)+' %',esc(labelText(g.partido))+' · bancada']
    ]);
    if(!focus)note('La elección de detalle está fuera del intervalo. Las tarjetas muestran la última elección del periodo.',true);
    const maxShare=Math.max(...total.map(r=>r.pct_sin_escanos));
    const topShare=Math.max(5,Math.ceil((maxShare+2)/5)*5);
    plot('Voto válido destinado a partidos sin diputaciones',tracesForShares(total),total,
      {height:485,subtitle:'Nacional · '+state.from+'–'+state.to,
       xTitle:'Elección',yTitle:'Porcentaje del voto válido (%)',
       layout:{yaxis:{range:[0,topShare],title:{text:'Porcentaje del voto válido (%)'},ticksuffix:' %',dtick:5},
               showlegend:false,hovermode:'closest',margin:{l:88,r:50,t:30,b:85}},
       note:'Cada punto indica la proporción de votos válidos destinados a partidos que no obtuvieron ninguna diputación. La parte restante (100 % menos ese valor) corresponde a partidos con diputaciones. Pase el cursor por los puntos para consultar ambas proporciones.'});
    table(total,['anio','pct_con_escanos','pct_sin_escanos'],'Evolución del voto válido por acceso a diputaciones (%)');
    note('El partido legislativo más votado no coincide necesariamente con el ganador de la Presidencia. Los porcentajes de escaños se calculan sobre el total de diputaciones de cada elección.');
    plot('Líder de votos legislativos · votos frente a escaños',compareTraces(leaders),leaders,{height:510,yTitle:'Porcentaje (%)',layout:{barmode:'group',yaxis:{range:[0,90],title:{text:'Porcentaje (%)'}}},note:'Pase el cursor sobre un año para identificar el partido. Comparación de porcentaje de sufragios válidos y de escaños del mismo partido.'});
    table(leaders,['anio','partido','votos','votos_pct','escanos','total_escanos','escanos_pct','brecha_pp'],'Líder legislativo · votos y representación parlamentaria');
    note('Para el partido presidencial electo se utiliza la candidatura ganadora definitiva, incluida la segunda ronda cuando corresponde. Se comparan sus votos legislativos y sus escaños, no sus votos presidenciales.');
    plot('Partido presidencial electo · voto legislativo y bancada',compareTraces(gov),gov,{height:510,yTitle:'Porcentaje (%)',layout:{barmode:'group',yaxis:{range:[0,90],title:{text:'Porcentaje (%)'}}},note:'El ganador de la Presidencia y el líder del voto legislativo pueden ser partidos diferentes; la tabla permite identificarlos.'});
    table(gov,['anio','presidencia','partido','votos','votos_pct','escanos','total_escanos','escanos_pct','brecha_pp','partido_lider_legislativo','coincide_lider_legislativo'],'Presidencia electa · partido, votos legislativos y bancada');
    add('<div class="notice">Fuente: Atlas Electoral de Costa Rica 1953–2026. Las series utilizan votos válidos legislativos, escaños y partido presidencial electo, con denominadores nacionales consistentes.</div>');
  };
  integrity();
  window.ATLAS_V7_REPRESENTACION_HTML_OK=true;
})();

'''

def instalar_representacion_v7_en_html():
    """Actualización idempotente de los recursos que YA utiliza el notebook V7.

    No necesita archivos nuevos, no sustituye los módulos existentes, y al
    ejecutar todas las celdas la aplicación completa incluye los gráficos.
    """
    recursos = PROJECT_DIR / 'recursos'
    js_path = recursos / 'atlas_final.js'
    plantilla_path = recursos / 'atlas_template.html'
    if not js_path.is_file() or not plantilla_path.is_file():
        raise FileNotFoundError(
            'No se encontraron los recursos originales del Atlas Electoral en ' + str(recursos) +
            '. Ejecute el notebook desde la raíz del proyecto habitual (con sus siete bases y recursos).')
    js_origen = js_path.read_text(encoding='utf-8')
    marcador = 'ATLAS_V7_REPRESENTACION_HTML'
    # Sustituir el módulo antiguo si ya se había ejecutado; nunca duplicar.
    if marcador in js_origen:
        patron = r'/\* ATLAS_V7_REPRESENTACION_HTML.*?window\.ATLAS_V7_REPRESENTACION_HTML_OK=true;\s*\}\)\(\);'
        nuevo, veces = re.subn(patron, lambda _: ATLAS_V7_INLINE_JS.strip(), js_origen, count=1, flags=re.S)
        if veces != 1:
            raise RuntimeError('Existe una versión anterior de V7 que no se pudo sustituir sin riesgo. No se modificó el JS.')
        js_origen = nuevo
    else:
        for exigido in ('views.representacion=', 'views.outcomes=', 'views.genero='):
            if exigido not in js_origen:
                raise RuntimeError('El archivo atlas_final.js no tiene la estructura habitual del proyecto: falta '+exigido)
        js_origen += '\n' + ATLAS_V7_INLINE_JS.strip() + '\n'
    js_path.write_text(js_origen, encoding='utf-8')
    plantilla = plantilla_path.read_text(encoding='utf-8')
    for anterior,nuevo in (
        ("['representacion','Votos y escaños']", "['representacion','Padrón y escaños']"),
        ("['representacion_voto','Voto y representación']", "['representacion_voto','Votos y escaños']"),
    ):
        plantilla = plantilla.replace(anterior,nuevo)
    opcion = "['representacion_voto','Votos y escaños'],"
    if opcion not in plantilla:
        anterior = "['representacion','Padrón y escaños'],"
        if anterior not in plantilla:
            raise RuntimeError('No se pudo localizar la entrada de representación del menú HTML original.')
        plantilla = plantilla.replace(anterior,anterior + '\n' + opcion,1)
    plantilla_path.write_text(plantilla,encoding='utf-8')
    # Integración focalizado : la primera carga debe utilizar la población visible del selector.
    genero = js_path.read_text(encoding='utf-8')
    genero = genero.replace("const pop=(typeof local!=='undefined' && local.sexpop)?local.sexpop:'';", 
                            "const pop=(typeof local!=='undefined' && local.sexpop)?local.sexpop:'Diputación propietaria';")
    genero = genero.replace("const result=previousGenero.apply(this,args);",
                            "local.sexpop=local.sexpop||'Diputación propietaria';\n    const result=previousGenero.apply(this,args);")
    genero = genero.replace("if(String(title).includes('a través del tiempo')){",
        "if(String(title).includes('a través del tiempo') || titleText==='Composición por sexo entre candidaturas electas' || titleText==='Composición por sexo entre candidaturas no electas'){")
    genero = genero.replace("const active=milestones.filter(m=>years.has(m.year));",
        "const active=['Diputación propietaria','Suplencia legislativa','Diputación electa'].includes(pop) ? milestones.filter(m=>years.has(m.year)) : [];")
    genero = genero.replace(
        "note:[nextOpts.note,'Líneas doradas: hitos de la normativa electoral sobre participación de mujeres. Su coincidencia con cambios en la serie es descriptiva; no demuestra causalidad por sí sola.'].filter(Boolean).join(' ')",
        "note:[nextOpts.note,active.length?'Líneas doradas: hitos de la normativa electoral relativos a candidaturas legislativas.':null].filter(Boolean).join(' ')")
    genero = genero.replace(
        "    }\n    return result;\n  };\n})();\n\n\n\n/* ATLAS_V7_REPRESENTACION_HTML",
        "    } else note(pop==='Diputación electa'?'La población seleccionada comprende solamente diputaciones electas. No corresponde dividirla en candidaturas electas y no electas.':'No se presentan gráficos de elección/no elección para esta población porque sus resultados no son comparables.');\n    return result;\n  };\n})();\n\n\n\n/* ATLAS_V7_REPRESENTACION_HTML")

    # Idempotencia de la normalización tras ejecutar varias veces en el mismo proyecto.
    genero = genero.replace("local.sexpop=local.sexpop||'Diputación propietaria';\n    local.sexpop=local.sexpop||'Diputación propietaria';\n", 
                           "local.sexpop=local.sexpop||'Diputación propietaria';\n")
    js_path.write_text(genero,encoding='utf-8')
    assert marcador in js_path.read_text(encoding='utf-8')
    assert opcion in plantilla_path.read_text(encoding='utf-8')
    print('✓ Módulo de votos y escaños actualizado desde el notebook.')
    print('  Menú: Padrón y escaños · Votos y escaños. Sexo y representación corregido.')

instalar_representacion_v7_en_html()


✓ Módulo de votos y escaños actualizado desde el notebook.
  Menú: Padrón y escaños · Votos y escaños. Sexo y representación corregido.


**Actualización ** · Exportación de tarjetas como PNG/SVG sin `foreignObject`, hitos legislativos en el gráfico de género y dos tasas descriptivas junto a los flujos. El parche se instala una sola vez y es compatible con copias sin necesidad de reemplazar fragmentos exactos de JavaScript. Para verlo, ejecute el notebook completo y abra el HTML regenerado.


## 2. Lectura e integración de las ocho bases

Los vínculos usan año, provincia y denominación partidaria normalizada. Las trayectorias de personas se vinculan por nombre completo normalizado. Se conservan las diferencias documentales y no se confunde una etiqueta compartida con identidad jurídica de partidos o personas.

La base activa es `Base_Final_Candidaturas_Diputaciones_1953-2026_ENTREGA.xlsx`. La hoja `Datos` y sus 11 columnas se validan antes de procesarla. Los cambios de nombres no requieren editar el notebook; se reemplaza el Excel con el mismo nombre y estructura y se ejecuta de nuevo desde el inicio.


## Conciliación de variantes nominales · Se aplica antes de integrar las bases y construir las trayectorias. Vincula una forma de un solo nombre con una única forma de varios nombres cuando aquel nombre es un componente exacto y coinciden los dos apellidos normalizados. Conserva las grafías originales; no modifica los Excel. Los apellidos compuestos se aprovechan desde campos expresos; las formas dudosas no se trocean arbitrariamente. Bloquea candidatos múltiples, documentos o sexo incompatibles y conflictos simultáneos de partido/cargo. La unión es nominal, no una verificación civil exhaustiva.

La tabla completa de decisiones queda en `salidas/datos/conciliacion_nombres.csv`, dentro de los datos integrados y en la sección Datos del HTML. La regla se ejecuta nuevamente cuando se actualicen las bases.

In [6]:
# Conciliación nominal ANTES de construir trayectorias e indicadores.
# Los Excel originales se leen y permanecen intactos.
import atlas_pipeline as _atlas_identity_pipeline
from collections import defaultdict as _identity_defaultdict
import re as _identity_re
import json as _identity_json
from pathlib import Path as _IdentityPath

_IDENTITY_PARTICLES = {'DE','DEL','LA','LAS','LOS','SAN','SANTA','VAN','VON','DA','DOS'}

def _identity_text(value):
    try:
        if value is None or pd.isna(value): return ''
    except (TypeError, ValueError): pass
    return _atlas_identity_pipeline.norm(value)

def _identity_document(value):
    text=str(value).strip()
    if not text or text.lower() in ('nan','none','nat','0','0.0'): return ''
    text=_identity_re.sub(r'\.0$', '', text)
    return ''.join(ch for ch in text if ch.isdigit())

def _identity_inventory(raw):
    records=[]
    for source in ('cand','formula','electas'):
        frame=raw[source]
        for index,row in frame.iterrows():
            original=row.get('Nombre completo')
            name=_identity_text(original)
            if not name: continue
            first=_identity_text(row.get('Primer apellido'))
            second=_identity_text(row.get('Segundo apellido'))
            role='Diputación' if source in ('cand','electas') else str(row.get('Cargo',''))
            year=row.get('Año',row.get('Año de elección'))
            try: year=int(year)
            except (ValueError,TypeError): year=None
            records.append({'source':source,'index':index,'name':name,'original':str(original),
                'surname1':first,'surname2':second,'sex':_identity_text(row.get('Sexo')),
                'document':_identity_document(row.get('Cédula')),'year':year,'role':role,
                'party':_atlas_identity_pipeline.party(row.get('Partido'))})
    return records

def _identity_resolve(raw):
    inventory=_identity_inventory(raw)
    by_name=_identity_defaultdict(list)
    explicit=_identity_defaultdict(set)
    suffixes=_identity_defaultdict(set)
    for r in inventory:
        by_name[r['name']].append(r)
        if r['surname1'] and r['surname2']:
            tail=r['surname1']+' '+r['surname2']
            if r['name'].endswith(' '+tail):
                given=r['name'][:-len(tail)].strip()
                explicit[r['name']].add((tuple(given.split()),r['surname1'],r['surname2']))
                suffixes[tail.split()[-1]].add((r['surname1'],r['surname2']))
    parts={}
    for name in by_name:
        if len(explicit[name])==1:
            given,s1,s2=next(iter(explicit[name]));parts[name]=(given,s1,s2,'Apellidos expresos en fuente');continue
        if len(explicit[name])>1: continue
        words=name.split()
        if len(words)<3 or any(len(w)==1 or '.' in w for w in words): continue
        possible=[]
        for s1,s2 in suffixes.get(words[-1],[]):
            tail=s1+' '+s2
            if name.endswith(' '+tail): possible.append((len(tail.split()),s1,s2))
        if possible:
            longest=max(r[0] for r in possible)
            best={(s1,s2) for length,s1,s2 in possible if length==longest}
            if len(best)!=1: continue
            s1,s2=next(iter(best));tail=s1+' '+s2
            given=tuple(name[:-len(tail)].strip().split())
            if given: parts[name]=(given,s1,s2,'Apellidos contrastados por sufijo expreso')
        elif not (set(words)&_IDENTITY_PARTICLES):
            parts[name]=(tuple(words[:-2]),words[-2],words[-1],'Dos últimos componentes sin partículas')
    long_names=_identity_defaultdict(list)
    for name,(given,s1,s2,method) in parts.items():
        if len(given)>=2: long_names[(s1,s2)].append(name)
    aliases={};audit=[]
    for short,(given,s1,s2,method) in sorted(parts.items()):
        if len(given)!=1: continue
        options=sorted(n for n in long_names[(s1,s2)] if given[0] in parts[n][0])
        if not options: continue
        reason='';target=options[0] if len(options)==1 else ''
        if len(options)>1: reason='Nombre abreviado compatible con varios nombres completos'
        else:
            rows=by_name[short]+by_name[target]
            docs={r['document'] for r in rows if r['document']}
            sexes={r['sex'] for r in rows if r['sex'] not in ('','SIN DATO','DESCONOCIDO')}
            if len(docs)>1: reason='Documentos de identidad distintos'
            elif len(sexes)>1: reason='Sexo incompatible entre registros'
            else:
                # No ocultar registros incompatibles del mismo año y cargo bajo una unión.
                left={(r['year'],r['role']):{x['party'] for x in by_name[short] if x['year']==r['year'] and x['role']==r['role']} for r in by_name[short]}
                right={(r['year'],r['role']):{x['party'] for x in by_name[target] if x['year']==r['year'] and x['role']==r['role']} for r in by_name[target]}
                if any(left[k].isdisjoint(right[k]) for k in left.keys()&right.keys()):reason='Partidos incompatibles en el mismo año y cargo'
        if target and not reason: aliases[short]=target
        audit.append({'nombre_abreviado_normalizado':short,'nombre_completo_normalizado':target,
            'primer_apellido':s1,'segundo_apellido':s2,'estado':'Conciliado nominalmente' if target and not reason else 'Pendiente; no se une',
            'regla':'Un nombre exacto contenido en los nombres de una única forma completa y dos apellidos coincidentes',
            'metodo_apellidos':method,'candidatos_completos':' | '.join(options),'motivo_no_union':reason,
            'filas_nombre_abreviado':len(by_name[short]),'filas_nombre_completo':len(by_name[target]) if target else None,
            'nivel_validacion':'Conciliación nominal; no equivale a validación civil individual'})
    # Validar también componentes completos: dos abreviaturas no deben unir
    # indirectamente documentos distintos a través de un nombre largo sin cédula.
    components=_identity_defaultdict(list)
    for short,target in aliases.items():components[target].append(short)
    for target,shorts in components.items():
        rows=[r for name in [target,*shorts] for r in by_name[name]]
        docs={r['document'] for r in rows if r['document']}
        sexes={r['sex'] for r in rows if r['sex'] not in ('','SIN DATO','DESCONOCIDO')}
        reason=''
        if len(docs)>1:reason='Documentos distintos en el conjunto completo de variantes'
        elif len(sexes)>1:reason='Sexo incompatible en el conjunto completo de variantes'
        if reason:
            for short in shorts:
                del aliases[short]
                row=next(r for r in audit if r['nombre_abreviado_normalizado']==short)
                row['estado']='Pendiente; no se une';row['motivo_no_union']=reason
    return aliases,audit,inventory

_identity_state={}
_identity_original_prepare=getattr(_atlas_identity_pipeline.prepare_people,'_atlas_identity_original',_atlas_identity_pipeline.prepare_people)
_identity_original_build=getattr(_atlas_identity_pipeline.build_dataset,'_atlas_identity_original',_atlas_identity_pipeline.build_dataset)

def _identity_prepare(raw,issues):
    aliases,audit,inventory=_identity_resolve(raw)
    canonical_text={}
    for r in inventory:canonical_text.setdefault(r['name'],r['original'])
    revised=dict(raw)
    for key in ('cand','formula','electas'):
        frame=raw[key].copy(deep=True)
        frame['nombre_original_fuente']=frame['Nombre completo']
        frame['Nombre completo']=frame['Nombre completo'].map(lambda v:canonical_text[aliases[_identity_text(v)]] if _identity_text(v) in aliases else v)
        revised[key]=frame
    result=list(_identity_original_prepare(revised,issues))
    c,e,f,events,people,moves,placeholders=result
    originals={r['id_evento']:r['nombre_original_fuente'] for frame in (c,f) for r in frame.to_dict('records')}
    original_electas={r['id_registro']:r['nombre_original_fuente'] for r in e.to_dict('records')}
    variants=_identity_defaultdict(set)
    for r in inventory:variants[aliases.get(r['name'],r['name'])].add(r['original'])
    targets=set(aliases.values())
    for frame in (c,e,f,events):
        frame['nombre_conciliado']=frame['nombre']
        frame['conciliacion_nombre']=frame['nombre_key'].map(lambda k:'Conciliado nominalmente por variante abreviada' if k in targets else 'Sin unión adicional')
    changed_ids=set(events.loc[events.nombre_key.isin(targets),'id_persona'])
    events.loc[events.id_persona.isin(changed_ids),'identidad']='Conciliación nominal unívoca: variante de nombres y dos apellidos; no validación civil'
    for frame in (people,):
        if 'identidad' in frame:frame.loc[frame.id_persona.isin(changed_ids),'identidad']='Conciliación nominal unívoca: variante de nombres y dos apellidos; no validación civil'
    _identity_state.clear();_identity_state.update({'audit':audit,'aliases':aliases,'originals':originals,'original_electas':original_electas,
        'variants':dict(variants),'pid_keys':dict(zip(events.id_persona,events.nombre_key))})
    assert len(events)==len(c)+len(f), 'La conciliación no puede agregar ni eliminar candidaturas.'
    return tuple(result)

_identity_prepare._atlas_identity_original=_identity_original_prepare
_atlas_identity_pipeline.prepare_people=_identity_prepare

def _identity_build(root,output):
    data=_identity_original_build(root,output)
    # Restituir grafías de fuente después de la presentación tipográfica de enrich().
    original_map=_identity_state['originals'];electas_map=_identity_state['original_electas'];variants=_identity_state['variants'];pid_keys=_identity_state['pid_keys']
    for key,rows in data.items():
        if not isinstance(rows,list):continue
        for row in rows:
            if not isinstance(row,dict):continue
            event=row.get('id_evento',row.get('id'))
            if event in original_map:row['nombre_original_fuente']=original_map[event];row['nombre_original']=original_map[event]
            if key=='electas' and row.get('id_registro') in electas_map:row['nombre_original_fuente']=electas_map[row['id_registro']];row['nombre_original']=electas_map[row['id_registro']]
            if row.get('id_origen') in original_map:row['nombre_original_origen']=original_map[row['id_origen']]
            if row.get('id_destino') in original_map:row['nombre_original_destino']=original_map[row['id_destino']]
            pid=row.get('id_persona')
            if pid in pid_keys:row['variantes_nombre']=' | '.join(sorted(variants.get(pid_keys[pid],())))
    data['conciliacion_nombres']=_identity_state['audit']
    data['meta']['identidad']='Identidades nominales: nombre normalizado y conciliación unívoca de un nombre abreviado con una forma de varios nombres y los mismos dos apellidos; se conservan variantes y se bloquean ambigüedades, documentos o sexo incompatibles. No equivale a validación civil exhaustiva.'
    data['meta']['conciliacion_nombres']={'regla':'Nombre abreviado de un solo nombre + dos apellidos coincidentes, candidato completo único','uniones':len(_identity_state['aliases']),'pendientes':sum(r['estado']!='Conciliado nominalmente' for r in data['conciliacion_nombres'])}
    output=_IdentityPath(output)
    for key,rows in data.items():
        if isinstance(rows,list):pd.DataFrame(rows).to_csv(output/(key+'.csv'),index=False,encoding='utf-8-sig')
    (output/'datos_integrados.json').write_text(_identity_json.dumps(data,ensure_ascii=False,allow_nan=False),encoding='utf-8')
    print('Conciliación de nombres:',data['meta']['conciliacion_nombres'])
    return data

_identity_build._atlas_identity_original=_identity_original_build
_atlas_identity_pipeline.build_dataset=_identity_build
build_dataset=_identity_build
print(': conciliación nominal instalada antes de generar datos, trayectorias, matrices y HTML.')


: conciliación nominal instalada antes de generar datos, trayectorias, matrices y HTML.


In [7]:
data = build_dataset(DATA_DIR, OUTPUT_DIR / "datos")
frames = {key: pd.DataFrame(rows) for key, rows in data.items() if isinstance(rows, list)}
# Octava base: identidades partidarias y participaciones por elección.
_parties = DATA_DIR / 'Base_Final_Partidos_Politicos_1953-2026.xlsx'
if not _parties.is_file():
    raise FileNotFoundError('Falta la octava base de partidos políticos en la raíz del paquete.')
_master = pd.read_excel(_parties, sheet_name='Partidos')
_participations = pd.read_excel(_parties, sheet_name='Participaciones')
assert _master['ID partido'].is_unique
assert not _participations[['Año','Denominación electoral']].duplicated().any()
data['partidos_maestros'] = json.loads(_master.to_json(orient='records',force_ascii=False))
data['participaciones_partidarias'] = json.loads(_participations.to_json(orient='records',force_ascii=False))
frames['partidos_maestros'] = _master
frames['participaciones_partidarias'] = _participations


display(pd.DataFrame(data["meta"]["fuentes"])[["base", "archivo", "bytes"]])
display(pd.DataFrame({"Tabla": list(frames), "Registros": [len(df) for df in frames.values()]}))


Conciliación de nombres: {'regla': 'Nombre abreviado de un solo nombre + dos apellidos coincidentes, candidato completo único', 'uniones': 136, 'pendientes': 7}


,base,archivo,bytes
0,leg,Base_Electoral_Diputaciones_1953-2026.xlsx,11349713
1,pres,Base_Electoral_Presidencial_1953-2026.xlsx,10318055
2,cand,Base_Final_Candidaturas_Diputaciones_1953-2026...,921446
3,formula,Base_Final_Candidaturas_Presidenciales_1953-20...,38190
4,electas,Base_Final_Diputaciones_Electas_1953-2026.xlsx,71918
5,diseno,Base_Final_Diseno_Papeletas_Diputaciones_1953-...,31861
6,cancelados,Base_partidos_cancelados.xlsx,65633


,Tabla,Registros
0,votos,3991
1,totales,336
2,escanos,2503
3,metricas,336
4,panel,2527
5,candidaturas,14870
6,electas,1059
7,formulas,530
8,eventos,15400
9,personas,13050


### Composición de las diputaciones por participación de sus partidos

Una fila por diputación electa. Se distingue participación presidencial observada de participación solo legislativa y, como lectura adicional, cobertura legislativa en siete o menos provincias. Esto no equivale a escala jurídica. La visualización pública se incorpora a la sección 07 · Presidencia y Asamblea, siempre por elección.

In [8]:
"""Funciones puras para incorporar dentro de ambos notebooks; no se requieren como dependencia."""
import pandas as pd

ARENA_CATEGORIES = ['Con candidatura presidencial', 'Solo legislativa']
COVERAGE_CATEGORIES = ['Con candidatura presidencial', 'Solo legislativa · siete provincias', 'Solo legislativa · una a seis provincias']  # legado de cálculo, no se visualiza en HTML
COMPOSITION_COLORS = {'Con candidatura presidencial':'#167d8d','Solo legislativa':'#b58a4d','Solo legislativa · siete provincias':'#b58a4d','Solo legislativa · una a seis provincias':'#775697'}

def clasificar_diputaciones(electas, oferta):
    """Una fila por escaño electo; unión validada por elección y denominación."""
    seats=electas.copy();supply=oferta.copy()
    required={'anio','partido','provincia','nombre'}
    if not required.issubset(seats):raise ValueError('Faltan columnas de diputaciones: '+str(required-set(seats)))
    if supply.duplicated(['anio','partido']).any():raise ValueError('Oferta duplicada por elección y partido; revisar la fuente antes de unir.')
    if 'id_registro' in seats and seats.id_registro.duplicated().any():raise ValueError('ID de escaño repetido.')
    seats['anio']=pd.to_numeric(seats['anio'],errors='raise').astype(int)
    supply['anio']=pd.to_numeric(supply['anio'],errors='raise').astype(int)
    joined=seats.merge(supply[['anio','partido','oferta','provincias_legislativas']],how='left',on=['anio','partido'],validate='many_to_one',indicator='cotejo_oferta')
    if len(joined)!=len(seats):raise AssertionError('La unión cambió el número de escaños.')
    if (joined['cotejo_oferta']!='both').any():
        raise ValueError('Hay partidos electos sin correspondencia en la oferta electoral. Revisar la unión.')
    def arena(r):
        if r['oferta']=='Presidencial y legislativa':return ARENA_CATEGORIES[0]
        if r['oferta']=='Solo legislativa':return ARENA_CATEGORIES[1]
        raise ValueError(f'Oferta electoral sin conciliación: {r["anio"]} · {r["partido"]} · {r["oferta"]!r}')
    def coverage(r):
        if r['arena_origen']==ARENA_CATEGORIES[0]:return COVERAGE_CATEGORIES[0]
        if r['arena_origen']==ARENA_CATEGORIES[1] and pd.notna(r['provincias_legislativas']):
            n=int(r['provincias_legislativas'])
            if n==7:return COVERAGE_CATEGORIES[1]
            if 1<=n<=6:return COVERAGE_CATEGORIES[2]
        raise ValueError(f'Cobertura legislativa sin conciliación: {r["anio"]} · {r["partido"]} · {r["provincias_legislativas"]!r}')
    joined['arena_origen']=joined.apply(arena,axis=1)
    joined['arena_cobertura_origen']=joined.apply(coverage,axis=1)
    return joined

def resumir_composicion(detalle, agrupador='anio', criterio='arena_origen'):
    categories=ARENA_CATEGORIES if criterio=='arena_origen' else COVERAGE_CATEGORIES
    groups=sorted(detalle[agrupador].dropna().unique())
    index=pd.MultiIndex.from_product([groups,categories],names=[agrupador,'categoria'])
    counts=detalle.groupby([agrupador,criterio],observed=True).size()
    counts.index=counts.index.set_names([agrupador,'categoria'])
    out=counts.reindex(index,fill_value=0).rename('diputaciones').reset_index()
    totals=detalle.groupby(agrupador).size().rename('total_diputaciones')
    out=out.merge(totals,on=agrupador,validate='many_to_one')
    out['porcentaje']=100*out.diputaciones/out.total_diputaciones
    out['criterio']=criterio
    assert out.groupby(agrupador).diputaciones.sum().equals(totals.rename('diputaciones'))
    assert out.groupby(agrupador).porcentaje.sum().sub(100).abs().max()<1e-8
    return out

def figura_composicion(resumen, eje='anio', titulo='Composición de las diputaciones electas'):
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    groups=list(dict.fromkeys(resumen[eje].astype(str)))
    categories=list(dict.fromkeys(resumen.categoria))
    table=resumen.assign(_eje=resumen[eje].astype(str)).set_index(['categoria','_eje'])
    counts=[[int(table.loc[(c,g),'diputaciones']) for g in groups] for c in categories]
    shares=[[float(table.loc[(c,g),'porcentaje']) for g in groups] for c in categories]
    fig=make_subplots(rows=2,cols=1,vertical_spacing=.20,subplot_titles=['Cantidad de diputaciones electas','Porcentaje del total de diputaciones electas'])
    scale=[[0,'#f5f8fc'],[.25,'#c4dced'],[.6,'#5b9dc7'],[1,'#185181']]
    for row,z,title in [(1,counts,'Escaños'),(2,shares,'%')]:
        labels=[[f'{counts[i][j]} · {shares[i][j]:.1f}%' for j in range(len(groups))] for i in range(len(categories))]
        fig.add_trace(go.Heatmap(x=groups,y=categories,z=z,zmin=0,zmax=max(1,max(map(max,counts))) if row==1 else 100,colorscale=scale,xgap=2,ygap=3,text=labels,texttemplate='%{text}',textfont={'size':11},colorbar={'title':title,'len':.39,'y':.80 if row==1 else .20},customdata=[[[counts[i][j],shares[i][j]] for j in range(len(groups))] for i in range(len(categories))],hovertemplate='%{y}<br>%{x}<br>%{customdata[0]} diputaciones · %{customdata[1]:.2f}%<extra></extra>'),row=row,col=1)
        fig.update_xaxes(type='category',categoryorder='array',categoryarray=groups,tickangle=-45,row=row,col=1)
        fig.update_yaxes(autorange='reversed',row=row,col=1)
    fig.update_layout(title=titulo,font={'family':'Arial','size':12,'color':'#173b4e'},paper_bgcolor='white',plot_bgcolor='white',height=760,width=1550,margin={'l':265,'r':95,'t':90,'b':90})
    return fig

detalle_composicion = clasificar_diputaciones(pd.DataFrame(data['camara_electa']), pd.DataFrame(data['oferta_partidos']))
# Los registros sin enlace nominal se conservan como registros, sin unir identidades por conjetura.
detalle_composicion['id_persona'] = detalle_composicion.apply(lambda r: r['id_persona'] if pd.notna(r.get('id_persona')) else 'registro_'+str(r['id_registro']), axis=1)
detalle_composicion['id_evento'] = detalle_composicion['id_registro']
data['composicion_diputaciones_detalle'] = json.loads(detalle_composicion.to_json(orient='records',force_ascii=False))
for criterio in ['arena_origen','arena_cobertura_origen']:
    resumen = resumir_composicion(detalle_composicion,criterio=criterio)
    data['composicion_diputaciones_'+criterio] = json.loads(resumen.to_json(orient='records',force_ascii=False))
    frames['composicion_diputaciones_'+criterio] = resumen
    resumen.to_csv(OUTPUT_DIR/'datos'/('composicion_diputaciones_'+criterio+'.csv'),index=False,encoding='utf-8-sig')
detalle_composicion.to_csv(OUTPUT_DIR/'datos'/'composicion_diputaciones_detalle.csv',index=False,encoding='utf-8-sig')
frames['composicion_diputaciones_detalle'] = detalle_composicion
display(resumir_composicion(detalle_composicion).head(12))
assert len(detalle_composicion)==len(data['electas'])
assert detalle_composicion.groupby('anio').size().tolist()==[45,45]+[57]*17
print('Composición legislativa: 1.059 registros de escaño clasificados; fuentes preservadas.')


,anio,categoria,diputaciones,total_diputaciones,porcentaje,criterio
0,1953,Con candidatura presidencial,41,45,91.111111,arena_origen
1,1953,Solo legislativa,4,45,8.888889,arena_origen
2,1958,Con candidatura presidencial,33,45,73.333333,arena_origen
3,1958,Solo legislativa,12,45,26.666667,arena_origen
4,1962,Con candidatura presidencial,57,57,100.000000,arena_origen
5,1962,Solo legislativa,0,57,0.000000,arena_origen
6,1966,Con candidatura presidencial,55,57,96.491228,arena_origen
7,1966,Solo legislativa,2,57,3.508772,arena_origen
8,1970,Con candidatura presidencial,57,57,100.000000,arena_origen
9,1970,Solo legislativa,0,57,0.000000,arena_origen


Composición legislativa: 1.059 registros de escaño clasificados; fuentes preservadas.


## 3. Cobertura y validación aritmética

Se comprueban los 19 procesos, las siete provincias, la suma de escaños y la coherencia de cada escrutinio. La asistencia a la jornada (padrón menos abstenciones) se concilia entre Presidencia y Diputaciones en primera ronda; puede diferir de los totales de papeletas contabilizadas, que se conservan por arena. En 1974 legislativo, el total de votos de la fuente excluye blancos y nulos: los emitidos analíticos son válidos + blancos + nulos, mientras `emitidos_fuente` conserva el original. Cuatro filas sin nombre se excluyen de conteos de candidaturas identificadas. La elección individual se obtiene del cruce con las declaratorias y conserva la bandera original como `electa_fuente`.

In [9]:
votos, totales, escanos = (frames[k] for k in ("votos", "totales", "escanos"))
candidaturas, electas, formulas = (frames[k] for k in ("candidaturas", "electas", "formulas"))
metricas, panel = frames["metricas"], frames["panel"]
eventos, personas = frames["eventos"], frames["personas"]

assert sorted(candidaturas.anio.unique()) == YEARS
assert set(candidaturas.provincia) == set(PROVINCES)
assert len(frames["papeletas"]) == len(YEARS) * len(PROVINCES)
assert not votos.duplicated(["anio", "provincia", "arena", "ronda", "partido"]).any()
assert np.allclose(totales.emitidos, totales.validos + totales.blancos + totales.nulos)
# Dos magnitudes distintas: asistencia efectiva a la jornada y papeletas escrutadas.
# Presidencia y Diputaciones de primera ronda comparten padrón y abstenciones
# una vez aplicadas las conciliaciones; válidos/blancos/nulos pertenecen a cada papeleta.
assert totales[['padron', 'abstenciones', 'emitidos']].notna().all().all()
assert (totales[['padron', 'abstenciones', 'emitidos']] >= 0).all().all()
assert (totales.abstenciones <= totales.padron).all(), 'Abstenciones superiores al padrón'

asistencia_jornada = totales.padron - totales.abstenciones
# Verificación sustantiva de la conciliación simultánea, por provincia y año.
primeras = totales[totales.ronda.eq(1) & totales.arena.isin(['Presidencial', 'Legislativa'])]
ref_presidencial = (primeras[primeras.arena.eq('Presidencial')]
                   .set_index(['anio','provincia'])[['padron','abstenciones']].sort_index())
ref_legislativa = (primeras[primeras.arena.eq('Legislativa')]
                  .set_index(['anio','provincia'])[['padron','abstenciones']].sort_index())
assert ref_presidencial.index.equals(ref_legislativa.index), 'Cobertura desigual entre arenas'
assert np.allclose(ref_presidencial, ref_legislativa), 'La conciliación de asistencia no coincide entre arenas'

# No se confunden los votos contabilizados en cada papeleta con la asistencia a la jornada.
# La diferencia se informa para auditoría, sin alterar válidos, blancos ni nulos.
_diferencia = (totales.emitidos - asistencia_jornada).round(6)
_reporte_diferencias = totales.loc[~np.isclose(_diferencia, 0),
    ['anio','provincia','arena','ronda','padron','emitidos','abstenciones']].copy()
_reporte_diferencias['asistencia_jornada'] = asistencia_jornada.loc[_reporte_diferencias.index]
_reporte_diferencias['diferencia_papeletas_asistencia'] = _diferencia.loc[_reporte_diferencias.index]
print(f'Conciliación de participación: {len(ref_presidencial)} territorios-año, dos arenas, mismo padrón y abstenciones.')
print(f'Registros con diferencia entre papeletas escrutadas y asistencia: {len(_reporte_diferencias)} (sin alterar los escrutinios).')
if len(_reporte_diferencias):
    display(_reporte_diferencias.head(12))
shares = votos.groupby(["anio", "provincia", "arena", "ronda"]).porcentaje.sum()
assert np.allclose(shares, 100)
seat_count = escanos[escanos.provincia.eq("Nacional")].groupby("anio").escanos.sum()
elected_count = electas.groupby("anio").size()
expected = pd.Series({y: 45 if y in (1953, 1958) else 57 for y in YEARS})
assert np.array_equal(seat_count.reindex(YEARS), expected)
assert np.array_equal(elected_count.reindex(YEARS), expected)
display(pd.DataFrame({"Escaños": seat_count, "Declaratorias": elected_count,
                      "Candidaturas identificadas": candidaturas.groupby("anio").size()}))
display(frames["integridad"])
print("Validación aritmética y de cobertura completada.")


Conciliación de participación: 152 territorios-año, dos arenas, mismo padrón y abstenciones.
Registros con diferencia entre papeletas escrutadas y asistencia: 150 (sin alterar los escrutinios).


,anio,provincia,arena,ronda,padron,emitidos,abstenciones,asistencia_jornada,diferencia_papeletas_asistencia
0,1953,Alajuela,Legislativa,1,55332.0,38010.0,17491.0,37841.0,169.0
1,1953,Cartago,Legislativa,1,34979.0,24553.0,10582.0,24397.0,156.0
2,1953,Guanacaste,Legislativa,1,26682.0,17064.0,9564.0,17118.0,-54.0
3,1953,Heredia,Legislativa,1,19657.0,14714.0,5052.0,14605.0,109.0
4,1953,Limón,Legislativa,1,11286.0,7118.0,4229.0,7057.0,61.0
5,1953,Puntarenas,Legislativa,1,30190.0,15034.0,15278.0,14912.0,122.0
6,1953,San José,Legislativa,1,115544.0,81777.0,33985.0,81559.0,218.0
7,1958,Alajuela,Legislativa,1,64212.0,43824.0,20468.0,43744.0,80.0
8,1958,Cartago,Legislativa,1,40607.0,27179.0,13380.0,27227.0,-48.0
9,1958,Guanacaste,Legislativa,1,32300.0,21176.0,11162.0,21138.0,38.0


,Escaños,Declaratorias,Candidaturas identificadas
anio,,,
1953,45.0,45,252
1958,45.0,45,437
1962,57.0,57,290
1966,57.0,57,311
1970,57.0,57,532
1974,57.0,57,752
1978,57.0,57,786
1982,57.0,57,647
1986,57.0,57,617


,base,anio,provincia,detalle,diferencia
0,cand,NaN,Varias,Filas sin nombre excluidas de conteos de perso...,4.0
1,cand/electas,1970.0,Limón,Bandera electa y cruce nominal no coinciden: R...,NaN
2,cand/electas,1978.0,San José,Bandera electa y cruce nominal no coinciden: J...,NaN
3,cand/electas,1978.0,San José,Bandera electa y cruce nominal no coinciden: J...,NaN
4,cand/electas,2002.0,Alajuela,Bandera electa y cruce nominal no coinciden: L...,NaN
5,cand/electas,2026.0,Alajuela,Bandera electa y cruce nominal no coinciden: K...,NaN
6,cand/electas,2026.0,Cartago,Bandera electa y cruce nominal no coinciden: J...,NaN
7,electas/cand,2002.0,Alajuela,Electa sin coincidencia nominal exacta en cand...,NaN
8,electas/cand,2026.0,Alajuela,Electa sin coincidencia nominal exacta en cand...,NaN
9,electas/cand,2026.0,Cartago,Electa sin coincidencia nominal exacta en cand...,NaN


Validación aritmética y de cobertura completada.


### Verificación de magnitudes y representación territorial

In [10]:
# Auditoría : representación territorial y cambios de asignación.
territorial = frames["representacion_territorial"].copy()
required_rep_cols = {
    "peso_escanos_pct", "peso_padron_pct", "brecha_representacion_pp",
    "indice_representacion", "electores_por_escano", "cambio_escanos",
    "balance_padron", "brecha_abs_pp", "cambio_asignacion"
}
assert required_rep_cols.issubset(territorial.columns)
assert territorial.groupby("anio").magnitud.sum().tolist() == [45, 45] + [57] * 17
cambios_asignacion = territorial[territorial.cambio_asignacion].copy()
assert cambios_asignacion.anio.nunique() >= 1
display(cambios_asignacion[["anio","provincia","magnitud","cambio_escanos"]].sort_values(["anio","provincia"]))


,anio,provincia,magnitud,cambio_escanos
14,1962,Alajuela,11.0,3.0
15,1962,Cartago,7.0,1.0
16,1962,Guanacaste,6.0,1.0
17,1962,Heredia,4.0,1.0
18,1962,Limón,3.0,1.0
19,1962,Puntarenas,6.0,1.0
20,1962,San José,20.0,4.0
21,1966,Alajuela,10.0,-1.0
24,1966,Heredia,3.0,-1.0
26,1966,Puntarenas,7.0,1.0


## 4. Participación y abstencionismo

Las tasas nacionales se calculan sumando numeradores y denominadores provinciales. La comparación entre arenas usa la primera ronda presidencial. La segunda ronda se conserva por separado. La cobertura son las siete provincias; no se agrega voto exterior.

In [11]:
nacional = metricas[metricas.provincia.eq("Nacional") & metricas.ronda.eq(1)].sort_values("anio")
legislativa = nacional[nacional.arena.eq("Legislativa")].copy()
provincial = metricas[metricas.provincia.ne("Nacional") & metricas.ronda.eq(1) & metricas.arena.eq("Legislativa")]
show(px.line(nacional, x="anio", y="abstencion_pct", color="arena", markers=True,
             labels={"anio": "Elección", "abstencion_pct": "Abstencionismo (%)", "arena": "Elección"}),
     "Participación histórica: abstencionismo presidencial y legislativo")
heat = provincial.pivot(index="provincia", columns="anio", values="abstencion_pct").reindex(PROVINCES)
show(go.Figure(go.Heatmap(x=heat.columns, y=heat.index, z=heat.values,
    colorscale=[[0,"#edf6f4"],[1,"#007f7a"]], colorbar=dict(title="%"))),
    "Abstencionismo legislativo por provincia y año")
rounds = metricas[metricas.provincia.eq("Nacional") & metricas.arena.eq("Presidencial")].copy()
rounds["Ronda"] = rounds.ronda.map({1: "Primera ronda", 2: "Segunda ronda"})
display(rounds[["anio", "Ronda", "padron", "emitidos", "abstencion_pct"]])


,anio,Ronda,padron,emitidos,abstencion_pct
11,1953,Primera ronda,293670.0,197489.0,32.751388
27,1958,Primera ronda,354780.0,229543.0,35.299904
43,1962,Primera ronda,483980.0,391406.0,19.127650
59,1966,Primera ronda,554627.0,451490.0,18.595741
75,1970,Primera ronda,675285.0,562766.0,16.662446
91,1974,Primera ronda,875043.0,699340.0,20.079356
107,1978,Primera ronda,1058445.0,860206.0,18.729268
123,1982,Primera ronda,1261127.0,991679.0,21.365652
139,1986,Primera ronda,1486474.0,1216300.0,18.175494
155,1990,Primera ronda,1692050.0,1384326.0,18.186460


## 5. Voto válido, blanco y nulo

Blancos y nulos se dividen entre votos emitidos; el abstencionismo se divide entre padrón. Son fenómenos diferentes. Los filtros del HTML permiten cambiar el indicador y comparar provincias.

In [12]:
no_validos = nacional.melt(id_vars=["anio", "arena"], value_vars=["blancos_pct", "nulos_pct"],
    var_name="Tipo", value_name="Porcentaje")
show(px.line(no_validos, x="anio", y="Porcentaje", color="arena", line_dash="Tipo", markers=True),
     "Votos blancos y nulos como proporción de los emitidos")
display(nacional[["anio", "arena", "blancos", "nulos", "invalidos_pct"]].tail(12))


,anio,arena,blancos,nulos,invalidos_pct
226,2006,Legislativa,14652.0,34286.0,2.942933
227,2006,Presidencial,8834.0,30422.0,2.360201
243,2010,Presidencial,6959.0,32555.0,2.025479
242,2010,Legislativa,20077.0,30806.0,2.608438
263,2014,Legislativa,13053.0,34801.0,2.282942
264,2014,Presidencial,5724.0,37969.0,2.084144
287,2018,Legislativa,12653.0,39746.0,2.390854
288,2018,Presidencial,5090.0,22937.0,1.286626
311,2022,Legislativa,18431.0,23199.0,1.965996
312,2022,Presidencial,11359.0,17636.0,1.369272


## 6. Concentración, fragmentación y competencia

NEP = 1 / suma de proporciones de voto al cuadrado. Las concentraciones del primero y de los dos primeros permiten examinar predominio y bipartidismo sin imponer periodizaciones. La volatilidad compara denominaciones nominales, no continuidad jurídica.

In [13]:
show(px.line(nacional, x="anio", y="nep", color="arena", markers=True,
             labels={"nep": "Número efectivo de partidos", "anio": "Elección"}),
     "Fragmentación en las arenas presidencial y legislativa")
concentration = legislativa.melt(id_vars=["anio"], value_vars=["concentracion_1", "concentracion_2"],
                                var_name="Indicador", value_name="Porcentaje")
show(px.line(concentration, x="anio", y="Porcentaje", color="Indicador", markers=True),
     "Concentración legislativa del primero y los dos primeros")
display(nacional[["anio", "arena", "ganador", "partidos_con_votos", "nep", "margen", "volatilidad_nominal"]].tail(12))


,anio,arena,ganador,partidos_con_votos,nep,margen,volatilidad_nominal
226,2006,Legislativa,LIBERACION NACIONAL,27,4.627774,11.196119,27.582924
227,2006,Presidencial,LIBERACION NACIONAL,14,2.981362,1.118786,36.002345
243,2010,Presidencial,LIBERACION NACIONAL,9,3.044466,21.850667,21.157840
242,2010,Legislativa,LIBERACION NACIONAL,18,4.780011,19.654810,19.624107
263,2014,Legislativa,LIBERACION NACIONAL,20,6.225324,2.224380,26.789220
264,2014,Presidencial,ACCION CIUDADANA,13,4.369871,0.916914,28.319710
287,2018,Legislativa,LIBERACION NACIONAL,27,7.672039,1.332900,35.959473
288,2018,Presidencial,RESTAURACION NACIONAL,13,5.512805,3.418955,49.040702
311,2022,Legislativa,LIBERACION NACIONAL,36,7.987608,9.784370,51.699259
312,2022,Presidencial,LIBERACION NACIONAL,25,6.114215,10.503152,64.914056


## 7. Relación presidencial–legislativa

La brecha es porcentaje presidencial menos legislativo, en puntos porcentuales. La divergencia compara las distribuciones completas. Estos agregados no identifican cuántas personas cruzaron su voto.

In [14]:
panel_nacional = panel[panel.provincia.eq("Nacional")]
ultimo = panel_nacional[panel_nacional.anio.eq(2026) & panel_nacional.brecha_pp.notna()].sort_values("brecha_pp")
show(px.bar(ultimo, x="brecha_pp", y="partido", orientation="h",
             labels={"brecha_pp": "Presidencial − legislativo (pp)", "partido": "Partido"}),
     "Brecha entre arenas por partido · 2026")
show(px.line(legislativa, x="anio", y="divergencia_arenas", markers=True,
             labels={"divergencia_arenas": "Divergencia (pp)"}),
     "Divergencia histórica entre distribuciones presidenciales y legislativas")


## 8. Votos, escaños y desproporcionalidad

Gallagher = raíz de la mitad de la suma de diferencias al cuadrado entre porcentajes de voto y escaños. Se utilizan distribuciones completas. La magnitud provincial permite analizar las circunscripciones sin asumir que todos los partidos presentaron listas completas.

In [15]:
representation = legislativa[["anio", "nep", "nep_escanos"]].melt(id_vars="anio", var_name="Distribución", value_name="NEP")
show(px.line(representation, x="anio", y="NEP", color="Distribución", markers=True),
     "Fragmentación de votos y escaños")
show(px.line(legislativa, x="anio", y="gallagher", markers=True,
             labels={"gallagher": "Gallagher"}), "Desproporcionalidad legislativa")
display(panel_nacional[panel_nacional.anio.eq(2026)][["partido", "porcentaje_Legislativa", "escanos", "escanos_pct", "desviacion_pp"]].sort_values("escanos", ascending=False))


,partido,porcentaje_Legislativa,escanos,escanos_pct,desviacion_pp
2474,PUEBLO SOBERANO,45.199059,31.0,54.385965,9.186906
2469,LIBERACION NACIONAL,23.249153,17.0,29.824561,6.575408
2466,FRENTE AMPLIO,12.334543,7.0,12.280702,-0.053842
2475,UNIDAD SOCIAL CRISTIANA,4.524633,1.0,1.754386,-2.770248
2456,AGENDA CIUDADANA,4.069753,1.0,1.754386,-2.315367
2455,ACTUEMOS YA,0.495633,0.0,0.000000,-0.495633
2468,JUSTICIA SOCIAL COSTARRICENSE,0.311048,0.0,0.000000,-0.311048
2477,UNION COSTARRICENSE DEMOCRATICA,0.183681,0.0,0.000000,-0.183681
2476,UNIDOS PODEMOS,1.195500,0.0,0.000000,-1.195500
2473,PROGRESO SOCIAL DEMOCRATICO,0.582875,0.0,0.000000,-0.582875


## 8.1. Voto válido y representación partidaria

Se calculan las proporciones de votos válidos correspondientes a organizaciones que obtuvieron al menos una diputación y a aquellas que no obtuvieron ninguna. Se trata de **representación partidaria directa de la opción votada**, no de representación jurídica individual.

Las tres series nuevas son: (1) porcentaje del voto válido a partidos con y sin diputaciones, mediante puntos y tallos proporcionales, sin barras apiladas ni líneas redundantes; (2) porcentaje de votos legislativos frente a porcentaje de escaños del partido con más votos legislativos; y (3) las mismas dos proporciones para el partido que ganó definitivamente la Presidencia. **No se presenta una gráfica de cantidades absolutas de votos.**

Para el cálculo se utilizan registros nacionales, sin sumar otra vez las provincias. Se cotejan los votos válidos y las bancadas con las tablas originales. El notebook genera PNG de alta resolución y CSV verificables en `salidas/graficas_articulo/`; el HTML recrea las series con filtros y descargas.


In [16]:
"""V7: controles de votos válidos, escaños y lecturas interactivas en el notebook.
El Atlas público consume los datos de salidas/datos; esta celda NO crea PNG,
figuras para el artículo, CSV independientes ni carpetas adicionales.
"""
import pandas as pd
import numpy as np
import plotly.graph_objects as go

REPRESENTADO = "#206E88"
SIN_ESCANO = "#D1744A"
ESCANOS = "#A98535"

def calcular_representacion_v7(frames):
    """Usa SOLO filas Nacional para no duplicar agregados provinciales.

    Define 'con escaños' cuando el partido obtuvo >= 1 diputación en cualquier
    provincia en esa misma elección. Eso NO identifica votantes individuales,
    preferencias personales ni escaños específicos para cada sufragio.
    """
    v = frames['votos'].copy()
    s = frames['escanos'].copy()
    v = v.loc[v.provincia.eq('Nacional') & v.arena.eq('Legislativa') & v.ronda.eq(1),
              ['anio', 'partido', 'votos']].copy()
    s = s.loc[s.provincia.eq('Nacional'), ['anio', 'partido', 'escanos']].copy()
    v['anio'] = v.anio.astype(int)
    s['anio'] = s.anio.astype(int)
    assert not v.duplicated(['anio', 'partido']).any(), 'Votos legislativos duplicados por partido/año'
    assert not s.duplicated(['anio', 'partido']).any(), 'Escaños duplicados por partido/año'
    v['votos'] = pd.to_numeric(v.votos, errors='raise').fillna(0)
    s['escanos'] = pd.to_numeric(s.escanos, errors='raise').fillna(0)
    assert (v.votos >= 0).all() and (s.escanos >= 0).all()
    v = v.merge(s, on=['anio', 'partido'], how='left', validate='one_to_one', indicator=True)
    # Sin fila de escaño == sin representación; nunca excluir del denominador.
    v['escanos'] = v.escanos.fillna(0)
    v['con_escanos'] = v.escanos.gt(0)
    assert v.anio.nunique() == 19 and v.anio.min() == 1953 and v.anio.max() == 2026

    base = v.groupby('anio', as_index=False).agg(votos_validos=('votos','sum'))
    con = v[v.con_escanos].groupby('anio').votos.sum().rename('votos_con_escanos')
    sin = v[~v.con_escanos].groupby('anio').votos.sum().rename('votos_sin_escanos')
    base = base.join(con, on='anio').join(sin, on='anio').fillna({'votos_con_escanos':0, 'votos_sin_escanos':0})
    base['pct_con_escanos'] = 100*base.votos_con_escanos/base.votos_validos
    base['pct_sin_escanos'] = 100*base.votos_sin_escanos/base.votos_validos
    base['partidos_con_escanos'] = v[v.con_escanos].groupby('anio').partido.nunique().reindex(base.anio,fill_value=0).to_numpy()
    base['partidos_sin_escanos'] = v[(~v.con_escanos)&v.votos.gt(0)].groupby('anio').partido.nunique().reindex(base.anio,fill_value=0).to_numpy()
    assert np.allclose(base.votos_con_escanos+base.votos_sin_escanos,base.votos_validos)
    assert np.allclose(base.pct_con_escanos+base.pct_sin_escanos,100)
    if 'totales' in frames:
        validos = frames['totales'].query('provincia == "Nacional" and arena == "Legislativa" and ronda == 1')
        chk = base.merge(validos[['anio','validos']],on='anio',validate='one_to_one')
        assert np.allclose(chk.votos_validos,chk.validos), 'Los votos por partido no suman los votos válidos oficiales'
    if 'metricas' in frames:
        metr = frames['metricas'].query('provincia == "Nacional" and arena == "Legislativa" and ronda == 1')
        chk = base.merge(metr[['anio','voto_sin_escanos_pct']],on='anio',validate='one_to_one')
        assert np.allclose(chk.pct_sin_escanos,chk.voto_sin_escanos_pct,atol=1e-8), 'Difiere del indicador original'

    escanos_total = s.groupby('anio').escanos.sum().rename('total_escanos').reset_index()
    # Partido legislativo ganador por pluralidad de votos legislativos, NO por Presidencia.
    lider = v.sort_values(['anio','votos','partido'],ascending=[True,False,True]).drop_duplicates('anio').copy()
    lider = lider.merge(base[['anio','votos_validos']],on='anio',validate='one_to_one').merge(escanos_total,on='anio',validate='one_to_one')
    lider['votos_pct'] = 100*lider.votos/lider.votos_validos
    lider['escanos_pct'] = 100*lider.escanos/lider.total_escanos
    lider['brecha_escanos_votos_pp'] = lider.escanos_pct-lider.votos_pct
    lider = lider[['anio','partido','votos','votos_pct','escanos','total_escanos','escanos_pct','brecha_escanos_votos_pp']].sort_values('anio')

    # Ganador definitivo presidencial: incluye resolución de segunda vuelta.
    gov = frames['presidencia_asamblea'][['anio','nombre','partido','escanos','escanos_pct']].copy()
    gov['anio'] = gov.anio.astype(int)
    assert not gov.duplicated('anio').any()
    pres = gov.merge(v[['anio','partido','votos']],on=['anio','partido'],how='left',validate='one_to_one')
    assert pres.votos.notna().all(), 'Ganador presidencial sin votos legislativos correspondientes'
    pres = pres.merge(base[['anio','votos_validos']],on='anio',validate='one_to_one').merge(escanos_total,on='anio',validate='one_to_one')
    pres['votos_pct'] = 100*pres.votos/pres.votos_validos
    pres['escanos_pct_verificado'] = 100*pres.escanos/pres.total_escanos
    assert np.allclose(pres.escanos_pct,pres.escanos_pct_verificado,atol=1e-8)
    pres['brecha_escanos_votos_pp'] = pres.escanos_pct-pres.votos_pct
    pres = pres.merge(lider[['anio','partido']].rename(columns={'partido':'ganador_legislativo'}),on='anio',validate='one_to_one')
    pres['coincide_ganador_legislativo'] = pres.partido.eq(pres.ganador_legislativo)

    # Primer lugar en PRIMERA vuelta, distinto del presidente definitivo en algunos años.
    prep = frames['votos'].copy()
    prep = prep.loc[prep.provincia.eq('Nacional') & prep.arena.eq('Presidencial') & prep.ronda.eq(1),['anio','partido','votos']]
    first = prep.sort_values(['anio','votos','partido'],ascending=[True,False,True]).drop_duplicates('anio')[['anio','partido']]
    first = first.rename(columns={'partido':'primer_lugar_primera_ronda_presidencial'})
    pres = pres.merge(first,on='anio',validate='one_to_one')
    pres['coincide_primera_ronda'] = pres.partido.eq(pres.primer_lugar_primera_ronda_presidencial)
    pres = pres[['anio','nombre','partido','votos','votos_pct','escanos','total_escanos','escanos_pct','brecha_escanos_votos_pp',
                 'ganador_legislativo','coincide_ganador_legislativo','primer_lugar_primera_ronda_presidencial','coincide_primera_ronda']].sort_values('anio')
    if 'concordancia_gobierno' in frames:
        chk = pres.merge(frames['concordancia_gobierno'][['anio','ganador_presidencial','ganador_voto_legislativo']],on='anio')
        assert (chk.partido == chk.ganador_presidencial).all() and (chk.ganador_legislativo == chk.ganador_voto_legislativo).all()
    assert len(base)==len(lider)==len(pres)==19
    return base, lider, pres

def figuras_interactivas_v7(base,lider,pres):
    """Tres figuras Plotly compatibles con el método show() del notebook."""
    years=base.anio.astype(int).tolist()
    vals=base.pct_sin_escanos.to_numpy(dtype=float)
    stems_x=[];stems_y=[]
    for yr,valor in zip(years,vals):
        stems_x.extend([yr,yr,None]);stems_y.extend([0,float(valor),None])
    f1=go.Figure()
    f1.add_trace(go.Scatter(x=stems_x,y=stems_y,mode='lines',showlegend=False,
                            hoverinfo='skip',line=dict(color='rgba(209,116,74,.45)',width=2.5)))
    f1.add_trace(go.Scatter(x=years,y=vals,mode='markers',showlegend=False,
                            marker=dict(color=SIN_ESCANO,size=12,line=dict(color='white',width=1.5)),
                            customdata=np.column_stack((base.pct_con_escanos.to_numpy(dtype=float),)),
                            hovertemplate='Elección %{x}<br><b>Sin diputaciones: %{y:.2f} %</b><br>Con diputaciones: %{customdata[0]:.2f} %<extra></extra>'))
    upper=max(5,float(np.ceil((vals.max()+2)/5)*5))
    f1.update_layout(height=485,hovermode='closest',showlegend=False,
                     title='Voto válido destinado a partidos sin diputaciones',
                     xaxis=dict(title='Elección',tickmode='array',tickvals=years,tickangle=-45),
                     yaxis=dict(title='Porcentaje del voto válido (%)',range=[0,upper],ticksuffix=' %',dtick=5),
                     margin=dict(l=82,r=45,t=65,b=92))
    def compare(df,label):
        fig=go.Figure()
        extra=np.array(df.partido).reshape(-1,1)
        for key,name,col in [('votos_pct','Votos legislativos válidos (%)',REPRESENTADO),('escanos_pct','Escaños (%)',ESCANOS)]:
            fig.add_trace(go.Bar(x=df.anio.astype(str),y=df[key],name=name,marker_color=col,customdata=extra,
                        hovertemplate='%{x} · %{customdata[0]}<br>%{y:.2f}%<extra>'+name+'</extra>'))
        fig.update_layout(barmode='group',height=520,title=label,hovermode='x unified',yaxis_title='Porcentaje (%)')
        return fig
    f2=compare(lider,'Partido más votado en diputaciones: voto válido y cuota de escaños')
    f3=compare(pres,'Partido presidencial electo: voto válido legislativo y cuota de escaños')
    return [f1,f2,f3]


def producir_v7(frames, show_fig=None, display_tables=False):
    base, lider, pres = calcular_representacion_v7(frames)
    if show_fig is not None:
        for figura in figuras_interactivas_v7(base, lider, pres):
            show_fig(figura)
    if display_tables:
        print('Voto válido por acceso a diputaciones, 2022 y 2026:')
        display(base.loc[base.anio.isin([2022,2026]), ['anio','pct_con_escanos','pct_sin_escanos']])
        print('Años en que el partido presidencial electo no encabezó el voto legislativo:')
        display(pres.loc[~pres.coincide_ganador_legislativo,
                ['anio','partido','ganador_legislativo','votos_pct','escanos_pct']])
    return {'por_anio':base,'lider_legislativo':lider,'partido_presidencial':pres}

V7_REPRESENTACION = producir_v7(frames, show_fig=show, display_tables=True)
assert len(V7_REPRESENTACION['por_anio']) == 19
print('Comprobaciones concluidas. Sin exportaciones editoriales adicionales.')


Voto válido por acceso a diputaciones, 2022 y 2026:


,anio,pct_con_escanos,pct_sin_escanos
17,2022,78.733226,21.266774
18,2026,89.377142,10.622858


Años en que el partido presidencial electo no encabezó el voto legislativo:


,anio,partido,ganador_legislativo,votos_pct,escanos_pct
1,1958,UNION NACIONAL,LIBERACION NACIONAL,21.366383,22.222222
3,1966,UNIFICACION NACIONAL,LIBERACION NACIONAL,43.159060,45.614035
15,2014,ACCION CIUDADANA,LIBERACION NACIONAL,23.481363,22.807018
16,2018,ACCION CIUDADANA,LIBERACION NACIONAL,16.249284,17.543860
17,2022,PROGRESO SOCIAL DEMOCRATICO,LIBERACION NACIONAL,15.035609,17.543860


Comprobaciones concluidas. Sin exportaciones editoriales adicionales.


### Registro de decisiones para la próxima revisión del artículo (sin modificar el Word todavía)

- Conservar la gráfica histórica de **número de partidos con votos y partidos con escaños**. Añadir las tres visualizaciones de en su bloque de oferta, votos y representación; la numeración deberá ajustarse **después** de decidir cuáles serán las figuras definitivas.
- En el orden de la sección de participación, intercambiar la posición actual de las figuras **5 y 6**: el **ranking provincial** debe seguir inmediatamente al gráfico de **abstencionismo provincial**, y luego debe aparecer la **distribución del padrón**. No hacerlo todavía en el Word.
- Explicar con precisión la dominancia provincial del PLN entre 1953 y 1974: encabeza **41 de las 42** combinaciones provincia–elección, con excepción de **Puntarenas, 1966** (Unificación Nacional). Es predominio en pluralidades legislativas provinciales, no ausencia de otros partidos ni partido único.
- En representación femenina, separar **candidaturas propietarias** (salto a partir de 2002), **suplencias** (variaciones e incremento desde fines de los noventa; 1962 no está documentado por categoría), **diputaciones efectivamente electas** (desde 2018 superan el 40 %) y **fórmulas presidenciales** (interpretación condicionada por denominadores reducidos y diferencias entre Presidencia y vicepresidencias). Los hitos normativos se identifican como coincidencias temporales, no como efectos causales estimados.
- En la redacción del artículo, citar cada nueva figura junto al párrafo que la interpreta y explicar expresamente la fórmula y los denominadores. Evitar el uso ambiguo de «votos sin representación»; preferir «votos a partidos sin escaños».


## 9. Presidencia y Asamblea

El partido ganador se identifica por la fórmula presidencial electa, no por el primer lugar en primera ronda. La mayoría se refiere a la composición inicial de la Asamblea. Los antecedentes y la elección siguiente se extraen del corpus; no describen cambios de bancada durante el mandato.

In [17]:
gobierno = frames["presidencia_asamblea"]
show(px.bar(gobierno, x="anio", y="escanos_pct", color="mayoria_inicial",
             hover_data=["nombre", "partido", "escanos"],
             labels={"escanos_pct": "Escaños del partido presidencial (%)", "mayoria_inicial": "Mayoría inicial"}),
     "Bancada inicial del partido presidencial electo")
display(gobierno[["anio", "nombre", "partido", "escanos", "mayoria_inicial", "antecedente_legislativo_electo", "eleccion_siguiente", "cambio_voto_legislativo_pp", "cambio_escanos"]])


,anio,nombre,partido,escanos,mayoria_inicial,antecedente_legislativo_electo,eleccion_siguiente,cambio_voto_legislativo_pp,cambio_escanos
0,1953,José Figueres Ferrer,LIBERACION NACIONAL,30.0,True,False,1958.0,-23.066848,-10.0
1,1958,Mario Echandi Jiménez,UNION NACIONAL,10.0,False,True,1962.0,-8.095996,-1.0
2,1962,Francisco José Orlich Bolmarcich,LIBERACION NACIONAL,29.0,True,True,1966.0,0.081978,0.0
3,1966,José Joaquín Trejos Fernández,UNIFICACION NACIONAL,26.0,False,False,1970.0,-7.265767,-4.0
4,1970,José Figueres Ferrer,LIBERACION NACIONAL,32.0,True,False,1974.0,-9.836745,-5.0
5,1974,Daniel Oduber Quirós,LIBERACION NACIONAL,27.0,False,True,1978.0,-2.020279,-2.0
6,1978,Rodrigo Carazo Odio,UNIDAD,27.0,False,True,1982.0,-14.331614,-9.0
7,1982,Luis Alberto Monge Álvarez,LIBERACION NACIONAL,33.0,True,True,1986.0,-7.317600,-4.0
8,1986,Óscar Arias Sánchez,LIBERACION NACIONAL,29.0,True,True,1990.0,-5.949433,-4.0
9,1990,Rafael Ángel Calderón Fournier,UNIDAD SOCIAL CRISTIANA,29.0,True,True,1994.0,-5.835341,-4.0


## 10. Permanencia partidaria y alcance territorial

Aparición y desaparición se refieren al corpus electoral, mientras inscripción y cancelación pertenecen al registro jurídico. La cobertura efectiva de candidaturas y votos no reemplaza la escala legal de inscripción. Un nombre compartido no garantiza una organización idéntica.

In [18]:
historia_partidos = frames["historia_partidos"]
alcance = frames["alcance_partidos"]
display(historia_partidos.sort_values("elecciones_observadas", ascending=False).head(20))
coverage = alcance[alcance.anio.eq(2026)].sort_values("provincias_con_candidaturas")
show(px.bar(coverage, x="provincias_con_candidaturas", y="partido", orientation="h",
             hover_data=["oferta_presidencial", "candidaturas", "dispersion_apoyo_pp"]),
     "Cobertura provincial de candidaturas legislativas · 2026")


,partido,primera_aparicion,ultima_aparicion,elecciones_observadas,primera_diputacion,escanos_acumulados,denominacion
63,LIBERACION NACIONAL,1953,2026,19,1953.0,467.0,Etiqueta nominal; continuidad jurídica no esta...
115,UNIDAD SOCIAL CRISTIANA,1986,2026,11,1986.0,163.0,Etiqueta nominal; continuidad jurídica no esta...
119,UNION AGRICOLA CARTAGINES,1970,2010,11,1974.0,5.0,Etiqueta nominal; continuidad jurídica no esta...
55,INDEPENDIENTE,1958,1998,8,1958.0,3.0,Etiqueta nominal; continuidad jurídica no esta...
57,INTEGRACION NACIONAL,1998,2026,8,1998.0,5.0,Etiqueta nominal; continuidad jurídica no esta...
23,AUTENTICO LIMONENSE,1978,2018,7,NaN,0.0,Etiqueta nominal; continuidad jurídica no esta...
69,MOVIMIENTO LIBERTARIO,1998,2022,7,1998.0,26.0,Etiqueta nominal; continuidad jurídica no esta...
1,ACCION CIUDADANA,2002,2022,6,2002.0,65.0,Etiqueta nominal; continuidad jurídica no esta...
46,FRENTE AMPLIO,2006,2026,6,2006.0,25.0,Etiqueta nominal; continuidad jurídica no esta...
95,RENOVACION COSTARRICENSE,1998,2018,6,1998.0,5.0,Etiqueta nominal; continuidad jurídica no esta...


## 11. Tamaño de las listas, propietarios y suplencias

Los 45 escaños de 1953 y 1958 y los 57 desde 1962 son magnitudes nacionales. No son un número obligatorio de candidaturas por partido: una organización puede presentar listas parciales o participar en ciertas provincias. Las filas sin nombre no cuentan como candidaturas identificadas.

In [19]:
oferta = candidaturas.groupby(["anio", "tipo"]).size().reset_index(name="Candidaturas")
show(px.line(oferta, x="anio", y="Candidaturas", color="tipo", markers=True),
     "Evolución de las candidaturas propietarias y suplencias")
display(panel[panel.anio.eq(2026) & panel.provincia.ne("Nacional")][["provincia", "partido", "propietarias", "suplencias", "magnitud", "cobertura_lista_pct"]].head(21))


,provincia,partido,propietarias,suplencias,magnitud,cobertura_lista_pct
2335,Alajuela,ACTUEMOS YA,NaN,NaN,12.0,NaN
2336,Alajuela,AGENDA CIUDADANA,12.0,3.0,12.0,100.000000
2337,Alajuela,ALIANZA COSTA RICA PRIMERO,12.0,3.0,12.0,100.000000
2338,Alajuela,ANTICORRUPCION COSTARRICENSE,NaN,NaN,12.0,NaN
2339,Alajuela,AQUI COSTA RICA MANDA,6.0,1.0,12.0,50.000000
2340,Alajuela,AVANZA,3.0,0.0,12.0,25.000000
2341,Alajuela,CENTRO DEMOCRATICO Y SOCIAL,10.0,3.0,12.0,83.333333
2342,Alajuela,COMPATRIOTAS,NaN,NaN,12.0,NaN
2343,Alajuela,DE LA CLASE TRABAJADORA,3.0,0.0,12.0,25.000000
2344,Alajuela,ESPERANZA NACIONAL,12.0,0.0,12.0,100.000000


## 12. Sexo, listas y acceso a la representación

Se compara la participación de mujeres entre propietarias, suplencias, encabezamientos y personas electas. Sexo procede de las bases aportadas. La asociación con cambios normativos necesita una cronología jurídica documentada; este notebook no atribuye efectos causales a reformas no incorporadas.

In [20]:
genero = frames["genero_renovacion"]
genero_nacional = genero[genero.provincia.eq("Nacional")]
gender_long = genero_nacional.melt(id_vars="anio", value_vars=["mujeres_propietarias_pct", "mujeres_suplentes_pct", "mujeres_encabezamientos_pct", "mujeres_electas_pct"], var_name="Categoría", value_name="Mujeres (%)")
show(px.line(gender_long, x="anio", y="Mujeres (%)", color="Categoría", markers=True),
     "De las listas a la representación: mujeres a lo largo del tiempo")
owners = candidaturas[candidaturas.tipo.eq("Propietaria")]
success = owners.groupby(["anio", "sexo"]).agg(candidaturas=("nombre", "size"), electas=("electa", "sum")).reset_index()
success["eleccion_pct"] = 100 * success.electas / success.candidaturas
show(px.line(success, x="anio", y="eleccion_pct", color="sexo", markers=True,
             labels={"eleccion_pct": "Candidaturas electas (%)", "sexo": "Sexo según fuente"}),
     "Acceso al escaño entre candidaturas propietarias por sexo")


## 13. Recurrencias, experiencia y movilidad

La experiencia es un antecedente electoral registrado, no experiencia política total. En 1953 los historiales están truncados por el inicio del corpus. Se distingue cambio de partido, provincia o cargo entre elecciones de cambios de bancada dentro del mandato. Coincidencias nominales no verifican exhaustivamente homónimos.

In [21]:
display(personas.sort_values(["elecciones", "candidaturas"], ascending=False).head(20))
show(px.line(genero_nacional, x="anio", y="experiencia_pct", markers=True,
             labels={"experiencia_pct": "Personas con antecedente electoral (%)"}),
     "Experiencia previa entre personas candidatas a diputaciones")
movimientos = frames["movimientos"]
display(movimientos.groupby(["dimension", "origen", "destino"]).size().reset_index(name="Cambios").sort_values("Cambios", ascending=False).head(25))
carreras = eventos[eventos.cargo.ne("Diputación") & eventos.experiencia_previa]
display(carreras[["anio", "nombre", "partido", "cargo", "participaciones_previas", "electa"]].head(20))


,id_persona,nombre,sexo,inicio,fin,elecciones,candidaturas,partidos,provincias,cargos,elecciones_exitosas,primera_eleccion,intentos_hasta_eleccion,identidad,nombre_original,variantes_nombre
750,0f1468f41ea0f8,Walter Enrique Muñoz Cespedes,Hombre,1998,2026,8,11,1,1,2,2,1998.0,1.0,Conciliación nominal unívoca: variante de nomb...,Walter Enrique Muñoz Cespedes,Walter Enrique Muñoz Cespedes | Walter Muñoz C...
1253,193c4d408f9bc2,Otto Guevara Guth,Hombre,1998,2022,7,8,2,1,2,2,1998.0,1.0,Coincidencia nominal exacta,Otto Guevara Guth,Otto Guevara Guth
1125,1694875a5cc529,Juan Guillermo Brenes Castillo,Hombre,1966,2002,6,6,2,1,1,3,1974.0,3.0,Coincidencia nominal exacta,Juan Guillermo Brenes Castillo,Juan Guillermo Brenes Castillo
2537,32c879539f79fb,Maria Yolanda Alpizar Sanchez,Mujer,2002,2026,6,6,3,2,1,0,NaN,NaN,Conciliación nominal unívoca: variante de nomb...,Maria Yolanda Alpizar Sanchez,Maria Yolanda Alpizar Sanchez | Yolanda Alpiza...
9395,ba0847319cbed4,Marvin Luis Wright Lindo,Hombre,1978,1998,6,6,1,1,1,0,NaN,NaN,Coincidencia nominal exacta,Marvin Luis Wright Lindo,Marvin Luis Wright Lindo
9473,bba4e9b6a76a46,Edgar Mora Garcia,Hombre,1953,1982,6,6,5,1,1,1,1953.0,1.0,Coincidencia nominal exacta,Edgar Mora Garcia,Edgar Mora Garcia | Edgar Mora García
7506,94ae1e2160e8bd,Óscar Andrés López Arias,Hombre,2006,2022,5,7,1,1,2,2,2006.0,1.0,Coincidencia nominal exacta,Óscar Andrés López Arias,Oscar Andres Lopez Arias | Óscar Andrés López ...
3387,4242bf22003a67,Ovidio Jovel Morera Molina,Hombre,1986,2002,5,6,2,1,2,0,NaN,NaN,Conciliación nominal unívoca: variante de nomb...,Ovidio Jovel Morera Molina,Jovel Morera Molina | Ovidio Jovel Morera Molina
7681,98427373dde399,Álvaro González Espinoza,Hombre,1974,1998,5,6,3,2,2,0,NaN,NaN,Coincidencia nominal exacta,Álvaro González Espinoza,Alvaro Gonzalez Espinoza | Álvaro González Esp...
7897,9cdb1e0ff6badd,Rolando Araya Monge,Hombre,1974,2022,5,6,4,1,2,1,1974.0,1.0,Coincidencia nominal exacta,Rolando Araya Monge,Rolando Araya Monge


,dimension,origen,destino,Cambios
0,Cargo,Diputación,Presidencia,51
1,Cargo,Diputación,Primera Vicepresidencia,35
180,Partido,INDEPENDIENTE,INDEPENDIENTE OBRERO,30
406,Partido,UNIDAD,UNIDAD SOCIAL CRISTIANA,29
2,Cargo,Diputación,Segunda Vicepresidencia,28
534,Provincia,San José,Alajuela,27
506,Provincia,Alajuela,San José,26
512,Provincia,Cartago,San José,25
522,Provincia,Heredia,San José,23
5,Cargo,Primera Vicepresidencia,Diputación,21


,anio,nombre,partido,cargo,participaciones_previas,electa
420,1958,Francisco José Orlich Bolmarcich,LIBERACION NACIONAL,Presidencia,1,False
517,1958,José Joaquín Peralta Esquivel,UNION NACIONAL,Primera Vicepresidencia,1,True
579,1958,Mario Echandi Jiménez,UNION NACIONAL,Presidencia,1,True
624,1958,Otto Cortés Fernández,LIBERACION NACIONAL,Primera Vicepresidencia,1,False
646,1958,Rafael París Steffens,LIBERACION NACIONAL,Segunda Vicepresidencia,1,False
794,1962,Enrique Obregón Valverde,ACCION DEMOCRATICA POPULAR,Presidencia,1,False
809,1962,Francisco José Orlich Bolmarcich,LIBERACION NACIONAL,Presidencia,2,True
913,1962,Manuel Escalante Durán,REPUBLICANO NACIONAL,Primera Vicepresidencia,1,False
948,1962,Otilio Ulate Blanco,UNION NACIONAL,Presidencia,1,False
966,1962,Rafael Ángel Calderón Guardia,REPUBLICANO NACIONAL,Presidencia,1,False


## 14. Fórmulas presidenciales, vicepresidencias y apellidos

El HTML permite consultar fórmulas y trayectorias individuales. Los apellidos se toman de campos expresamente separados de las bases de electas y fórmulas; su repetición no demuestra parentesco ni familias políticas.

In [22]:
composition = formulas.groupby(["anio", "cargo"]).sexo.agg(lambda s: 100 * s.eq("Mujer").mean()).reset_index(name="Mujeres (%)")
show(px.line(composition, x="anio", y="Mujeres (%)", color="cargo", markers=True),
     "Mujeres en Presidencia y vicepresidencias")
surnames = pd.concat([electas[["nombre", "apellido1", "apellido2"]], formulas[["nombre", "apellido1", "apellido2"]]])
surnames = surnames.melt(id_vars="nombre", value_vars=["apellido1", "apellido2"], value_name="Apellido").dropna(subset=["Apellido"])
surname_counts = surnames.groupby("Apellido").nombre.nunique().sort_values(ascending=False).head(20)
display(surname_counts.rename("Nombres completos distintos").to_frame())


,Nombres completos distintos
Apellido,
Vargas,53
Rodríguez,45
Rojas,45
Jiménez,40
Mora,35
González,34
Castro,30
Ramírez,28
Quesada,27


## 15. Evolución del diseño de las papeletas

Se utilizan las 133 observaciones provincia–elección y las características de la base aportada. Sus fuentes son imágenes reconstruidas. Se examinan nombres, suplencias, banderas, disposición, casillas, fondos, encabezados y tipografía; la correlación con voto nulo o blanco no establece un efecto del diseño.

In [23]:
papeletas = frames["papeletas"]
show(px.scatter(papeletas, x="anio", y="provincia", color="Familia de diseño",
                symbol="Familia de diseño", hover_data=["Variante de diseño", "Nombres de candidaturas visibles", "Fondo celeste visible"]),
     "Familias de diseño de papeletas por provincia y elección")
display(papeletas[papeletas.provincia.eq("San José")][["anio", "Familia de diseño", "Variante de diseño", "Nombres de candidaturas visibles", "Posición de casilla", "Fondo celeste visible", "Tipo de fuente"]])


,anio,Familia de diseño,Variante de diseño,Nombres de candidaturas visibles,Posición de casilla,Fondo celeste visible,Tipo de fuente
0,1953,1953 · Columnas iniciales,Encabezado horizontal con serifas,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
7,1958,1958–1994 · Columnas nominales,Reconstrucción de columnas con suplentes,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
14,1962,1958–1994 · Columnas nominales,Lista sin sección de suplencia visible,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
21,1966,1958–1994 · Columnas nominales,Columnas con sustitutos,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
28,1970,1958–1994 · Columnas nominales,Columnas con sustitutos,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
35,1974,1958–1994 · Columnas nominales,Columnas con sustitutos,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
42,1978,1958–1994 · Columnas nominales,Columnas con sustitutos,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
49,1982,1958–1994 · Columnas nominales,Provincia bajo título central,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
56,1986,1958–1994 · Columnas nominales,Provincia bajo título central,Sí,Debajo de los nombres,No,Imagen reconstruida aportada
63,1990,1958–1994 · Columnas nominales,Provincia bajo título central,Sí,Debajo de los nombres,No,Imagen reconstruida aportada


## 16. Cancelaciones y reutilización de nombres

La última cancelación del listado aportado es de junio de 2022. La coincidencia temporal y nominal no verifica continuidad ideológica u organizativa. El HTML incluye también la tabla de coincidencias electorales del archivo original de cancelaciones.

In [24]:
cancelados = frames["cancelados"]
repetidos = cancelados[cancelados["Nombre repetido"].eq("Sí")].groupby("Nombre del partido").size().sort_values()
show(px.bar(x=repetidos.values, y=repetidos.index, orientation="h",
             labels={"x": "Registros", "y": "Denominación partidaria"}),
     "Nombres repetidos en registros de cancelación")
display(cancelados[["Nombre del partido", "Escala", "Provincia", "Fecha de inscripción", "Fecha de cancelación", "Nombre repetido"]].head(20))
display(frames["coincidencias_registrales"].head(10))


,Nombre del partido,Escala,Provincia,Fecha de inscripción,Fecha de cancelación,Nombre repetido
0,Republicano Nacional,Nacional,None,1947-11-07,1952-01-19,Sí
1,Compactación Nacional,Nacional,None,1949-08-13,1952-05-13,No
2,Unión Nacional,Nacional,None,1947-11-04,1952-10-17,Sí
3,Progresista Independiente,Nacional,None,1952-10-25,1956-10-25,No
4,Demócrata,Nacional,None,1952-10-25,1956-10-26,Sí
5,Republicano Nacional Independiente,Nacional,None,1953-03-20,1957-03-20,No
6,Movimiento Democrático Oposicionista,Nacional,None,1956-06-28,1960-06-28,No
7,Demócrata Auténtico,Nacional,None,1957-01-12,1961-01-12,No
8,Independiente,Nacional,None,1957-01-23,1961-01-23,No
9,Republicano Nacional,Nacional,None,1961-01-30,1965-01-30,Sí


,ID registro,Nombre del partido,Nombre comparado,Tipo de denominación,Año electoral,Provincia,Relación temporal,Bases con coincidencia,Nombres en bases electorales
0,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Alajuela,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
1,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Cartago,Posterior al intervalo,Candidaturas legislativas; Resultados legislat...,REPUBLICANO NACIONAL
2,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Guanacaste,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
3,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Heredia,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
4,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Limón,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
5,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,Puntarenas,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
6,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1958,San José,Posterior al intervalo,Resultados legislativos,REPUBLICANO NACIONAL
7,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1962,Nacional,Posterior al intervalo,Candidaturas presidenciales,Partido Republicano Nacional (PRN)
8,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1974,Alajuela,Posterior al intervalo,Candidaturas legislativas; Resultados legislat...,REPUBLICANO NACIONAL
9,CR-PC-001,Republicano Nacional,Republicano Nacional,Nombre principal,1974,Cartago,Posterior al intervalo,Candidaturas legislativas; Resultados legislat...,REPUBLICANO NACIONAL


## 17. Laboratorio de asociaciones exploratorias

Las correlaciones se calculan con pares completos. No se seleccionan por significancia ni se interpretan causalmente. Las observaciones tienen dependencia temporal y territorial. El HTML permite cambiar las variables, el año y la provincia y descargar los pares usados.

In [25]:
asociaciones = frames["asociaciones"]
display(asociaciones[asociaciones.ambito.eq("Nacional")].head(15))
show(px.scatter(provincial, x="nep", y="abstencion_pct", color="provincia", hover_data=["anio"],
                labels={"nep": "Número efectivo de partidos", "abstencion_pct": "Abstencionismo (%)"}),
     "Fragmentación y abstencionismo: observaciones provincia–elección")


,ambito,variable_x,variable_y,observaciones,pearson,spearman,interpretacion
0,Nacional,abstencion_pct,invalidos_pct,19,-0.059895,-0.384211,Exploratoria; dependencia temporal y territori...
1,Nacional,abstencion_pct,nep,19,0.702281,0.659649,Exploratoria; dependencia temporal y territori...
2,Nacional,abstencion_pct,concentracion_2,19,-0.786267,-0.798246,Exploratoria; dependencia temporal y territori...
3,Nacional,abstencion_pct,margen,19,0.158949,0.100000,Exploratoria; dependencia temporal y territori...
4,Nacional,abstencion_pct,gallagher,19,0.551144,0.449123,Exploratoria; dependencia temporal y territori...
5,Nacional,abstencion_pct,mujeres_electas_pct,19,0.678097,0.413370,Exploratoria; dependencia temporal y territori...
6,Nacional,abstencion_pct,mujeres_candidatas_pct,19,0.629977,0.384211,Exploratoria; dependencia temporal y territori...
7,Nacional,abstencion_pct,divergencia_arenas,19,0.743896,0.756140,Exploratoria; dependencia temporal y territori...
8,Nacional,abstencion_pct,volatilidad_nominal,18,0.265913,0.329205,Exploratoria; dependencia temporal y territori...
9,Nacional,invalidos_pct,nep,19,-0.474615,-0.729825,Exploratoria; dependencia temporal y territori...


## Lectura visual y congruencia

Se conservan los módulos y todas las descargas. La actualización añade rankings provinciales, diferencias pequeñas entre arenas, composición completa por sexo, movilidad provincial con nombres, coincidencia entre ganadores y primera bancada, y secuencias registrales de nombres reutilizados. Las variantes `Unidad Socialcristiana` y `Org Socialista de los Trabajadores` se unifican con sus denominaciones desarrolladas antes de calcular las métricas. Los nombres personales se presentan con capitalización uniforme, conservando `nombre_original`. No se inventan tildes o edades.

Los colores documentados se extraen de las divisas del TSE. Son tonos representativos para visualización, no códigos HEX oficiales ni verificación de continuidad histórica. Las referencias y banderas están en `recursos/colores/` y `recursos/colores_partidos.json`; el HTML muestra las fuentes en Metodología. Para nombres sin divisa documentada se utiliza una paleta de identificación sin carácter oficial.

In [26]:
gaps = frames["brechas_participacion"]
gap_national = gaps[gaps.provincia.eq("Nacional")]
show(px.bar(gap_national, x="anio", y="diferencia_participacion_pp",
            labels={"diferencia_participacion_pp":"Participación legislativa − presidencial (pp)"}),
     "Diferencias de participación entre arenas: escala de puntos porcentuales")
rankings = frames["rankings_provinciales"]
r = rankings[rankings.arena.eq("Legislativa")]
fig = px.line(r, x="anio", y="rango_participacion", color="provincia", markers=True,
              hover_data=["participacion_pct"], color_discrete_map=PROVINCE_COLORS)
fig.update_yaxes(autorange="reversed", dtick=1, title="Puesto · 1 = mayor participación")
show(fig, "Posición de cada provincia en participación legislativa")


### Presidencia electa, voto legislativo y primera bancada

Se distingue el ganador final de la Presidencia, el partido con más votos legislativos y la fuerza con más escaños. Primera bancada no significa mayoría absoluta ni presidencia del Directorio legislativo.

In [27]:
agreement = frames["concordancia_gobierno"]
display(agreement[["anio","nombre","ganador_presidencial","ganador_voto_legislativo","primera_bancada","escanos","escanos_primera_bancada","coincide_primera_bancada"]])
assert len(agreement) == len(YEARS)
assert agreement.escanos_primera_bancada.ge(agreement.escanos).all()
print("Elecciones con Presidencia y primera bancada diferentes:", int((~agreement.coincide_primera_bancada).sum()))


,anio,nombre,ganador_presidencial,ganador_voto_legislativo,primera_bancada,escanos,escanos_primera_bancada,coincide_primera_bancada
0,1953,José Figueres Ferrer,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,30.0,30.0,True
1,1958,Mario Echandi Jiménez,UNION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,10.0,20.0,False
2,1962,Francisco José Orlich Bolmarcich,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,29.0,29.0,True
3,1966,José Joaquín Trejos Fernández,UNIFICACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,26.0,29.0,False
4,1970,José Figueres Ferrer,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,32.0,32.0,True
5,1974,Daniel Oduber Quirós,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,27.0,27.0,True
6,1978,Rodrigo Carazo Odio,UNIDAD,UNIDAD,UNIDAD,27.0,27.0,True
7,1982,Luis Alberto Monge Álvarez,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,33.0,33.0,True
8,1986,Óscar Arias Sánchez,LIBERACION NACIONAL,LIBERACION NACIONAL,LIBERACION NACIONAL,29.0,29.0,True
9,1990,Rafael Ángel Calderón Fournier,UNIDAD SOCIAL CRISTIANA,UNIDAD SOCIAL CRISTIANA,UNIDAD SOCIAL CRISTIANA,29.0,29.0,True


Elecciones con Presidencia y primera bancada diferentes: 5


### Movilidad provincial y composición por sexo

Cada movimiento relaciona dos participaciones observadas de una identidad nominal. El año del movimiento es el de destino. La matriz cuenta transiciones; la tabla permite inspeccionar nombres, años y provincias. La edad no se calcula porque las fuentes no contienen fechas de nacimiento.

In [28]:
moves = frames["movimientos"]
province_moves = moves[moves.dimension.eq("Provincia")]
matrix = pd.crosstab(province_moves.origen, province_moves.destino).reindex(index=PROVINCES, columns=PROVINCES, fill_value=0)
fig = go.Figure(go.Heatmap(x=PROVINCES, y=PROVINCES, z=matrix.values,
    texttemplate="%{z}", colorscale=[[0,"#f1f5f9"],[1,"#007f7a"]],
    hovertemplate="Origen: %{y}<br>Destino: %{x}<br>%{z} cambios<extra></extra>"))
fig.update_layout(xaxis_title="Provincia de destino", yaxis_title="Provincia de origen")
show(fig, "Movimientos entre provincias de candidatura")
assert matrix.to_numpy().sum() == len(province_moves)
display(province_moves[["nombre","sexo","anio_origen","origen","anio","destino"]].head(25))
sex = frames["composicion_sexo"]
sex_owners = sex[sex.provincia.eq("Nacional") & sex.poblacion.eq("Candidaturas legislativas") & sex.categoria.eq("Propietaria")]
show(px.bar(sex_owners, x="anio", y="porcentaje", color="sexo", barmode="stack",
            hover_data=["registros","total_categoria"], color_discrete_map=SEX_COLORS),
     "Mujeres y hombres entre candidaturas propietarias")


,nombre,sexo,anio_origen,origen,anio,destino
0,Gabriela Rios Cascante,Mujer,2014,Guanacaste,2018,San José
4,Iris Zamora Zumbado,Mujer,2010,Alajuela,2018,San José
36,Alvaro Escamilla Gutierrez,Hombre,2006,Guanacaste,2014,San José
45,Mauricio Alvarado Delgadillo,Hombre,1986,San José,2006,Puntarenas
49,Carmen Elena Quesada Santamaria,Mujer,2022,Limón,2026,San José
55,Carlos Manuel Hernandez Herrera,Hombre,1966,Cartago,1994,San José
63,German Ramon Meza Sibaja,Hombre,1994,San José,2002,Alajuela
68,Laura Guido Perez,Mujer,2014,San José,2018,Cartago
73,José Rafael Cordero Croceri,Hombre,1974,Cartago,1990,Puntarenas
75,Marcial Aguiluz Orellana,Hombre,1958,Puntarenas,1970,San José


### Cambios de oferta y secuencias de nombres partidarios

La oferta observada distingue participación presidencial y legislativa y el número de provincias; no deduce cambios de escala jurídica. Las fechas de inscripción y cancelación permiten separar intervalos superpuestos de inscripciones posteriores con igual nombre, sin establecer continuidad ideológica.

In [29]:
display(frames["cambios_oferta"].head(25))
reuse = frames["reutilizacion_nombres"]
display(reuse[["nombre_partido","cancelacion_anterior","inscripcion_posterior","dias_entre_registros","relacion","escala_anterior","escala_posterior"]])
assert (reuse["dias_entre_registros"].gt(0) == reuse["relacion"].eq("Inscripción posterior a cancelación")).all()
print("Identidades nominales; fechas y registros originales conservados.")


,partido,anio_origen,anio,oferta_anterior,oferta_actual,alcance_anterior,alcance_actual,provincias_antes,provincias_despues
0,ACCESIBILIDAD SIN EXCLUSION,2006,2010,Solo legislativa,Presidencial y legislativa,Una provincia,Siete provincias,1,7
1,ALIANZA DEMOCRATA CRISTIANA,2014,2018,Solo legislativa,Presidencial y legislativa,Una provincia,Siete provincias,1,7
2,ALIANZA DEMOCRATICA NACIONALISTA,2006,2018,Presidencial y legislativa,Solo legislativa,Siete provincias,Una provincia,7,1
3,ALIANZA PATRIOTICA,2010,2014,Presidencial y legislativa,Solo legislativa,Siete provincias,Varias provincias,7,2
4,DEMOCRATA,1953,1966,Presidencial y legislativa,Solo legislativa,Siete provincias,Siete provincias,7,7
5,DEMOCRATA,1966,1974,Solo legislativa,Presidencial y legislativa,Siete provincias,Siete provincias,7,7
6,FRENTE AMPLIO,2006,2010,Solo legislativa,Presidencial y legislativa,Una provincia,Siete provincias,1,7
7,INDEPENDIENTE,1958,1974,Presidencial y legislativa,Solo legislativa,Siete provincias,Siete provincias,7,7
8,INDEPENDIENTE,1974,1978,Solo legislativa,Presidencial y legislativa,Siete provincias,Siete provincias,7,7
9,INDEPENDIENTE,1978,1982,Presidencial y legislativa,Presidencial y legislativa,Siete provincias,Varias provincias,7,6


,nombre_partido,cancelacion_anterior,inscripcion_posterior,dias_entre_registros,relacion,escala_anterior,escala_posterior
0,Demócrata,1956-10-26,1965-03-08,3055,Inscripción posterior a cancelación,Nacional,Nacional
1,Demócrata,1984-02-01,1997-07-23,4921,Inscripción posterior a cancelación,Nacional,Nacional
2,Nuevo Partido Socialista,2014-03-11,2017-06-14,1191,Inscripción posterior a cancelación,Provincial,Provincial
3,Nuevo Partido Socialista,2018-03-06,2021-07-28,1240,Inscripción posterior a cancelación,Provincial,Provincial
4,Nuevo Partido Socialista,2022-06-08,2021-07-28,-315,Intervalos superpuestos,Provincial,Provincial
5,Republicano Nacional,1952-01-19,1953-03-20,426,Inscripción posterior a cancelación,Nacional,Provincial
6,Republicano Nacional,1957-03-20,1956-10-05,-166,Intervalos superpuestos,Provincial,Provincial
7,Republicano Nacional,1960-10-05,1956-10-30,-1436,Intervalos superpuestos,Provincial,Provincial
8,Republicano Nacional,1960-10-30,1961-01-30,92,Inscripción posterior a cancelación,Provincial,Nacional
9,Republicano Nacional,1965-01-30,1971-06-30,2342,Inscripción posterior a cancelación,Nacional,Nacional


Identidades nominales; fechas y registros originales conservados.


## Mapas, perfiles y representación territorial
Autoría: **Juan Ignacio Garbanzo Fallas — politólogo y científico de datos**.
La cartografía provincial se incorpora al HTML sin servicios externos. Perfiles distingue la denominación electoral de los registros jurídicos, sin asignar continuidad no documentada. El laboratorio añade cruces categóricos y el ranking de apellidos muestra todas las etiquetas.
La representación territorial usa el padrón como denominador; no equivale a población censal.

In [30]:
territorial = frames["representacion_territorial"]
assert len(territorial) == 133
assert territorial.groupby("anio").magnitud.sum().tolist() == [45,45]+[57]*17
assert len(data["geografia"]["features"]) == 7
show(px.line(territorial, x="anio", y="magnitud", color="provincia", markers=True, line_shape="hv", labels={"magnitud":"Escaños", "anio":"Elección"}), "Distribución provincial de escaños")
show(px.line(territorial, x="anio", y="indice_representacion", color="provincia", markers=True), "Peso en escaños / peso en padrón")
display(territorial[["anio","provincia","magnitud","cambio_escanos","electores_por_escano","brecha_representacion_pp"]])


,anio,provincia,magnitud,cambio_escanos,electores_por_escano,brecha_representacion_pp
0,1953,Alajuela,8.0,NaN,6916.500000,-1.063779
1,1953,Cartago,6.0,NaN,5829.833333,1.422345
2,1953,Guanacaste,5.0,NaN,5336.400000,2.025403
3,1953,Heredia,3.0,NaN,6552.333333,-0.026901
4,1953,Limón,2.0,NaN,5643.000000,0.601355
...,...,...,...,...,...,...
128,2026,Guanacaste,5.0,1.0,55717.600000,1.169620
129,2026,Heredia,5.0,-1.0,75558.200000,-1.537502
130,2026,Limón,5.0,0.0,62755.600000,0.209330
131,2026,Puntarenas,6.0,1.0,58501.000000,0.947812


## 18. Base final y auditoría de resultados · El resultado de cada candidatura se toma de su campo `Electa`; la lista separada de electas sirve de cotejo sin desmarcar personas por abreviaturas nominales. La unidad de los porcentajes se declara en cada tabla: **candidatura-evento** o **persona nominal**. Una derrota en una candidatura no se interpreta como el resultado final de toda la carrera: se distingue si la persona había sido electa antes, resultó electa después o nunca registra una elección exitosa en el corpus.

La temporalidad se calcula por **año electoral**. Cuando una persona aparece en dos candidaturas en el mismo año, ninguna se considera antecedente temporal de la otra. Las cédulas y las identidades históricas incompletas impiden equiparar automáticamente nombre con persona civil.

In [31]:
from IPython.display import display

# Autorreparación 1.7: si el kernel conserva `frames` de una integración anterior,
# reconstruir las tablas de outcomes de la versión actual antes de analizarlas.
_required_outcomes = {
    "resultados_candidaturas", "resultados_personas", "transiciones_resultados",
    "puentes_cargos", "antecedentes_cargos_electos", "camara_electa", "ejecutivo_electo"
}
_missing_outcomes = sorted(_required_outcomes.difference(frames))
if _missing_outcomes:
    print("Reconstruyendo tablas de outcomes ausentes del kernel:", ", ".join(_missing_outcomes))
    from atlas_outcomes import build_outcomes
    from atlas_pipeline import norm, records

    _events = frames["eventos"].copy()
    _electas = frames["electas"].copy()
    _formulas = frames["formulas"].copy()
    for _df in (_events, _electas, _formulas):
        _df["nombre_key"] = _df["nombre"].map(norm)

    _rebuilt = build_outcomes(_events, _electas, _formulas)
    for _key, _df in _rebuilt.items():
        frames[_key] = _df.copy()
        data[_key] = records(_df)
        _df.to_csv(OUTPUT_DIR / "datos" / f"{_key}.csv", index=False, encoding="utf-8-sig")

    _still_missing = sorted(_required_outcomes.difference(frames))
    if _still_missing:
        raise RuntimeError("No se pudieron reconstruir las tablas de outcomes: " + ", ".join(_still_missing))
    print("Outcomes reconstruidos correctamente para esta sesión.")

outcome = frames["resultados_candidaturas"]
people_outcome = frames["resultados_personas"]
transitions = frames["transiciones_resultados"]
bridges = frames["puentes_cargos"]
antecedents = frames["antecedentes_cargos_electos"]
chamber = frames["camara_electa"]

assert len(chamber) == 1059
assert chamber.groupby("anio").size().sort_index().tolist() == [45, 45] + [57] * 17
assert int(frames["candidaturas"].electa.sum()) == 1059
assert len(frames["candidaturas"]) + data["meta"]["filas_sin_nombre"] == data["meta"]["filas_fuentes"]["cand"]
assert len(data["geografia"]["features"]) == 7
assert (bridges.anio_origen < bridges.anio).all()
assert (bridges.anio_origen < bridges.anio).all()  # La repetición de cargo es válida si el año posterior es mayor.
print('Puentes al mismo cargo en elecciones posteriores:', int((bridges.cargo_origen == bridges.cargo_destino).sum()))
assert len(antecedents) == int(outcome.electa.sum())
assert set(antecedents.antecedente).issubset({"Ya había resultado electa antes", "Había competido antes, pero aún no había sido electa", "No tenía candidaturas anteriores en estas bases"})

print("Candidaturas en fuente:", len(frames["candidaturas"]) + data["meta"]["filas_sin_nombre"])
print("Candidaturas con nombre:", len(frames["candidaturas"]), "| diputaciones electas:", len(chamber))
print("Personas nominales:", len(people_outcome), "| transiciones consecutivas:", len(transitions))
print("Puentes entre cargos:", len(bridges), "| antecedentes de outcomes electos:", len(antecedents))

rates = []
for label, subset in [("Todas", outcome), ("Primer año documentado", outcome[outcome.primera_candidatura]),
                      ("Año electoral posterior", outcome[outcome.regresa]),
                      ("Ya electa anteriormente", outcome[outcome.electa_antes])]:
    n = int(subset.electa.sum()); den = len(subset)
    rates.append({"Grupo": label, "Electas": n, "Candidaturas": den, "Porcentaje": 100*n/den if den else None})
display(pd.DataFrame(rates))

# Ejemplo reproducible del nuevo puente Diputación -> Presidencia.
bridge_example = bridges[(bridges.cargo_origen == "Diputación") & (bridges.cargo_destino == "Presidencia")]
bridge_people = bridge_example.groupby("id_persona").agg(nombre=("nombre", "last"),
    pares=("id_destino", "size"), destino_electo=("electa_destino", "max")).reset_index()
display(bridge_people.sort_values(["destino_electo", "pares"], ascending=False).head(20))

show(px.bar(pd.DataFrame(rates), x="Grupo", y="Porcentaje", text="Electas",
            labels={"Porcentaje":"Candidaturas electas (%)", "Grupo":"Condición previa"}),
     "Éxito electoral descriptivo por experiencia")
print("Las asociaciones son descriptivas. No atribuir causalidad, identidad civil o parentesco a coincidencias nominales.")

Puentes al mismo cargo en elecciones posteriores: 2745
Candidaturas en fuente: 14874
Candidaturas con nombre: 14870 | diputaciones electas: 1059
Personas nominales: 13050 | transiciones consecutivas: 2325
Puentes entre cargos: 3194 | antecedentes de outcomes electos: 1116


,Grupo,Electas,Candidaturas,Porcentaje
0,Todas,1116,15400,7.246753
1,Primer año documentado,776,13121,5.914183
2,Año electoral posterior,340,2279,14.918824
3,Ya electa anteriormente,153,377,40.583554


,id_persona,nombre,pares,destino_electo
22,55b570dd15b317,Luis Alberto Monge Álvarez,4,True
25,5ed5c12bb01e4e,Rodrigo Carazo Odio,4,True
11,14bf391b934645,Rafael Ángel Calderón Fournier,3,True
45,b33964bddee8d1,Mario Echandi Jiménez,3,True
53,dbe07284e6ddb3,Daniel Oduber Quirós,3,True
9,11bf4727a7d350,Francisco José Orlich Bolmarcich,2,True
20,45488e796aaf7c,Óscar Arias Sánchez,2,True
21,4fac866f683a09,Abel Pacheco de la Espriella,2,True
35,8640f03f7ceb18,Miguel Ángel Rodríguez Echeverría,2,True
31,7de24a499c9b84,Laura Chinchilla Miranda,1,True


Las asociaciones son descriptivas. No atribuir causalidad, identidad civil o parentesco a coincidencias nominales.


### 18.1 Auditoría adicional de claridad: candidatura, persona y carrera completa

Estas comprobaciones sostienen las vistas de **Éxito electoral**, **Explorador de personas**, **Sexo y representación** y **Fórmulas presidenciales**. Se verifican denominadores, el contexto temporal de candidaturas no electas, la recurrencia por años electorales y los porcentajes de elección por cargo y sexo. Las asociaciones son descriptivas.

In [32]:
# Auditoría vectorizada: la condición "electa antes/después" depende de años
# estrictamente anteriores/posteriores, nunca del orden de candidaturas simultáneas.
_year = (outcome.groupby(["id_persona","anio"], as_index=False)
         .agg(electa_anio=("electa","max"))
         .sort_values(["id_persona","anio"]))
_prior_cum = _year.groupby("id_persona")["electa_anio"].cummax()
_year["prior_expected"] = _prior_cum.groupby(_year["id_persona"]).shift(fill_value=False).astype(bool)
_rev = _year.iloc[::-1].copy()
_later_cum = _rev.groupby("id_persona")["electa_anio"].cummax()
_rev["later_expected"] = _later_cum.groupby(_rev["id_persona"]).shift(fill_value=False).astype(bool)
_year["later_expected"] = _rev["later_expected"].iloc[::-1].to_numpy()
_audit = outcome.merge(_year[["id_persona","anio","prior_expected","later_expected"]],
                       on=["id_persona","anio"], how="left", validate="many_to_one")
assert _audit.electa_antes.eq(_audit.prior_expected).all()
assert _audit.electa_despues.eq(_audit.later_expected).all()

# Resumen directo por cargo: candidaturas, electas, no electas y tasa.
by_role = (outcome.groupby("cargo")
           .agg(candidaturas=("id_evento","size"), electas=("electa","sum"))
           .reset_index())
by_role["no_electas"] = by_role.candidaturas - by_role.electas
by_role["tasa_eleccion_pct"] = 100 * by_role.electas / by_role.candidaturas
display(by_role.sort_values("candidaturas", ascending=False))

# Resumen por sexo dentro de cada cargo, con denominador explícito.
by_role_sex = (outcome[outcome.sexo.isin(["Mujer","Hombre"])]
               .groupby(["cargo","sexo"])
               .agg(candidaturas=("id_evento","size"), electas=("electa","sum"))
               .reset_index())
by_role_sex["no_electas"] = by_role_sex.candidaturas - by_role_sex.electas
by_role_sex["tasa_eleccion_pct"] = 100 * by_role_sex.electas / by_role_sex.candidaturas
display(by_role_sex)

# Las candidaturas no electas se clasifican por lo que ocurre en la carrera completa.
losses = outcome[~outcome.electa].copy()
losses["contexto_carrera"] = np.select(
    [losses.electa_despues, (~losses.electa_despues) & losses.electa_antes],
    ["Resultó electa después", "Ya había resultado electa antes"],
    default="Nunca resultó electa en el corpus"
)
display(losses.groupby("contexto_carrera").agg(candidaturas=("id_evento","size"), personas=("id_persona","nunique")).reset_index())

# Rutas completas de personas hasta su primera elección exitosa.
display(people_outcome.groupby("ruta_primera_eleccion").size().sort_values(ascending=False).rename("personas").to_frame())
print("Auditoría 1.7 completada: temporalidad por año electoral y denominadores explícitos.")

,cargo,candidaturas,electas,no_electas,tasa_eleccion_pct
0,Diputación,14870,1059,13811,7.121722
1,Presidencia,178,19,159,10.674157
2,Primera Vicepresidencia,176,19,157,10.795455
3,Segunda Vicepresidencia,176,19,157,10.795455


,cargo,sexo,candidaturas,electas,no_electas,tasa_eleccion_pct
0,Diputación,Hombre,10296,833,9463,8.090521
1,Diputación,Mujer,4572,226,4346,4.943132
2,Presidencia,Hombre,163,17,146,10.429448
3,Presidencia,Mujer,15,2,13,13.333333
4,Primera Vicepresidencia,Hombre,97,15,82,15.463918
5,Primera Vicepresidencia,Mujer,79,4,75,5.063291
6,Segunda Vicepresidencia,Hombre,110,14,96,12.727273
7,Segunda Vicepresidencia,Mujer,66,5,61,7.575758


,contexto_carrera,candidaturas,personas
0,Nunca resultó electa en el corpus,13825,12093
1,Resultó electa después,272,210
2,Ya había resultado electa antes,187,139


,personas
ruta_primera_eleccion,
Nunca resultó electa en el corpus,12087
Resultó electa en su primer año documentado,776
Resultó electa después de 1 elección previa sin éxito,152
Resultó electa después de 2 elecciones previas sin éxito,30
Resultó electa después de 3 elecciones previas sin éxito,5


Auditoría 1.7 completada: temporalidad por año electoral y denominadores explícitos.


## 19. Aplicación interactiva · El HTML contiene comparación provincial y temporal, composición de la cámara por elección, perfiles nominales, coincidencias de apellidos, resultados por candidatura y persona, transiciones, puentes entre cargos, representación territorial y un centro de descarga de datos. Incluye la biblioteca gráfica, los datos y la geometría provincial dentro del mismo archivo, por lo que puede abrirse sin conexión.


## 20. Generación de la aplicación HTML · El resultado conserva los módulos, filtros de año y territorio, controles específicos por módulo, búsqueda y ordenamiento de tablas, trayectorias individuales, explorador de datos y laboratorio de relaciones. Cada gráfico ofrece PNG, SVG y CSV; las tablas descargan todas las filas de la búsqueda. El Explorador de datos permite además descargar paquetes temáticos de varios CSV o una selección personalizada en JSON. Las vistas pueden guardarse y recargarse como JSON. Plotly y los datos están dentro del HTML: no necesita servidor ni conexión.

La autoría y el contexto IFED–TSE identifican el producto de la Práctica Profesional. Es un entregable interactivo para consulta y análisis y no sustituye las publicaciones oficiales del Tribunal.


### Aplicación y datos derivados
La publicación final deja `Atlas_Electoral_Costa_Rica_1953_2026.html` en la raíz del proyecto. Los pasos intermedios se ensamblan en `salidas/`; el HTML intermedio se retira después de publicar correctamente la aplicación principal. Los datos derivados y el ZIP descargable de fuentes conservan sus carpetas y vínculos relativos.


In [33]:
# Conservación de recursos y de la aplicación pública.
# No reinstalar extensiones previas ni sobrescribir JavaScript en cada ejecución.
assert (PROJECT_DIR/'recursos'/'atlas_final.js').is_file()
assert (PROJECT_DIR/'recursos'/'atlas_navegacion.js').is_file()
assert (PROJECT_DIR/'recursos'/'atlas_trayectorias.js').is_file()
assert (PROJECT_DIR/'recursos'/'atlas_template.html').is_file()
print('Módulos del Atlas verificados; la arquitectura estable se conserva sin parches destructivos.')


Módulos del Atlas verificados; la arquitectura estable se conserva sin parches destructivos.


## 21. Movimientos y títulos auditables
Dos diagramas independientes en Personas y trayectorias: cargos y provincias, con selector de población para la vista provincial. El módulo Éxito electoral permite Todos en cargo anterior y posterior y contextualiza las tasas en título, subtítulo y PNG. Sin exportación de figuras históricas por separado.


In [34]:
assert (PROJECT_DIR/'recursos'/'atlas_final.js').is_file()
assert (PROJECT_DIR/'recursos'/'atlas_trayectorias.js').is_file()
data['meta']['version']='V22.0 · archivo presidencial, hemiciclo y datos accesibles'
print(': módulos verificados; métricas y visualizaciones preservadas; sin PNG externos.')


: módulos verificados; métricas y visualizaciones preservadas; sin PNG externos.


In [35]:
# Ajustes puntuales y reversibles durante la generación del HTML.
# 1. Añade porcentajes VISIBLES en los nodos de TODOS los Sankey.
# 2. Mantiene también la lectura por cursor dentro de cada conexión.
# 3. Añade a las tres poblaciones presidenciales los hitos históricos de género.
# El paquete y los recursos originales quedan intactos. No crea PNG externos.
from tempfile import TemporaryDirectory
from pathlib import Path
import shutil
from atlas_pipeline import render_app

ATLAS_V21_AJUSTES_JS = r'''/* ATLAS V21.2: porcentajes visibles en Sankey + referencias visuales de género.
 * Añadido al HTML durante la ejecución del notebook sin modificar recursos/.
 */
(function atlasV21DosCambiosVisibles(){
  if(typeof plot!=='function') throw new Error('Falta la función de gráficos del Atlas.');
  const plotAnterior=plot;
  const porcentaje=(n,total)=>total>0?(100*n/total).toLocaleString('es-CR',{minimumFractionDigits:1,maximumFractionDigits:1})+' %':'—';
  const limpiarEtiqueta=texto=>String(texto)
    .replace(/\s*[·|]\s*[0-9.,]+\s*%\s*(del total)?\s*$/i,'')
    .replace(/\s*\([^)]*%[^)]*\)\s*$/,'')
    .trim();
  const hitos=[
    {year:1998,label:'Cuota 40 %'},
    {year:2002,label:'Puestos elegibles'},
    {year:2014,label:'Paridad vertical'},
    {year:2018,label:'Paridad horizontal'}
  ];
  plot=function(titulo,trazas,filas,opciones={}){
    let actualizadas=trazas;
    let configuracion={...opciones};
    if(Array.isArray(trazas)&&trazas.some(t=>t?.type==='sankey')){
      actualizadas=trazas.map(t=>{
        if(t?.type!=='sankey') return t;
        const nodos=t.node||{}, enlaces=t.link||{};
        const etiquetas=Array.isArray(nodos.label)?nodos.label:[];
        const origen=Array.isArray(enlaces.source)?enlaces.source:[];
        const destino=Array.isArray(enlaces.target)?enlaces.target:[];
        const pesos=Array.isArray(enlaces.value)?enlaces.value:[];
        if(!etiquetas.length||!pesos.length) return t;
        const salidas=etiquetas.map(()=>0), entradas=etiquetas.map(()=>0);
        for(let i=0;i<pesos.length;i++){
          const s=Number(origen[i]), d=Number(destino[i]), v=Number(pesos[i]);
          if(!Number.isInteger(s)||!Number.isInteger(d)||s<0||d<0||s>=etiquetas.length||d>=etiquetas.length||!Number.isFinite(v)||v<0) continue;
          salidas[s]+=v;
          entradas[d]+=v;
        }
        const total=salidas.reduce((a,b)=>a+b,0);
        if(total<=0) return t;
        const nombres=etiquetas.map((texto,i)=>{
          const base=limpiarEtiqueta(texto);
          if(salidas[i]>0) return `${base} · ${porcentaje(salidas[i],total)}`;
          if(entradas[i]>0) return `${base} · ${porcentaje(entradas[i],total)}`;
          return base;
        });
        const hoverOriginal=enlaces.hovertemplate;
        const hovers=pesos.map((v,i)=>{
          const s=Number(origen[i]), d=Number(destino[i]);
          const base=Array.isArray(hoverOriginal)?hoverOriginal[i]:hoverOriginal;
          const original=(typeof base==='string'&&base)?base:'%{value}<extra></extra>';
          const anexo=`<br>Participación dentro del origen: ${porcentaje(Number(v),salidas[s]||0)}`+
            `<br>Participación dentro del destino: ${porcentaje(Number(v),entradas[d]||0)}`;
          return original.includes('<extra>') ? original.replace('<extra>',anexo+'<extra>') : original+anexo+'<extra></extra>';
        });
        return {
          ...t,
          node:{...nodos,label:nombres},
          link:{...enlaces,hovertemplate:hovers}
        };
      });
      configuracion={
        ...configuracion,
        note:[
          configuracion.note,
          'Los porcentajes visibles junto a cada nodo representan la proporción del peso total de las conexiones mostradas. En el cursor de cada enlace también se informa qué proporción representa dentro de su origen y dentro de su destino. Se conservan los conteos y filtros originales; un enlace no equivale necesariamente a una persona única.'
        ].filter(Boolean).join(' ')
      };
    }
    const poblacion=String(typeof local!=='undefined'?(local.sexpop||''):'');
    const esGeneroTemporal=String(titulo).startsWith('Composición por sexo a través del tiempo');
    if(esGeneroTemporal&&['Presidencia','Primera Vicepresidencia','Segunda Vicepresidencia'].includes(poblacion)){
      const anios=new Set((Array.isArray(actualizadas)?actualizadas:[]).flatMap(t=>Array.isArray(t.x)?t.x.map(Number):[]));
      const visibles=hitos.filter(h=>anios.has(h.year));
      const distribucion=[1.09,1.19,1.09,1.19];
      if(visibles.length){
        const layout=configuracion.layout||{};
        const formaNueva=visibles.map(h=>({type:'line',xref:'x',yref:'paper',x0:h.year,x1:h.year,y0:0,y1:1,line:{color:'#b28744',width:1.7,dash:'dot'}}));
        const notaNueva=visibles.map((h,i)=>({xref:'x',yref:'paper',x:h.year,y:distribucion[i],text:`${h.year} · ${h.label}`,showarrow:false,xanchor:'center',font:{size:10,color:'#755c32'},bgcolor:'rgba(255,250,241,.98)',bordercolor:'rgba(178,135,68,.4)',borderpad:3}));
        configuracion={
          ...configuracion,
          height:Math.max(560,configuracion.height||0),
          layout:{...layout,shapes:[...(layout.shapes||[]),...formaNueva],annotations:[...(layout.annotations||[]),...notaNueva],margin:{...(layout.margin||{}),t:175}},
          note:[configuracion.note,'Hitos de referencia de la normativa de género en elecciones costarricenses (1998, 2002, 2014, 2018). Las etiquetas se muestran como contexto histórico comparativo: las reglas legislativas de cuotas, puestos elegibles y paridad no deben suponerse automáticamente aplicables a las candidaturas presidenciales o vicepresidenciales. La coincidencia temporal no establece causalidad.'].filter(Boolean).join(' ')
        };
      }
    }
    return plotAnterior(titulo,actualizadas,filas,configuracion);
  };
})();
'''
# Módulo autónomo añadido durante render, sin tocar recursos/.
ATLAS_V21_AJUSTES_JS += r'''
/* ATLAS_V21_4_DOBLE_POSTULACION_CLARIDAD
 * Extensión autónoma que no altera bases, otros módulos ni recursos permanentes.
 * Doble postulación: misma identidad nominal + mismo año de elección,
 * Un cargo de la fórmula presidencial (Presidencia, primera o segunda
 * vicepresidencia) + Diputación propietaria en la misma elección.
 * Nacional no se trata como provincia; cada persona-año se cuenta una vez.
 */
(function atlasV213Dobles(){
 'use strict';
 if(typeof views==='undefined'||typeof plot!=='function'||typeof D==='undefined'||!Array.isArray(D.eventos)) return;
 const nation='Costa Rica';
 const executiveRoles=['Presidencia','Primera Vicepresidencia','Segunda Vicepresidencia'];
 const typeOrder=['Presidencia','Primera Vicepresidencia','Segunda Vicepresidencia'];
 const isElected=v=>v===true||v===1||String(v).toLowerCase()==='true';
 const fmt=(n,d=0)=>Number(n).toLocaleString('es-CR',{maximumFractionDigits:d,minimumFractionDigits:d});
 let cached=null;
 function observations(){
  if(cached)return cached;
  const events=D.eventos.filter(r=>r&&r.id_persona&&Number.isInteger(Number(r.anio)));
  const by=new Map();
  for(const ev of events){const key=String(ev.id_persona)+'|'+String(ev.anio);if(!by.has(key))by.set(key,[]);by.get(key).push(ev)}
  const out=[];
  for(const [key,list] of by){
   const national=list.filter(r=>executiveRoles.includes(r.cargo));
   const legislative=list.filter(r=>r.cargo==='Diputación'&&r.tipo==='Propietaria'&&P.includes(r.provincia));
   if(!national.length||!legislative.length)continue;
   const provinces=uniq(legislative.map(r=>r.provincia));
   const parties=uniq([...national,...legislative].map(r=>r.partido));
   const nationalRoles=typeOrder.filter(role=>national.some(r=>r.cargo===role));
   // Se registra cada persona-elección una sola vez y no se infiere provincia del cargo nacional.
   // Concurrencias de varios cargos nacionales quedan marcadas para auditoría.
   const firstN=national[0],firstL=legislative[0];
   out.push({id_persona:firstN.id_persona,anio:Number(firstN.anio),nombre:firstN.nombre,
    identidad:firstN.identidad,sexo:firstN.sexo,provincia:provinces.length===1?provinces[0]:'Ambigua',
    partido:parties.length===1?parties[0]:'Partidos distintos',
    partido_nacional:firstN.partido,partido_presidencial:firstN.partido,partido_legislativo:firstL.partido,
    cargo_nacional:nationalRoles.length===1?nationalRoles[0]:'Varios cargos nacionales',
    cargos_nacionales:nationalRoles.join(' | '),
    tipo_doble:nationalRoles.length===1?`${nationalRoles[0]} + Diputación`:'Múltiples cargos nacionales + Diputación',
    mismo_partido:national.some(p=>legislative.some(l=>p.partido===l.partido)),
    electa_cargo_nacional:national.some(r=>isElected(r.electa)),
    electa_presidencia:national.some(r=>r.cargo==='Presidencia'&&isElected(r.electa)),
    electa_primera_vicepresidencia:national.some(r=>r.cargo==='Primera Vicepresidencia'&&isElected(r.electa)),
    electa_segunda_vicepresidencia:national.some(r=>r.cargo==='Segunda Vicepresidencia'&&isElected(r.electa)),
    electa_diputacion:legislative.some(r=>isElected(r.electa)),
    numero_candidaturas_nacionales:national.length,numero_candidaturas_presidenciales:national.length,
    numero_candidaturas_legislativas:legislative.length,
    id_nacional:firstN.id_evento,id_presidencial:firstN.id_evento,id_legislativo:firstL.id_evento,
    orden_lista:firstL.orden,ambiguo:provinces.length!==1||nationalRoles.length!==1||legislative.length!==1});
  }
  out.sort((a,b)=>a.anio-b.anio||a.nombre.localeCompare(b.nombre,'es'));
  const prior=new Map();
  for(const x of out){const earlier=prior.get(x.id_persona);
   x.repeticion_doble=Boolean(earlier);
   x.anio_doble_anterior=earlier?earlier.anio:null;
   x.continuidad_partido=earlier?(earlier.partido===x.partido?'Mantiene partido':'Cambia partido'):'Primera doble postulación';
   x.continuidad_provincia=earlier?(earlier.provincia===x.provincia?'Mantiene provincia':'Cambia provincia'):'Primera doble postulación';
   prior.set(x.id_persona,x);
  }
  cached=out;
  return out;
 }
 function draw(){
  add('<div class="extension-heading"><span>DOBLE POSTULACIÓN · FÓRMULA PRESIDENCIAL Y DIPUTACIÓN</span></div>');
  add('<div class="clarity-box"><b>Doble postulación:</b> una misma persona figura en la misma elección como candidata a <b>diputación propietaria</b> y además a <b>Presidencia, primera vicepresidencia o segunda vicepresidencia</b>. Se identifica cada combinación por separado. No basta participar en dos cargos nacionales sin diputación. <b>Reiteración:</b> una persona vuelve a presentar una doble postulación en otra elección posterior; no significa reelección. Cada persona–elección cuenta una sola vez; la provincia corresponde EXCLUSIVAMENTE a su candidatura legislativa. La clasificación se basa en registros nominales y debe contrastarse documentalmente, especialmente ante coincidencias de nombres.</div>');
  const all=observations();window.atlas231AllDoubles=all;
  const provinces=['Todas las provincias',...P];
  const electionResult=[['Todos','Todos los resultados'],['Diputación electa','Diputación electa'],['Ninguna electa','Ninguna candidatura electa'],['Cargo nacional electo','Cargo nacional electo'],['Ambos cargos electos','Ambos cargos electos']];
  const doubleTypes=[['Todos','Todos los tipos'],...typeOrder.map(role=>[role+' + Diputación',role+' + diputación'])];
  const repeatOptions=[['Todos','Todas las dobles postulaciones'],['Primera','Primera doble postulación de la persona'],['Repetida','Persona que ya había tenido doble postulación en otra elección']];
  const continuity=['Todas','Primera doble postulación','Mantiene partido','Cambia partido'];
  const provincesCont=['Todas','Primera doble postulación','Mantiene provincia','Cambia provincia'];
  controls([
   ['dobProv','Provincia de la candidatura a diputación',provinces,local.dobProv||provinces[0]],
   ['dobType','Tipo de doble postulación',doubleTypes,local.dobType||'Todos'],
   ['dobOutcome','Resultado combinado',electionResult,local.dobOutcome||'Todos'],
   ['dobRepeat','¿Volvió a postularse simultáneamente a los dos cargos?',repeatOptions,local.dobRepeat||'Todos'],
   ['dobPartyChange','Partido entre dobles postulaciones',continuity,local.dobPartyChange||'Todas'],
   ['dobProvChange','Provincia entre dobles postulaciones',provincesCont,local.dobProvChange||'Todas']
  ]);
  const date=all.filter(r=>r.anio>=state.from&&r.anio<=state.to);
  const filtered=date.filter(r=>
   (local.dobProv===undefined||local.dobProv==='Todas las provincias'||r.provincia===local.dobProv)&&
   (!local.dobType||local.dobType==='Todos'||r.tipo_doble===local.dobType)&&
   (!local.dobOutcome||local.dobOutcome==='Todos'||
    (local.dobOutcome==='Diputación electa'&&r.electa_diputacion)||
    (local.dobOutcome==='Cargo nacional electo'&&r.electa_cargo_nacional)||
    (local.dobOutcome==='Ninguna electa'&&!r.electa_diputacion&&!r.electa_cargo_nacional)||
    (local.dobOutcome==='Ambos cargos electos'&&r.electa_diputacion&&r.electa_cargo_nacional))&&
   (!local.dobRepeat||local.dobRepeat==='Todos'||(local.dobRepeat==='Repetida'?r.repeticion_doble:!r.repeticion_doble))&&
   (!local.dobPartyChange||local.dobPartyChange==='Todas'||r.continuidad_partido===local.dobPartyChange)&&
   (!local.dobProvChange||local.dobProvChange==='Todas'||r.continuidad_provincia===local.dobProvChange));
  window.atlas225ActiveDoubles=filtered;
  const people=uniq(filtered.map(r=>r.id_persona)).length;
  const yearFirst=all.length?Math.min(...all.map(r=>r.anio)):null;
  kpis([
   ['Casos persona–elección',fmt(filtered.length),'Cada persona en una elección cuenta una vez'],
   ['Personas nominales distintas',fmt(people),'No equivale a candidatos en años distintos'],
   ['Diputaciones electas',fmt(filtered.filter(r=>r.electa_diputacion).length),'Resultados simultáneos de las dobles postulaciones'],
   ['Cargos nacionales electos',fmt(filtered.filter(r=>r.electa_cargo_nacional).length),'Presidencia o vicepresidencia según el tipo de doble postulación']
  ]);
  const scope=[local.dobProv&&local.dobProv!=='Todas las provincias'?local.dobProv:'Costa Rica',
   local.dobType&&local.dobType!=='Todos'?local.dobType:null,
   local.dobOutcome&&local.dobOutcome!=='Todos'?local.dobOutcome:'Todos los resultados',
   local.dobRepeat&&local.dobRepeat!=='Todos'?local.dobRepeat:null,
   local.dobPartyChange&&local.dobPartyChange!=='Todas'?local.dobPartyChange:null,
   local.dobProvChange&&local.dobProvChange!=='Todas'?local.dobProvChange:null].filter(Boolean).join(' · ');
  add(`<div class="clarity-box"><b>Primera observación en el corpus completo:</b> ${yearFirst??'sin casos'} · <b>Lectura activa:</b> ${esc(scope)} · ${state.from}–${state.to}. Las cifras se basan en las candidaturas registradas en las bases de esta versión, no en un inventario jurídico exhaustivo de toda la historia electoral.</div>`);
  const ys=yearsVisible();
  const longitudinal=ys.map(anio=>{
   const gg=filtered.filter(r=>r.anio===anio);
   const nFormula=uniq(D.eventos.filter(r=>r.anio===anio&&executiveRoles.includes(r.cargo)).map(r=>r.id_persona)).length;
   return {anio,postulaciones_dobles:gg.length,personas:gg.length,porcentaje_integrantes_formula:nFormula?100*gg.length/nFormula:null,integrantes_formula:nFormula,electas_diputacion:gg.filter(r=>r.electa_diputacion).length,electas_cargo_nacional:gg.filter(r=>r.electa_cargo_nacional).length};
  });
  plot('Doble postulación · fórmula presidencial y diputación · evolución histórica',
   [{type:'scatter',mode:'lines+markers',name:'Postulaciones dobles',x:ys,y:longitudinal.map(r=>r.postulaciones_dobles),line:{color:'#087f87',width:3},marker:{size:9},customdata:longitudinal.map(r=>[r.integrantes_formula,r.porcentaje_integrantes_formula]),hovertemplate:'%{x}<br>%{y} dobles postulaciones<br>Personas en fórmula nacional: %{customdata[0]}<br>Proporción: %{customdata[1]:.1f}%<extra></extra>'},
    {type:'scatter',mode:'lines+markers',name:'Reiteraciones (ambos cargos, en otra elección posterior)',x:ys,y:ys.map(y=>filtered.filter(r=>r.anio===y&&r.repeticion_doble).length),line:{color:'#bc8b5b',width:2,dash:'dash'},marker:{size:7},hovertemplate:'%{x}<br>%{y} episodios nuevos de personas que ya habían tenido una doble postulación<extra></extra>'}],longitudinal,
   {height:430,subtitle:`${scope} · ${state.from}–${state.to}`,yTitle:'Casos persona–elección',note:'Primera línea: personas que se presentaron simultáneamente a diputación propietaria y a uno de los tres cargos nacionales de la fórmula. Segunda línea: entre ellas, cuántas ya habían realizado una doble postulación en una elección anterior (reiteraciones, no personas nuevas). La leyenda y el filtro distinguen Presidencia, primera y segunda vicepresidencia. Un cero indica ausencia de casos registrados, no prohibición legal.'});
  const tipoCounts=typeOrder.map(cargo=>({tipo:cargo+' + Diputación',casos:filtered.filter(r=>r.cargo_nacional===cargo).length}));
  plot('Composición de la doble postulación por cargo nacional',
   [{type:'bar',orientation:'h',x:tipoCounts.map(r=>r.casos),y:tipoCounts.map(r=>r.tipo),
     marker:{color:['#24738f','#99619c','#c38c55']},
     text:tipoCounts.map(r=>`${r.casos} · ${filtered.length?fmt(100*r.casos/filtered.length,1):'0,0'} %`),textposition:'outside',cliponaxis:false}],
    tipoCounts,{height:320,subtitle:scope,xTitle:'Casos persona–elección',layout:{margin:{l:235,r:110,t:35,b:65}},
    note:'El desglose identifica el cargo nacional concurrente con diputación. Cada persona–elección se asigna una vez; se auditan por separado eventuales concurrencias de varios cargos nacionales.'});
  const geos=P.map(p=>({provincia:p,registros:filtered.filter(r=>r.provincia===p).length,personas:uniq(filtered.filter(r=>r.provincia===p).map(r=>r.id_persona)).length}));
  const heatRows=geos.flatMap(g=>ys.map(anio=>({provincia:g.provincia,anio,casos:filtered.filter(r=>r.anio===anio&&r.provincia===g.provincia).length})));
  heatRows.push(...ys.map(anio=>({provincia:nation,anio,casos:filtered.filter(r=>r.anio===anio).length})));
  const totals=geos.map(g=>({provincia:g.provincia,total:g.registros}));
  const xYears=ys.map(String),yProvinces=[...P];
  const matrixRows=geos.map(g=>ys.map(anio=>filtered.filter(r=>r.anio===anio&&r.provincia===g.provincia).length));
  const totalByProvince=geos.map(g=>g.registros),totalByYear=ys.map(anio=>filtered.filter(r=>r.anio===anio).length);
  const matrixAudit=[...heatRows,
    ...P.map((provincia,i)=>({provincia,anio:'Total histórico',casos:totalByProvince[i],tipo:'Total por provincia'})),
    ...ys.map((anio,i)=>({provincia:nation,anio,casos:totalByYear[i],tipo:'Total país por elección'})),
    {provincia:nation,anio:'Total histórico',casos:filtered.length,tipo:'Total país y periodo'}];
  if(filtered.length){
   // V21.7: UNA SOLA escala cromática para cuerpo, columna total y fila Costa Rica.
   // 0 es un valor observado; se conserva y se visualiza como 0.
   // ÚNICA excepción: esquina país/periodo (gran total, sin escala).
   const xAll=[...xYears,'Total histórico'];
   const yAll=[...yProvinces,nation];
   const escalaMax=Math.max(1,...matrixRows.flat(),...totalByProvince,...totalByYear);
   // Una sola paleta teal: un mismo valor siempre tiene exactamente el mismo
   // color en toda la matriz. Los puntos de control evitan perder contraste
   // para valores pequeños, sin cambiar los conteos ni los ceros.
   const puntosColor=[
     [0,'#f0f6f7'],[1,'#e4f0f1'],[2,'#cbe4e5'],[4,'#9dccd0'],
     [6,'#70b5bc'],[8,'#449ca6'],[10,'#27818c'],[12,'#186b79'],
     [20,'#125164'],[33,'#0b354d']
   ];
   const colorscale=[];
   for(const [valor,color] of puntosColor){
     if(valor<=escalaMax) colorscale.push([valor/escalaMax,color]);
   }
   if(colorscale[colorscale.length-1][0]<1){
     colorscale.push([1,puntosColor.find(p=>p[0]>escalaMax)?.[1]||'#0b354d']);
   }
   const grid=[
     ...matrixRows.map((fila,i)=>[...fila,totalByProvince[i]]),
     [...totalByYear,null]
   ];
   const texto=grid.map((fila,i)=>fila.map((n,j)=>
     n===null?'':(n>=8?'':String(n))));
   const base={type:'heatmap',x:xAll,y:yAll,z:grid,text:texto,
      texttemplate:'%{text}',textfont:{size:11,color:'#173849'},
      zmin:0,zmax:escalaMax,colorscale,
      xgap:1.5,ygap:1.5,
      colorbar:{title:{text:'Casos'},len:.74,thickness:15,
        tickmode:'array',tickvals:[0,1,2,4,8,12,20,33].filter(v=>v<=escalaMax)},
      hovertemplate:'%{y} · %{x}<br>%{z} casos persona–elección<extra></extra>'};
   const formas=[],etiquetas=[];
   // V21.8: en lugar de anotaciones blancas (que desaparecían al exportar),
   // dibujamos los valores altos como una traza de texto encima del heatmap.
   // El calor y el denominador NO cambian; se conserva la etiqueta "0".
   const textoIntenso={x:[],y:[],text:[]};
   for(let i=0;i<grid.length;i++)for(let j=0;j<grid[i].length;j++){
     const n=grid[i][j];
     if(n!==null && n>=8){
       textoIntenso.x.push(xAll[j]);
       textoIntenso.y.push(yAll[i]);
       textoIntenso.text.push(String(n));
     }
   }
   const capaTextoIntenso={
      type:'scatter',mode:'text',x:textoIntenso.x,y:textoIntenso.y,
      text:textoIntenso.text,textposition:'middle center',
      textfont:{size:11,color:'#ffffff',family:'Arial, sans-serif'},
      hoverinfo:'skip',showlegend:false,cliponaxis:false
   };
   // Solo esta esquina se mantiene completamente BLANCA y aislada.
   formas.push({type:'rect',xref:'x',yref:'y',x0:ys.length-.5,x1:ys.length+.5,
    y0:P.length-.5,y1:P.length+.5,fillcolor:'#ffffff',
    line:{color:'#15558c',width:2.6},layer:'above'});
   etiquetas.push({xref:'x',yref:'y',x:'Total histórico',y:nation,
    text:`<b>${filtered.length}</b>`,showarrow:false,
    font:{size:17,color:'#13578e'},xanchor:'center',yanchor:'middle'});
   // Separadores estructurales: mismos ejes y tamaños de celda.
   formas.push({type:'line',xref:'x',yref:'y',x0:ys.length-.5,x1:ys.length-.5,
    y0:-.5,y1:P.length+.5,line:{color:'#335d77',width:2.1},layer:'above'});
   formas.push({type:'line',xref:'x',yref:'y',x0:-.5,x1:ys.length+.5,
    y0:P.length-.5,y1:P.length-.5,line:{color:'#335d77',width:2.1},layer:'above'});
   plot('Doble postulación por provincia y elección · total Costa Rica',
    [base,capaTextoIntenso],matrixAudit,
    {height:615,subtitle:`${scope} · ${state.from}–${state.to}`,
     xTitle:'Elección',yTitle:'Provincia de la candidatura legislativa',
     layout:{xaxis:{type:'category',tickangle:-45,
       categoryorder:'array',categoryarray:xAll,
       range:[-.55,xAll.length-.45]},
      yaxis:{type:'category',autorange:'reversed',
       categoryorder:'array',categoryarray:yAll,
       range:[yAll.length-.48,-.52]},
      margin:{l:125,r:95,b:115,t:35},
      shapes:formas,annotations:etiquetas},
     note:'El mismo valor de casos utiliza el mismo color en toda la matriz: provincias por elección, totales históricos provinciales y totales nacionales anuales comparten una única escala. Los ceros son valores válidos y se muestran como 0. La casilla blanca de la esquina inferior derecha muestra exclusivamente el total país–periodo, sin degradado. Cada celda anual cuenta casos persona–elección con diputación propietaria y al menos un cargo de la fórmula nacional (Presidencia, primera o segunda vicepresidencia). La provincia es la de la candidatura legislativa.'});
   const ordered=geos.slice().sort((a,b)=>b.registros-a.registros);
   plot('Concentración territorial de las dobles postulaciones',
    [{type:'bar',orientation:'h',x:ordered.map(r=>r.registros),y:ordered.map(r=>r.provincia),text:ordered.map(r=>`${r.registros} · ${fmt(100*r.registros/filtered.length,1)} %`),textposition:'outside',cliponaxis:false,marker:{color:ordered.map(r=>provinceColors[r.provincia]||'#27818b')},hovertemplate:'%{y}<br>%{x} casos persona–elección<extra></extra>'}],ordered,
    {height:410,subtitle:`${scope}`,xTitle:'Dobles postulaciones documentadas',note:'Provincia de la candidatura a diputación; no equivale al lugar de origen o residencia de las personas.'});
   // Tasas por provincia: numerador = candidaturas a diputación electas;
   // denominador = casos persona–elección con doble postulación en la provincia.
   const provinceSuccess=geos.map(g=>{
     const observationsHere=filtered.filter(r=>r.provincia===g.provincia);
     const n=observationsHere.length,won=observationsHere.filter(r=>r.electa_diputacion).length;
     return {provincia:g.provincia,casos:n,electas:won,no_electas:n-won,tasa_eleccion_pct:n?100*won/n:null};
   }).filter(g=>g.casos>0).sort((a,b)=>a.tasa_eleccion_pct-b.tasa_eleccion_pct);
   if(provinceSuccess.length)plot('Éxito legislativo entre las dobles postulaciones · por provincia',
    [{type:'bar',orientation:'h',x:provinceSuccess.map(r=>r.tasa_eleccion_pct),
      y:provinceSuccess.map(r=>r.provincia),text:provinceSuccess.map(r=>`${fmt(r.tasa_eleccion_pct,1)} % · ${r.electas}/${r.casos}`),
      textposition:'outside',cliponaxis:false,marker:{color:provinceSuccess.map(r=>provinceColors[r.provincia]||'#167f83')},
      customdata:provinceSuccess.map(r=>[r.electas,r.casos]),
      hovertemplate:'%{y}<br>%{customdata[0]} diputaciones electas de %{customdata[1]} dobles postulaciones · %{x:.1f}%<extra></extra>'}],provinceSuccess,
    {height:420,subtitle:`${scope}`,xTitle:'Porcentaje que resultó electo a diputación (%)',layout:{margin:{l:130,r:120,t:40,b:75},xaxis:{range:[0,110]}},
      note:'Descriptivo y sensible a tamaños muestrales reducidos. No implica que la doble postulación produzca la elección.'});
   const o=[
    {resultado:'Ninguna candidatura electa',casos:filtered.filter(r=>!r.electa_diputacion&&!r.electa_cargo_nacional).length},
    {resultado:'Solo diputación electa',casos:filtered.filter(r=>r.electa_diputacion&&!r.electa_cargo_nacional).length},
    {resultado:'Solo cargo nacional electo',casos:filtered.filter(r=>r.electa_cargo_nacional&&!r.electa_diputacion).length},
    {resultado:'Ambos cargos electos',casos:filtered.filter(r=>r.electa_diputacion&&r.electa_cargo_nacional).length}
   ];
   plot('Resultados de las postulaciones dobles · ambos cargos',
    [{type:'bar',orientation:'h',x:o.map(r=>r.casos),y:o.map(r=>r.resultado),text:o.map(r=>`${r.casos} · ${fmt(100*r.casos/filtered.length,1)} %`),textposition:'outside',cliponaxis:false,marker:{color:['#95a6b1','#14857f','#c58351','#684c93']}}],o,
    {height:365,subtitle:`${scope}`,xTitle:'Casos persona–elección',layout:{margin:{l:205,r:80,t:40,b:75}},note:'Resultados simultáneos del cargo nacional específico (Presidencia o vicepresidencia) y diputación. Las proporciones tienen como denominador los casos del filtro activo; “electa” procede de las bases y requiere contraste documental.'});
  }else note('No se observaron dobles postulaciones que cumplan esta combinación de filtros. Revise rango temporal, provincia o resultado.',true);
  const nRepeat=uniq(filtered.filter(r=>r.repeticion_doble).map(r=>r.id_persona)).length;
  const episodes=filtered.filter(r=>r.repeticion_doble).length;
  add(`<div class="clarity-box"><b>Qué significa reiteración:</b> una persona presenta diputación y un cargo de la fórmula presidencial en una elección y vuelve a presentarse <b>simultáneamente a diputación y a un cargo de la fórmula</b> en otra elección posterior. El cargo nacional podría cambiar de una elección a la siguiente. En la selección activa hay <b>${fmt(episodes)} episodios de reiteración</b> correspondientes a <b>${fmt(nRepeat)} personas nominales</b>. Ejemplo: si alguien realiza dobles postulaciones en tres elecciones, aporta <b>tres casos al total</b>, de los cuales <b>dos son reiteraciones</b>. Los cambios de partido o provincia se calculan entre dos episodios de doble postulación, no entre cualquier candidatura de la trayectoria.</div>`);
  atlas225DoublePanel(filtered);
  table(filtered.map(r=>({...r,resultado_cargo_nacional:r.electa_cargo_nacional?'Electa':'No electa',resultado_legislativo:r.electa_diputacion?'Electa':'No electa',reiteracion_de_doble_postulacion:r.repeticion_doble?'Sí · posterior':'No · primera'})),
   ['anio','nombre','sexo','tipo_doble','cargo_nacional','cargos_nacionales','partido_nacional','partido_legislativo','provincia','orden_lista','resultado_cargo_nacional','resultado_legislativo','reiteracion_de_doble_postulacion','anio_doble_anterior','continuidad_partido','continuidad_provincia','ambiguo'],
   'Auditoría nominal · dobles postulaciones simultáneas');
  add('<div class="clarity-box"><b>Sobre el posible beneficio electoral:</b> estos datos permiten calcular tasas descriptivas de éxito simultáneo; no permiten atribuir resultados a la doble postulación sin considerar posición de lista, magnitud provincial, voto del partido, selección de candidaturas y contexto político. Se distingue siempre una persona única de una persona–elección.</div>');
 }
 // V21.6 · Movimientos provinciales estrictamente legislativos.
 // No se usa población presidencial ni selector de población. Se valida que
 // AMBOS extremos del movimiento tengan una candidatura a diputación
 // documentada (misma identidad nominal, año y provincia).
 function movimientosLegislativos(){
   const raw=period(D.movimientos,false),scope=state.province;
   const incluir=(r,dim)=>scope==='Nacional'||(dim==='Provincia'?
     (r.origen===scope||r.destino===scope):true);
   // Se conserva el Sankey de cambios entre cargos, independiente del provincial.
   add('<div class="atlas17-subhead">Movimientos entre participaciones · cargos</div>');
   const cargos=raw.filter(r=>r.dimension==='Cargo'&&incluir(r,'Cargo'));
   pairedFlow(cargos,'Cargo',`Cambios de un cargo al siguiente · ${state.from}–${state.to}`);
   table(cargos,['nombre','anio_origen','origen','anio','destino','id_persona'],'Movimientos entre cargos · registros de origen y destino');
   add('<div class="atlas17-subhead">Movimientos entre participaciones · provincias</div>');
   const identificados=new Set((D.eventos||[])
     .filter(e=>e.cargo==='Diputación'&&P.includes(e.provincia))
     .map(e=>`${String(e.id_persona)}|${Number(e.anio)}|${e.provincia}`));
   const todos=raw.filter(r=>r.dimension==='Provincia'&&incluir(r,'Provincia'));
   const provincias=todos.filter(r=>
     identificados.has(`${String(r.id_persona)}|${Number(r.anio_origen)}|${r.origen}`)&&
     identificados.has(`${String(r.id_persona)}|${Number(r.anio)}|${r.destino}`));
   const nPeople=uniq(provincias.map(r=>r.id_persona)).length;
   add(`<div class="clarity-box"><b>Solo candidaturas legislativas.</b> `+
     `${num(provincias.length,0)} movimientos entre provincias, correspondientes a `+
     `${num(nPeople,0)} personas nominales en el intervalo seleccionado. `+
     `Cada transición exige una candidatura a diputación documentada en origen y destino. `+
     `${num(todos.length-provincias.length,0)} movimientos del registro general `+
     `no cumplen esa doble validación y quedan excluidos. `+
     `No representa migración residencial, doble postulación ni candidaturas presidenciales.</div>`);
   pairedFlow(provincias,'Provincia',`Diputaciones · siete provincias · ${state.from}–${state.to}`);
   if(provincias.length){
     const cells=[];for(const origen of P)for(const destino of P)
       cells.push({origen,destino,cambios:provincias.filter(r=>r.origen===origen&&r.destino===destino).length});
     plot('Matriz de movimientos provinciales · diputaciones',
       [{type:'heatmap',x:P,y:P,
         z:P.map(o=>P.map(d=>cells.find(r=>r.origen===o&&r.destino===d).cambios)),
         texttemplate:'%{z}',colorscale:[[0,'#edf4f6'],[1,'#167f83']],
         colorbar:{title:{text:'Cambios'}},
         hovertemplate:'Origen: %{y}<br>Destino: %{x}<br>%{z} movimientos<extra></extra>'}],
       cells,{height:450,subtitle:`Diputaciones · ${state.from}–${state.to}`,
         xTitle:'Provincia de destino',yTitle:'Provincia de origen',
         note:'Solamente movimientos entre candidaturas a diputación verificadas en ambas elecciones. La diagonal es cero: se cuentan cambios de provincia. No es una medición de migración residencial.'});
   }
   table(provincias,['nombre','sexo','anio_origen','origen','anio','destino','id_persona'],
     'Personas y movimientos provinciales · diputaciones');
   // Mantener la lectura partidaria opcional preexistente: no crear otras vistas provinciales.
   const optParty=local.movParty||'Ocultar';
   controls([['movParty','Lectura adicional de movimientos entre partidos',['Ocultar','Mostrar'],optParty]]);
   if(optParty==='Mostrar'){
     const parties=raw.filter(r=>r.dimension==='Partido'&&incluir(r,'Partido'));
     add('<div class="atlas17-subhead">Movimientos entre participaciones · partidos</div>');
     pairedFlow(parties,'Partido',`Cambios de denominación partidaria · ${state.from}–${state.to}`);
     table(parties,['nombre','anio_origen','origen','anio','destino','id_persona'],'Registros de movimientos partidarios');
   }
 }
 enhancedMobility=movimientosLegislativos;
 const beforeMobility=enhancedMobility;
 enhancedMobility=function(...args){const x=beforeMobility.apply(this,args);draw();return x;};
 window.ATLAS_V21_4_DOBLES_OK=true;
})();


'''

HTML_PATH = OUTPUT_DIR / 'Atlas_Electoral_Costa_Rica_1953_2026.html'
RECURSOS = PROJECT_DIR / 'recursos'
with TemporaryDirectory(prefix='atlas_v21_render_') as carpeta_temporal:
    temporal = Path(carpeta_temporal)
    for nombre in ('atlas_template.html','atlas_exploracion.js','atlas_final.js','atlas_navegacion.js','atlas_trayectorias.js'):
        shutil.copyfile(RECURSOS / nombre, temporal / nombre)
    # La extensión se coloca al FINAL, después de las capas /.
    # Así no sobrescribe los recursos permanentes ni duplica visualizaciones.
    ultima = temporal / 'atlas_trayectorias.js'
    ultima.write_text(ultima.read_text(encoding='utf-8')+'\n'+ATLAS_V21_AJUSTES_JS,encoding='utf-8')
    render_app(data, temporal / 'atlas_template.html', HTML_PATH)
(OUTPUT_DIR/'Atlas_Electoral_Figuras_Historicas.html').unlink(missing_ok=True)
print('Aplicación pública actualizada:', HTML_PATH.resolve())
print('Sankey: porcentajes visibles en ambos lados del nodo y lectura adicional por cursor.')
print('Género presidencial: hitos comparativos visibles en Presidencia y Vicepresidencias.')
print('Únicas salidas: HTML público y datos. No se crean figuras externas.')


Aplicación pública actualizada: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Sankey: porcentajes visibles en ambos lados del nodo y lectura adicional por cursor.
Género presidencial: hitos comparativos visibles en Presidencia y Vicepresidencias.
Únicas salidas: HTML público y datos. No se crean figuras externas.


### 21.9 · Ajustes de presentación no destructivos

Se mejoran **exclusivamente** tipografía de controles, ayudas de lectura, estados de foco y distribución de encabezados en pantallas pequeñas. La gráfica Plotly, sus escalas, filtros, trazas, colores, cálculos, descarga, título y tamaños **no se modifican**. La hoja CSS es autosuficiente y se inserta una sola vez al final de la generación del HTML. Los recursos originales permanecen intactos.

In [36]:
# Atlas · Refinamiento conservador de interfaz. CSS estático.
# Este bloque no modifica los recursos externos ni el contenido de las figuras.
from pathlib import Path
from tempfile import NamedTemporaryFile
import os

HTML_PUBLICO = OUTPUT_DIR / "Atlas_Electoral_Costa_Rica_1953_2026.html"
assert HTML_PUBLICO.is_file(), "Ejecutá primero la celda que genera el HTML público."
MARCA = "<!-- ATLAS_V21_9_INTERFAZ -->"
CSS_PUBLICO = r"""
<style id="atlas-v21-9-interfaz">
/* Ajustes de accesibilidad y jerarquía para elementos externos a Plotly. */
.navbutton:focus-visible, .button:focus-visible, .exports button:focus-visible,
.intro-button:focus-visible, .intro-link:focus-visible,
select:focus-visible, input:focus-visible, summary:focus-visible,
a:focus-visible {outline:3px solid #157e87;outline-offset:3px;box-shadow:0 0 0 3px #ffffff;}
.navbutton {transition:background-color .15s ease,color .15s ease;}
.section-head h2 {line-height:1.22;letter-spacing:-.55px;}
.section-head p {line-height:1.65;}
.scope {font-weight:650;letter-spacing:.12px;}
.controls {border:1px solid #dce7ea;row-gap:14px;}
.controls label {line-height:1.4;}
.controls select {min-width:125px;}
.card-head > div:first-child {min-width:0;}
.card-head h3 {overflow-wrap:break-word;line-height:1.38;}
.card-head .mini {line-height:1.55;display:block;}
.exports button:hover {border-color:#9dbdc7;background:#f0f6f7;color:#204858;}
.chart-note {line-height:1.65;border-top:1px solid #edf1f3;padding-top:10px;margin-top:12px;}
.reading-help > summary {line-height:1.5;}
.notice {line-height:1.62;}
.tablefooter {gap:12px;flex-wrap:wrap;}
@media(max-width:750px){
  .card-head {flex-wrap:wrap;row-gap:10px;}
  .card-head .exports {margin-left:auto;}
  .section-head {flex-wrap:wrap;}
  .controls {gap:12px;}
  .controls label {flex:1 1 170px;min-width:0;}
}
@media(prefers-reduced-motion:reduce){.navbutton {transition:none;}}
</style>
"""

# Idempotencia: no repetir la hoja ni crear archivos secundarios.
contenido = HTML_PUBLICO.read_text(encoding="utf-8")
if MARCA not in contenido:
    cierre = contenido.find("</head>")
    if cierre < 0:
        raise ValueError("HTML público sin </head>: no se aplica una transformación insegura.")
    contenido = contenido[:cierre] + MARCA + "\n" + CSS_PUBLICO + "\n" + contenido[cierre:]
    HTML_PUBLICO.write_text(contenido,encoding="utf-8")
    estado = "Insertada una vez"
else:
    estado = "Ya presente; sin duplicación"
assert HTML_PUBLICO.read_text(encoding="utf-8").count(MARCA)==1
print('Interfaz :',estado)
print("HTML público:",HTML_PUBLICO.resolve())
print("Sin nuevas carpetas, JS permanente ni figuras externas.")


Interfaz : Insertada una vez
HTML público: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Sin nuevas carpetas, JS permanente ni figuras externas.


## 22.0 · Archivo presidencial, hemiciclo y recursos de consulta\nEsta actualización amplía módulos de consulta pública sin rediseñar las visualizaciones Plotly aprobadas. Los recursos de la interfaz son `recursos/atlas_presentacion.js` y `recursos/atlas_presentacion.css`; el último bloque integra ambos en el HTML autónomo.\n

In [37]:
# Mejoras públicas no destructivas, con recursos fuente auditables.
# Se ejecuta DESPUÉS de la generación estable . No modifica el JavaScript
# de gráficas aprobadas: conserva sus traces, cálculo, exportación, nombres y filtros.
from pathlib import Path

html_path = OUTPUT_DIR / 'Atlas_Electoral_Costa_Rica_1953_2026.html'
resource_dir = PROJECT_DIR / 'recursos'
assert html_path.is_file() and (resource_dir/'atlas_presentacion.js').is_file() and (resource_dir/'atlas_presentacion.css').is_file()
html = html_path.read_text(encoding='utf-8')
js = (resource_dir/'atlas_presentacion.js').read_text(encoding='utf-8')
css = (resource_dir/'atlas_presentacion.css').read_text(encoding='utf-8')
marker_js = '/* ATLAS_V22_PUBLIC_PRODUCT'
marker_css = '/* ATLAS_V22_PUBLIC_PRESENTATION'
js_hook = "window.addEventListener('resize',()=>document.querySelectorAll('.js-plotly-plot').forEach(el=>Plotly.Plots.resize(el)));setup();"
assert js_hook in html, 'No se encontró el punto de integración del HTML V21.9.'
# La celda es idempotente: si se reejecuta, NO incorpora una segunda copia.
if marker_js not in html:
    html = html.replace(js_hook, js + '\n' + js_hook, 1)
if marker_css not in html:
    assert '</head>' in html
    html = html.replace('</head>', '<style id="atlas-v22-interfaz">\n' + css + '\n</style>\n</head>', 1)
html_path.write_text(html,encoding='utf-8')
assert html.count(marker_js)==1 and html.count(marker_css)==1
print('Atlas público:',html_path.resolve())
print('Módulos ampliados: fórmulas, Asamblea histórica, diccionario, laboratorio y metodología.')
print('Visualizaciones originales, Excel fuente, cálculos y exportaciones sin modificaciones.')


Atlas público: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Módulos ampliados: fórmulas, Asamblea histórica, diccionario, laboratorio y metodología.
Visualizaciones originales, Excel fuente, cálculos y exportaciones sin modificaciones.


### Mapas provinciales sincronizados (corrección de ubicación y lectura)

- **Sección 07 · Presidencia y Asamblea:** ubica los mapas de ganadores provinciales **inmediatamente después del gráfico de composición de la Asamblea**, no al final de todas las tablas. Muestra en paralelo resultados presidenciales de primera ronda y legislativos, con colores partidarios y detalle al pasar el cursor. Actualiza los dos mapas cuando se cambia el año, incluida la reproducción histórica.
- **Sección 18 · Mapas provinciales:** mantiene los valores de participación y la comparación con la elección anterior. Si el usuario solicita "Cambio desde la elección previa" en 1953, muestra el detalle observado y explica por qué no existe una diferencia histórica calculable. En años posteriores la variación se expresa en puntos porcentuales con colores divergentes según signo.

**Datos y módulos restantes preservados.** No modifica los siete archivos Excel ni las gráficas de otras secciones. Se puede reejecutar esta celda para reemplazar una versión antigua del mismo parche sin duplicarlo.

**Ajuste de encuadre ** ambos mapas provinciales se reajustan cuando terminan de dibujarse *y ya se encuentran dentro de su cuadrícula*, con escala geográfica uniforme y centrado continental. Se preservan los colores, las etiquetas, los datos y la exportación original.


In [38]:
# Mapas provinciales sincronizados y contraste histórico (extensión autónoma).
# No altera recursos fuente, siete Excel, gráficas previas ni crea figuras externas.
# Se inserta una sola vez dentro del HTML público generado por la celda .
from pathlib import Path
from tempfile import NamedTemporaryFile
import os

HTML_V22_MAPAS = OUTPUT_DIR / "Atlas_Electoral_Costa_Rica_1953_2026.html"
assert HTML_V22_MAPAS.is_file(), "Primero ejecute el generador HTML V22 del notebook."
V22_MAPS_JS = '/* ATLAS_V22_1_MAPAS_HISTORICOS_Y_PROVINCIALES | Mejora no destructiva. */\n(function(){\n \'use strict\';\n if(typeof views===\'undefined\'||typeof D===\'undefined\'||typeof P===\'undefined\'||typeof plot!==\'function\')return;\n const yearsAll=(typeof Y!==\'undefined\'?Y:(D.meta?.años||[])).map(Number).sort((a,b)=>a-b);\n const signedValue=x=>x!==null&&x!==undefined&&x!==\'\'&&Number.isFinite(Number(x));\n const numeric=x=>signedValue(x)?Number(x):null;\n const nearRows=(arr,province,year)=>arr.find(r=>r.provincia===province&&Number(r.anio)===year);\n // Sección 18: un recorte 2022–2026 no debe eliminar 2018 como referencia para 2022.\n if(typeof atlas14MapRows===\'function\'){\n  const originalMapRows=atlas14MapRows;\n  atlas14MapRows=function(mode,arena,party,key){\n   if(mode!==\'Cambio desde la elección previa\'&&mode!==\'Brecha presidencial–legislativa\')return originalMapRows(mode,arena,party,key);\n   const foco=Number(state.focus),idx=yearsAll.indexOf(foco);\n   const anterior=idx>0?yearsAll[idx-1]:null;\n   if(mode===\'Brecha presidencial–legislativa\'){\n    return P.map(provincia=>{\n     let p=null,l=null;\n     if(party===\'Todos\'){\n      p=(D.metricas||[]).find(r=>r.provincia===provincia&&Number(r.anio)===foco&&r.arena===\'Presidencial\'&&Number(r.ronda)===1)?.participacion_pct;\n      l=(D.metricas||[]).find(r=>r.provincia===provincia&&Number(r.anio)===foco&&r.arena===\'Legislativa\'&&Number(r.ronda)===1)?.participacion_pct;\n     }else{\n      const r=(D.panel||[]).find(r=>r.provincia===provincia&&Number(r.anio)===foco&&r.partido===party);\n      p=r?.porcentaje_Presidencial;l=r?.porcentaje_Legislativa;\n     }\n     return {anio:foco,provincia,valor:signedValue(p)&&signedValue(l)?numeric(p)-numeric(l):null,etiqueta:party===\'Todos\'?\'Participación presidencial − legislativa (pp)\':\'Voto presidencial − legislativo (pp)\'};\n    });\n   }\n   const base=party===\'Todos\'?(key===\'indice_representacion\'?(D.representacion_territorial||[]):(D.metricas||[]).filter(r=>r.arena===arena&&Number(r.ronda)===1)):(D.panel||[]).filter(r=>r.partido===party);\n   return P.map(provincia=>{\n    const actual=nearRows(base,provincia,foco)?.[key],previo=anterior===null?null:nearRows(base,provincia,anterior)?.[key];\n    return {anio:foco,provincia,anio_anterior:anterior,valor:signedValue(actual)&&signedValue(previo)?numeric(actual)-numeric(previo):null,\n      etiqueta:`Cambio de ${metricLabels[key]||key} · ${anterior??\'sin elección anterior\'} → ${foco}${/(_pct|porcentaje)/i.test(key)?\' (pp)\':\'\'}`};\n   });\n  };\n }\n // Paleta divergente idéntica en celdas y leyenda: magenta negativo, verde positivo.\n if(typeof atlas14MapFigure===\'function\'){\n  atlas14MapFigure=function(rows,titleText,opts={}){\n   const vals=rows.filter(r=>signedValue(r.valor)).map(r=>Number(r.valor)),lo=vals.length?Math.min(...vals):0,hi=vals.length?Math.max(...vals):0;\n   const divergent=!!opts.delta||lo<0,span=Math.max(1e-8,Math.abs(lo),Math.abs(hi));\n   const hexToRgb=h=>[1,3,5].map(i=>parseInt(h.slice(i,i+2),16));\n   const mix=(a,b,t)=>{const c=hexToRgb(a),d=hexToRgb(b);return \'rgb(\'+c.map((v,i)=>Math.round(v+(d[i]-v)*Math.max(0,Math.min(1,t)))).join(\',\')+\')\';};\n   const color=v=>{\n    if(!signedValue(v))return \'#e4e8eb\';\n    v=Number(v);\n    if(divergent)return v<0?mix(\'#f2f6f6\',\'#b64b78\',Math.abs(v)/span):mix(\'#f2f6f6\',\'#168572\',v/span);\n    return mix(\'#edf7f7\',\'#097c87\',(v-lo)/Math.max(hi-lo,1e-8));\n   };\n   const tr=[];\n   for(const f of D.geografia?.features||[]){\n    const name=f.properties.NAME_1,row=rows.find(r=>r.provincia===name),v=row?.valor;\n    const polygons=f.geometry.type===\'MultiPolygon\'?f.geometry.coordinates:[f.geometry.coordinates];\n    for(const poly of polygons){const ring=poly[0];if(!ring?.length)continue;\n     tr.push({type:\'scatter\',mode:\'lines\',x:ring.map(p=>p[0]),y:ring.map(p=>p[1]),fill:\'toself\',fillcolor:color(v),line:{color:\'#fff\',width:1},showlegend:false,hoveron:\'fills\',hoverinfo:\'text\',\n      text:`${name} · ${state.focus}<br>${row?.etiqueta||opts.label||\'Indicador\'}: ${signedValue(v)?num(v,2):\'Sin dato\'}`});\n    }\n   }\n   const barColors=divergent?[[0,\'#b64b78\'],[.5,\'#f2f6f6\'],[1,\'#168572\']]:[[0,\'#edf7f7\'],[1,\'#097c87\']];\n   tr.push({type:\'scatter\',mode:\'markers\',x:[null],y:[null],showlegend:false,marker:{size:0,color:[0],cmin:divergent?-span:lo,cmax:divergent?span:(hi===lo?lo+1:hi),colorscale:barColors,showscale:true,colorbar:{title:{text:opts.label||\'Valor\'},thickness:15}}});\n   plot(titleText,tr,rows,{height:590,subtitle:`${state.focus} · ${opts.subtitle||\'\'}`,\n    layout:{xaxis:{visible:false,range:[-86.2,-82.4]},yaxis:{visible:false,scaleanchor:\'x\',scaleratio:1,range:[8,11.35]},hovermode:\'closest\',showlegend:false,margin:{l:18,r:80,t:25,b:18}},\n    note:!vals.length?\'No existe un valor comparable para esta elección y selección. El primer proceso de la serie no tiene elección anterior.\':(opts.note||\'La misma escala cromática se aplica al mapa y a su leyenda. Consulte el valor exacto al pasar el cursor.\')});\n  };\n }\n // La primera elección no tiene comparación anterior. Si se solicita un cambio\n // para 1953, se ofrece el detalle real y una explicación visible, no un mapa vacío.\n if(typeof views.mapas===\'function\'){\n  const previousMapsView=views.mapas;\n  views.mapas=function(...args){\n   const first=Number(state.focus)===yearsAll[0] && local.mapmode===\'Cambio desde la elección previa\';\n   if(first)local.mapmode=\'Detalle provincial\';\n   const out=previousMapsView.apply(this,args);\n   if(first){\n    const area=document.getElementById(\'content\');\n    const msg=document.createElement(\'p\');msg.className=\'notice atlas-v22-primera-eleccion\';\n    msg.textContent=`${yearsAll[0]} es el primer año registrado: no existe una elección anterior con la cual calcular el cambio. Se muestra el valor provincial observado. Para comparar, seleccione ${yearsAll[1]} o una elección posterior.`;\n    const controls=area?.querySelector(\'.controls\');\n    if(controls)controls.insertAdjacentElement(\'afterend\',msg);\n    else area?.prepend(msg);\n   }\n   return out;\n  };\n }\n // Sección 07. Mapas del partido con mayor votación provincial, por arena, en primera ronda.\n const customColors={\n  \'NUEVA REPUBLICA\':\'#39bced\',\'PUEBLO SOBERANO\':\'#0da4b5\',\n  \'RESTAURACION NACIONAL\':\'#1d84d6\',\'UNIDAD SOCIAL CRISTIANA\':\'#0057b8\',\n  \'UNIDAD\':\'#6d7d88\',\'UNIFICACION NACIONAL\':\'#a7b2ba\',\n  \'REPUBLICANO\':\'#caa11a\',\'REPUBLICANO NACIONAL\':\'#caa11a\'\n };\n function partyColor(name){\n  const normalized=typeof norm===\'function\'?norm(name):String(name||\'\').normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toUpperCase();\n  return customColors[normalized]||(typeof semanticColor===\'function\'?semanticColor(name):\'#748996\')||\'#748996\';\n }\n function winnersOfYear(arena,year){\n  const data=(D.metricas||[]).filter(r=>Number(r.anio)===year&&r.arena===arena&&Number(r.ronda)===1&&P.includes(r.provincia));\n  return P.map(p=>{\n   const r=data.find(v=>v.provincia===p),ganador=r?.ganador||null;\n   const vote=(D.votos||[]).find(v=>Number(v.anio)===year&&v.provincia===p&&v.arena===arena&&Number(v.ronda)===1&&v.partido===ganador);\n   return {anio:year,arena,provincia:p,partido:ganador||\'Sin dato\',porcentaje:numeric(r?.concentracion_1),votos:numeric(vote?.votos),fuente:\'Primera ronda / voto provincial\'};\n  });\n }\n function generateProvinceMap(arena,year){\n  const rows=winnersOfYear(arena,year),tr=[],seen=new Set();\n  const palette=[...new Set(rows.map(r=>r.partido))].filter(p=>p!==\'Sin dato\');\n  for(const f of D.geografia?.features||[]){\n   const name=f.properties.NAME_1,rr=rows.find(r=>r.provincia===name),p=rr?.partido||\'Sin dato\';\n   const color=p===\'Sin dato\'?\'#e4e8eb\':partyColor(p);\n   const polygons=f.geometry.type===\'MultiPolygon\'?f.geometry.coordinates:[f.geometry.coordinates];\n   for(const poly of polygons){const ring=poly[0];if(!ring?.length)continue;\n    const hover=`${name} · ${year}<br>Primera fuerza ${arena===\'Presidencial\'?\'presidencial\':\'legislativa\'}: ${p}`+\n     `<br>Votos válidos: ${rr?.votos===null?\'No disponible\':num(rr?.votos,0)}`+\n     `<br>Cuota de votos válidos: ${rr?.porcentaje===null?\'Sin dato\':num(rr.porcentaje,2)+\' %\'}`;\n    tr.push({type:\'scatter\',mode:\'lines\',x:ring.map(v=>v[0]),y:ring.map(v=>v[1]),fill:\'toself\',fillcolor:color,\n      line:{color:\'#fff\',width:1.35},showlegend:false,hoveron:\'fills\',hoverinfo:\'text\',text:hover});\n   }\n  }\n  const nombre=arena===\'Presidencial\'?\'Resultados presidenciales por provincia\':\'Resultados legislativos por provincia\';\n  const old=new Set(document.querySelectorAll(\'#content .card\'));\n  plot(nombre,tr,rows,{height:460,subtitle:`${year} · ${arena===\'Presidencial\'?\'Primera ronda\':\'Diputaciones\'} · siete provincias`,\n   layout:{xaxis:{visible:false,range:[-86.2,-82.4]},yaxis:{visible:false,scaleanchor:\'x\',scaleratio:1,range:[8,11.35]},margin:{l:5,r:5,t:10,b:15},showlegend:false,hovermode:\'closest\'},\n   note:arena===\'Presidencial\'?\'El color indica el partido más votado en primera ronda en cada provincia. No necesariamente corresponde al ganador final de la Presidencia si hubo segunda ronda.\':\'El color indica el partido con más votos para diputaciones en cada provincia; no implica que obtuvo todos los escaños provinciales.\'});\n  const newCard=[...document.querySelectorAll(\'#content .card\')].find(node=>!old.has(node));\n  if(newCard){newCard.classList.add(\'atlas-v22-provincia-card\');\n   const legend=document.createElement(\'div\');legend.className=\'atlas-v22-provincia-leyenda\';\n   for(const p of palette){const item=document.createElement(\'span\');item.className=\'atlas-v22-provincia-partido\';\n    const swatch=document.createElement(\'i\');swatch.style.backgroundColor=partyColor(p);item.append(swatch,document.createTextNode(typeof labelText===\'function\'?labelText(p):p));legend.append(item);}\n   newCard.appendChild(legend);\n  }\n  return newCard;\n }\n function applyFormulaColors(){\n  const section=[...document.querySelectorAll(\'#content .card\')].find(c=>/Fórmula presidencial electa/i.test(c.querySelector(\'h3\')?.textContent||\'\'));\n  if(!section)return;\n  const elected=(D.ejecutivo_electo||[]).filter(r=>Number(r.anio)===Number(state.focus));\n  section.querySelectorAll(\'.assembly-person\').forEach((element,i)=>{\n   const candidate=elected[i];if(!candidate)return;\n   element.style.borderLeft=\'4px solid \'+partyColor(candidate.partido);\n   const spans=[...element.querySelectorAll(\'span\')];\n   if(spans.length){const s=spans[spans.length-1];s.style.color=partyColor(candidate.partido);s.style.fontWeight=\'750\';}\n  });\n }\n let currentYear=null;\n function refreshYear(force){\n  if(state.tab!==\'gobierno\')return;\n  const year=Number(state.focus);\n  if(!yearsAll.includes(year))return;\n  const active=document.getElementById(\'atlas-v22-map-pair\');\n  if(active&&currentYear===year&&!force)return;\n  if(active)active.remove();\n  currentYear=year;\n  const host=document.createElement(\'section\');host.id=\'atlas-v22-map-pair\';\n  host.className=\'atlas-v22-provincia-wrap atlas-v23-mapa-preparando\';\n  host.innerHTML=`<div class="atlas-v22-provincia-head"><div><h3>Mapa electoral provincial · ${year}</h3><p>Primera fuerza por votos en cada provincia · recorrido histórico sincronizado con la Asamblea.</p></div></div><div class="atlas-v22-provincia-grid"></div>`;\n  document.querySelector(\'#content\')?.appendChild(host);\n  const grid=host.querySelector(\'.atlas-v22-provincia-grid\');\n  const left=generateProvinceMap(\'Presidencial\',year),right=generateProvinceMap(\'Legislativa\',year);\n  if(left)grid.appendChild(left);if(right)grid.appendChild(right);\n  // Después de mover ambas tarjetas del contenido general a una cuadrícula,\n  // Plotly conserva por un instante el ancho original de página completa.\n  // Esperar a que sus newPlot finalicen y reajustar en el contenedor real evita\n  // que el mapa se abra cortado, desplazado o pequeño en el PNG exportado.\n  const newMapPlots=typeof pending!==\'undefined\'?pending.slice(-2):[];\n  const centerProvincialMaps=async()=>{\n   if(!host.isConnected||state.tab!==\'gobierno\'||Number(state.focus)!==year)return;\n   const nodes=[...grid.querySelectorAll(\'.atlas-v22-provincia-card .js-plotly-plot\')];\n   for(const gd of nodes){\n    if(!gd.isConnected||!gd._fullLayout)continue;\n    await Plotly.Plots.resize(gd);\n    if(!gd.isConnected)continue;\n    // Límites continentales del GeoJSON: se preserva la proporción cartográfica.\n    // Isla del Coco no altera el encuadre de las siete provincias continentales.\n    const size=gd._fullLayout?._size||{};\n    const w=Math.max(150,Number(size.w)||gd.clientWidth-10);\n    const h=Math.max(150,Number(size.h)||gd.clientHeight-25);\n    const midLon=(-85.96069335937489-82.55322265625)/2;\n    const midLat=(8.037637710571346+11.21691799163824)/2;\n    const spanLon=85.96069335937489-82.55322265625;\n    const spanLat=11.21691799163824-8.037637710571346;\n    const unitsPerPixel=Math.max(spanLon*1.08/w,spanLat*1.08/h);\n    const dx=unitsPerPixel*w/2,dy=unitsPerPixel*h/2;\n    await Plotly.relayout(gd,{\n     \'xaxis.range\':[midLon-dx,midLon+dx],\n     \'yaxis.range\':[midLat-dy,midLat+dy],\n     \'xaxis.autorange\':false,\'yaxis.autorange\':false\n    });\n   }\n  };\n  Promise.allSettled(newMapPlots)\n   .then(()=>new Promise(resolve=>requestAnimationFrame(()=>requestAnimationFrame(resolve))))\n   .then(centerProvincialMaps)\n   .then(()=>host.classList.remove(\'atlas-v23-mapa-preparando\'))\n   .catch(err=>{host.classList.remove(\'atlas-v23-mapa-preparando\');console.warn(\'Ajuste visual de mapas V22:\',err)});\n  // Insertar inmediatamente después del hemiciclo, sin alterar el trazado original.\n  const hemi=[...document.querySelectorAll(\'#content > .card\')].find(e=>\n   /Composición partidaria de la Asamblea|Hemiciclo histórico|Hemiciclo/i.test(e.querySelector(\'h3\')?.textContent||\'\'));\n  if(hemi){hemi.insertAdjacentElement(\'afterend\',host);}\n  else{const formula=[...document.querySelectorAll(\'#content > .card\')].find(e=>/Fórmula presidencial electa/i.test(e.querySelector(\'h3\')?.textContent||\'\'));\n   if(formula)formula.insertAdjacentElement(\'afterend\',host);}\n  host.dataset.anio=String(year);\n  applyFormulaColors();\n }\n if(typeof views.gobierno===\'function\'){\n  const oldGov=views.gobierno;\n  views.gobierno=function(...args){const out=oldGov.apply(this,args);currentYear=null;refreshYear(true);return out;};\n }\n // Reproducir historia modifica el año: sincronizar SIN crear otro temporizador.\n const updateOnRender=typeof render===\'function\'?render:null;\n if(updateOnRender){render=async function(...args){const result=await updateOnRender.apply(this,args);if(state.tab===\'gobierno\')refreshYear(false);return result;};}\n const main=document.getElementById(\'content\');\n if(main&&typeof MutationObserver!==\'undefined\'){\n  const observer=new MutationObserver(()=>{\n   if(state.tab===\'gobierno\'&&yearsAll.includes(Number(state.focus))&&currentYear!==Number(state.focus))refreshYear(false);\n  });\n  observer.observe(main,{subtree:true,childList:true,characterData:true});\n }\n window.ATLAS_V22_1_MAPAS_OK=true;\n})();\n'
V22_MAPS_CSS = '/* ATLAS_V22_1_MAPAS_CSS: solo módulo de mapas históricos. */\n.atlas-v22-provincia-wrap{margin:22px 0 28px;}\n.atlas-v22-provincia-head{margin:0 0 12px;padding:1px 3px;}\n.atlas-v22-provincia-head h3{font-size:19px;line-height:1.35;letter-spacing:-.3px;color:#163e4d;}\n.atlas-v22-provincia-head p{font-size:12px;color:#526d7b;margin:3px 0 0;}\n.atlas-v22-provincia-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:16px;align-items:stretch;}\n.atlas-v22-provincia-card{min-width:0;box-sizing:border-box;margin:0;}\n.atlas-v22-provincia-card .card-head h3{font-size:15px;line-height:1.35;}\n.atlas-v22-provincia-card .chart{min-height:460px;}\n.atlas-v22-provincia-card .chart-note{line-height:1.52;}\n.atlas-v22-provincia-leyenda{margin:10px 0 0;display:flex;align-items:center;gap:8px 13px;flex-wrap:wrap;border-top:1px solid #e6edef;padding-top:12px;}\n.atlas-v22-provincia-partido{display:inline-flex;align-items:center;gap:6px;font-size:11px;line-height:1.3;color:#314d5b;}\n.atlas-v22-provincia-partido i{width:12px;height:12px;display:inline-block;border-radius:2px;flex-shrink:0;border:1px solid rgba(0,0,0,.1);}\n@media(max-width:1120px){.atlas-v22-provincia-grid{grid-template-columns:1fr;}.atlas-v22-provincia-card .chart{min-height:460px;}}\n'
V22_MAPS_JS_MARK = "/* ATLAS_V22_1_MAPAS_HISTORICOS_Y_PROVINCIALES"
V22_MAPS_CSS += '\n#atlas-v22-map-pair.atlas-v23-mapa-preparando .atlas-v22-provincia-card .chart{visibility:hidden!important}\n'
V22_MAPS_CSS_MARK = "/* ATLAS_V22_1_MAPAS_CSS"
V22_HOOK = "window.addEventListener('resize',()=>document.querySelectorAll('.js-plotly-plot').forEach(el=>Plotly.Plots.resize(el)));setup();"

html = HTML_V22_MAPAS.read_text(encoding="utf-8")
if V22_MAPS_JS_MARK in html:
    # Refrescar la extensión antigua () si esta celda se reejecuta aislada.
    start = html.index(V22_MAPS_JS_MARK)
    end = html.find("\n})();", start)
    assert end > start, "No se reconoció el final de la extensión V22 anterior."
    html = html[:start] + V22_MAPS_JS + html[end + len("\n})();"):]
else:
    assert V22_HOOK in html, "No se encontró el punto de integración de V22; no se ha modificado el HTML."
    html = html.replace(V22_HOOK,V22_MAPS_JS+"\n"+V22_HOOK,1)
if V22_MAPS_CSS_MARK not in html:
    assert "</head>" in html, "No se encontró la cabecera HTML."
    html = html.replace("</head>","<style id=\"atlas-v22-mapas-historicos\">\n"+V22_MAPS_CSS+"\n</style>\n</head>",1)
assert html.count(V22_MAPS_JS_MARK)==1 and html.count(V22_MAPS_CSS_MARK)==1
# Escritura atómica: no deja un HTML a medias en caso de error.
with NamedTemporaryFile("w",encoding="utf-8",dir=HTML_V22_MAPAS.parent,
                        prefix=".atlas_v22_mapas_",suffix=".html",delete=False) as temp:
    temp.write(html)
    temp_path=Path(temp.name)
os.replace(temp_path,HTML_V22_MAPAS)
print('· dos mapas centrados y redimensionados en sus tarjetas, sincronizados con la historia.')
print("Mapas provinciales · cambio vs elección anterior y colores corregidos (solo sección 18).")
print("HTML generado:",HTML_V22_MAPAS.resolve())
print("No se generaron carpetas de figuras ni se modificaron archivos fuente.")


· dos mapas centrados y redimensionados en sus tarjetas, sincronizados con la historia.
Mapas provinciales · cambio vs elección anterior y colores corregidos (solo sección 18).
HTML generado: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
No se generaron carpetas de figuras ni se modificaron archivos fuente.


**Comprobación editorial:** no se encontraron instrucciones personales de edición ni mensajes dirigidos al autor en la plantilla pública del Atlas. Las notas metodológicas que indican las limitaciones de los indicadores se conservan por razones de transparencia. La presentación no cambia el orden de las secciones ni las interpretaciones; es una corrección visual acotada.

## Alcance de la El recorrido cubre las dimensiones disponibles en las ocho bases y preserva la agenda ampliada en `recursos/agenda.md`. Los análisis de ideología, parentesco, efectos causales, cronología jurídica exacta y cambios de bancada durante un mandato necesitan fuentes adicionales. Las trayectorias son nominales y la cobertura presidencial corresponde a las siete provincias de las bases. La aplicación permite explorar sin convertir cada asociación en un hallazgo académico.

En representación territorial, la serie longitudinal disponible permite comparar el **peso provincial en escaños** con el **peso provincial en el padrón electoral**. Esa comparación es descriptiva: no sustituye una medición basada en población ni la regla jurídica de asignación. recupera la identificación visual de los años en que cambió la asignación provincial y documenta las provincias afectadas.


## claridad final de recurrencia, éxito y exploración

Esta revisión amplía **Éxito electoral** con dos lecturas separadas: (1) el universo completo, incluyendo personas nunca electas; y (2) el subconjunto de personas que sí resultaron electas, para observar con claridad cuántas ganaron en su primera elección documentada, cuántas necesitaron intentos previos y cuántas volvieron a resultar electas en años posteriores. También añade estadísticas explícitas de recurrencia, búsqueda avanzada del Explorador de personas, tarjetas auditables para antecedentes y una ampliación del Panorama histórico. Ninguna de estas mejoras modifica las ocho bases fuente.

## versión final y pestaña de presentación

La conserva íntegramente la arquitectura analítica de y agrega una pestaña **00 · Presentación** como punto de entrada para el público. La pestaña explica qué es la aplicación, quién la desarrolló, el contexto IFED–TSE, el periodo y bases integradas, la metodología de Political Data Science, las principales dimensiones disponibles, cómo usar filtros y descargas y las cautelas necesarias para interpretar los resultados.

No se modifican las ocho bases Excel fuente ni los cálculos de . El cambio funcional de es de orientación, accesibilidad y presentación del producto final.


## Auditoría de doble postulación y movilidad provincial

La matriz aplica una **única escala cromática** a provincias por elección, totales históricos por provincia y totales nacionales por elección. **Los ceros son conteos válidos**. La casilla final del total país–periodo es la única que queda blanca y fuera de la escala. La movilidad provincial permanece restringida a candidaturas legislativas documentadas en ambos extremos.

**(etiquetas)**: restaura los valores altos (2022: 10 en San José y 12 en Costa Rica; total histórico de San José: 33) mediante una capa de texto de alto contraste visible en el HTML y sus exportaciones, sin alterar la escala de color ni los conteos.


## Ampliación académica dos correcciones solicitadas

1. **Padrón y escaños:** Ranking provincial histórico 1–7 por cada elección; columna final «Ranking global» (ranking del promedio aritmético de brechas firmadas), escala cromática única y eje categórico con todas las elecciones.
2. **Éxito electoral:** la profundización representa los antecedentes mediante dos distribuciones horizontales por cargo, normalizadas al 100 % y conectadas al visor nominal.


In [39]:
# Corrección focalizada para la aplicación pública.
# Preserva el generador, todos los recursos fuente, las siete bases y las gráficas existentes.
# Reemplaza SOLO la extensión académica de ranking/antecedentes; no genera PNG externos.
from pathlib import Path
from tempfile import NamedTemporaryFile
import os
ATLAS_V22_NEW_CHARTS_JS = '/* ATLAS_V22_7_MATRIZ_EXPLORATORIA · mejora focalizada, no altera las siete bases. */\n(function(){\n \'use strict\';\n if(typeof views===\'undefined\'||typeof D===\'undefined\'||typeof plot!==\'function\')return;\n if(window.ATLAS_V22_7_ANTECEDENTES_OK)return;\n const finite=v=>v!==null&&v!==undefined&&v!==\'\'&&Number.isFinite(Number(v));\n const pctSafe=(a,b)=>b?100*a/b:0;\n const fmt=v=>Number(v).toLocaleString(\'es-CR\',{maximumFractionDigits:1,minimumFractionDigits:1});\n const provinces=[\'San José\',\'Alajuela\',\'Cartago\',\'Heredia\',\'Guanacaste\',\'Puntarenas\',\'Limón\'];\n const roles=[\'Diputación\',\'Presidencia\',\'Primera Vicepresidencia\',\'Segunda Vicepresidencia\'];\n const categories=[\'Ya había resultado electa antes\',\'Había competido antes, pero aún no había sido electa\'];\n const roleColors={\'Diputación\':\'#118578\',\'Presidencia\':\'#4176aa\',\'Primera Vicepresidencia\':\'#9464a4\',\'Segunda Vicepresidencia\':\'#c28a54\'};\n const roleLabel=rs=>{\n  if(!rs.length)return \'Sin antecedente verificable\';\n  if(rs.length===1)return rs[0];\n  const leg=rs.includes(\'Diputación\');\n  const nat=roles.filter(r=>r!==\'Diputación\'&&rs.includes(r));\n  if(leg&&nat.length===1)return \'Doble postulación: \'+nat[0]+\' + Diputación\';\n  if(leg&&nat.length>1)return \'Doble postulación con varios cargos nacionales: \'+nat.join(\' + \')+\' + Diputación\';\n  return \'Concurrencia de cargos nacionales: \'+rs.join(\' + \')+\' (sin diputación)\';\n };\n const colorRole=r=>roleColors[r]||(r.startsWith(\'Doble postulación:\')?\'#0c8c8f\':r.startsWith(\'Doble postulación con\')?\'#006e77\':\'#71838d\');\n const isElected=v=>v===true||String(v).toLowerCase()===\'true\'||v===1;\n\n // Representación relativa al PADRÓN, no a la población censal, que es el denominador legal.\n // Serie completa por elección; última columna = ranking del promedio aritmético histórico.\n if(typeof views.representacion===\'function\'){\n  const earlier=views.representacion;\n  views.representacion=function(...args){\n   const result=earlier.apply(this,args);\n   const records=(D.representacion_territorial||[]).filter(r=>r.anio>=state.from&&r.anio<=state.to&&provinces.includes(r.provincia)&&finite(r.brecha_representacion_pp));\n   if(!records.length)return result;\n   const years=[...new Set(records.map(r=>Number(r.anio)))].sort((a,b)=>a-b);\n   const registry=new Map(records.map(r=>[r.provincia+\'|\'+r.anio,Number(r.brecha_representacion_pp)]));\n   const annual=[],averages=[];\n   for(const year of years){\n     const sorted=provinces.filter(p=>registry.has(p+\'|\'+year)).map(p=>({provincia:p,valor:registry.get(p+\'|\'+year)}))\n       .sort((a,b)=>b.valor-a.valor||provinces.indexOf(a.provincia)-provinces.indexOf(b.provincia));\n     sorted.forEach((r,i)=>annual.push({anio:year,provincia:r.provincia,puesto:i+1,brecha_pp:r.valor}));\n   }\n   for(const provincia of provinces){\n     const matches=years.map(year=>registry.get(provincia+\'|\'+year)).filter(finite);\n     averages.push({provincia,media_pp:matches.length?matches.reduce((a,b)=>a+b,0)/matches.length:null,elecciones:matches.length});\n   }\n   averages.sort((a,b)=>(b.media_pp??-Infinity)-(a.media_pp??-Infinity)||provinces.indexOf(a.provincia)-provinces.indexOf(b.provincia));\n   averages.forEach((r,i)=>r.puesto_global=r.media_pp===null?null:i+1);\n   const annualByKey=new Map(annual.map(r=>[r.provincia+\'|\'+r.anio,r]));\n   const avgByProv=new Map(averages.map(r=>[r.provincia,r]));\n   const x=[...years.map(String),\'Ranking global\'];\n   const signed=provinces.map(p=>[...years.map(y=>registry.has(p+\'|\'+y)?registry.get(p+\'|\'+y):null),avgByProv.get(p)?.media_pp??null]);\n   const ranks=provinces.map(p=>[...years.map(y=>annualByKey.get(p+\'|\'+y)?.puesto??null),avgByProv.get(p)?.puesto_global??null]);\n   const maxAbs=Math.max(1,...signed.flat().filter(finite).map(v=>Math.abs(v)));\n   plot(\'Ranking provincial histórico de representación relativa al padrón\',[\n    {type:\'heatmap\',x,y:provinces,z:signed,customdata:ranks,\n     text:ranks.map(row=>row.map(v=>v===null?\'—\':String(v))),texttemplate:\'%{text}\',textfont:{size:12,color:\'#173644\'},\n     colorscale:[[0,\'#a75266\'],[.5,\'#f2f5f6\'],[1,\'#1b9b83\']],zmin:-maxAbs,zmax:maxAbs,zmid:0,xgap:2,ygap:2,\n     colorbar:{title:{text:\'Brecha<br>(pp)\'},thickness:16},\n     hovertemplate:\'%{y} · %{x}<br>Puesto %{customdata} de 7<br>Brecha escaños − padrón: %{z:.2f} pp<extra></extra>\'}\n   ],[...annual,...averages],{\n     height:535,subtitle:`${state.from}–${state.to} · 1 = mayor sobrerrepresentación relativa · último campo: promedio global`,\n     xTitle:\'Elección / ranking global\',yTitle:\'Provincia\',\n     layout:{xaxis:{type:\'category\',categoryorder:\'array\',categoryarray:x,tickangle:-50},\n       shapes:[{type:\'line\',xref:\'x\',yref:\'paper\',x0:years.length-.5,x1:years.length-.5,y0:0,y1:1,line:{color:\'#214a62\',width:2}}],\n       margin:{l:120,r:110,t:35,b:118}},\n     note:\'Los números 1–7 indican el puesto de cada provincia en cada elección (1 = mayor brecha positiva). El color muestra la BRECHA firmada (puntos porcentuales) y utiliza la misma escala en todas las columnas, incluido el promedio. La columna «Ranking global» ordena el promedio simple de brechas por provincia dentro del intervalo seleccionado. Ante empates, se utiliza el orden provincial fijo. IMPORTANTE: el padrón electoral no es el criterio jurídico de reparto de diputaciones; esta es una comparación descriptiva.\'\n   });\n   table(annual,[\'anio\',\'provincia\',\'puesto\',\'brecha_pp\'],\'Ranking de representación relativa al padrón · cada elección\');\n   table(averages,[\'puesto_global\',\'provincia\',\'media_pp\',\'elecciones\'],\'Ranking global · media de brechas en pp\');\n   return result;\n  };\n }\n\n // Distribuye frecuencias de antecedentes, no porcentajes de personas superpuestas.\n function priorDetail(){\n   const selectedRole=local.anteCargo||\'Presidencia\';\n   const unit=local.anteUnit||\'Personas distintas · primera elección en el cargo\';\n   const personsMode=String(unit).startsWith(\'Personas distintas\');\n   let candidates=(D.antecedentes_cargos_electos||[]).filter(r=>r.cargo===selectedRole&&Number(r.anio)>=state.from&&Number(r.anio)<=state.to);\n   if(state.province!==\'Nacional\'&&selectedRole===\'Diputación\')candidates=candidates.filter(r=>r.provincia===state.province);\n   const byPerson=new Map();for(const r of candidates.slice().sort((a,b)=>Number(a.anio)-Number(b.anio)||String(a.id_evento).localeCompare(String(b.id_evento))))if(!byPerson.has(r.id_persona))byPerson.set(r.id_persona,r);\n   const population=personsMode?[...byPerson.values()]:candidates;\n   const histories=new Map();for(const row of D.resultados_candidaturas||[]){if(!histories.has(row.id_persona))histories.set(row.id_persona,[]);histories.get(row.id_persona).push(row)}\n   const selected=categories.includes(local.v222Anterior)?local.v222Anterior:\'Ambos grupos\';\n   const groups=selected===\'Ambos grupos\'?categories:[selected];\n   add(\'<div class="extension-heading">PROFUNDIZACIÓN · DISTRIBUCIÓN DE ANTECEDENTES POR CARGO</div>\');\n   controls([[\'v222Anterior\',\'Antecedentes para explorar\',[[\'Ambos grupos\',\'Ambas distribuciones\'],[categories[0],\'Cargos ganados anteriormente\'],[categories[1],\'Cargos disputados sin elección previa\']],selected]]);\n   if(local.v227AnteCell&&local.v227AnteCell.scope!==atlas227AnteScope())delete local.v227AnteCell;\n   add(\'<div class="clarity-box"><b>Cómo leer estas distribuciones:</b> cada gráfica reparte el 100 % de sus antecedentes entre cuatro cargos. No reparte personas. Una persona puede aportar varios antecedentes; una doble postulación aporta una candidatura a cada cargo. Pulse una barra o su punto para ver las personas correspondientes, sin repetir tarjetas.</div>\');\n   const summary=[],details=[];\n   for(const groupName of groups){\n     const targets=population.filter(r=>r.antecedente===groupName);\n     const groupDetails=[];\n     for(const target of targets){\n       const seen=new Set();\n       for(const previous of histories.get(target.id_persona)||[]){\n         if(Number(previous.anio)>=Number(target.anio)||!roles.includes(previous.cargo))continue;\n         if(groupName===categories[0]&&!isElected(previous.electa))continue;\n         const eventId=previous.id_evento||[previous.anio,previous.cargo,previous.partido,previous.provincia,previous.tipo,previous.orden].join(\'|\');\n         if(seen.has(eventId))continue;seen.add(eventId);\n         groupDetails.push({id_persona:target.id_persona,nombre:target.nombre,id_resultado_electo:target.id_evento,anio_electo:target.anio,cargo_electo:selectedRole,grupo:groupName,id_antecedente:eventId,anio_antecedente:previous.anio,cargo_anterior:previous.cargo,electa_antecedente:isElected(previous.electa),partido_antecedente:previous.partido,provincia_antecedente:previous.provincia,nombre_original_antecedente:previous.nombre_original_fuente||previous.nombre_original||previous.nombre,unidad:unit});\n       }\n     }\n     const total=groupDetails.length;\n     const rows=roles.map(role=>({cargo_electo:selectedRole,unidad:unit,grupo:groupName,cargo_anterior:role,antecedentes:groupDetails.filter(r=>r.cargo_anterior===role).length,total_antecedentes:total,personas_grupo:new Set(targets.map(r=>r.id_persona)).size,resultados_grupo:targets.length}));\n     // Redondeo por restos mayores: los cuatro porcentajes visibles suman 100,0 %.\n     const ticks=rows.map(r=>total?Math.floor(1000*r.antecedentes/total):0);\n     if(total){const order=rows.map((r,i)=>({i,rest:1000*r.antecedentes/total-ticks[i]})).sort((a,b)=>b.rest-a.rest||a.i-b.i);let remaining=1000-ticks.reduce((a,b)=>a+b,0);for(let i=0;i<remaining;i++)ticks[order[i].i]++}\n     rows.forEach((r,i)=>{r.porcentaje=total?100*r.antecedentes/total:null;r.porcentaje_visible=total?ticks[i]/10:null});\n     summary.push(...rows);details.push(...groupDetails);\n     const won=groupName===categories[0];\n     const title=(won?\'Cargos ganados anteriormente\':\'Cargos disputados sin elección previa\')+\' · quienes resultaron electos en \'+selectedRole;\n     if(!total){add(\'<article class="card"><div class="card-head"><h3>\'+esc(title)+\'</h3></div><div class="empty">Sin antecedentes para esta selección.</div></article>\');continue}\n     const y=roles,values=rows.map(r=>r.porcentaje),custom=rows.map(r=>[groupName,r.cargo_anterior,r.antecedentes,total]);\n     const annotations=rows.map(r=>({xref:\'paper\',yref:\'y\',x:1.025,y:r.cargo_anterior,xanchor:\'left\',showarrow:false,text:\'<b>\'+fmt(r.porcentaje_visible)+\' %</b>\',font:{size:23,color:colorRole(r.cargo_anterior)},align:\'left\'}));\n     const sourceNote=won?\'100 % = todas las elecciones anteriores ganadas por este grupo, sumadas por cargo.\':\'100 % = todas las candidaturas anteriores de este grupo, que aún no había ganado ningún cargo.\';\n     const unitNote=personsMode?\'Personas distintas: se toma la primera elección de cada persona en el cargo dentro del intervalo y se cuenta su historia estrictamente anterior.\':\'Resultados electos: se cuenta la historia estrictamente anterior a cada resultado; un antecedente puede contribuir a más de un resultado posterior.\';\n     plot(title,[\n       {type:\'bar\',orientation:\'h\',x:roles.map(()=>100),y,width:.20,marker:{color:\'#edf2f5\'},text:roles.map(()=>\'\'),textposition:\'none\',hoverinfo:\'skip\',showlegend:false},\n       {type:\'bar\',orientation:\'h\',x:values,y,width:.20,marker:{color:roles.map(colorRole)},text:roles.map(()=>\'\'),textposition:\'none\',customdata:custom,showlegend:false,hovertemplate:\'%{y}<br>%{x:.2f} % del total de antecedentes<extra></extra>\'},\n       {type:\'scatter\',mode:\'markers\',x:values,y,marker:{size:12,color:roles.map(colorRole),line:{color:\'#fff\',width:2}},customdata:custom,showlegend:false,hovertemplate:\'%{y}<br>%{x:.2f} % del total de antecedentes<extra></extra>\'}\n     ],rows,{height:330,subtitle:`${state.from}–${state.to} · ${state.province} · ${unit}`,xTitle:\'\',layout:{barmode:\'overlay\',showlegend:false,annotations,margin:{l:205,r:118,t:22,b:28},xaxis:{type:\'linear\',range:[0,100],showgrid:false,zeroline:false,showticklabels:false,fixedrange:true},yaxis:{type:\'category\',categoryorder:\'array\',categoryarray:roles,autorange:\'reversed\',showgrid:false,zeroline:false,ticks:\'\',tickfont:{size:14},automargin:true,fixedrange:true}},note:sourceNote+\' \'+unitNote+\' Cada ocurrencia de un cargo cuenta una vez. Las dobles postulaciones no forman una categoría adicional. Los porcentajes visibles se redondean a una decimal y suman 100,0 %; el CSV conserva también el porcentaje sin redondear. Antecedentes disponibles desde 1953.\',onClick:evt=>{const d=evt.points?.[0]?.customdata;if(!Array.isArray(d))return;local.anteCategory=d[0];local.antepage=0;local.v227AnteCell={group:d[0],kind:\'role\',value:d[1],scope:atlas227AnteScope(),label:d[0]+\' · \'+d[1]};render()}});\n   }\n   window.atlas229AntecedentDistributions={summary,details};\n   table(summary,[\'cargo_electo\',\'unidad\',\'grupo\',\'cargo_anterior\',\'antecedentes\',\'total_antecedentes\',\'porcentaje\',\'porcentaje_visible\',\'personas_grupo\',\'resultados_grupo\'],\'Distribuciones de antecedentes · porcentajes y denominadores\');\n   table(details,[\'id_persona\',\'nombre\',\'id_resultado_electo\',\'anio_electo\',\'cargo_electo\',\'grupo\',\'id_antecedente\',\'anio_antecedente\',\'cargo_anterior\',\'electa_antecedente\',\'partido_antecedente\',\'provincia_antecedente\',\'nombre_original_antecedente\',\'unidad\'],\'Auditoría de antecedentes · cada cargo que aporta al porcentaje\');\n }\n if(typeof views.outcomes===\'function\'){\n   const earlier=views.outcomes;\n   views.outcomes=function(...args){\n     const oldPlot=plot;\n     let integrated=false;\n     const intercept=function(...plotArgs){\n       const answer=oldPlot.apply(this,plotArgs);\n       if(!integrated&&String(plotArgs[0]||\'\').startsWith(\'Cómo llegaban quienes resultaron electos en \')){\n         integrated=true;\n         plot=oldPlot;\n         try{priorDetail();}finally{plot=intercept;}\n       }\n       return answer;\n     };\n     plot=intercept;\n     try{return earlier.apply(this,args);}finally{plot=oldPlot;}\n   };\n }\n window.ATLAS_V22_7_ANTECEDENTES_OK=true;\n})();'
page=OUTPUT_DIR / 'Atlas_Electoral_Costa_Rica_1953_2026.html'
assert page.is_file(), 'Ejecute primero la sección que construye el HTML público.'
source=page.read_text(encoding='utf-8')
oldmarker='/* ATLAS_V22_1_AMPLIACIONES_ACADEMICAS'
newmarker='/* ATLAS_V22_7_MATRIZ_EXPLORATORIA'
previous_v226_marker='/* ATLAS_V22_6_MATRIZ_EXCLUYENTE'
previous_v225_marker='/* ATLAS_V22_5_ANTECEDENTES_CORREGIDOS'
previous_matrix_marker='/* ATLAS_V22_4_MATRIZ_PROCEDENCIA'
previous_marker='/* ATLAS_V22_3_RANKING_Y_ANTECEDENTES'
intermediate_marker='/* ATLAS_V22_2_RANKING_Y_ANTECEDENTES'
hook="window.addEventListener('resize',()=>document.querySelectorAll('.js-plotly-plot').forEach(el=>Plotly.Plots.resize(el)));setup();"
if any(marker in source for marker in (oldmarker, intermediate_marker, previous_marker, previous_matrix_marker, previous_v225_marker, previous_v226_marker, newmarker)):
    begin=min(source.index(marker) for marker in (oldmarker, intermediate_marker, previous_marker, previous_matrix_marker, previous_v225_marker, previous_v226_marker, newmarker) if marker in source)
    end=source.find('\n})();',begin)
    if end<0: raise RuntimeError('Bloque V22 anterior incompleto, se conserva el HTML sin modificaciones.')
    source=source[:begin]+ATLAS_V22_NEW_CHARTS_JS+source[end+len('\n})();'):]
elif newmarker not in source:
    if hook not in source: raise RuntimeError('No se encontró el punto de inserción estable de V22.2.')
    source=source.replace(hook,ATLAS_V22_NEW_CHARTS_JS+'\n'+hook,1)
assert source.count(newmarker)==1
assert oldmarker not in source
with NamedTemporaryFile('w',encoding='utf-8',dir=page.parent,prefix='.atlas_v22_2_',suffix='.html',delete=False) as f:
    f.write(source); temp=Path(f.name)
os.replace(temp,page)
print(': dos distribuciones de antecedentes por cargo; cada una suma 100 %, con selección nominal y CSV.')
print('Se retiró la gráfica adicional de continuidad partidaria hacia Presidencia/vicepresidencias.')
print('HTML actualizado:',page.resolve())


: dos distribuciones de antecedentes por cargo; cada una suma 100 %, con selección nominal y CSV.
Se retiró la gráfica adicional de continuidad partidaria hacia Presidencia/vicepresidencias.
HTML actualizado: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html


## Distribuciones horizontales de antecedentes

La profundización sustituye la matriz por **dos distribuciones**: cargos ganados anteriormente y cargos disputados por quienes todavía no habían ganado ningún cargo. Se conservan los cuatro cargos electos, las dos unidades de análisis y los controles del Atlas.

Cada gráfico suma **100 % del total de antecedentes** del grupo, no de sus personas. Se cuenta cada evento anterior por cargo; una doble postulación aporta a ambos cargos sin categoría adicional. En personas distintas se toma la primera elección en el cargo dentro del intervalo; en resultados electos, cada resultado aporta su historia anterior, por lo que un antecedente puede contribuir a distintos resultados posteriores. Los porcentajes visibles emplean redondeo por restos mayores a una decimal para sumar 100,0 %; el CSV incluye también los valores sin redondear.

Los porcentajes aparecen en una columna fija de gran tamaño y permanecen visibles con cualquier configuración de etiquetas. El clic selecciona las personas de ese cargo/grupo en el visor nominal, una tarjeta por persona. Las dos tablas inferiores permiten descargar el resumen y cada antecedente que aporta al porcentaje. Los grupos sin antecedentes indican que no hay datos y no muestran una distribución ficticia.

In [40]:
# Selección nominal y exportaciones. Solo modifica el HTML generado.
from pathlib import Path
from tempfile import NamedTemporaryFile
import os
page = OUTPUT_DIR / 'Atlas_Electoral_Costa_Rica_1953_2026.html'
source = page.read_text(encoding='utf-8')
ATLAS225_HELPERS = '/* ATLAS_V22_5_NOMINAL_HELPERS_BEGIN */\n// Una tarjeta por ID nominal; los registros relacionados se agrupan dentro de ella.\nvar atlas225ExportRegistry=new Map(),atlas225Tables=new Map(),atlas225PanelSequence=0;\nfunction atlas225Bool(v){return v===true||v===1||String(v).toLowerCase()===\'true\'}\nfunction atlas225UniqueRows(rows){const by=new Map();for(const r of rows){const k=r.id_evento||r.id_origen+\'|\'+r.id_destino+\'|\'+r.id_persona+\'|\'+r.anio_origen+\'|\'+r.anio+\'|\'+(r.tipo_doble||r.cargo||\'\');if(!by.has(k))by.set(k,r);}return [...by.values()]}\nfunction atlas225People(rows){return group(rows,\'id_persona\').map(([id_persona,g])=>({id_persona,nombre:g[0].nombre,rows:atlas225UniqueRows(g),pares:atlas225UniqueRows(g),targets:atlas225UniqueRows(g)})).sort((a,b)=>String(a.nombre).localeCompare(String(b.nombre),\'es\'))}\nfunction atlas225CaseLine(r,mode){\n if(mode===\'double\')return `<div class="career-pair"><b>${r.anio} · ${esc(r.cargos_nacionales||r.cargo_nacional)}</b><br>${esc(labelText(r.partido_nacional))} · ${atlas225Bool(r.electa_cargo_nacional)?\'Electa\':\'No electa\'}<br>Diputación · ${esc(r.provincia)} · ${esc(labelText(r.partido_legislativo))} · <b>${atlas225Bool(r.electa_diputacion)?\'Electa\':\'No electa\'}</b> <span class="mini">${r.repeticion_doble?\'Reiteración de doble postulación\':\'Primera doble postulación observada\'}</span></div>`;\n if(mode===\'antecedent\')return `<div class="career-pair"><b>Resultado analizado: ${r.anio} · ${esc(r.cargo)}</b> <span class="mini">${esc(labelText(r.partido))} · ${esc(r.provincia)} · ${esc(r.antecedente)}</span></div>`;\n return `<div class="career-pair"><span class="${atlas225Bool(r.electa_origen)?\'wintext\':\'losstext\'}">${r.anio_origen} · ${esc(r.cargo_origen)} · ${esc(r.resultado_origen)}</span> <span class="mini">${esc(labelText(r.partido_origen))} · ${esc(r.provincia_origen)}</span> → <span class="${atlas225Bool(r.electa_destino)?\'wintext\':\'losstext\'}">${r.anio} · ${esc(r.cargo_destino)} · ${esc(r.resultado_destino)}</span> <span class="mini">${esc(labelText(r.partido_destino))} · ${esc(r.provincia_destino)}</span></div>`;\n}\nfunction atlas225Pack(rows,page=0,per=48,mode=\'pair\'){\n const people=atlas225People(rows),slice=people.slice(page*per,(page+1)*per);\n const html=\'<div class="person-result-grid">\'+slice.map(p=>{\n   const cases=p.rows.slice().sort((a,b)=>Number(a.anio)-Number(b.anio)||Number(a.anio_origen||0)-Number(b.anio_origen||0));\n   let history=\'\';\n   if(mode===\'antecedent\'){\n     const maxYear=Math.max(...cases.map(r=>Number(r.anio)));\n     const hist=atlas225UniqueRows((D.resultados_candidaturas||[]).filter(r=>r.id_persona===p.id_persona&&Number(r.anio)<=maxYear));\n     history=\'<details><summary>Historia documentada hasta el resultado analizado</summary>\'+hist.sort((a,b)=>a.anio-b.anio||String(a.cargo).localeCompare(String(b.cargo),\'es\')).map(atlas18EventLine).join(\'\')+\'</details>\';\n   }\n   return `<div class="person-result" data-person-id="${esc(p.id_persona)}">${atlasPersonButton(p)}<span class="mini"><b>${cases.length} ${cases.length===1?\'caso\':\'casos\'} de esta selección</b> · una sola tarjeta por persona</span><details open><summary>Ver ${cases.length===1?\'el caso\':\'los \'+cases.length+\' casos\'} de la selección</summary>${cases.map(r=>atlas225CaseLine(r,mode)).join(\'\')}</details>${history}</div>`;\n }).join(\'\')+\'</div>\';\n return {people,html};\n}\nfunction atlas225AttachExports(root,rows,base){\n if(!root)return;let heading=root.querySelector(\'h3\');if(!heading)return;\n if(!root.id)root.id=\'atlas225_people_\'+(++atlas225PanelSequence);\n let head=heading.closest(\'.card-head\');if(!head){head=document.createElement(\'div\');head.className=\'card-head\';heading.parentNode.insertBefore(head,heading);head.appendChild(heading)}\n let tools=head.querySelector(\'.exports\');if(!tools){tools=document.createElement(\'div\');tools.className=\'exports\';head.appendChild(tools)}\n for(const kind of [\'PNG\',\'SVG\',\'CSV\']){\n   let button=[...tools.querySelectorAll(\'button\')].find(b=>b.textContent.trim()===kind);\n   if(!button){button=document.createElement(\'button\');button.type=\'button\';button.textContent=kind;tools.appendChild(button)}\n   button.disabled=kind===\'CSV\'?!rows.length:!root.querySelector(\'.person-result\');\n   button.title=kind===\'CSV\'?\'Todos los casos de la selección, agrupables por id_persona; incluye todas las páginas\':\'Personas y casos de la página visible\';\n   button.onclick=kind===\'CSV\'?()=>csv(rows,base+\'.csv\'):()=>atlasV5ExportNode(root.id,kind.toLowerCase(),base);\n }\n const mini=head.querySelector(\'.mini\');if(mini&&rows.some(r=>r.cargo_origen)){mini.textContent=state.from+\'–\'+state.to+\' · \'+state.province+\' · \'+[...new Set(rows.map(r=>r.cargo_origen+\' → \'+r.cargo_destino))].join(\' / \')}\n let help=root.querySelector(\'[data-nominal-explanation]\');if(!help){help=document.createElement(\'p\');help.className=\'chart-note\';help.dataset.nominalExplanation=\'yes\';help.textContent=\'Cada persona aparece una sola vez. Sus casos están agrupados dentro de la tarjeta. El CSV incluye todos los casos y personas de la selección, aunque haya varias páginas.\';head.insertAdjacentElement(\'afterend\',help)}\n}\nfunction atlas225NominalPanel(heading,rows,mode,key,caption){\n const people=atlas225People(rows),per=36,page=Math.min(Math.max(0,Number(local[key+\'Page\']||0)),Math.max(0,Math.ceil(people.length/per)-1));\n if(people.length>per)controls([[key+\'Page\',\'Página de personas · \'+people.length+\' personas distintas\',atlas18PageOptions(people.length,per),page]]);\n const pack=atlas225Pack(rows,page,per,mode),rootId=\'atlas225_\'+key;\n add(`<section class="card" id="${rootId}"><h3>${esc(heading)}</h3><p class="chart-note"><b>${pack.people.length} ${pack.people.length===1?\'persona distinta\':\'personas distintas\'} · ${rows.length} ${rows.length===1?\'caso\':\'casos\'}</b><br>${esc(caption||\'\')}</p>${rows.length?pack.html:\'<p class="empty">No hay personas en esta selección.</p>\'}</section>`);\n atlas225ExportRegistry.set(heading,{rows,base:\'Atlas_\'+key+\'_\'+state.from+\'_\'+state.to});\n atlas225AttachExports($(rootId),rows,\'Atlas_\'+key+\'_\'+state.from+\'_\'+state.to);\n return pack;\n}\nfunction atlas225BridgePanel(bridges,bFrom,bTo,caption,bridgeTitle){\n const defs=[[\'Mantiene partido\',r=>!r.cambia_partido],[\'Cambia partido\',r=>!!r.cambia_partido],[\'Mantiene provincia\',r=>r.provincia_origen!==\'Nacional\'&&r.provincia_destino!==\'Nacional\'&&!r.cambia_provincia],[\'Cambia provincia\',r=>r.provincia_origen!==\'Nacional\'&&r.provincia_destino!==\'Nacional\'&&!!r.cambia_provincia]];\n const scope=JSON.stringify([bFrom,bTo,caption,bridges.map(r=>[r.id_origen,r.id_destino])]);\n if(local.v225BridgeScope!==scope){local.v225BridgeScope=scope;local.v225BridgePick=\'Todos\';local.v225BridgePage=0}\n const mobil=defs.map(([grupo,fn])=>{const g=bridges.filter(fn),electas=g.filter(r=>r.resultado_destino===\'Electa\').length;return {grupo,pares:g.length,personas:new Set(g.map(r=>r.id_persona)).size,electas,no_electas:g.length-electas,porc_electas:g.length?100*electas/g.length:null,porc_no_electas:g.length?100*(g.length-electas)/g.length:null}}).filter(r=>r.pares);\n const options=[[\'Todos\',\'Todos los casos del puente\'],...mobil.flatMap(r=>[\'Electa\',\'No electa\'].map(o=>[r.grupo+\'|\'+o,r.grupo+\' · \'+o]))];\n if(!options.some(o=>o[0]===local.v225BridgePick))local.v225BridgePick=\'Todos\';\n if(mobil.length)plot(bridgeTitle,[\'Electa\',\'No electa\'].map((outcome,i)=>({type:\'bar\',name:outcome,y:mobil.map(r=>r.grupo),x:mobil.map(r=>i?r.porc_no_electas:r.porc_electas),orientation:\'h\',marker:{color:i?\'#b98e60\':\'#168278\'},customdata:mobil.map(r=>[r.grupo,outcome,i?r.no_electas:r.electas,r.pares]),text:mobil.map(r=>`${num(i?r.porc_no_electas:r.porc_electas,1)} % · ${i?r.no_electas:r.electas}`),textposition:\'inside\',hovertemplate:\'%{customdata[0]} · %{customdata[1]}<br>%{customdata[2]} de %{customdata[3]} pares · %{x:.1f}%<extra></extra>\'})),mobil,{height:380,subtitle:caption,xTitle:\'Porcentaje de pares de candidaturas dentro de cada grupo (%)\',layout:{barmode:\'stack\',xaxis:{range:[0,100]}},note:\'Pulse un segmento para ver sus personas, una tarjeta por persona con todos sus pares de la selección. El resultado corresponde a la candidatura posterior exacta. Cada barra suma 100% de los pares de su grupo; una persona puede aportar varios pares.\',onClick:evt=>{const d=evt.points?.[0]?.customdata;if(Array.isArray(d)){local.v225BridgePick=d[0]+\'|\'+d[1];local.v225BridgePage=0;render()}}});\n table(mobil,[\'grupo\',\'pares\',\'personas\',\'electas\',\'no_electas\',\'porc_electas\',\'porc_no_electas\'],\'Resultados de las rutas filtradas por movilidad\');\n controls([[\'v225BridgePick\',\'Personas del segmento seleccionado\',options,local.v225BridgePick]]);\n let selected=bridges;\n if(local.v225BridgePick!==\'Todos\'){const [g,o]=local.v225BridgePick.split(\'|\');const def=defs.find(d=>d[0]===g);selected=bridges.filter(r=>def[1](r)&&r.resultado_destino===o)}\n const people=atlas225People(selected);\n add(`<div class="selected-detail"><b>${esc(local.v225BridgePick===\'Todos\'?\'Todos los casos del puente\':local.v225BridgePick.replace(\'|\',\' · \'))}</b><br>${people.length} personas distintas · ${selected.length} pares de candidaturas</div>`);\n atlas225NominalPanel(\'Quiénes forman este puente de carrera\',selected,\'pair\',\'v225Bridge\',caption);\n table(selected,[\'id_persona\',\'nombre\',\'anio_origen\',\'cargo_origen\',\'resultado_origen\',\'anio\',\'cargo_destino\',\'resultado_destino\',\'partido_origen\',\'partido_destino\',\'provincia_origen\',\'provincia_destino\',\'cambia_partido\',\'cambia_provincia\',\'id_origen\',\'id_destino\'],\'Pares de candidaturas que forman el puente\');\n}\nfunction atlas225DoubleScope(rows){return JSON.stringify(rows.map(r=>[r.id_persona,r.anio]))}\nfunction atlas225SelectDouble(pick,rows){local.v225DoubleScope=atlas225DoubleScope(rows);local.v225DoublePick=pick;local.v225DoublePage=0;render()}\nfunction atlas225DoubleSelection(rows){\n if(local.v225DoubleScope!==atlas225DoubleScope(rows)){local.v225DoubleScope=atlas225DoubleScope(rows);local.v225DoublePick=null;local.v225DoublePage=0}\n const p=local.v225DoublePick;if(!p)return {rows,caption:\'Todas las dobles postulaciones de los filtros activos\'};\n let out=rows;\n if(p.year!==undefined)out=out.filter(r=>r.anio===Number(p.year));\n if(p.province)out=out.filter(r=>r.provincia===p.province);\n if(p.type)out=out.filter(r=>r.tipo_doble===p.type);\n if(p.repeat)out=out.filter(r=>r.repeticion_doble);\n if(p.result===\'Solo diputación electa\')out=out.filter(r=>r.electa_diputacion&&!r.electa_cargo_nacional);\n if(p.result===\'Solo cargo nacional electo\')out=out.filter(r=>r.electa_cargo_nacional&&!r.electa_diputacion);\n if(p.result===\'Ambos cargos electos\')out=out.filter(r=>r.electa_diputacion&&r.electa_cargo_nacional);\n if(p.result===\'Ninguna candidatura electa\')out=out.filter(r=>!r.electa_diputacion&&!r.electa_cargo_nacional);\n if(p.won)out=out.filter(r=>r.electa_diputacion);\n return {rows:out,caption:p.label};\n}\nfunction atlas225DoublePanel(rows){\n const s=atlas225DoubleSelection(rows);\n add(`<div class="selected-detail"><b>Selección de doble postulación:</b> ${esc(s.caption)}<br>Pulse un dato de las gráficas para cambiar las personas mostradas. <button class="button" id="atlas225ClearDouble">Ver todos los casos del filtro</button></div>`);\n if($(\'atlas225ClearDouble\'))$(\'atlas225ClearDouble\').onclick=()=>{local.v225DoublePick=null;local.v225DoublePage=0;render()};\n atlas225NominalPanel(\'Personas de la doble postulación seleccionada\',s.rows,\'double\',\'v225Double\',s.caption);\n}\nfunction atlas225DoubleClick(title,p,rows){\n if(!p)return;\n if(title.includes(\'evolución histórica\')){const year=Number(p.x);if(Number.isFinite(year))atlas225SelectDouble({year,repeat:p.curveNumber===1,label:(p.curveNumber===1?\'Reiteraciones\':\'Dobles postulaciones\')+\' · \'+year},rows);return}\n if(title===\'Composición de la doble postulación por cargo nacional\'){atlas225SelectDouble({type:String(p.y),label:String(p.y)},rows);return}\n if(title.startsWith(\'Doble postulación por provincia y elección\')){const pick={label:String(p.y)+\' · \'+String(p.x)};if(P.includes(p.y))pick.province=p.y;if(p.x!==\'Total histórico\'&&Number.isFinite(Number(p.x)))pick.year=Number(p.x);atlas225SelectDouble(pick,rows);return}\n if(title===\'Concentración territorial de las dobles postulaciones\'){atlas225SelectDouble({province:p.y,label:\'Provincia legislativa: \'+p.y},rows);return}\n if(title.startsWith(\'Éxito legislativo entre las dobles postulaciones\')){atlas225SelectDouble({province:p.y,won:true,label:\'Diputaciones electas · \'+p.y+\' (numerador de la tasa)\'},rows);return}\n if(title===\'Resultados de las postulaciones dobles · ambos cargos\')atlas225SelectDouble({result:p.y,label:p.y},rows);\n}\n(function atlas225Install(){\n if(window.ATLAS_V22_5_NOMINAL_OK)return;\n const style=document.createElement(\'style\');style.id=\'atlas225-nominal-style\';style.textContent=\'.person-result .career-pair>.mini{display:block;margin-top:3px}.person-result details{margin-top:8px}.person-result details summary{cursor:pointer;line-height:1.45}.person-result .career-pair{line-height:1.5;margin-top:7px}\';document.head.appendChild(style);\n atlas18FlowCards=(rows,page=0,per=48)=>atlas225Pack(rows,page,per,\'pair\');\n atlas18AntecedentCards=(rows,page=0,per=48)=>atlas225Pack(rows,page,per,\'antecedent\');\n atlas17CareerCards=people=>atlas225Pack(people.flatMap(p=>p.pares||[]),0,people.length||1,\'pair\').html;\n const basePlot=plot;\n plot=function(t,traces,rows,opts={}){\n   const isDouble=String(t).startsWith(\'Doble postulación · fórmula\')||String(t).startsWith(\'Doble postulación por provincia\')||[\'Composición de la doble postulación por cargo nacional\',\'Concentración territorial de las dobles postulaciones\',\'Éxito legislativo entre las dobles postulaciones · por provincia\',\'Resultados de las postulaciones dobles · ambos cargos\'].includes(t);\n   if(isDouble){const active=(window.atlas225ActiveDoubles||[]).slice(),priorClick=opts.onClick;opts={...opts,onClick:evt=>{if(priorClick)priorClick(evt);atlas225DoubleClick(t,evt.points?.[0],active)}};\n     if(String(t).startsWith(\'Doble postulación por provincia\'))opts.layout={...opts.layout,annotations:(opts.layout?.annotations||[]).map(a=>({...a,captureevents:true}))};\n     if(String(t).startsWith(\'Éxito legislativo\'))opts.note=(opts.note||\'\')+\' Al pulsar se muestran los casos electos del numerador; las demás candidaturas están en el gráfico de resultados y la tabla.\';\n   }\n   return basePlot(t,traces,rows,opts);\n };\n const baseTable=table;\n table=function(rows,columns,heading=\'Datos de la selección\'){atlas225Tables.set(heading,{rows,columns});return baseTable(rows,columns,heading)};\n function finish(){\n   const targets=[[\'Personas del flujo seleccionado\',\'Registros que sustentan el flujo seleccionado\'],[\'Personas del antecedente seleccionado\',\'Nombres y antecedentes que forman los porcentajes\'],[\'Personas en el antecedente seleccionado\',\'Casos y nombres que forman los porcentajes de antecedentes\']];\n   const cards=[...document.querySelectorAll(\'#content .card\')];\n   for(const [heading,tableHeading] of targets){const item=atlas225Tables.get(tableHeading);if(item)atlas225ExportRegistry.set(heading,{rows:item.rows,base:\'Atlas_\'+heading.replaceAll(\' \',\'_\')+\'_\'+state.from+\'_\'+state.to})}\n   for(const card of cards){const heading=card.querySelector(\'h3\')?.textContent.trim();const info=atlas225ExportRegistry.get(heading);if(info)atlas225AttachExports(card,info.rows,info.base)}\n   // El contrato común de plot proporciona CSV a todas las figuras, incluso sin datos.\n   for(const chart of document.querySelectorAll(\'#content .chart[id]\')){\n     const article=chart.closest(\'article.card\');if(!article)continue;\n     let tools=article.querySelector(\'.exports\');if(!tools)continue;\n     let button=[...tools.querySelectorAll(\'button\')].find(b=>b.textContent.trim()===\'CSV\');\n     if(!button){button=document.createElement(\'button\');button.textContent=\'CSV\';button.type=\'button\';tools.appendChild(button)}\n     const rows=chartRows[chart.id]||[];button.disabled=!rows.length;button.title=rows.length?\'Datos completos de esta gráfica con los filtros activos\':\'Sin datos para exportar\';button.onclick=()=>csv(rows,(chart.atlasFilename||\'Atlas_\'+state.tab+\'_\'+chart.id)+\'.csv\');\n     if(chart.atlasTitle?.startsWith(\'Doble postulación por provincia\')&&typeof chart.on===\'function\'&&!chart.atlas225AnnotationClick){chart.atlas225AnnotationClick=true;chart.on(\'plotly_clickannotation\',()=>atlas225SelectDouble({label:\'Total país y periodo\'},(window.atlas225ActiveDoubles||[]).slice()))}\n   }\n }\n const baseRender=render;\n render=async function(...args){atlas225ExportRegistry=new Map();atlas225Tables=new Map();atlas225PanelSequence=0;const out=await baseRender.apply(this,args);finish();return out};\n window.ATLAS_V22_5_NOMINAL_OK=true;\n})();\nfunction atlas227AnteScope(){return JSON.stringify([local.anteCargo||\'Presidencia\',local.anteUnit||\'Personas distintas · primera elección en el cargo\',state.from,state.to,state.province,local.v222Anterior||\'Ambos grupos\'])}\nfunction atlas227History(target){return (D.resultados_candidaturas||[]).filter(r=>r.id_persona===target.id_persona&&Number(r.anio)<Number(target.anio))}\nfunction atlas227HasDouble(history,role){const years=[...new Set(history.map(r=>Number(r.anio)))];return years.some(year=>{const same=history.filter(r=>Number(r.anio)===year);return same.some(r=>r.cargo===\'Diputación\'&&r.tipo===\'Propietaria\')&&same.some(r=>r.cargo===role)})}\nfunction atlas227Matches(target,kind,value){const history=atlas227History(target);if(kind===\'double\')return target.antecedente===\'Había competido antes, pero aún no había sido electa\'&&atlas227HasDouble(history,value);const eligible=target.antecedente===\'Ya había resultado electa antes\'?history.filter(r=>atlas225Bool(r.electa)):history;return eligible.some(r=>r.cargo===value)}\nfunction atlas227FilterAntecedentRows(rows){const pick=local.v227AnteCell;if(!pick)return rows;if(pick.scope!==atlas227AnteScope()||pick.group!==local.anteCategory){delete local.v227AnteCell;return rows}return rows.filter(r=>atlas227Matches(r,pick.kind,pick.value))}\n(function atlas227Install(){\n if(window.ATLAS_V22_7_ANTECEDENT_CLICKS)return;\n const previousPlot=plot;\n plot=function(title,traces,rows,opts={}){\n  if(String(title).startsWith(\'Cómo llegaban quienes resultaron electos en \')){\n   const before=opts.onClick;opts={...opts,onClick:evt=>{if(before)before(evt);const p=evt.points?.[0],name=p?.data?.name||traces[p?.curveNumber]?.name;\n    if(name){local.anteCategory=name;delete local.v227AnteCell;local.antepage=0;render()}}};\n  }\n  return previousPlot(title,traces,rows,opts);\n };\n const previousRender=render;\n render=async function(...args){const out=await previousRender.apply(this,args);\n   const card=[...document.querySelectorAll(\'#content .card\')].find(c=>c.querySelector(\'h3\')?.textContent.trim()===\'Personas del antecedente seleccionado\');\n   if(card){const scope=card.querySelector(\'.card-head .mini\');if(scope)scope.textContent=state.from+\'–\'+state.to+\' · \'+state.province+\' · Cargo electo: \'+(local.anteCargo||\'Presidencia\')+\' · \'+(local.anteUnit||\'Personas distintas · primera elección en el cargo\');const pick=local.v227AnteCell,info=document.createElement(\'div\');info.className=\'chart-note\';info.dataset.antecedentSelection=\'yes\';info.innerHTML=\'<b>Selección:</b> \'+esc(pick?pick.label:(local.anteCategory===\'Todos\'?\'Todas las categorías\':local.anteCategory||\'Todas las categorías\'))+\' <button class="button" id="atlas227ClearAnte">Ver todas las personas</button>\';card.querySelector(\'.card-head\')?.insertAdjacentElement(\'afterend\',info);const button=document.getElementById(\'atlas227ClearAnte\');if(button)button.onclick=()=>{local.anteCategory=\'Todos\';delete local.v227AnteCell;local.antepage=0;render()};\n   }\n   return out;\n };\n window.ATLAS_V22_7_ANTECEDENT_CLICKS=true;\n})();\n\n\n/* ATLAS_V22_8_IDENTITY_AUDIT */\n(function(){\n if(window.ATLAS_V22_8_IDENTITY_AUDIT_OK||!views.datos)return;\n const original=views.datos;\n views.datos=function(...args){const result=original.apply(this,args);const rows=D.conciliacion_nombres||[];\n add(\'<div class="extension-heading">CONCILIACIÓN NOMINAL · VARIANTES DE NOMBRES</div>\');\n note(\'Se conserva la grafía de fuente. La coincidencia de un nombre y dos apellidos se concilia únicamente con una forma completa compatible y sin contradicciones registradas. Es una inferencia nominal, no una validación civil exhaustiva.\');\n kpis([[\'Variantes conciliadas\',rows.filter(r=>r.estado===\'Conciliado nominalmente\').length,\'Una forma abreviada vinculada a una forma completa\'],[\'Casos pendientes\',rows.filter(r=>r.estado!==\'Conciliado nominalmente\').length,\'No se fusionan automáticamente\']]);\n table(rows,[\'nombre_abreviado_normalizado\',\'nombre_completo_normalizado\',\'primer_apellido\',\'segundo_apellido\',\'estado\',\'regla\',\'metodo_apellidos\',\'candidatos_completos\',\'motivo_no_union\',\'filas_nombre_abreviado\',\'filas_nombre_completo\',\'nivel_validacion\'],\'Auditoría de conciliación de nombres · decisiones y pendientes\');\n return result;};\n window.ATLAS_V22_8_IDENTITY_AUDIT_OK=true;\n})();\n\n/* ATLAS_V22_5_NOMINAL_HELPERS_END */\n'
BRIDGE_START = " const bridgeTitle=(bFrom==='Todos'&&bTo==='Todos'&&bFromResult==='Todos'&&bToResult==='Todos')?"
BRIDGE_END = ' add(\'<div class="atlas17-subhead">Antecedentes de quienes sí resultaron electos</div>\');'
BRIDGE_REPLACEMENT = " // ATLAS_V22_5_BRIDGE_PANEL: solo resultados exactos y personas únicas.\n const bridgeTitle=(bFrom==='Todos'&&bTo==='Todos'&&bFromResult==='Todos'&&bToResult==='Todos')?\n  'Resultados de candidaturas posteriores · todas las trayectorias':\n  `Resultados posteriores · ${fromDesc} → ${toDesc}`;\n atlas225BridgePanel(bridges,bFrom,bTo,bridgeCaption,bridgeTitle);\n\n"
if '// ATLAS_V22_5_BRIDGE_PANEL:' not in source:
    assert source.count(BRIDGE_START) == 1, 'No se identificó el bloque de puente compatible; se conserva el HTML.'
    begin = source.index(BRIDGE_START)
    end = source.index(BRIDGE_END, begin)
    source = source[:begin] + BRIDGE_REPLACEMENT + source[end:]
ANTE_ROWS_OLD="const anteRows=analytic.filter(r=>anteSel==='Todos'||r.antecedente===anteSel),selectedN=anteRows.length"
ANTE_ROWS_NEW="const anteRows=atlas227FilterAntecedentRows(analytic.filter(r=>anteSel==='Todos'||r.antecedente===anteSel)),selectedN=anteRows.length"
if ANTE_ROWS_NEW not in source:
    assert source.count(ANTE_ROWS_OLD)==1, 'No se identificó el visor de antecedentes compatible.'
    source=source.replace(ANTE_ROWS_OLD,ANTE_ROWS_NEW,1)
start_marker='/* ATLAS_V22_5_NOMINAL_HELPERS_BEGIN */'
end_marker='/* ATLAS_V22_5_NOMINAL_HELPERS_END */'
if start_marker in source:
    begin=source.index(start_marker); end=source.index(end_marker,begin)+len(end_marker)
    source=source[:begin]+ATLAS225_HELPERS+source[end:]
else:
    hook="window.addEventListener('resize',()=>document.querySelectorAll('.js-plotly-plot').forEach(el=>Plotly.Plots.resize(el)));setup();"
    assert source.count(hook)==1, 'No se identificó el punto de integración estable.'
    source=source.replace(hook,ATLAS225_HELPERS+'\n'+hook,1)
assert source.count(start_marker)==1
assert source.count('// ATLAS_V22_5_BRIDGE_PANEL:')==1
with NamedTemporaryFile('w',encoding='utf-8',dir=page.parent,prefix='.atlas_v225_',suffix='.html',delete=False) as stream:
    stream.write(source); temporary=Path(stream.name)
os.replace(temporary,page)
print(': cada persona aparece una vez; sus casos quedan agrupados.')
print('Resultados posteriores y doble postulación: clic para seleccionar personas; CSV con todos los casos de la selección.')
print('Solo se retiró la gráfica «Trayectorias: resultados…» del puente; se conserva la auditoría nominal y el resto del Atlas.')


: cada persona aparece una vez; sus casos quedan agrupados.
Resultados posteriores y doble postulación: clic para seleccionar personas; CSV con todos los casos de la selección.
Solo se retiró la gráfica «Trayectorias: resultados…» del puente; se conserva la auditoría nominal y el resto del Atlas.


## Matrices públicas por elección

Se conservan las dos matrices de tres cargos, los ceros, los totales, el visor nominal y el CSV. Cada columna representa una elección; no se aplica agrupación ni bandas de investigación. Se mantiene el antecedente de 1990, su marca roja y los hitos normativos documentados, sin atribuir ilegalidad.

In [41]:
# Extensión final autocontenida: solo actualiza el HTML generado.
from pathlib import Path
from tempfile import NamedTemporaryFile
import os
page=OUTPUT_DIR / 'Atlas_Electoral_Costa_Rica_1953_2026.html'
source=page.read_text(encoding='utf-8')
EXTENSION_V231 = '/* ATLAS_V23_1_1990_NORMATIVA_Y_MATRICES */\n(function(){\n if(window.ATLAS231_OK)return;\n const RED=\'#b44343\',PURPLE=\'#70588c\';\n const roles=[\'Presidencia\',\'Primera Vicepresidencia\',\'Segunda Vicepresidencia\'];\n const source96=\'https://ride.tse.go.cr/xmlui/handle/123456789/10604\';\n const source09=\'https://www.tse.go.cr/juris/electorales/4226-E8-2009.htm\';\n const legal=\'1996: art. IV del acta TSE 11041 (5 de diciembre) y Ley 7653, que adicionó el art. 74 bis. En 2009, resolución 4226-E8-2009. El caso de 1990 es anterior a esos hitos; su inclusión documental no supone una declaración de legalidad o ilegalidad.\';\n const pre=r=>Number(r.anio)<1996;\n const oldPlot=plot;\n plot=function(title,traces,rows,opts={}){\n   const active=window.atlas225ActiveDoubles||[],earlier=active.filter(pre);\n   if(String(title).startsWith(\'Doble postulación · fórmula\')&&String(title).includes(\'evolución histórica\')){\n     const x=traces[0]?.x||[],has1996=x.some(v=>Number(v)<=1994)&&x.some(v=>Number(v)>=1998);\n     const annotations=[...(opts.layout?.annotations||[])],shapes=[...(opts.layout?.shapes||[])];\n     if(has1996){shapes.push({type:\'line\',xref:\'x\',yref:\'paper\',x0:1996,x1:1996,y0:0,y1:1,line:{color:PURPLE,width:2,dash:\'dot\'}});annotations.push({xref:\'x\',yref:\'paper\',x:1996,y:1.10,showarrow:false,text:\'<b>1996 · Ley 7653 / art. 74 bis</b>\',font:{color:PURPLE,size:12},bgcolor:\'#fff\'})}\n     if(earlier.length){const year=1990,value=active.filter(r=>Number(r.anio)===year).length;traces=[...traces,{type:\'scatter\',mode:\'markers\',x:[year],y:[value],name:\'1990 · antecedente anterior a la reforma\',marker:{size:15,color:\'#fff\',line:{color:RED,width:3},symbol:\'circle\'},hovertemplate:\'1990 · antecedente anterior a la reforma de 1996<br>Segunda Vicepresidencia: Independiente<br>Diputación: Alianza Nacional Cristiana · Limón<br>Situación jurídica histórica pendiente de expediente<extra></extra>\'}];annotations.push({xref:\'x\',yref:\'y\',x:1990,y:value,ax:-65,ay:-70,showarrow:true,arrowcolor:RED,text:\'<b>1990 · dos partidos</b><br>Anterior a la reforma\',font:{size:12,color:RED},bgcolor:\'#fff\',bordercolor:RED,borderpad:4})}\n     opts={...opts,note:(opts.note||\'\')+\' \'+legal,layout:{...opts.layout,annotations,shapes,margin:{l:90,r:60,t:75,b:75}}};\n   }\n   const isGeo=String(title).startsWith(\'Doble postulación por provincia y elección\');\n   if(earlier.length&&(isGeo||title===\'Composición de la doble postulación por cargo nacional\'||title===\'Concentración territorial de las dobles postulaciones\')){\n     const shapes=[...(opts.layout?.shapes||[])];\n     if(isGeo){const x=traces[0]?.x||[],y=traces[0]?.y||[],xi=x.map(String).indexOf(\'1990\'),yi=y.indexOf(\'Limón\');if(xi>=0&&yi>=0)shapes.push({type:\'rect\',xref:\'x\',yref:\'y\',x0:xi-.46,x1:xi+.46,y0:yi-.46,y1:yi+.46,line:{color:RED,width:3},fillcolor:\'rgba(0,0,0,0)\',layer:\'above\'})}\n     else {const t=traces[0],label=title.startsWith(\'Composición\')?\'Segunda Vicepresidencia + Diputación\':\'Limón\',yi=t?.y?.indexOf(label),value=t?.x?.[yi];if(yi>=0&&Number(value)>0)shapes.push({type:\'rect\',xref:\'x\',yref:\'y\',x0:Number(value)-1,x1:Number(value),y0:yi-.33,y1:yi+.33,line:{color:RED,width:2},fillcolor:\'rgba(180,67,67,.17)\',layer:\'above\'})}\n     opts={...opts,layout:{...opts.layout,shapes},note:(opts.note||\'\')+\' Rojo: un episodio de 1990, anterior a la reforma de 1996, incluido por transparencia documental. No significa ilegalidad acreditada. \'+legal};\n   }\n   return oldPlot(title,traces,rows,opts);\n };\n const priorView=views.trayectorias;\n views.trayectorias=function(...args){\n   const result=priorView.apply(this,args);\n   const root=$(\'content\'),existingNodes=new Set(root.children);\n   const all=window.atlas231AllDoubles||[];\n   const province=local.dobProv&&local.dobProv!==\'Todas las provincias\'?local.dobProv:\'Todas las provincias\';\n   const records=all.filter(r=>r.anio>=state.from&&r.anio<=state.to&&(province===\'Todas las provincias\'||r.provincia===province));\n   add(\'<div class="extension-heading">DOBLE POSTULACIÓN · COMPARACIÓN ENTRE CARGOS Y ELECCIONES</div>\');\n   note(\'Estas dos matrices siempre muestran los tres tipos. Solo utilizan el periodo y la provincia legislativa; ignoran los filtros de tipo, resultado, reiteración y continuidad de las gráficas anteriores. El éxito cuenta diputaciones electas, no una tasa ni un cargo ejecutivo ganado.\');\n   const buckets=yearsVisible().map(y=>({label:String(y),from:y,to:y}));\n   const labels=[...buckets.map(p=>p.label),\'Total histórico\'];\n   const scope=JSON.stringify([state.from,state.to,province]);\n   if(local.v231MatrixScope!==scope){local.v231MatrixScope=scope;local.v231MatrixPick=null}\n   const roleMatch=(r,role)=>(r.cargos_nacionales||r.cargo_nacional||\'\').split(\' | \').includes(role);\n   const title=(success)=>success?\'Dobles postulaciones · diputaciones electas por cargo y elección\':\'Dobles postulaciones · todos los casos por cargo y elección\';\n   window.atlas231MatrixAudits=[];\n   const allMax=Math.max(1,...roles.map(role=>records.filter(r=>roleMatch(r,role)).length));\n   for(const success of [false,true]){\n     const selected=success?records.filter(r=>r.electa_diputacion):records;\n     const audit=[];\n     const z=roles.map(role=>[...buckets.map(p=>selected.filter(r=>roleMatch(r,role)&&r.anio>=p.from&&r.anio<=p.to).length),selected.filter(r=>roleMatch(r,role)).length]);\n     const custom=roles.map((role,i)=>[...buckets,{label:\'Total histórico\',from:state.from,to:state.to}].map((p,j)=>{audit.push({cargo_nacional:role,eleccion:p.label,desde:Math.max(p.from,state.from),hasta:Math.min(p.to,state.to),casos:z[i][j],solo_diputaciones_electas:success,provincia:province});return[role,p.from,p.to,success,p.label]}));\n     const shapes=[],annotations=[];\n     const a=buckets.findIndex(b=>b.from===1994),b=buckets.findIndex(b=>b.from===1998);\n     if(a>=0&&b>=0)shapes.push({type:\'line\',xref:\'x\',yref:\'paper\',x0:(a+b)/2,x1:(a+b)/2,y0:0,y1:1,line:{color:PURPLE,width:2,dash:\'dot\'}});\n     const col=buckets.findIndex(b=>b.from===1990);\n     if(!success&&col>=0&&records.some(pre))shapes.push({type:\'rect\',xref:\'x\',yref:\'y\',x0:col-.45,x1:col+.45,y0:1.55,y1:2.45,fillcolor:\'rgba(0,0,0,0)\',line:{color:RED,width:3},layer:\'above\'});\n     roles.forEach((role,i)=>labels.forEach((label,j)=>annotations.push({xref:\'x\',yref:\'y\',x:j,y:role,showarrow:false,text:\'<b>\'+z[i][j]+\'</b>\',font:{size:15,color:z[i][j]>allMax*.55?\'#fff\':\'#153e55\'}})));\n     plot(title(success),[{type:\'heatmap\',x:labels,y:roles,z,customdata:custom,zmin:0,zmax:allMax,colorscale:[[0,\'#f5f8fc\'],[.2,\'#c4dced\'],[.5,\'#5b9dc7\'],[1,\'#185181\']],text:z.map(row=>row.map(String)),texttemplate:\' \',xgap:2,ygap:3,colorbar:{title:{text:\'Casos\'},thickness:12},hovertemplate:\'%{y}<br>%{x}: %{z} casos<extra></extra>\'}],audit,{height:385,subtitle:`${state.from}–${state.to} · ${province} · Elecciones individuales · tres tipos siempre visibles`,layout:{xaxis:{type:\'category\',categoryorder:\'array\',categoryarray:labels,tickangle:-45,range:[-.5,labels.length-.5],autorange:false},yaxis:{type:\'category\',autorange:false,range:[2.5,-.5],automargin:true,tickfont:{size:13}},margin:{l:210,r:90,t:35,b:100},shapes,annotations},note:\'Casos persona–elección por modalidad. Cada celda incluye ceros; la última columna suma el intervalo mostrado. Ambas matrices usan la misma escala de conteos, incluidos sus totales. Pulse una celda para ver personas únicas con sus casos. \'+legal,onClick:evt=>{const d=evt.points?.[0]?.customdata;if(!Array.isArray(d))return;local.v231MatrixPick={role:d[0],from:d[1],to:d[2],success:d[3],label:d[0]+\' · \'+d[4]+(d[3]?\' · Diputación electa\':\' · Todos los casos\')};render()}});\n     window.atlas231MatrixAudits.push({success,audit,z});\n   }\n   const pick=local.v231MatrixPick;\n   const selected=pick?records.filter(r=>roleMatch(r,pick.role)&&r.anio>=pick.from&&r.anio<=pick.to&&(!pick.success||r.electa_diputacion)):records;\n   atlas225NominalPanel(\'Personas de las matrices por cargo\',selected,\'double\',\'v231Matrix\',pick?pick.label:\'Todos los casos de estas matrices; una tarjeta por persona\');\n   add(\'<div class="extension-heading">DOBLE POSTULACIÓN · PARTIDOS DE AMBAS CANDIDATURAS</div>\');\n   const diff=records.filter(r=>!r.mismo_partido),same=records.filter(r=>r.mismo_partido);\n   kpis([[\'Mismo partido\',same.length,\'Episodios en la selección de las matrices\'],[\'Partidos distintos\',diff.length,\'Episodios en la selección de las matrices\'],[\'Personas con partidos distintos\',new Set(diff.map(r=>r.id_persona)).size,\'Identidades nominales conciliadas\'],[\'Anterior a la reforma de 1996\',records.filter(pre).length,\'Se conserva en el total observado\']]);\n   note(\'<b>Hitos normativos:</b> <a href="\'+source96+\'" target="_blank" rel="noopener">Acta TSE 11041, art. IV (1996)</a> y Ley 7653 / art. 74 bis; <a href="\'+source09+\'" target="_blank" rel="noopener">resolución 4226-E8-2009</a>. No se ha acreditado una declaración de ilegalidad para el episodio de 1990. El rojo distingue su anterioridad y singularidad partidaria, no un fallo jurídico.\');\n   table(diff.map(r=>({...r,anterior_reforma_1996:pre(r),calificacion_juridica:\'No determinada para el caso histórico\'})),[\'id_persona\',\'nombre\',\'anio\',\'cargo_nacional\',\'provincia\',\'partido_nacional\',\'partido_legislativo\',\'electa_cargo_nacional\',\'electa_diputacion\',\'anterior_reforma_1996\',\'calificacion_juridica\'],\'Auditoría de dobles postulaciones con partidos distintos\');\n   window.atlas231PartyAudit={records,same,diff};\n   // Conservar los nodos y eventos; agrupar el bloque dentro de la sección 11.\n   const anchor=[...root.querySelectorAll(\':scope > .card\')].find(n=>n.querySelector(\'h3\')?.textContent===\'Intentos hasta la primera elección observada\');\n   if(anchor)for(const node of [...root.children].filter(n=>!existingNodes.has(n)))root.insertBefore(node,anchor);\n   return result;\n };\n window.ATLAS231_OK=true;\n})();\n'
start='/* ATLAS_V23_1_1990_NORMATIVA_Y_MATRICES */'
end='/* ATLAS_V23_1_END */'
block=EXTENSION_V231+'\n'+end
if start in source:
    a=source.index(start);b=source.index(end,a)+len(end);source=source[:a]+block+source[b:]
else:
    hook="window.addEventListener('resize',()=>document.querySelectorAll('.js-plotly-plot').forEach(el=>Plotly.Plots.resize(el)));setup();"
    assert source.count(hook)==1, 'No se encontró el punto de integración del HTML.'
    source=source.replace(hook,block+'\n'+hook,1)
with NamedTemporaryFile('w',encoding='utf-8',dir=page.parent,suffix='.html',delete=False) as f:
    f.write(source);temporary=Path(f.name)
os.replace(temporary,page)
print(': 1990 documentado, hitos de 1996, dos matrices de tres cargos y auditoría partidaria.')


: 1990 documentado, hitos de 1996, dos matrices de tres cargos y auditoría partidaria.


## Ubicación del bloque de diputaciones

La composición de diputaciones por participación de su partido pasa a «Partidos y territorio», después de la oferta partidaria por arena. No se introduce la periodización de cuatro épocas en el HTML público. Esta modificación de ubicación todavía requiere ejecución e inspección visual con las ocho bases y los recursos del paquete.


In [42]:
# Paso 89 protegido.
# La integración se ejecuta de forma indivisible al final de este notebook.
print("Paso de integración protegido; ninguna sección se elimina anticipadamente.")


Paso de integración protegido; ninguna sección se elimina anticipadamente.


## Matrices y exploración territorial auditables

El eje horizontal representa las elecciones de 1953 a 2026. La arena electoral distingue solo legislativa, presidencial y legislativa, y solo presidencial. La cobertura distingue listas uniprovinciales, multiprovinciales y nacionales.

En las matrices categóricas, cada porcentaje usa la **suma de las filas de su misma columna**. Los totales históricos suman cantidades; los promedios son medias aritméticas de esas cantidades. Para provincias, la referencia es Costa Rica porque la presencia de un partido en varias provincias no constituye varias identidades. En partidos se puede explorar la excepción de participación por provincia: un color ocre señala cuántos partidos menos participaron que el máximo de ese año. Candidaturas y escaños preservan la matriz de conteos y porcentajes, sin comparaciones artificiales contra una mediana.

Los destacados de máximos y mínimos se calculan a partir de porcentajes, con restricciones para no señalar ceros, series planas ni empates masivos. El visor de líneas permite Porcentajes, Absolutos o Ambos cuando solo hay una línea; con varias líneas solo se puede elegir Porcentajes o Absolutos. Las cohortes históricas son excluyentes, y se auditan las sumas por cobertura. Todas las casillas permiten consultar sus registros nominales y exportar resultados.


In [43]:
# Paso 91 protegido.
# La integración se ejecuta de forma indivisible al final de este notebook.
print("Paso de integración protegido; ninguna sección se elimina anticipadamente.")


Paso de integración protegido; ninguna sección se elimina anticipadamente.


## Consulta nominal en sexo y representación

La consulta nominal se integra en las gráficas originales de sexo y representación, conservando los hitos normativos existentes. Se evita duplicar las gráficas. La selección respeta la población, el año, el sexo y los filtros activos.


In [44]:
# Paso 93 protegido.
# La integración se ejecuta de forma indivisible al final de este notebook.
print("Paso de integración protegido; ninguna sección se elimina anticipadamente.")


Paso de integración protegido; ninguna sección se elimina anticipadamente.


## Comparación nacional de abstencionismo

Se mantienen los rankings provinciales legislativo y presidencial con una fila adicional **Costa Rica · total** que clasifica las elecciones por su tasa nacional oficial, sin promediar provincias. «Global» clasifica los promedios de las tasas provinciales entre provincias; la esquina inferior derecha queda sin etiqueta, valor ni interacción.

La escala de color distingue siete posiciones con contraste accesible. Todas las figuras del Atlas conservan las funciones de exportación.


In [45]:
# Paso 95 protegido.
# La integración se ejecuta de forma indivisible al final de este notebook.
print("Paso de integración protegido; ninguna sección se elimina anticipadamente.")


Paso de integración protegido; ninguna sección se elimina anticipadamente.


In [46]:
# Paso 96 protegido.
# La integración se ejecuta de forma indivisible al final de este notebook.
print("Paso de integración protegido; ninguna sección se elimina anticipadamente.")


Paso de integración protegido; ninguna sección se elimina anticipadamente.


## Sección 07 · escrutinio provincial interactivo

Elección, provincia y arena permiten explorar votos, porcentajes, diputaciones y personas. Pulsar una provincia en el mapa actualiza la selección y el resto del panel. El código se incorpora junto con la sección 08 solamente después de pasar las validaciones.


In [47]:
# Ensamblado ATÓMICO de las secciones 07 y 08; una sola publicación.
from pathlib import Path
from tempfile import TemporaryDirectory, NamedTemporaryFile
import gzip, base64, json, shutil, os

V2320_PASOS = {'89': 'H4sIAAAAAAACA21Ty27bMBC86ysWyEFS4sqNgxZtAh1cRwEMxA/4EbQoAoKWVjYBilRJKo/PybnoKYceeot/rEupaYs2PBBDDnc53B0ewNXgJDk+hqfvsEAnDDeQfRxdrpfjq+Ekm64ykNyCM2ihFNvG0IIrR4hAgZDrqtZW5GL/qJLgAJZ+S1k0NxycLjhFg8W85UEbsRWKSx8458aJQlu4B4fGCEekTiCTUO0fi0ZqSqY3Vkt0GnZ8s//GQShndNHkFNeqIr1fGnSEnr5agQ6hNvpGqFzQVh8axcHfLuzTjyQoja6g5m4nxQYEqTaORLhdRzis6lJIfGamvMJihX7Bzf0FMcEvRtug5ltMZ+vVfL1i5+NFPxw6ksMyiTk9g0s20tZxthA5Z8fv35ywwevB22TnKhkGfk59gsQgL5jDOxehynUh1DYNG1e+ehfGAYUbl4b9QxiuLodLRk1iAzaaTeaz5Xg0nk3ZfP3hcjwask9stjjPpnDYDwNUxX8x2fS85UQJbVIqIngNpwHQ4KnHiVAF3kUtH59t/t6jlD0eH0lUHsZtUPsEP30+5ddHLdicXrfUrXC7F2oXhbdh799X9gphukrU3KByPduUpbhLw65SvQKp95hecGkxBm9CStnJ9sOvklsyDkb+fHzmqpr5/qa+q1FLK1LSadaW6l1LnmP0fK7n746D2pCtonDSue6P58ijreu3Zv9QUifJblJUZN9CnxEE4rpTLzqZbPf7IyTU0J+nA7ZuaAMAAA==', '91': '...', '93': 'H4sIAAAAAAACA6VZ627i2hn9n6dAPT+Y0aQZY0KCj5ojBYKNDSFcbWNVinzDNmxfirnZ79RffYS+WNfetoFk5rSn7UjRGLy993dda33ml9pzQgLb3Nb2kVlLY2LWDm5ew3/uaedGafDPf0Q1lW/etWomqfXnr8Pa1vXcyN2aTnx3s9rGYW3nhskqIG4tCJN4u6uNzNB15i79YG4zEXduyjtxepOYnvv0tpiPF/P3F3n6vf68I2b63iOuvcN68t6N0535PoVN7w2h1XznOf7hzt+FpH5jpqm7expP35Ret3x669r7bRqn9e91k2104JvvrXdmYfwexWEQmeRundZv6BZP9PS7rWs67zv498WN7NgJIu+pvt+t/tyuf735pTaDH14EU2qOSwKLOgp/op1bc0lNmdVIbJvk11oSO9t//h0hc4vQ2XESmFWksHobxNu7ym/LTN2H+xsc5j4V13fWw73j0i++1IdZ7E30abrUXz11LefD9fJB7o+IJanrJR+9pPy9Z4XCztRavtXtbAytxdlh2zOkhWdJYi5LTnZ8EENTG/l21jnYIYnYWknFvSmhe7lSey9340M38ENHa60diRyswEvcY+wpTTU3Zp3c6SvJsjmJ3vCdqS1jhyd7QxKaw424mPfE+ev6taFzjjjfqLN5QxjPmqe20ydHQxNCuW8khu7k4zknYD88p8KeFjG6TnvVn7ZsST0YQedoSQIn0zW80bDCEWdqwl653qc3EbBHBBvhp5FYobpRMpIZ+rRhh/eP3aCztvhW7nSxV7PTgP9sDztvPCrcNLH7KrfURsnxcbSX+50Ma2DX1EdMfCWP48Gc+zbtttZL7G9qU9/pqxnivjdCkjqSmg36sjCO5D3ic7RyDk46EzsUjqaGfXXEZ2Ynw+fYk7vP0aKpwh91v+SJL0snYvAkZ/boxv961tzROynyuza1XjSYHQc4x/vBL0nFmhEHWx4G3Y0w3kz2hnYiyyb8zMpnGh3EbJQYpX1X9oRWU9k52glrW8jRiTN0ObbX3Dc7gK26El38VnPYTZCnj36jTiwNvsw6GmqO2H2F2DzZVHa/oa7G81ZvGBqoxQbNUY7rxOpPiR34Ge5lwxC1GjoHVi+w1dQaxA57nhqStaF/sj3bpN3gGWc7c0NzUDdTXxaNZMmrx082XuIRCpmlq+nP4r3kxczg26zOFrzKau33fBno3HXN5WWcidFTfJtfCOy8KGUxX/Kow+bEs18aqcWP0Kut3NB9WqftVdfpmXqn4Uji2rTfcgu98KmuUnxu2jkX25JwpPlhPRqNLFts4HmP5U9rcMlg5iW0R+Vup+oLjz37KZ/zSN0tQ+b3HjWYWNnm21hH3ehTgnxkFs+FyqbVsDSF5WUYintTayfjl0YOOzlj1uKsbBkWsX/2Bv3y2Yj2Fe3NEWH541o+8nSwQjGlvYw6or7xZhFDxPm83/l7Y7YJFY/VE/48YJWAPjnubV69tyUxAx60V9LpgFylwwj41e/4djRtFfUyPQCX8uosxA8YMGW19OM+m2QwR376LI9X2Neh+LRZsLj5yMNoO9CLPKKmOEdX9hUGo3Y2k1AERnrJpzp+rGJD8RK20HwfKL7ZElmbGf3sJef9JMO3sGeROzwnkXDQ76CO072rqb4dIFYa8iEdf5qXq31bhiZmA10hFFOoL0vtlBvzwn7URWIEhU/DUEDNjQ6WtmD1Pi1xQkFeZV0+44YSGJ9q9tln64FNDvCz4qHz+gsmewbyjl74GIvCzgOeRe80qjo5XGFXqESX8/8fX7AnYgvbwykwQ6SxBrZh/0BE7Krz/l97OwS2fcbyCHH7Q2dcamqaOVr5fZ9jfbyaxJ4RAeua08SiMZYo1pPUzgp+RgzOnEfjMNbKvghFjmIreJkHjt0vqA/5DvHy/MKfa96k9TaieMLJfeXgNF8F4OLeENl16f+mXCdujK4XjaVp4lDswxnApQB1wQEXc8q3Q80nSw32hnYwJh1xQUbj+Uacy72pKPdOHbnnzI73ojRbjDqyOBLnPfVlwhG6pjeZvXR32XM0aBa1T3t1UPbgMCJkqcvflezjGtYf6ziSXwdN9M2e4rmDGFsZjU2D8jCwyk4Gmd0e8rB7fR8NyhqCdgL2+MAsBzx0Alc4nNlNI8SBcTZ0z8HOUw+4G1gVPq+hi7I0/ognwnHVP/7QEzZ9XiIEGJcVvJ1Gw6Dzipop+IKv9Aj6udnBuh57xkH8Kb8NkdcKw+i1oQ9ftpqwgW8tWTrXvLdsqrTGDzSPdsT2AuYCjwO74oNznUBLISYLoeSF3VJ/Zpww6Kf7YQD+lchGhy12hL3D3gOwKzFEAWcrOfbwV/0T6rqFc5XMUAW6L7D8lCDnhw/3Zif6HHFCFTlpV/uUnwVoPxnYNV2h/iMT3AN7yu8madUbJR8/nPUHcP2z5lGyE9UEDWCvbxTnlNjc+65wFz30sWcdYTCrappi7VmHgluhf3I5Gy2oH8iDdpqgt4n1wrTPFU/YybLQ5tnrWn1DrSWU5xaSCOw6xtAuZQzFQmtjX+QHtgGXzhqeceOGcTzOt6EVh13Ht6IpakNlel7JjtH5nNzo4zvooXY07F/3iO0dtzary2seGOftCLHeUDw6+4s4LVHr4BcC/ZEiT5bCMx6JcPYefgR2uKDXjLPo+YwvURfoI/D4meOZNlGATbbEni1yRNfw57jjGvXEqzz1xcocB+veijMwg/Q3NMeYCQhspHh0FauPc865Z1Fnm0HXbuMZHvFEXEb5eKasTWpjt8FZvJAuQzGT1/ftZQRslQSKrcjvQpBDpeH0p8idDGyf4PNHnJzwJ4oxwXgDvReBKyp9B+2Ent1XOR53hQB9jrPuv+F6Y+rLb8oZI0sslWRBAVZe5W6OeL7AJ+yLvuRRd5KcLAPUWThKGX7PmD5+rLDl07OTAuOZDnhE/OgslUFnPq4KHTO44DjVzUI21ovZaxg6wDZwaI51H+sduoscmL74VNsVL1H8AB4RcMHe4HGmrhzG2mVGG3Tv9+CbzGi+lpz0fBhmnZel3mF8izweLKn9H+fRt6yINfbnKT46mHmZVtcwI3Q76fGhgb6q6sNPkNeifjRwM8Vm2GJA03/MgYA4OGVfng7OS4PNmSUmsusPvvTBD7QHwcVLqvmBLYwbqO5mmvtVcJtc5Sf+2gdZpJqT2kXnpk5kp28++HNNZ3bkLLExE9B+xj06S1981O5TyldL/tSgugH32ZxPZ/fPPVHh4TBre5d5jnLAFabBd7mnYs8m+mXaoDGHzkDcyc7UqRbqeUv77US5GXX9kX+kCuuO+2vtI88EhgFDfkN5FfzhY4adAjM2V9r1U4zPa07AdBEz22sxv2Sng92nOqaVDy4aruKlg7Upe4BquKi6f61NOgfUCvK12INvE8Q0H2ct61XkHpmtenH/rBfXP9tjBB2nrI05F090KF19swf3dGivuzO/sjcZN8v854kFDQA+V3N9Rqzi3An0GNUY6pujNaB9lPbqpeOA+4o+/PHMBuIMTn4VhhyrhQz8vaG1gFz45TuYh2HWwnw24qoaHPxsL4nkE621dzRg0Ex4YfqmCRyu9AZyUe53GNKaa05i1C0wbpGcZ4vreRSYh7mvsu/7An4jhtGgjPUwwjxJKDdDD2Z+bsHvJb/YI9c7QwcXZ/aHfVke1DSa6+rfMGtSTpArPtFnLcoZqaPRXvDZ/PPvc8X0nkDzY4WtBmoTul1wdWn6OOpzm3O8xLN9R4uH/uu27sF13y8z7pUvYYGdVY2j/lJDn3jwi0NMdsa8gdie0s8+DYF7qAPy4yzRKbUGMOpz/mbkMjPMHLHUJOjf+x/6W8mv/VfpfPCfzt1bf/zst8va/+58u0n5V83t7Go/9XfqYbb5OEdCe1yu42geXrDr43n/bmZpPNIaHRa1kJ7zBGxhGKNzH3iwyuulDy/vQaymkkDTMz1Hcf93sb7QwWesAvbvoQvQYzuh6slqf+Sm5fLn2Qk9pKaUYwpObfl2v5NSXDHRt0Op4Mg3nvLVIlY49hzTrVf469FaNTQno9jwBn526MwLrKA9b+sq+FymfRtZGtPVf+xdL31HEvZY31u014OCX3/QXuBT6ILMxjrKJ1TTXM+cb83zLF5q7skjnVER57A469RyoSmt5kT4HO+3Jle9a2nS9ytW095P1OnrRKX6Rl1NOfVtqipjnROGY72chSMuYXn1np7qX++q1+LVK/md6T3V/5La2yDZ1QLn6U/sRT/fbP25eMv/p9/qN8GqhmW1IKrR1/y/3tTwz3JX8da93brp7ol+e5cmJNh9wbrbxle24v3WXO3c7RNdUt6t/+V7cdJv9WoV++Gg2OwbW89+h9jualhrxU72W706t/iNgZ21dRNi2u6X85pbnPut/teo/o16R6/OJ9HLchGOPAY7/yc/oXypH+u3n3+vuHWCbfGbRmJu3Wh367jE3blPoklS9zbdr1bB6ale/HrytWam7KeaIjr06u64DXbuF3q7cDXCsU9jc+d/Ybfpx683cXr2hn5xS8/7epNsgwjh6sZRuof0qJU/tdQct5a6p5j9EtJoIDLpziSmY9bc6Nd6+ey/APaKMFZ1GgAA', '95': 'H4sIAAAAAAACA5VY2ZKjyBV9r6/A4QdVRbfVCC0l9bgetIFACG2IzTNRwSZAYmtAYonwV82DI/zo+THfBKSSqmrG446oFiSZebdzz72Zf8UEot1sdbD//Auj/cS0IjXCDNNPogB+MFeNMSOIMU9NIkc3Y8zMnDiBz+jRxSLVPzq+hfmq7gS+6qIlpmvq6M2Mmw/7KPCwUE1s19EwxwuDKMFW8Fp9SEwv3DuuefnCqZ5p8CZ6UaOchC8P9ZcgfghVy3xZ7vjVjn+d0JtvjWECyr1OQVoC893XcRAn6uvG0dXX1qDbfiVwote0E89tPKD/X9AGzchUjdfEzJJH09cDA3R/aZyS/d/6jacHWB4lLw0M/umBHyfY/uS6sqlG8cs/ms1mjp6anho+bsEXvvX09ecGxS5HQ/bnxi8/NR5M30CL//mzf1mvB174Ui5bmMh9LaLxEAbxC1KnuXd847EUiSQH4c0o7PQV5j09OHsMfv+OY0GEoTnVUzlRD05+Uq//y0vr+wNSO1Kd2MQ28MXxzGkUBdFjgwswGAtPRoCdNHBOhOKmucGPk4n99m/f0VGc3yJ5jS8KZQi7O7oTQnR/+9VvYluztMyMzr/9iraZ8QsWU2Fy5ARRE1xYh0tTY7PXeYjM0FV104PtXqqhptbrGCY43nxs0OOhRY/7Z3raHa1xd8UfyYWEC4vNjptsd4PlRuD2woEuFvy6R1OZreejVPcGhCp2D6pIFvSMc+kZY2te4spix9KpQU5TQqyI3EHVl4V2iOdvMriz3t7YNMmEMjG1dGJn6R55UgnBpinurPmbXN6OXG08OmhE94chdnEtHx2V7chOn8mznqeW5vUtmRicDJBJz0a51lZCzeNCWVqcxg6SU64tjPHIM8QsFkXB1v3FQMw7J9YHJElMwXotWx/bPOwBa/WQHRszjRo4spj6El/rW+rD4TTlFmuRKYxJK1Kk42DF476M3inhJBPuWXMHqdxe+8s0sNA6pJveXld24Zkte0KsF61Qx0kHxr+VerjDmMHvfZFGumVQA7zSIVx/2G/GnA3PBZnCVha7Eey5ru1IV3xXY53OyaxsPWkimc5noKvPJLI0PCmEgM9nw4hpp/68fQxXeVfTIV6mgH9bEd2WRqUhSxmxRkAcx11fkdaBPj5+Y/1RS5E4XGsN/No/9fzMmH/wE0koEmODrGIlDGsdhoOVG59Yp9tVRDLX8y7SJzB5/Avsd1bbQsp6hmuADfPc6DN50p3z/ZNI/AndvJarUK4ttUcHY9L/pvlCrJF4+N5vmkfiqjg4yWK2laWNC377H7pYPu+Rh3rN7+r0B3LWWpsLDQqwAbjTPKFNT8s4nUB+CNi0NZKxAT81Dq0AsIIr2yycgy7zWZIjPdkZwirdq2LKaarQGsy3xzqXhhbrAe5nYI9jnyH+gPlWuuDpQMu75XqYe9KJQQ72BPI2c8q9CfpUycKDOTGtn93+nppW2HFaTpUjdFjlEvqr/DSn+rG6BUy5ybmeA+9xJm2Pf+xzwCpwhb8SfgcHn/utxixgtYBchzyeb7Py90N+VtxTrMRLng7KXPiQL3l2539YA7lvGEv/yiHuDuVse/H8jkcKrS3k5T4Vfu5z2XvDPUuWNn6MM1XlDOS1bXiwl2jAPp/bsfEG+ZoYxFqbHuiVPlUujW3QU69i4INttQ8Nj4wNcQdj8RUbqii/jfOtyn8S4xozIdecUfl+iRn8ndl8RALngl5dH/gWYsFV+o9HEBvhBPzr6gRna9TUWpjz4bGYdpf5qEDjJnAwcETF307J3T7ofUTrDGpnKdTuDbN5HzjfddG45rmEImYu1JNCI7IzcLmte1OEIbCdQTwdazXf6xSJ6oJd4t0jy/oC8ytuykeEKgH/e5nLWhebkE6blu51LPg2ksG/jKOEwN0Druhn80L+MZ8psJ/gsvw0ZPEWxMlCfu7MZ5cakbmAD1wd4xk7mYZLv5p/8dv+E9wD3gpVZGJFqDEpVb9veT5osxRgmu+iWF7iSgD2W5D/P1COXvPuUnfGsqeC3ite9xgvQPG85BDU2K4Pen9jcr33Pt6rfO9YDNQll8N1yAMlv8g5hvPPc4hXKfKoSKhW6oflhHMW4hpn8tSnC9uRi43DbfWYyTlbJtbpktr57Fg/KBP3sOBtp5xHbDxZlLuLat5B5t2DfBiW82T+2JE9us208A/5JRNkrPAthL9oR9i2UnKjEkMecRX3gJ7QV9S+00TieOby7IBwo7t4LOZ2OC8gjoWesIdhuphM4xpHBvDTRV4lZ9tNDYkL5mScsXd2cxrntj6pa1275CTw70ayU9gTv+GaOz6rsENXOerYz2BPqBVhOYe97VX44KbGpD4L+IZY2uzwil8bYVYpe6zBqapx4QdeEQnXqG1AXLaH+RBrYY/ySibKHAaeCT/jvUs96Ylv8UjBx/icGl05+66/EtOyR2FmCeQp9D4Sfnm/8iFDoN+Ni+aDvFAnWmemCHyUv+DDgyqBXYQ/iYmOv99Oxkk+qvuuzLqxXTPbm65O7XoMYYM/N8i3PjuzekpZ4+v8nB17dz1WnvWW/gcOjU2vFWqH4MRtsx7K7+VBh+f0ykkVDgFHPuAV/Lasf9mZ7YPc3sLJugpBpssDHeuEfTba9TwP8QHnshSMAaeDvTB/VFzHr/bAH/CWOVv33vMDW45vIM4t8D3UYsAFIyW4Itn4HnIIYtwxKAVwuO6ZUBtNb9eDGl9jm+4x+SKTD5zDUQvwZ3YwEB8SrSP4zAY7zobEHJCssgbd6lPrDOOJDj29gXydC8/mJaZS0tnzmaMfumtDFKDX7VtXvarxnYw4fyw8y230fZAooD/g+rTwlAHDZ8iuXOY7fZYAGwCjK8cYSDd+r/rx0g9brW0U7BjV0kEOuQ+5sI5RDb/pDUioMxCjjXGDk2fAAWCNFIHjQXZY4XKbBYro+ir4+8L9q2KHc3yAL/lhbJIb6DEgfo5BapRwqPEItak81ySKCLmSjy79rn/rN4hRF3oBfOknHVmyQ8itZ2PmpihuwC0QRwN0PEI+c8jnPnAFzGeAWyBWBPCER3YZ4C3URyjQm5liNQaYQf2BUnFJBjZzUVmLIS4sv0v3d3gdWrCuo0qL3luevMm737/yJ9gKPSRjm4CpNz3daowPS97lKdIB3ihYn4tBvjsfQ//oCai3gTpvhe91gFzKFcI9gS/i5WFaAB/ny8OixYLfF4Ucy4cgW/DC4A53wMHAtSnUmJ7YTnBTGqE4gLyuC3p3QJ4H7x147tbPKeQU9Lh6bE6Gvev5Bs5eSnsTsNsOyLOypf+hZqN87y4mQQo8kYF+8Vs/e4kl1EQ+fL7JpZybrLscbwHXGKEy2wTLAzN4bzf8QY/XCa7YaiVXDJR7go/NGeMqh5L3wFYa9nMB03KPaUOPA/FAtla6WaVuJg88s+13al3Py/EHmf+fzgKOeD2UJPzG7tFJA25eepfa1b+TweDl2R74SYAzwSdnfIqDesyFigc1/XLOn21sHc6u0ONZd2dikKGJfRgX0Dn9TFMbtB7qOsyta1P6zJ3osZXRExxhCfKEDllntIA+z1PFzH5/VyAToB/hwjfUa5a6ulX+InlCQVeyXHq2mkSgP+z3qU7aW921NEKBvV3oc4fBYjsc0FTLRjnDIpwiG0F3TRSgl+3fnL/R/OOJnl7OpKPy7AX4q+4jiPJcetarewsc8eS7O5MY8B/T6EzgcZ+tSSvOHRS1P1y9TbZgLdgN8QfeBv1z5FPIkYMKPtUlwUW+hR7wkPZaNuvovXu8G1ceL+2s43Zv0yhJexBrCvpIqorpJ3Ub2b3gcWbEj4GPPVTrUUzTO1+htQZFFrK0uJdBoTghLGTI1vpOB84Rb/0KYGRQ1GeJUGkLSC/IIQ6dD866c4l5pQvTLs/7L42n5uU6q75FbEZxEjnh49OXxs9+ffOI/vvH9zCIf/lycyX2pRxG13rff0E3fTfXeg1uOKaX3JB9nbLTMXqcbl8Fot3qNP70ZZ9zucy9udy7v53FqtlmrEeO5lQ3emqk2845QHd5qZPYn1zJPjbSxtf3l6dfDSeqLlhDNQLbvhogJzFfSNWNza/xab93spdGdRv7hKkxFpyS8JRUhlTPzTRyEvMRTXn66cZNr+j6+AVdGj/WE33Q6ekhiJv1rMf3s78iTZ4ewshBvtwMuTnNUVtsMsWGoy0/5ZBD6e1iibFTit6yQ54WlpiMrTbTLT1Bn4csRnPj5Wa13Awny+33Rr3jfwGhG7V0LRcAAA==', '96': 'H4sIAAAAAAACA4VWzXLbNhC+8ynQ5ACykZlKjp1ECtNRZbn1VLY8kuJ2pu1oIHKpwKYABgBtqYkfpg+QQyeH3usX6y4p0nKSmfogg8D+YPf7drGP2UVnP2wfsn//YQO9yo1eCMMybdnq7lNS0ELc/S0SmeBqwwzsGVAJGPknilmpGDj5rgAnLEsz7YRyYEMvNXrFcuHeZnLB5CrXxrFz/KwOHKzyVGZQn5yJFSQzoA9hNsd44m1PTLPS1svFEqLxm9n5m9n86GTylPddJux8mEHsUDObD7R1Yj6RsZi3Xx7szzvfdQ7Dt26VcY9+IzIQGhDJ3MHa+aBinUi1jHjh0r0XPPAMvCukgSR67zH84zMDlil5DRn+z2ApbSacvNaWd/k+ZWwqwQHDpF1LFUvMAd6VoXQcy7tPircqM31MmSi3y3uyRKaAW6iQCLQ01ZlGG2BlUm5mtd5AL8C4wogd34IlgHl/6BaNnBaZk83OvQmhEDpBRiyLEd+MoEL5/mzUn3b2n8+Hv84m/fmgf3Z0ctSfvZn0p7XuOHYC/SFSgikRS63QbpeXWWaUZcqAQ8wbbxOhrjChTCJDcqSJBKQDqggCCp1dSLj5XBbDQYIhuXZcnPUHJ+Oz/mg+HA0HtBxO50jT9rNa+TgrLlGFdMvcbjC2ba5Q/Xg0/mV+NDye3idSWcxPmbu0UrWQCyOSEkok7s1wnWfagLm4d3IOhliKoArjMIlGltL0tTk3ujzbET9B9EgsIc7olcRQgOQVstseazMtCYAh7ujMqJYIMLYDHeZLqUJk583WOV3ggbMfRCZUDEwbuQS1hz6dVHoby/bwon3wRSgSAVmWcWMGKpay9nPUI0N05VqyuaEwl2V1N0nANaypJsWiCrD6ojtuBsIkeM3DWnt091EBKdTc317BiJi4bHe4MZIKdlVndx8dth8EVlG1FlQ3yPz74upyW2d0Jl0GO7oN3lscGCjUXOt7d4dTWJ/L+KrWOK2aXeWiFHxlYyNzx2QSPdoqHexhrsHoR68/V8OIjMQAK/i+ovpsrxGp1G+9lbSWus9vRBCWYuOgRWslzBUYzBKru1EosWFaP2AyZdtTpR1JUFf7w6PtylbXCGmBTQok4gqGxmjj85FgAuteigrt3VaCrLn7Cw3VjaHL+BPeYjy81FL5W6NB4C20dpHh/GuRvdgTxgiFb8Cj156fFqpExA+whfICL2OdkbHjPY/VZ+hQulKAYUj+DXYLfRM2DWn88zdR5EwB7MMH9vDwYP7j8Gw4Gd/LVFYYscQ6BhRxlOi4WGEthjH2egf4PtCXzwU1WB70Ko1SNiRqDYhhykX8TCP4LBZmKQxeM9bGIL8EacPDJ3EHb2R0Q5uQTeCaEMCOCZcQF1XKE8gIMcA0XoX8gX/rNhmEsbUzvEiEvSWhF6nb7uTrXozvguk+ftnuHHQOegsRXy2NLlTSfZy+hEM47KV48W77Wb5mdmORI3uFrK03OVjoZBPi40Lt2C99Bj0D2ChVKXnr7Qo7qqOoeldZ866WE0LnO+r2EyjrqpoFmpb7gP69Laxuk4NOWTUtRFHEa/h5UO35hAT6R9nmAvQ4b6YOUSONTAvKBg+ac0zP8BoXI4nxYin6/Gh8usVvhNKQ8Baxq/VeYwPsEkNuyQ1kCEpFu553G9Dvq6cVl18jravxwEBoi4Vv/o/l4bff3yu3OG+RdivNxNKSjWlQmaMfDCjPRAw+f/WUkEB5qqUn/HfVbLQD70a6t1+ZgnyOr+XnU0orkaaaZHJ6+1wL2YXTQHQsMMaWLdJUriNejT0BQxzdKu+WpMBFeIM4gU+HQc/VviIazXw6pg4UeNo2925kWuQy8HIjqZR+mp2O2EV/hCPD0biaIF+W08DOwNRipnriW1+8uUgeKpdy6AFzjVtdvnXwH6i4ooGQCgAA', '98': '...'}

def _finalizar_atlas_publico_v2320():
    destino = OUTPUT_DIR / 'Atlas_Electoral_Costa_Rica_1953_2026.html'
    if not destino.is_file():
        raise FileNotFoundError('No se generó el HTML principal. Ejecute todas las celdas desde el inicio.')
    # Una copia local: nada se retira del HTML utilizado por el usuario hasta
    # que todas las modificaciones y las comprobaciones concluyen.
    with TemporaryDirectory(prefix='atlas_v2320_ensamblado_') as carpeta:
        temporal = Path(carpeta)
        html_temporal = temporal / destino.name
        with destino.open('rb') as entrada, html_temporal.open('wb') as salida:
            shutil.copyfileobj(entrada, salida, 16 * 1024 * 1024)
        contexto = globals().copy()
        contexto['OUTPUT_DIR'] = temporal
        for numero in (89, 91, 93, 95, 96, 98):
            bloque = gzip.decompress(base64.b64decode(V2320_PASOS[str(numero)])).decode('utf-8')
            try:
                exec(compile(bloque, f'<integracion-atomica-v2320-{numero}>', 'exec'), contexto, contexto)
            except Exception as exc:
                raise RuntimeError(
                    f'Error al integrar el módulo {numero}: {exc}. '
                    'El HTML anterior NO se ha modificado.'
                ) from exc
        texto = html_temporal.read_text(encoding='utf-8')
        texto = texto.replace('Versión pública 22.0', 'Versión pública 23.31')
        texto = texto.replace('Atlas Electoral V23.20 · Resultados y cobertura territorial',
                              'Atlas Electoral V23.31 · Resultados y cobertura territorial')
        html_temporal.write_text(texto, encoding='utf-8')
        requeridos = {
            'Sección 08 · matrices':'atlas237DetailsMarker',
            'Sección 08 · cobertura':'denominatorV19',
            'Sección 08 · partidos':'atlas237FlowParty',
            'Provincias exactas auditadas':'atlas-v2326-provincias',
            'Pestañas internas':'atlas-v2326-worktabs',
            'Sección 08 · trayectorias':'flowExplorerV14',
            'Sección 08 · nombres':'atlas237Personas',
            'Sección 07 · nuevo visor':'ATLAS_V2319_RESULTADOS',
            'Sección 07 · datos geográficos':'atlas-v2317-json',
            'Sección 07 · interacción':'meta:{provincia:prov}',
            'Sección 17 · perfiles':'partyProfilesV14',
            'Validación y arranque':'atlas238-arranque',
            'Ranking':'NACIONAL_ELECCIONES_V2314',
        }
        ausentes = [nombre for nombre, etiqueta in requeridos.items() if etiqueta not in texto]
        if ausentes:
            raise RuntimeError('Integración incompleta: '+', '.join(ausentes)+
                               '. El HTML anterior permanece intacto.')
        if texto.count('<script id="atlas234-territorio">') != 1:
            raise RuntimeError('Módulo territorial duplicado o ausente. No se publicó el HTML.')
        if texto.count('<script id="atlas-v2317-json"') != 1:
            raise RuntimeError('Datos de la sección 07 duplicados o ausentes. No se publicó el HTML.')
        # Escribir primero un archivo temporal junto al destino para permitir
        # os.replace() atómico. Ante fallo de escritura el destino se conserva.
        publica = None
        try:
            with NamedTemporaryFile('wb', dir=destino.parent, suffix='.html',
                                    prefix='atlas_v2320_publicacion_', delete=False) as salida:
                publica = Path(salida.name)
                with html_temporal.open('rb') as entrada:
                    shutil.copyfileobj(entrada, salida, 16 * 1024 * 1024)
            os.replace(publica, destino)
            publica = None
        finally:
            if publica is not None:
                publica.unlink(missing_ok=True)
    print('ATLAS COMPLETO: secciones 07 y 08 integradas; HTML verificado:', destino)
    print('Archivo publicado solo después de validar ambas secciones y el resto del Atlas.')

_finalizar_atlas_publico_v2320()


Módulo obsoleto de tres gráficas eliminado; el resto de Partidos y territorio se conserva.
Matrices territoriales V23.19 instaladas en: /var/folders/0g/80cd8wkd0s52qc7shs3l3v5c0000gn/T/atlas_v2320_ensamblado_r_379e19/Atlas_Electoral_Costa_Rica_1953_2026.html
Consulta nominal de sexo V23.11 instalada en: /var/folders/0g/80cd8wkd0s52qc7shs3l3v5c0000gn/T/atlas_v2320_ensamblado_r_379e19/Atlas_Electoral_Costa_Rica_1953_2026.html
RANKINGS DE ABSTENCIONISMO LEGISLATIVO Y PRESIDENCIAL INCORPORADOS: /var/folders/0g/80cd8wkd0s52qc7shs3l3v5c0000gn/T/atlas_v2320_ensamblado_r_379e19/Atlas_Electoral_Costa_Rica_1953_2026.html
HTML VALIDADO V23.19 · tres niveles, ranking, flujos separados y sexo preservado: /var/folders/0g/80cd8wkd0s52qc7shs3l3v5c0000gn/T/atlas_v2320_ensamblado_r_379e19/Atlas_Electoral_Costa_Rica_1953_2026.html
Sección 07: mapa por partido/provincia y mesas completas añadidos al HTML: /var/folders/0g/80cd8wkd0s52qc7shs3l3v5c0000gn/T/atlas_v2320_ensamblado_r_379e19/Atlas_Electoral_Cost

## Integración territorial reproducible — provincias, cantones y distritos

Esta capa amplía **la aplicación pública principal**; no sustituye sus gráficas históricas.

- Preserva las secciones y los filtros existentes de la versión .
- Concilia la abstención común de la jornada electoral desde la base presidencial para la primera ronda y diputaciones, conservando votos válidos, blancos y nulos de cada papeleta.
- Integra resultados por provincia, cantón y distrito, las segundas vueltas documentadas y matrices de liderazgos cantonales.
- Ofrece rankings de indicadores territoriales, consulta detallada de resultados, ventajas de voto territorial sobre el porcentaje nacional y exportaciones CSV/PNG/SVG.
- Mantiene los años en el eje horizontal de las matrices históricas y distingue observaciones territoriales faltantes de votos cero.

Las modificaciones se incorporan **al HTML público de `salidas/`**, conservando los Excel originales y el resto de las visualizaciones. La ejecución es idempotente: si se vuelve a ejecutar, sustituye únicamente el bloque anterior.

In [48]:
# Integración complementaria al generador existente; requiere los dos Excel actualizados.
from pathlib import Path
from tempfile import NamedTemporaryFile
import os, json

_ATLAS24_PY = '"""Atlas V24.0: ampliación territorial reproducible sobre fuentes TSE.\nConcilia la abstención común de jornada mediante los resultados presidenciales,\nsin alterar los originales ni mezclar los votos válidos, blancos o nulos por arena.\n"""\nfrom pathlib import Path\nimport json, hashlib, unicodedata, collections\nimport pandas as pd\nimport numpy as np\n\nYEARS=[1953,1958]+list(range(1962,2027,4))\nPROVINCES=[\'San José\',\'Alajuela\',\'Cartago\',\'Heredia\',\'Guanacaste\',\'Puntarenas\',\'Limón\']\nCAT={\'ABSTENCIONISMO\':\'abst\',\'VOTOS EN BLANCO\':\'blancos\',\'VOTOS NULOS\':\'nulos\'}\n\ndef norm(s):\n    return \'\'.join(c for c in unicodedata.normalize(\'NFD\',str(s).upper().strip()) if unicodedata.category(c)!=\'Mn\')\ndef cleanname(s):\n    return str(s).strip()\ndef province(s):\n    return next((p for p in PROVINCES if norm(p)==norm(s)),str(s))\n\ndef assemble(root, output):\n    root=Path(root); output=Path(output); output.mkdir(parents=True,exist_ok=True)\n    inputs={\n        \'L\':(root/\'Base_Electoral_Diputaciones_1953-2026.xlsx\',\'Detalle_diputaciones\'),\n        \'P\':(root/\'Base_Electoral_Presidencial_1953-2026.xlsx\',\'Detalle_primera_vuelta\'),\n        \'R\':(root/\'Base_Electoral_Presidencial_1953-2026.xlsx\',\'Detalle_segunda_vuelta\')}\n    sources=[]; dfs=[]\n    for a,(p,sheet) in inputs.items():\n        if not p.exists():raise FileNotFoundError(p)\n        d=pd.read_excel(p,sheet_name=sheet,usecols=[\'Año\',\'Provincia\',\'Canton\',\'Distrito\',\'Código\',\'Partido\',\'Votos\'])\n        d=d.rename(columns={\'Año\':\'year\',\'Provincia\':\'prov\',\'Canton\':\'canton\',\'Distrito\':\'district\',\'Código\':\'code\',\'Partido\':\'party\',\'Votos\':\'votes\'})\n        d[\'year\']=pd.to_numeric(d.year,errors=\'raise\').astype(\'int16\')\n        d[\'code\']=d.code.astype(\'int64\').astype(str).str.zfill(5)\n        if not d.code.str.fullmatch(r\'[1-7][0-9]{4}\').all():raise ValueError(\'Código distrital inválido: \'+a)\n        d[\'cantcode\']=d.code.str[:3]\n        d[\'prov\']=d.prov.map(province)\n        d[\'party\']=d.party.map(cleanname)\n        d[\'votes\']=pd.to_numeric(d.votes,errors=\'raise\').astype(\'int64\')\n        if (d.votes<0).any():raise ValueError(\'Votos negativos en \'+a)\n        d[\'arena\']=a\n        d[\'kind\']=d.party.map(lambda x:CAT.get(norm(x),\'party\'))\n        d=d.drop_duplicates(subset=[\'year\',\'arena\',\'code\',\'party\',\'votes\',\'prov\',\'canton\',\'district\'],keep=\'first\') if False else d\n        dfs.append(d)\n        sources.append({\'archivo\':p.name,\'hoja\':sheet,\'filas\':len(d),\'sha256\':hashlib.sha256(p.read_bytes()).hexdigest()})\n    src=pd.concat(dfs,ignore_index=True)\n    # 2026 es referencia de orden, pero se conservan códigos históricos ausentes actualmente.\n    id_rows=(src[src.arena==\'P\'][[\'year\',\'prov\',\'canton\',\'district\',\'code\',\'cantcode\']]\n             .drop_duplicates().sort_values(\'year\'))\n    latest_cant=(id_rows.drop_duplicates(\'cantcode\',keep=\'last\')[[\'cantcode\',\'prov\',\'canton\']]\n                 .sort_values([\'prov\',\'cantcode\']))\n    latest_dist=(id_rows.drop_duplicates(\'code\',keep=\'last\')[[\'code\',\'cantcode\',\'prov\',\'canton\',\'district\']]\n                 .sort_values([\'prov\',\'cantcode\',\'code\']))\n    prov_order={p:i for i,p in enumerate(PROVINCES)}\n    latest_cant[\'pidx\']=latest_cant.prov.map(prov_order)\n    latest_cant=latest_cant.sort_values([\'pidx\',\'cantcode\'])\n    latest_dist[\'pidx\']=latest_dist.prov.map(prov_order)\n    latest_dist=latest_dist.sort_values([\'pidx\',\'cantcode\',\'code\'])\n    # Archivo oficial disponible para abstencionismo compartido. Sin imputación ad hoc.\n    presfile=inputs[\'P\'][0]\n    audit=pd.read_excel(presfile,sheet_name=\'Participacion_distrital\')\n    audit=audit[[\'Año\',\'Código\',\'Abst. común (presidencial)\',\'Papeletas pres. (válidos+blancos+nulos)\',\'Papeletas leg. (válidos+blancos+nulos)\',\'Estado\']].copy()\n    audit.columns=[\'year\',\'code\',\'shared_abst\',\'pres_papers\',\'leg_papers\',\'status\']\n    audit[\'code\']=audit.code.astype(\'int64\').astype(str).str.zfill(5)\n    audit=audit.groupby([\'year\',\'code\'],as_index=False).agg(shared_abst=(\'shared_abst\',\'sum\'),pres_papers=(\'pres_papers\',\'sum\'),leg_papers=(\'leg_papers\',\'sum\'),status=(\'status\',\'first\'))\n    if audit.duplicated([\'year\',\'code\']).any():raise ValueError(\'Clave de abstención duplicada\')\n    # Distritos por elección; se conserva el universo de cada año.\n    units=id_rows.drop_duplicates([\'year\',\'code\'])[[\'year\',\'code\',\'cantcode\',\'prov\',\'canton\',\'district\']]\n    rows=[]; parties=[]\n    for area in (\'P\',\'L\',\'R\'):\n        di=src[src.arena==area]\n        if di.empty:continue\n        grouped=di.groupby([\'year\',\'code\',\'kind\'],as_index=False).votes.sum()\n        wide=grouped.pivot_table(index=[\'year\',\'code\'],columns=\'kind\',values=\'votes\',aggfunc=\'sum\',fill_value=0).reset_index()\n        for k in [\'party\',\'blancos\',\'nulos\',\'abst\']:\n            if k not in wide:wide[k]=0\n        wide=wide.merge(units,on=[\'year\',\'code\'],how=\'left\',validate=\'one_to_one\')\n        wide=wide.merge(audit,on=[\'year\',\'code\'],how=\'left\',validate=\'one_to_one\')\n        wide[\'arena\']=area\n        wide[\'validos\']=wide[\'party\'].astype(int)\n        wide[\'emitidos\']=wide.validos+wide.blancos+wide.nulos\n        wide[\'abstenciones\']=wide.shared_abst.where(wide.shared_abst.notna(),wide.abst)\n        if area==\'R\':\n            # Segunda ronda tiene asistencia distinta; NO tomar abstención de primera ronda.\n            wide[\'abstenciones\']=wide.abst\n            wide[\'padron_analitico\']=wide.emitidos+wide.abstenciones\n            wide[\'asistencia_estimada\']=wide.emitidos\n        else:\n            wide[\'padron_analitico\']=wide.shared_abst+wide.pres_papers\n            wide[\'asistencia_estimada\']=wide.pres_papers\n        wide[\'abst_pct\']=100*wide.abstenciones/wide.padron_analitico.replace(0,np.nan)\n        wide[\'part_pct\']=100*wide.asistencia_estimada/wide.padron_analitico.replace(0,np.nan)\n        wide[\'blancos_pct\']=100*wide.blancos/wide.emitidos.replace(0,np.nan)\n        wide[\'nulos_pct\']=100*wide.nulos/wide.emitidos.replace(0,np.nan)\n        wide[\'noval_pct\']=100*(wide.blancos+wide.nulos)/wide.emitidos.replace(0,np.nan)\n        wide[\'valid_pct\']=100*wide.validos/wide.emitidos.replace(0,np.nan)\n        wide[\'emip_pct\']=100*wide.emitidos/wide.padron_analitico.replace(0,np.nan)\n        wide[\'level\']=\'d\';wide[\'key\']=wide.code\n        rows.append(wide)\n        pv=di[di.kind==\'party\'].groupby([\'year\',\'code\',\'party\'],as_index=False).votes.sum()\n        pv=pv.merge(units,on=[\'year\',\'code\'],how=\'left\',validate=\'many_to_one\')\n        pv[\'arena\']=area;pv[\'level\']=\'d\';pv[\'key\']=pv.code\n        parties.append(pv)\n    district=pd.concat(rows,ignore_index=True)\n    party=pd.concat(parties,ignore_index=True)\n    measures=[\'validos\',\'blancos\',\'nulos\',\'abstenciones\',\'emitidos\',\'padron_analitico\',\'asistencia_estimada\']\n    metric_levels=[district]\n    party_levels=[party]\n    # Agrupación por claves administrativas estables dentro de cada elección.\n    for lev,key in [(\'c\',\'cantcode\'),(\'p\',\'prov\')]:\n        gr=district.groupby([\'year\',\'arena\',key],as_index=False)[measures].sum(min_count=1)\n        labels=(district.sort_values(\'year\').drop_duplicates([\'year\',\'arena\',key],keep=\'last\')\n                [[\'year\',\'arena\',key,\'prov\',\'canton\'] if lev==\'c\' else [\'year\',\'arena\',\'prov\']])\n        gr=gr.merge(labels,on=[\'year\',\'arena\',key],how=\'left\',validate=\'one_to_one\',suffixes=(\'\',\'_r\'))\n        gr[\'key\']=gr[key];gr[\'level\']=lev\n        for col in [\'abst\',\'part\',\'blancos\',\'nulos\',\'noval\',\'valid\',\'emip\']:\n            num={\'abst\':\'abstenciones\',\'part\':\'asistencia_estimada\',\'blancos\':\'blancos\',\'nulos\':\'nulos\',\'noval\':None,\'valid\':\'validos\',\'emip\':\'emitidos\'}[col]\n            numerator=gr.blancos+gr.nulos if col==\'noval\' else gr[num]\n            divisor=gr.padron_analitico if col in [\'abst\',\'part\',\'emip\'] else gr.emitidos\n            gr[col+\'_pct\']=100*numerator/divisor.replace(0,np.nan)\n        metric_levels.append(gr)\n        par=party.groupby([\'year\',\'arena\',key,\'party\'],as_index=False).votes.sum()\n        par[\'key\']=par[key];par[\'level\']=lev\n        party_levels.append(par)\n    national=district.groupby([\'year\',\'arena\'],as_index=False)[measures].sum(min_count=1)\n    national[\'key\']=\'CR\';national[\'level\']=\'n\';national[\'prov\']=\'Costa Rica\'\n    for col in [\'abst\',\'part\',\'blancos\',\'nulos\',\'noval\',\'valid\',\'emip\']:\n        num={\'abst\':\'abstenciones\',\'part\':\'asistencia_estimada\',\'blancos\':\'blancos\',\'nulos\':\'nulos\',\'noval\':None,\'valid\':\'validos\',\'emip\':\'emitidos\'}[col]\n        numerator=national.blancos+national.nulos if col==\'noval\' else national[num]\n        divisor=national.padron_analitico if col in [\'abst\',\'part\',\'emip\'] else national.emitidos\n        national[col+\'_pct\']=100*numerator/divisor.replace(0,np.nan)\n    metric_levels.append(national)\n    natparty=party.groupby([\'year\',\'arena\',\'party\'],as_index=False).votes.sum()\n    natparty[\'key\']=\'CR\';natparty[\'level\']=\'n\';party_levels.append(natparty)\n    metrics=pd.concat(metric_levels,ignore_index=True)\n    ptable=pd.concat(party_levels,ignore_index=True)\n    denom=metrics[[\'year\',\'arena\',\'level\',\'key\',\'validos\']]\n    ptable=ptable.merge(denom,on=[\'year\',\'arena\',\'level\',\'key\'],how=\'left\',validate=\'many_to_one\')\n    ptable[\'share\']=100*ptable.votes/ptable.validos.replace(0,np.nan)\n    winners=(ptable.sort_values([\'year\',\'arena\',\'level\',\'key\',\'votes\',\'party\'],ascending=[True,True,True,True,False,True])\n             .drop_duplicates([\'year\',\'arena\',\'level\',\'key\'],keep=\'first\'))\n    top2=ptable.sort_values(\'votes\',ascending=False).groupby([\'year\',\'arena\',\'level\',\'key\']).head(2)\n    # Diferencia de voto presidente/diputaciones, por distribución completa y mismas claves.\n    both=ptable[(ptable.arena.isin([\'L\',\'P\'])) & ptable.level.isin([\'c\',\'p\',\'n\'])]\n    pivot=both.pivot_table(index=[\'year\',\'level\',\'key\',\'party\'],columns=\'arena\',values=\'share\',aggfunc=\'sum\',fill_value=0).reset_index()\n    for c in [\'L\',\'P\']:\n        if c not in pivot:pivot[c]=0\n    divergence=pivot.assign(delta=lambda d:(d.P-d.L).abs()).groupby([\'year\',\'level\',\'key\'],as_index=False).delta.sum()\n    divergence[\'quiebre_pp\']=divergence.delta/2\n    # Solo variables reproducibles y ámbito correcto. No se infieren escaños cantonales.\n    mcols=[\'year\',\'arena\',\'level\',\'key\',\'validos\',\'blancos\',\'nulos\',\'abstenciones\',\'emitidos\',\'padron_analitico\',\'asistencia_estimada\',\'abst_pct\',\'part_pct\',\'blancos_pct\',\'nulos_pct\',\'noval_pct\',\'valid_pct\',\'emip_pct\']\n    for c in mcols:\n        if c not in metrics:metrics[c]=np.nan\n    metrics[mcols].to_csv(output/\'territorio_metricas.csv\',index=False,encoding=\'utf-8-sig\')\n    ptable[[\'year\',\'arena\',\'level\',\'key\',\'party\',\'votes\',\'share\']].to_csv(output/\'territorio_votos_partidarios.csv\',index=False,encoding=\'utf-8-sig\')\n    winners[[\'year\',\'arena\',\'level\',\'key\',\'party\',\'votes\',\'share\']].to_csv(output/\'territorio_liderazgos.csv\',index=False,encoding=\'utf-8-sig\')\n    divergence.to_csv(output/\'territorio_quiebre.csv\',index=False,encoding=\'utf-8-sig\')\n    # Proteger diferencias territoriales: códigos que existieron, pero no en 2026.\n    years_c=units.groupby(\'cantcode\').year.apply(lambda x: sorted(x.unique().tolist())).to_dict()\n    years_d=units.groupby(\'code\').year.apply(lambda x: sorted(x.unique().tolist())).to_dict()\n    def records(df,fields):\n        d=df[fields].copy().replace({np.nan:None})\n        return json.loads(d.to_json(orient=\'values\',force_ascii=False,double_precision=5))\n    summary={\n        \'version\':\'24.0\',\n        \'years\':YEARS,\n        \'provinces\':PROVINCES,\n        \'cantons\':records(latest_cant,[\'cantcode\',\'prov\',\'canton\']),\n        \'districts\':records(latest_dist,[\'code\',\'cantcode\',\'prov\',\'canton\',\'district\']),\n        \'canton_years\':years_c,\'district_years\':years_d,\n        \'metrics_fields\':mcols,\n        \'metrics\':records(metrics,mcols),\n        \'votes_fields\':[\'year\',\'arena\',\'level\',\'key\',\'party\',\'votes\',\'share\'],\n        \'votes\':records(ptable,[\'year\',\'arena\',\'level\',\'key\',\'party\',\'votes\',\'share\']),\n        \'leaders\':records(winners,[\'year\',\'arena\',\'level\',\'key\',\'party\',\'votes\',\'share\']),\n        \'quiebre\':records(divergence,[\'year\',\'level\',\'key\',\'quiebre_pp\']),\n        \'sources\':sources,\n        \'notes\':{\'abstencion\':\'Participación y abstención de jornada electoral conciliadas a partir de la base presidencial para los resultados de primera ronda y diputaciones.\',\n                 \'ronda\':\'Ronda 2 usa su propia abstención y sus propias papeletas. Los votos por candidatura legislativa no existen; solo votos por partido.\',\n                 \'provenance\':\'Los partidos se agrupan por rótulo electoral de la fuente, sin presumir continuidad jurídica de organizaciones homónimas.\',\n                 \'territorio\':\'Cantones y distritos se mantienen con sus códigos históricos; los territorios sin observación se distinguen de cero votos.\'}}\n    js=json.dumps(summary,ensure_ascii=False,separators=(\',\',\':\')).replace(\'</\',\'<\\\\/\')\n    (output/\'territorio_base.json\').write_text(js,encoding=\'utf-8\')\n    audit_report={\n        \'n_cantones_1953\':int(units[units.year==1953].cantcode.nunique()),\n        \'n_cantones_2026\':int(units[units.year==2026].cantcode.nunique()),\n        \'n_distritos_1953\':int(units[units.year==1953].code.nunique()),\n        \'n_distritos_2026\':int(units[units.year==2026].code.nunique()),\n        \'n_distritos_historicos\':int(units.code.nunique()),\n        \'n_cantones_historicos\':int(units.cantcode.nunique()),\n        \'segundas_rondas\':sorted(int(y) for y in src[src.arena==\'R\'].year.unique()),\n        \'n_metricas\':len(metrics),\'n_votos_agrupados\':len(ptable),\n        \'nonexisting_or_notobserved\':\'Los años sin registro de un territorio se diferencian de cero votos; se utiliza el catálogo observado por elección.\',\n        \'nota_criterios\':\'La matriz distingue ausencia territorial observada de cero votos. La abstención es común para Presidencia y Diputaciones.\',\n        \'entradas\':sources}\n    (output/\'territorio_auditoria.json\').write_text(json.dumps(audit_report,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    print(\'V24 TERRITORIO\',json.dumps({k:v for k,v in audit_report.items() if isinstance(v,(int,list))},ensure_ascii=False))\n    return summary,audit_report\n\nif __name__==\'__main__\':\n    import sys\n    assemble(sys.argv[1],Path(sys.argv[1])/\'salidas\'/\'datos\')\n'
_ATLAS24_JS = '/* Atlas Electoral V24.0 — capa adicional, preserva las visualizaciones V23.31. */\n(()=>{\n  \'use strict\';\n  if(window.atlasV24Territorial)return;\n  const node=document.getElementById(\'atlas-v24-data\');\n  if(!node)return;\n  const V=JSON.parse(node.textContent);\n  const years=V.years;\n  const era=y=>y<=1974?0:y<=1998?1:y<=2022?2:3;\n  const eraColors=[\'#a9d1c6\',\'#aec2e5\',\'#edc7a5\',\'#d4bde6\'];\n  const es=(x)=>String(x??\'\').replace(/[&<>"\']/g,s=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[s]));\n  const fmt=(n,d=1)=>Number.isFinite(Number(n))?Number(n).toLocaleString(\'es-CR\',{minimumFractionDigits:d,maximumFractionDigits:d}):\'—\';\n  const safe=(str)=>String(str??\'\').replace(/[^a-z0-9]+/gi,\'_\').slice(0,65);\n  const idx=(arr,k)=>new Map(arr.map(r=>[k(r),r]));\n  const terms={abst_pct:\'Abstencionismo\',part_pct:\'Participación\',valid_pct:\'Voto válido\',noval_pct:\'Voto no válido\',blancos_pct:\'Votos blancos\',nulos_pct:\'Votos nulos\',emip_pct:\'Votos emitidos\'};\n  const names={P:\'Presidencia · primera ronda\',L:\'Diputaciones\',R:\'Presidencia · segunda ronda\'};\n  const runoffYears=new Set(V.metrics.filter(r=>r[1]===\'R\').map(r=>r[0]));\n  const regions=new Map(V.cantons.map(r=>[r[0],{code:r[0],province:r[1],name:r[2]}]));\n  const districts=new Map(V.districts.map(r=>[r[0],{code:r[0],parent:r[1],province:r[2],canton:r[3],name:r[4]}]));\n  const mrows=V.metrics.map(r=>Object.fromEntries(V.metrics_fields.map((f,i)=>[f,r[i]])));\n  const prows=V.votes.map(r=>Object.fromEntries(V.votes_fields.map((f,i)=>[f,r[i]])));\n  const leader=V.leaders.map(r=>({year:r[0],arena:r[1],level:r[2],key:String(r[3]),party:r[4],votes:r[5],share:r[6]}));\n  const metricIndex=idx(mrows,r=>`${r.year}/${r.arena}/${r.level}/${r.key}`);\n  const leaderIndex=idx(leader,r=>`${r.year}/${r.arena}/${r.level}/${r.key}`);\n  const voteByLocation=new Map();\n  for(const r of prows){const k=`${r.year}/${r.arena}/${r.level}/${r.key}`;if(!voteByLocation.has(k))voteByLocation.set(k,[]);voteByLocation.get(k).push(r)}\n  for(const v of voteByLocation.values())v.sort((a,b)=>b.votes-a.votes);\n  const pick=(year,arena,level,key)=>metricIndex.get(`${year}/${arena}/${level}/${key}`);\n  const list=(year,arena,level,key)=>voteByLocation.get(`${year}/${arena}/${level}/${key}`)||[];\n  const color=s=>typeof semanticColor===\'function\'?semanticColor(s):\'#268b94\';\n  function dl(filename,content,mime){const url=URL.createObjectURL(new Blob([content],{type:mime}));const a=document.createElement(\'a\');a.href=url;a.download=filename;document.body.append(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),3000)}\n  function csvData(rows,filename){if(!rows?.length)return;const cols=Object.keys(rows[0]);const quote=x=>\'"\'+String(x??\'\').replace(/"/g,\'""\')+\'"\';dl(filename,\'\\ufeff\'+[cols.map(quote).join(\',\'),...rows.map(r=>cols.map(k=>quote(r[k])).join(\',\'))].join(\'\\r\\n\'),\'text/csv;charset=utf-8\')}\n  function actions(id){return `<div class="v24-actions"><button data-export="${id}:png">PNG</button><button data-export="${id}:svg">SVG</button><button data-export="${id}:csv">CSV</button></div>`}\n  function listenExport(host,id,fig,rows,filename){for(const b of host.querySelectorAll(\'[data-export^="\'+id+\':"]\'))b.addEventListener(\'click\',async()=>{const fmt=b.dataset.export.split(\':\')[1];if(fmt===\'csv\')return csvData(rows,filename+\'.csv\');if(typeof Plotly!==\'undefined\'&&fig){await Plotly.downloadImage(fig,{format:fmt,filename,width:1400,height:780,scale:fmt===\'png\'?2:1})}})}\n  function section(target,id,title,desc){if(!target||document.getElementById(id))return null;const c=document.createElement(\'section\');c.id=id;c.className=\'v24-section\';c.innerHTML=`<div class="v24-head"><div><span class="v24-eyebrow">ATLAS ELECTORAL · ANÁLISIS TERRITORIAL</span><h3>${title}</h3><p>${desc}</p></div></div>`;target.append(c);return c}\n  const select=(id,label,opts)=>`<label>${label}<select id="${id}">${opts.map(([v,t])=>`<option value="${es(v)}">${es(t)}</option>`).join(\'\')}</select></label>`;\n  function redrawPlot(id,traces,layout){const e=document.getElementById(id);if(!e||!window.Plotly)return;Plotly.react(e,traces,{font:{family:\'Arial, sans-serif\',color:\'#183646\'},paper_bgcolor:\'#fff\',plot_bgcolor:\'#fff\',margin:{l:185,r:35,t:38,b:90},...layout},{responsive:true,displaylogo:false,toImageButtonOptions:{format:\'png\',scale:2}})}\n  function bindInputs(host,callback){host.querySelectorAll(\'select,input\').forEach(n=>n.addEventListener(\'change\',callback))}\n  const renderRegions=(year,arena)=>{\n    return V.cantons.filter(r=>V.canton_years[r[0]]?.includes(year)).map(r=>r[0]);\n  };\n  function resultsPanel(){\n    const root=document.getElementById(\'content\'); const sec=section(root,\'atlas24-results\',\'Resultados por provincia, cantón y distrito\',\'Consulte el escrutinio territorial de cada elección, compare partidos y descargue los datos del territorio seleccionado. Los mapas provinciales permanecen en la vista principal.\');\n    if(!sec)return;\n    sec.innerHTML+=`<div class="v24-controls">${select(\'v24r-year\',\'Elección\',years.map(y=>[y,y]))}${select(\'v24r-arena\',\'Elección y ronda\',Object.entries(names).map(([a,n])=>[a,n]))}${select(\'v24r-level\',\'División territorial\',[[\'p\',\'Provincia\'],[\'c\',\'Cantón\'],[\'d\',\'Distrito\']])}${select(\'v24r-province\',\'Provincia\',V.provinces.map(x=>[x,x]))}${select(\'v24r-canton\',\'Cantón\',[])}${select(\'v24r-district\',\'Distrito\',[])}</div>\n    <div class="v24-kpis" id="v24r-kpis"></div><div class="v24-chart-head"><div><h4 id="v24r-title">Voto válido por partido</h4><p>Porcentajes sobre votos válidos en la arena y territorio seleccionados</p></div>${actions(\'v24r\')}</div><div id="v24r-plot" class="v24-plot"></div><div class="v24-tablewrap"><table id="v24r-table"></table></div>`;\n    const el=id=>document.getElementById(id);let current=[];\n    function populate(){let year=Number(el(\'v24r-year\').value),p=el(\'v24r-province\').value;\n      const rOption=el(\'v24r-arena\').querySelector(\'option[value="R"]\');if(rOption){rOption.disabled=!runoffYears.has(year);rOption.hidden=!runoffYears.has(year)}if(!runoffYears.has(year)&&el(\'v24r-arena\').value===\'R\')el(\'v24r-arena\').value=\'P\';\n      let cc=[...regions.values()].filter(c=>c.province===p&&V.canton_years[c.code]?.includes(year));\n      el(\'v24r-canton\').innerHTML=cc.map(c=>`<option value="${c.code}">${es(c.name)}</option>`).join(\'\');\n      const c=el(\'v24r-canton\').value;\n      el(\'v24r-district\').innerHTML=[...districts.values()].filter(d=>d.parent===c&&V.district_years[d.code]?.includes(year)).map(d=>`<option value="${d.code}">${es(d.name)}</option>`).join(\'\');\n    }\n    function draw(){\n      const y=+el(\'v24r-year\').value,a=el(\'v24r-arena\').value,l=el(\'v24r-level\').value,p=el(\'v24r-province\').value;\n      el(\'v24r-canton\').closest(\'label\').hidden=l===\'p\';el(\'v24r-district\').closest(\'label\').hidden=l!==\'d\';\n      const k=l===\'p\'?p:l===\'c\'?el(\'v24r-canton\').value:el(\'v24r-district\').value;\n      const geo=l===\'p\'?p:l===\'c\'?regions.get(k)?.name:districts.get(k)?.name;\n      const rows=list(y,a,l,k),m=pick(y,a,l,k);\n      current=rows.map((r,i)=>({posicion:i+1,anio:y,arena:names[a],provincia:p,territorio:geo||k,codigo:k,partido:r.party,votos:r.votes,porcentaje_validos:+Number(r.share||0).toFixed(4)}));\n      el(\'v24r-title\').textContent=`${names[a]} · ${geo||p} · ${y}`;\n      el(\'v24r-kpis\').innerHTML=[[\'Votos válidos\',m?.validos,0],[\'Participación\',m?.part_pct,1,\'%\'],[\'Abstencionismo\',m?.abst_pct,1,\'%\'],[\'Voto no válido\',m?.noval_pct,2,\'%\']].map(([t,n,d,suffix=\'\'])=>`<div><small>${t}</small><b>${fmt(n,d)}${suffix}</b></div>`).join(\'\');\n      el(\'v24r-table\').innerHTML=\'<thead><tr><th>Partido</th><th>Votos</th><th>% válidos</th></tr></thead><tbody>\'+rows.map(r=>`<tr><td><span class="v24-dot" style="background:${color(r.party)}"></span>${es(r.party)}</td><td>${fmt(r.votes,0)}</td><td>${fmt(r.share,2)} %</td></tr>`).join(\'\')+\'</tbody>\';\n      const drawrows=rows.slice(0,18).reverse();\n      redrawPlot(\'v24r-plot\',[{type:\'bar\',orientation:\'h\',x:drawrows.map(r=>r.share),y:drawrows.map(r=>r.party),marker:{color:drawrows.map(r=>color(r.party))},customdata:drawrows.map(r=>r.votes),hovertemplate:\'%{y}<br>%{x:.2f}% · %{customdata:,} votos<extra></extra>\',text:drawrows.map(r=>`${fmt(r.share,1)} %`),textposition:\'outside\'}],{height:Math.max(400,drawrows.length*35+135),xaxis:{title:\'Porcentaje de votos válidos\'},yaxis:{automargin:true},margin:{l:210,r:68,t:15,b:75}})\n    }\n    el(\'v24r-year\').value=state.focus;el(\'v24r-province\').value=state.province===\'Nacional\'?\'San José\':state.province;populate();draw();\n    el(\'v24r-year\').addEventListener(\'change\',()=>{populate();draw()});el(\'v24r-province\').addEventListener(\'change\',()=>{populate();draw()});\n    el(\'v24r-canton\').addEventListener(\'change\',()=>{const y=+el(\'v24r-year\').value,c=el(\'v24r-canton\').value;el(\'v24r-district\').innerHTML=[...districts.values()].filter(d=>d.parent===c&&V.district_years[d.code]?.includes(y)).map(d=>`<option value="${d.code}">${es(d.name)}</option>`).join(\'\');draw()});\n    for(const id of [\'v24r-arena\',\'v24r-level\',\'v24r-district\'])el(id).addEventListener(\'change\',draw);\n    sec.querySelectorAll(\'[data-export^="v24r:"]\').forEach(b=>b.addEventListener(\'click\',async()=>{const type=b.dataset.export.split(\':\')[1];const fn=\'resultados_\'+el(\'v24r-year\').value+\'_\'+safe(el(\'v24r-province\').value);if(type===\'csv\')return csvData(current,fn+\'.csv\');await Plotly.downloadImage(el(\'v24r-plot\'),{format:type,filename:fn,width:1450,height:Math.max(650,current.slice(0,18).length*38+150),scale:type===\'png\'?2:1})}));\n  }\n  function rankingPanel(){\n    const sec=section(document.getElementById(\'content\'),\'atlas24-rankings\',\'Comparador histórico de participación electoral\',\'Identifique territorios con los valores mayores o menores por elección. Cambie la arena, la variable y la escala sin perder las series históricas.\');if(!sec)return;\n    sec.innerHTML+=`<div class="v24-controls">${select(\'v24p-year\',\'Elección\',years.map(y=>[y,y]))}${select(\'v24p-arena\',\'Arena\',Object.entries(names).map(([a,n])=>[a,n]))}${select(\'v24p-level\',\'Unidad territorial\',[[\'p\',\'Provincias\'],[\'c\',\'Cantones\'],[\'d\',\'Distritos\']])}${select(\'v24p-metric\',\'Indicador\',Object.entries(terms).map(([k,v])=>[k,v]))}${select(\'v24p-order\',\'Orden\',[[\'top\',\'Valores mayores\'],[\'bottom\',\'Valores menores\']])}</div><div class="v24-chart-head"><div><h4 id="v24p-title"></h4><p>Porcentajes por territorio · Top 10</p></div>${actions(\'v24p\')}</div><div id="v24p-plot" class="v24-plot"></div><div class="v24-tablewrap"><table id="v24p-table"></table></div>`;\n    const el=id=>document.getElementById(id);let selected=[];\n    function label(r){return r.level===\'p\'?r.key:r.level===\'c\'?`${regions.get(String(r.key))?.name||r.key} · ${regions.get(String(r.key))?.province||\'\'}`:`${districts.get(String(r.key))?.name||r.key} · ${districts.get(String(r.key))?.canton||\'\'}`}\n    function draw(){const y=+el(\'v24p-year\').value,a=el(\'v24p-arena\').value,l=el(\'v24p-level\').value,m=el(\'v24p-metric\').value,order=el(\'v24p-order\').value;\n      const all=mrows.filter(r=>r.year===y&&r.arena===a&&r.level===l&&r[m]!=null).sort((x,z)=>order===\'top\'?z[m]-x[m]:x[m]-z[m]);\n      selected=all.map((r,i)=>({posicion:i+1,eleccion:y,arena:names[a],territorio:label(r),codigo:r.key,indicador:terms[m],porcentaje:r[m],validos:r.validos,emitidos:r.emitidos,abstenciones:r.abstenciones}));\n      const top=selected.slice(0,10);el(\'v24p-title\').textContent=`${terms[m]} · ${names[a]} · ${y}`;\n      el(\'v24p-table\').innerHTML=\'<thead><tr><th>Posición</th><th>Territorio</th><th>%</th></tr></thead><tbody>\'+top.map(r=>`<tr><td>${r.posicion}</td><td>${es(r.territorio)}</td><td><strong>${fmt(r.porcentaje,2)} %</strong></td></tr>`).join(\'\')+\'</tbody>\';\n      redrawPlot(\'v24p-plot\',[{type:\'bar\',orientation:\'h\',x:top.map(x=>x.porcentaje).reverse(),y:top.map(x=>x.territorio).reverse(),marker:{color:\'#208d95\'},text:top.map(x=>`${fmt(x.porcentaje,1)} %`).reverse(),textposition:\'outside\',hovertemplate:\'%{y}<br>%{x:.2f}%<extra></extra>\'}],{height:530,margin:{l:255,r:45,t:20,b:65},xaxis:{title:\'Porcentaje\'}})\n    }\n    el(\'v24p-year\').value=state.focus;draw();bindInputs(sec,draw);\n    sec.querySelectorAll(\'[data-export^="v24p:"]\').forEach(b=>b.addEventListener(\'click\',async()=>{let type=b.dataset.export.split(\':\')[1];if(type===\'csv\')return csvData(selected,\'ranking_territorial_\'+el(\'v24p-year\').value+\'.csv\');await Plotly.downloadImage(el(\'v24p-plot\'),{format:type,filename:\'ranking_territorial_\'+el(\'v24p-year\').value,width:1450,height:780,scale:type===\'png\'?2:1})}));\n  }\n  function cantonMatrix(){\n    const sec=section(document.getElementById(\'content\'),\'atlas24-canton\',\'Liderazgos por cantón y elección\',\'Matriz de partidos con más votos. Las provincias agrupan sus cantones y cada columna representa una elección; los años de ausencia territorial se muestran con un guion.\');if(!sec)return;\n    sec.innerHTML+=`<div class="v24-controls">${select(\'v24c-arena\',\'Arena\',[[\'L\',\'Diputaciones\'],[\'P\',\'Presidencia\']])}${select(\'v24c-level\',\'Cobertura territorial\',[[\'c\',\'84 cantones por provincia\'],[\'p\',\'7 provincias\']])}</div><div class="v24-chart-head"><div><h4>Liderazgo territorial · 1953–2026</h4><p>Color por partido ganador; consultar cifra y porcentaje colocando el cursor en cada casilla.</p></div>${actions(\'v24c\')}</div><div class="v24-matrixwrap" id="v24c-matrix"></div>`;\n    const el=id=>document.getElementById(id);let rows=[];\n    function geo(level){if(level===\'p\')return V.provinces.map(p=>({province:p,name:p,key:p}));return V.provinces.flatMap(p=>V.cantons.filter(c=>c[1]===p).map(c=>({province:p,name:c[2],key:c[0]})))}\n    const cell=(r,y,arena,l)=>{const live=l===\'p\'||V.canton_years[r.key]?.includes(y);if(!live)return null;return leaderIndex.get(`${y}/${arena}/${l}/${r.key}`)||null};\n    function makeSVG(){const g=geo(el(\'v24c-level\').value),l=el(\'v24c-level\').value,a=el(\'v24c-arena\').value;\n      const cw=68,rh=27,left=315,top=90,w=left+cw*years.length+30,h=top+g.length*rh+50;\n      let s=`<svg xmlns="http://www.w3.org/2000/svg" width="${w}" height="${h}" viewBox="0 0 ${w} ${h}"><rect width="100%" height="100%" fill="white"/><text x="16" y="26" font-family="Arial" font-size="20" font-weight="700" fill="#183646">Liderazgos territoriales · ${a===\'L\'?\'Diputaciones\':\'Presidencia\'}</text>`;\n      years.forEach((y,i)=>{s+=`<rect x="${left+i*cw}" y="53" width="${cw-2}" height="5" fill="${eraColors[era(y)]}"/><text x="${left+i*cw+cw/2}" y="77" font-family="Arial" font-size="12" fill="#34505f" text-anchor="middle">${y}</text>`});\n      let last=\'\';g.forEach((r,i)=>{const y=top+i*rh;if(r.province!==last){last=r.province;s+=`<rect x="0" y="${y}" width="100%" height="1.5" fill="#b1c7d0"/>`}\n        s+=`<text x="10" y="${y+19}" font-size="12" font-family="Arial" font-weight="${r.province===r.name?\'bold\':\'normal\'}" fill="#23475a">${es(r.name)}</text>`;\n        years.forEach((yr,j)=>{const x=left+j*cw,v=cell(r,yr,a,l),fill=v?color(v.party):\'#e8edf0\';s+=`<rect x="${x}" y="${y}" width="${cw-2}" height="${rh-2}" rx="2" fill="${fill}" stroke="#fff"/>`;if(!v)s+=`<text x="${x+cw/2}" y="${y+19}" font-size="11" text-anchor="middle" font-family="Arial" fill="#82959f">—</text>`});\n      });return s+\'</svg>\'}\n    function draw(){const g=geo(el(\'v24c-level\').value),l=el(\'v24c-level\').value,a=el(\'v24c-arena\').value;\n      rows=g.flatMap(r=>years.map(y=>{const v=cell(r,y,a,l);return {eleccion:y,arena:names[a],provincia:r.province,territorio:r.name,codigo:r.key,estado:v?\'Con resultados\':\'Sin registro territorial\',partido_ganador:v?.party||\'\',votos_ganador:v?.votes??\'\',porcentaje_validos:v?.share??\'\'}}));\n      let last=\'\',s=\'<table class="v24-matrix"><thead><tr><th>Provincia</th><th>Cantón / provincia</th>\'+years.map(y=>`<th><span style="border-bottom:4px solid ${eraColors[era(y)]}">${y}</span></th>`).join(\'\')+\'</tr></thead><tbody>\';\n      for(const r of g){if(r.province!==last){last=r.province;s+=`<tr class="v24-group"><th colspan="${2+years.length}">${es(last)}</th></tr>`}\n        s+=`<tr><td>${es(r.province)}</td><th>${es(r.name)}</th>`+years.map(y=>{const v=cell(r,y,a,l);return v?`<td class="v24-cell" title="${es(v.party)} · ${fmt(v.votes,0)} votos · ${fmt(v.share,2)}%" style="--cell:${color(v.party)}"><span>${fmt(v.share,0)}%</span></td>`:`<td class="v24-cell missing" title="Sin observación territorial">—</td>`}).join(\'\')+\'</tr>\';\n      }el(\'v24c-matrix\').innerHTML=s+\'</tbody></table>\';\n    }\n    draw();bindInputs(sec,draw);\n    sec.querySelectorAll(\'[data-export^="v24c:"]\').forEach(b=>b.addEventListener(\'click\',()=>{const t=b.dataset.export.split(\':\')[1];if(t===\'csv\')return csvData(rows,\'liderazgo_cantonal_historico.csv\');const svg=makeSVG();if(t===\'svg\')return dl(\'liderazgo_cantonal_historico.svg\',svg,\'image/svg+xml\');const blob=new Blob([svg],{type:\'image/svg+xml\'}),url=URL.createObjectURL(blob),im=new Image();im.onload=()=>{const canvas=document.createElement(\'canvas\');canvas.width=im.naturalWidth;canvas.height=im.naturalHeight;canvas.getContext(\'2d\').drawImage(im,0,0);canvas.toBlob(png=>{if(png)dl(\'liderazgo_cantonal_historico.png\',png,\'image/png\');URL.revokeObjectURL(url)})};im.src=url}));\n  }\n  function strongholdPanel(){\n    const sec=section(document.getElementById(\'content\'),\'atlas24-bastiones\',\'Bastiones electorales y ventajas territoriales\',\'Compare el porcentaje local del partido con su porcentaje nacional en la misma elección. La diferencia se expresa en puntos porcentuales y puede explorarse por cantón o distrito.\');if(!sec)return;\n    const allParties=[...new Set(prows.filter(r=>r.arena===\'P\'||r.arena===\'L\').map(r=>r.party))].sort((a,b)=>a.localeCompare(b,\'es\'));\n    sec.innerHTML+=`<div class="v24-controls">${select(\'v24b-year\',\'Elección\',years.map(y=>[y,y]))}${select(\'v24b-arena\',\'Arena\',[[\'L\',\'Diputaciones\'],[\'P\',\'Presidencia\']])}${select(\'v24b-level\',\'División\',[[\'c\',\'Cantones\'],[\'d\',\'Distritos\'],[\'p\',\'Provincias\']])}${select(\'v24b-party\',\'Partido\',allParties.map(n=>[n,n]))}</div><div class="v24-chart-head"><div><h4 id="v24b-title">Ventajas electorales territoriales</h4><p>Porcentaje territorial menos porcentaje nacional del mismo partido</p></div>${actions(\'v24b\')}</div><div id="v24b-plot" class="v24-plot"></div><div class="v24-tablewrap"><table id="v24b-table"></table></div>`;\n    const el=id=>document.getElementById(id);let current=[];\n    function lbl(l,k){return l===\'c\'?(regions.get(String(k))?.name+\' · \'+regions.get(String(k))?.province):l===\'d\'?(districts.get(String(k))?.name+\' · \'+districts.get(String(k))?.canton):k}\n    function draw(){let y=+el(\'v24b-year\').value,a=el(\'v24b-arena\').value,l=el(\'v24b-level\').value,p=el(\'v24b-party\').value;\n      let national=list(y,a,\'n\',\'CR\').find(r=>r.party===p)?.share||0;\n      const available=prows.filter(r=>r.year===y&&r.arena===a&&r.level===l&&r.party===p).map(r=>({eleccion:y,partido:p,arena:names[a],territorio:lbl(l,r.key),codigo:r.key,votos:r.votes,cuota_territorial:r.share,cuota_nacional:national,diferencia_pp:r.share-national})).sort((x,z)=>z.diferencia_pp-x.diferencia_pp);\n      current=available;const top=current.slice(0,15);el(\'v24b-title\').textContent=`${p} · ${y} · ventajas territoriales`;\n      el(\'v24b-table\').innerHTML=\'<thead><tr><th>Territorio</th><th>Votos</th><th>% local</th><th>% nacional</th><th>Ventaja (pp)</th></tr></thead><tbody>\'+top.map(r=>`<tr><td>${es(r.territorio)}</td><td>${fmt(r.votos,0)}</td><td>${fmt(r.cuota_territorial,2)} %</td><td>${fmt(r.cuota_nacional,2)} %</td><td><strong>${r.diferencia_pp>=0?\'+\':\'\'}${fmt(r.diferencia_pp,2)}</strong></td></tr>`).join(\'\')+\'</tbody>\';\n      const t=top.slice().reverse();redrawPlot(\'v24b-plot\',[{type:\'bar\',orientation:\'h\',x:t.map(r=>r.diferencia_pp),y:t.map(r=>r.territorio),marker:{color:color(p)},text:t.map(r=>`${r.diferencia_pp>=0?\'+\':\'\'}${fmt(r.diferencia_pp,1)} pp`),textposition:\'outside\',hovertemplate:\'%{y}<br>%{x:+.2f} puntos porcentuales<extra></extra>\'}],{height:600,margin:{l:290,r:75,t:20,b:65},xaxis:{title:\'Diferencia frente al porcentaje nacional (pp)\',zeroline:true,zerolinewidth:2}})\n    }\n    el(\'v24b-year\').value=state.focus;const defaultParties=prows.filter(r=>r.year===state.focus&&r.arena===\'L\'&&r.level===\'n\');el(\'v24b-party\').value=defaultParties.sort((a,b)=>b.votes-a.votes)[0]?.party||allParties[0];draw();bindInputs(sec,draw);\n    sec.querySelectorAll(\'[data-export^="v24b:"]\').forEach(b=>b.addEventListener(\'click\',async()=>{let type=b.dataset.export.split(\':\')[1];if(type===\'csv\')return csvData(current,\'bastiones_\'+el(\'v24b-year\').value+\'.csv\');await Plotly.downloadImage(el(\'v24b-plot\'),{format:type,filename:\'bastiones_\'+el(\'v24b-year\').value,width:1450,height:800,scale:type===\'png\'?2:1})}));\n  }\n  // Añade nuevos módulos a las mismas secciones; las gráficas aprobadas siguen intactas.\n  for(const [key,extras] of [[\'participacion\',[rankingPanel]],[\'gobierno\',[resultsPanel]],[\'partidos\',[cantonMatrix,strongholdPanel]]]){\n    const before=views[key];if(typeof before!==\'function\')continue;\n    views[key]=function(...args){let result=before.apply(this,args);for(const ext of extras){try{ext()}catch(err){console.error(\'Atlas V24\',key,err)}}return result};\n  }\n  window.atlasV24Territorial={version:\'24.0\',metrics:mrows.length,votes:prows.length};\n})();\n'
_ATLAS24_CSS = '/* Atlas Electoral V24.0 · presentación territorial adicional */\n:root{--v24-ink:#183646;--v24-teal:#007e83;--v24-border:#cddfe2}\n.v24-section{margin:30px 0 26px;padding:24px;background:linear-gradient(165deg,#fff 15%,#f4faf9 100%);border:1px solid #cfe0e2;border-radius:18px;box-shadow:0 10px 35px #1836460b;overflow:hidden}\n.v24-head{display:flex;align-items:start;justify-content:space-between;gap:20px;margin-bottom:20px;padding-bottom:16px;border-bottom:1px solid #dce8eb}\n.v24-eyebrow{font-size:10px;letter-spacing:2px;font-weight:800;color:var(--v24-teal)}\n.v24-head h3{font-size:22px;line-height:1.23;margin:8px 0;color:var(--v24-ink);letter-spacing:-.45px}\n.v24-head p,.v24-chart-head p{font-size:12px;line-height:1.55;color:#537080;margin:5px 0;max-width:850px}\n.v24-controls{display:flex;gap:12px;align-items:end;flex-wrap:wrap;margin:10px 0 20px;background:#e9f3f4;border:1px solid #d7e6e7;padding:15px;border-radius:11px}\n.v24-controls label{min-width:130px;max-width:320px;flex:1 1 135px;color:#355d6b;font-size:11px;font-weight:750}\n.v24-controls label select{width:100%;background:#fff;border:1px solid #bed4db;border-radius:7px;padding:9px 11px;min-height:40px}\n.v24-kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:11px;margin:10px 0 19px}\n.v24-kpis>div{border:1px solid #d5e5e7;background:#fff;border-radius:10px;padding:15px 17px;border-top:3px solid #218d91}\n.v24-kpis small{display:block;font-size:11px;color:#607b87;font-weight:600}.v24-kpis b{display:block;font-size:24px;color:#17394a;margin-top:6px;letter-spacing:-.7px}\n.v24-chart-head{display:flex;align-items:start;justify-content:space-between;gap:15px;margin-top:20px}.v24-chart-head h4{font-size:16px;margin:0;color:#1c4657}\n.v24-actions{display:flex;gap:5px;flex-shrink:0}.v24-actions button{border:1px solid #bdd5dc;background:#fff;color:#245264;border-radius:7px;padding:7px 10px;font-weight:700;font-size:10px}\n.v24-actions button:hover{color:white;background:#087d80;border-color:#087d80}\n.v24-plot{min-height:370px;width:100%;background:#fff;border:1px solid #e2ebee;border-radius:10px;margin-top:12px}\n.v24-tablewrap{max-height:440px;overflow:auto;background:white;border:1px solid #dce9ed;border-radius:9px;margin:12px 0 4px}.v24-tablewrap table{width:100%;border-collapse:collapse;font-size:12px}.v24-tablewrap th,.v24-tablewrap td{padding:9px 12px;border-bottom:1px solid #eef2f4}.v24-tablewrap th{background:#eaf4f5;position:sticky;top:0;z-index:2;text-align:left}.v24-tablewrap tr:nth-child(even){background:#f6fafb}\n.v24-dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:9px}\n.v24-matrixwrap{overflow:auto;max-height:720px;border-radius:10px;border:1px solid #d4e3e5;background:#fff;margin-top:14px}\n.v24-matrix{white-space:nowrap;border-collapse:separate;border-spacing:2px;font-size:10px}\n.v24-matrix thead th{z-index:3;top:0;position:sticky;background:#e8f0f3;min-width:57px;padding:8px 4px;text-align:center;color:#2b5060}\n.v24-matrix thead th:nth-child(1),.v24-matrix thead th:nth-child(2){text-align:left}\n.v24-matrix td:nth-child(1){background:#f2f7f8;color:#52707b;position:sticky;left:0;z-index:1;min-width:85px;padding:6px}\n.v24-matrix tbody th:not(.v24-group th){position:sticky;left:87px;z-index:1;background:#fff;padding:6px 9px;text-align:left;min-width:160px;font-size:11px}\n.v24-matrix .v24-group th{background:#d6e9e9!important;color:#194b53;text-align:left;padding:8px 10px;font-size:11px;letter-spacing:.3px}\n.v24-matrix .v24-cell{text-align:center;width:58px;min-width:58px;height:28px;color:#fff;background:var(--cell);border-radius:3px;cursor:help;text-shadow:0 1px 2px #102f4b80;font-weight:750}\n.v24-matrix .v24-cell.missing{color:#83969d;text-shadow:none;background:#edf2f4;font-weight:400}\n.v24-matrix .v24-cell:hover{filter:saturate(1.4) brightness(.88);outline:2px solid #1a6d77;outline-offset:0}\n@media(max-width:750px){.v24-section{padding:14px}.v24-kpis{grid-template-columns:repeat(2,minmax(0,1fr))}.v24-head h3{font-size:18px}.v24-chart-head{flex-wrap:wrap}.v24-kpis b{font-size:20px}}\n@media print{.v24-controls,.v24-actions{display:none}.v24-section{box-shadow:none;break-inside:avoid}.v24-matrixwrap{max-height:none}}\n'

_namespace = {'__name__':'atlas_v24_embebido'}
exec(compile(_ATLAS24_PY, 'atlas_territorio.py', 'exec'), _namespace)
_atlas24, _atlas24_audit = _namespace['assemble'](PROJECT_DIR, OUTPUT_DIR/'datos')

_final_html = OUTPUT_DIR/'Atlas_Electoral_Costa_Rica_1953_2026.html'
if not _final_html.is_file(): raise FileNotFoundError('La versión pública base debe generarse antes de la extensión .')
_page = _final_html.read_text(encoding='utf-8')
_v24_begin = '<!-- ATLAS_V24_EXTENSION_BEGIN -->'
_v24_end = '<!-- ATLAS_V24_EXTENSION_END -->'
if _v24_begin in _page:
    start=_page.index(_v24_begin)
    finish=_page.index(_v24_end,start)+len(_v24_end)
    _page=_page[:start]+_page[finish:]
_js_data=json.dumps(_atlas24,ensure_ascii=False,separators=(',',':')).replace('</','<\\/')
_content = (_v24_begin+'\n<style id="atlas-v24-css">\n'+_ATLAS24_CSS+'\n</style>\n'
            +'<script type="application/json" id="atlas-v24-data">'+_js_data+'</script>\n'
            +'<script id="atlas-v24-js">\n'+_ATLAS24_JS+'\n</script>\n'+_v24_end)
assert _page.count('</body>')==1, 'No es seguro insertar V24 en un HTML de estructura desconocida.'
_page=_page.replace('</body>',_content+'\n</body>',1)
with NamedTemporaryFile(mode='w',encoding='utf-8',dir=_final_html.parent,
                       prefix='atlas_v24_verificado_',suffix='.html',delete=False) as f:
    f.write(_page); _tmp=Path(f.name)
os.replace(_tmp,_final_html)
assert _final_html.read_text(encoding='utf-8').count(_v24_begin)==1
print('Atlas HTML generado:',_final_html)
print('Módulos nuevos: escrutinio territorial, rankings, matriz cantonal y bastiones electorales.')
print('Registros territoriales:', _atlas24_audit['n_metricas'],
      '· observaciones partidarias:',_atlas24_audit['n_votos_agrupados'])


V24 TERRITORIO {"n_cantones_1953": 65, "n_cantones_2026": 84, "n_distritos_1953": 324, "n_distritos_2026": 492, "n_distritos_historicos": 508, "n_cantones_historicos": 84, "segundas_rondas": [2002, 2014, 2018, 2022], "n_metricas": 21469, "n_votos_agrupados": 214104, "entradas": [{"archivo": "Base_Electoral_Diputaciones_1953-2026.xlsx", "hoja": "Detalle_diputaciones", "filas": 118986, "sha256": "ff423c026422dd28eeef21ef212b0c02cb6082fec663a73dfa072301a1e00a77"}, {"archivo": "Base_Electoral_Presidencial_1953-2026.xlsx", "hoja": "Detalle_primera_vuelta", "filas": 104128, "sha256": "e2a9448829f1d271f7393cb623b2d7228fbd9506b5fcf372dcce9b6e3d22c3a1"}, {"archivo": "Base_Electoral_Presidencial_1953-2026.xlsx", "hoja": "Detalle_segunda_vuelta", "filas": 9555, "sha256": "e2a9448829f1d271f7393cb623b2d7228fbd9506b5fcf372dcce9b6e3d22c3a1"}]}
Atlas HTML generado: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Módulos nuevos: escrutinio terr

## Ensamblado de integración territorial en los módulos existentes

Este paso reemplaza únicamente el módulo territorial en el HTML generado previamente; conserva las vistas históricas aprobadas y utiliza `recursos/atlas_integracion_territorial.js` y `.css`. Incluye catálogo cromático normalizado y candidaturas documentadas, sin modificar los ocho Excel de entrada.

In [49]:
'Integra los datos territoriales y sus recursos visuales en la aplicación.'
from pathlib import Path
import json,re,csv,hashlib,os,tempfile,unicodedata

ROOT=Path(PROJECT_DIR)
HTML=ROOT/'salidas/Atlas_Electoral_Costa_Rica_1953_2026.html'
JS=ROOT/'recursos/atlas_integracion_territorial.js'
CSS=ROOT/'recursos/atlas_integracion_territorial.css'
assert HTML.is_file() and JS.is_file() and CSS.is_file(), 'Falta el HTML V24 o los recursos V25'

def norm(s):
 s=''.join(c for c in unicodedata.normalize('NFD',str(s)) if unicodedata.category(c)!='Mn')
 s=''.join(c for c in s.upper() if c.isalnum())
 return s[7:] if s.startswith('PARTIDO') else s

palette={}
for item in json.loads((ROOT/'recursos/colores_partidos.json').read_text()):
 for label in (item.get('nombre'),item.get('partido')):
  if label and item.get('color'):palette[norm(label)]=item['color']
# Conservar nombres documentados; equivalencias habituales del corpus.
extra={'LIBERACIONNACIONAL':'#079648','UNIDADSOCIALCRISTIANA':'#0074B8','ACCIONCIUDADANA':'#F3BB25','FRENTEAMPLIO':'#E4B626','AGENDACIUDADANA':'#DB4C35'}
for label,col in extra.items():palette.setdefault(label,col)
# Paleta auxiliar estable para partidos sin referencia cromática documental.
# Un color por denominación normalizada, independiente de vista y elección.
import colorsys
territorial_file=ROOT/'salidas/datos/territorio_base.json'
territorial=json.loads(territorial_file.read_text(encoding='utf-8'))
all_party_names=sorted({str(r[4]) for r in territorial['votes']},key=norm)
unknown_keys=sorted({norm(x) for x in all_party_names if norm(x) not in palette})
for i,k in enumerate(unknown_keys):
 hue=((i*0.618033988749895)+0.113)%1
 light=.39+(i%4)*.038
 sat=.55+(i%3)*.035
 rgb=colorsys.hls_to_rgb(hue,light,sat)
 palette[k]='#'+''.join(f'{round(255*x):02X}' for x in rgb)

nom=[]
for ar,filename in [('P','formulas.csv'),('L','candidaturas.csv')]:
 with (ROOT/'salidas/datos'/filename).open(encoding='utf-8-sig',newline='') as f:
  for r in csv.DictReader(f):
   if not r.get('nombre'):continue
   nom.append({'year':int(r['anio']),'arena':ar,'province':r.get('provincia') or 'Nacional',
      'party':r.get('partido',''),'name':r['nombre'],'office':r.get('cargo') if ar=='P' else 'Diputación '+r.get('tipo',''),
      'elected':str(r.get('electa','')).strip().lower() in ('true','1','si','sí')})

html=HTML.read_text(encoding='utf-8')
# Idempotencia sin alterar ninguna otra extensión, incluyendo el generador base .
if 'id="atlas-v25-js"' in html:
 html=re.sub(r'<script id="atlas-v25-js">.*?</script>', '<script id="atlas-v24-js">/* módulo reemplazado para reconstrucción */</script>',html,flags=re.S)
if 'id="atlas-v25-css"' in html:
 html=html.replace('id="atlas-v25-css"','id="atlas-v24-css"')

start_tag='<script type="application/json" id="atlas-v24-data">'
pos=html.find(start_tag)
assert pos>=0, 'No hay datos territoriales V24 embebidos'
start=pos+len(start_tag)
end=html.find('</script>',start)
assert end>start
v24=json.loads(html[start:end]);v24['version']='25.0';v24['party_colors']=palette;v24['nomination_rows']=nom
blob=json.dumps(v24,ensure_ascii=False,separators=(',',':')).replace('</','<\\/')
(ROOT/'salidas/datos/territorio_integrado.json').write_text(json.dumps(v24,ensure_ascii=False,separators=(',',':')),encoding='utf-8')
html=html[:start]+blob+html[end:]
old_js=re.search(r'<script id="atlas-v24-js">.*?</script>',html,flags=re.S)
assert old_js, 'No se encontró el módulo anterior para sustituir sin duplicar'
new_js='<script id="atlas-v25-js">\n'+JS.read_text(encoding='utf-8')+'\n</script>'
html=html[:old_js.start()]+new_js+html[old_js.end():]
old_css=re.search(r'<style id="atlas-v24-css">.*?</style>',html,flags=re.S)
assert old_css
new_css='<style id="atlas-v25-css">\n'+CSS.read_text(encoding='utf-8')+'\n</style>'
html=html[:old_css.start()]+new_css+html[old_css.end():]
# Un mecanismo de Integración explícito, sin ejecutar scripts ni listeners adicionales.
anchor='else comparisonDraw();}'
if html.count(anchor)==1:
 html=html.replace(anchor,"else comparisonDraw();root.dispatchEvent(new CustomEvent('atlas-results-updated',{detail:{year:state.year,province:state.province,arena:state.arena}}));}",1)
else:
 assert html.count("root.dispatchEvent(new CustomEvent('atlas-results-updated'")==1, 'No se localizó de forma inequívoca la actualización del visor original'
# En los colores de respaldo del explorador existente se prioriza el mismo catálogo conciliado.
fall='return `hsl(${Math.abs(n)%360} 45% 46%)`;'
if fall in html:
 html=html.replace(fall,"return (window.atlasPartyColor?.(name)||'#7c90a2');",1)
else:
 assert "window.atlasPartyColor?.(name)" in html
# Presentación pública: ocho fuentes actuales, novena base prevista.
html=html.replace('Siete bases históricas','Ocho bases históricas').replace('siete bases históricas','ocho bases históricas')
html=html.replace('siete bases aportadas','ocho bases integradas').replace('Siete bases conectadas','Ocho bases integradas').replace('Siete bases relacionables','Ocho bases relacionables')
html=html.replace('Versión pública 23.31 · Recurso de consulta','Versión pública 25.0 · Recurso de consulta')
html=html.replace(' · Politólogo <span class="tag">Producto interactivo',' · Ciencia Política · Economía · Ciencia de Datos <span class="tag">Producto interactivo')
assert html.count('id="atlas-v25-js"')==1 and html.count('id="atlas-v24-js"')==0
assert html.count('id="atlas-v25-css"')==1
assert html.count('id="atlas-v24-data"')==1
# Publicar solamente cuando los artefactos están completos, mediante sustitución atómica.
fd,tmp=tempfile.mkstemp(prefix='atlas_v25_',suffix='.html',dir=HTML.parent)
try:
 with os.fdopen(fd,'w',encoding='utf-8') as f:f.write(html)
 os.replace(tmp,HTML)
except Exception:
 Path(tmp).unlink(missing_ok=True);raise

stats={'version':'25.0','proyecto':'Atlas Electoral de Costa Rica 1953-2026','n_years':len(v24['years']),
       'n_cantons':len(v24['cantons']),'n_districts':len(v24['districts']),
       'n_votes':len(v24['votes']),'n_metrics':len(v24['metrics']),
       'n_nomination_rows':len(nom),'n_catalog_colors':len(palette),
       'html_size':HTML.stat().st_size,
       'html_sha256':hashlib.sha256(HTML.read_bytes()).hexdigest(),
       'source_sha256':{str(p.name):hashlib.sha256(p.read_bytes()).hexdigest() for p in [JS,CSS]}}
(ROOT/'salidas/datos/control_integracion_territorial.json').write_text(json.dumps(stats,ensure_ascii=False,indent=2),encoding='utf-8')
print('ATLAS CONCILIADO',json.dumps({k:v for k,v in stats.items() if not k.endswith('sha256')},ensure_ascii=False))
print('HTML',HTML)


ATLAS CONCILIADO {"version": "25.0", "proyecto": "Atlas Electoral de Costa Rica 1953-2026", "n_years": 19, "n_cantons": 84, "n_districts": 508, "n_votes": 214104, "n_metrics": 21469, "n_nomination_rows": 15400, "n_catalog_colors": 136, "html_size": 110549647}
HTML /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html


## Edición Resultados históricos integrados

La aplicación mantiene las gráficas aprobadas y amplía los resultados de la misma elección dentro de la sección 07. Las matrices históricas de la sección 02 permiten comparar cantones, provincias y las diez primeras posiciones de cada elección. La sección 08 conserva la visualización provincial y permite consultar resultados cantonales y bastiones partidarios relevantes.

Las tasas de abstención y participación de la misma jornada se concilian con los consolidados presidenciales, respetando los resultados de votos válidos, nulos y blancos de cada papeleta. Las ocho bases Excel de origen no se sobrescriben.

**Ejecutar desde la carpeta del paquete completo.** Se necesitan los scripts `construir_interfaz.py`, `conciliar_resultados.py`, `completar_interfaz.py` y `publicar_aplicacion.py`, junto con `recursos/` y las ocho bases. Esta celda final publica el HTML en la máquina del autor; el notebook se entrega sin HTML precompilado para facilitar su descarga y ejecución.

In [50]:
from pathlib import Path
import subprocess, sys
_ROOT = Path(PROJECT_DIR) if 'PROJECT_DIR' in globals() else Path.cwd()
_SCRIPTS = ('construir_interfaz.py','conciliar_resultados.py','completar_interfaz.py','publicar_aplicacion.py')
for item in _SCRIPTS:
    ruta=_ROOT/item
    if not ruta.is_file():
        raise FileNotFoundError(f'Recurso de generación faltante: {ruta}. Utilice el paquete completo.')
for item in _SCRIPTS:
    subprocess.run([sys.executable, str(_ROOT/item)],cwd=str(_ROOT),check=True)
_HTML=_ROOT/'salidas/Atlas_Electoral_Costa_Rica_1953_2026.html'
_FUENTES=_ROOT/'salidas/Atlas_Bases_Electorales_Originales.zip'
assert _HTML.is_file() and _FUENTES.is_file()
print('Atlas Electoral publicado:',_HTML)
print('Fuentes originales descargables:',_FUENTES)


JS 68853 bytes
Conciliación : 304 registros provinciales/nacionales, 19 elecciones, dos arenas; 0 diferencias en indicadores comunes.
San José 1953: (33985, 115544) San José 2026: (328270, 1191100)
JS bytes 77579 CSS bytes 23197
Atlas publicado: Atlas_Electoral_Costa_Rica_1953_2026.html · fuentes: Atlas_Bases_Electorales_Originales.zip
Atlas Electoral publicado: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Fuentes originales descargables: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Bases_Electorales_Originales.zip


## Atlas Electoral Integración territorial en el mismo notebook público

Esta es la continuación directa del notebook público **no busca, lee ni ejecuta notebooks anteriores**. Las celdas previas construyen el HTML público con las bases y recursos que ya utiliza el proyecto. La celda siguiente incorpora la exploración territorial al **mismo archivo HTML**.

**Recursos adicionales**: `recursos/atlas_fastxlsx.py`, `recursos/atlas_datos_territoriales.py`, `recursos/atlas_integracion.py` y `recursos/atlas_explorador_territorial.js`. Se añaden a la carpeta `recursos/` sin sustituir los archivos . Se mantienen las ocho bases originales y los scripts de publicación ya empleados por (`construir_interfaz.py`, `conciliar_resultados.py`, `completar_interfaz.py`, `publicar_aplicacion.py`).

**Ejecución**: abrir este único `.ipynb` en la raíz del proyecto y ejecutar *Kernel → Restart Kernel and Run All Cells*. No ejecutar ningún notebook auxiliar antes ni después. La investigación de cuatro épocas es un notebook separado, no una dependencia para producir la aplicación pública.

In [51]:
# ampliación integrada al mismo HTML creado por las celdas anteriores.
# No se ejecutan ni se abren notebooks de versiones anteriores.
from pathlib import Path
import json
ROOT_V28 = Path(PROJECT_DIR).resolve()
RECURSOS_NUEVOS_V28 = (
    'atlas_fastxlsx.py',
    'atlas_datos_territoriales.py',
    'atlas_integracion.py',
    'atlas_explorador_territorial.js',
)
_faltan_v28 = [str(ROOT_V28/'recursos'/nombre) for nombre in RECURSOS_NUEVOS_V28
               if not (ROOT_V28/'recursos'/nombre).is_file()]
if _faltan_v28:
    raise FileNotFoundError('Recursos nuevos V28 ausentes en recursos/:\n'+'\n'.join(_faltan_v28))
_html_v28=ROOT_V28/'salidas'/'Atlas_Electoral_Costa_Rica_1953_2026.html'
if not _html_v28.is_file():
    raise FileNotFoundError(f'No se produjo el HTML en las celdas anteriores: {_html_v28}')
from recursos.atlas_integracion import install as integrar_v28
_v28_estado=integrar_v28(ROOT_V28,html=_html_v28)
assert _html_v28.is_file()
print('Aplicación pública integrada en:',_html_v28)
print('Validación de incorporación:',json.dumps(_v28_estado,ensure_ascii=False,indent=2))

Aplicación pública integrada en: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Validación de incorporación: {
  "html": "/Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html",
  "dataset_bytes": 6278334,
  "javascript_bytes": 26360,
  "integracion_unica": true
}


## Publicación final y presentación
La última celda reúne la aplicación completa, corrige el ciclo de vida de las gráficas, mantiene un único visor de bastiones y publica el HTML principal en la raíz. La navegación interna organiza los temas de cada sección y facilita el acceso a vistas relacionadas. La consulta de rondas se muestra únicamente en elecciones presidenciales.

Ajuste de presentación: la barra «En esta sección» ocupa su propio espacio y no flota encima de filtros, indicadores ni gráficas. No se modifica el contenido de las vistas ni se incorporan épocas a la aplicación pública.

La participación separa indicador y unidad (porcentaje o cantidad), conservando sus denominadores. El resumen histórico de primeros lugares complementa las tres modalidades de bastiones existentes. Las diferencias territoriales mantienen el detalle nominal en las descargas y usan siglas habituales en pantalla.

La consulta de apellidos utiliza campos documentados y distingue personas de candidaturas repetidas. Fórmulas presidenciales conserva los tres cargos de la fórmula nacional sin un selector de arena legislativa. Perfiles utiliza una sola selección y la bandera de referencia, con sus antecedentes y resultados integrados. Un único laboratorio reúne indicadores provinciales, escalas territoriales, dispersión, ajuste, rangos, series, distribuciones, agrupamientos y cruces categóricos. Los mapas permiten comparar dos elecciones y abrir el escrutinio provincial en la sección 7.

La presentación de fórmulas conserva exclusivamente el archivo histórico con colores y enlaces nominales. Apellidos incorpora persona de referencia y coincidencias de campos documentados. Perfiles reúne accesos por partido, visualizaciones territoriales, candidaturas, representación y personas. La cartografía calcula sus valores, rankings y series mediante un único motor; distingue brecha partidaria firmada y distancia general entre distribuciones de voto. El laboratorio añade correlaciones multivariables, componentes principales y asociación categórica descriptiva. Las tablas y gráficos nominales enlazan al explorador de personas mediante identidades documentadas o búsqueda nominal explícita. El visor y los cálculos de la sección 7 permanecen idénticos.


In [52]:
# Atlas · Integración final legible e incorporada al notebook.
# Conserva los recursos existentes; no incorpora archivos auxiliares nuevos.
from pathlib import Path
from tempfile import NamedTemporaryFile
import re, os

_ATLAS37_PROFILE_COMPLEMENT = 'function atlasPartyProfiles(){\n const container=document.getElementById(\'content\');if(!container||document.getElementById(\'atlas37-profile-territory\'))return;\n const selected=typeof local!==\'undefined\'?(local.profileparty||\'LIBERACION NACIONAL\'):\'LIBERACION NACIONAL\',api=window.Atlas32Territory;\n const sec=block(\'Primeros lugares cantonales y distritales\',\'La misma organización seleccionada arriba, sin un segundo perfil ni un segundo selector de partido.\',\'atlas37-profile-territory\');\n sec.innerHTML+=`<div class="a26-chart-title"><h4>${esc(selected)}</h4>${buttons}</div><div class="a25-tablewrap"><table data37-profile-table></table></div><p class="a26-footnote">Se cuentan primeros lugares exclusivos; los empates no se adjudican. El intervalo y la provincia siguen los filtros generales. Las candidaturas, escaños, resultados y antecedentes registrales permanecen en este mismo perfil.</p>`;\n let records=[];for(const y of years.filter(y=>y>=state.from&&y<=state.to))for(const a of [1,0]){const scope=state.province===\'Nacional\'?null:String(provinces.indexOf(state.province)+1),counts={};for(const [level,key] of [[\'c\',\'cantones\'],[\'d\',\'distritos\']]){const units=api?.leaders(y,a,1,level)||[];counts[key]=units.filter(g=>(!scope||g.id[0]===scope)&&!g.tie&&norm(api.name(g.winner))===norm(selected)).length;counts[key+\'_observados\']=units.filter(g=>!scope||g.id[0]===scope).length}records.push({anio:y,partido:selected,arena:a===1?\'Diputaciones\':\'Presidencia · primera ronda\',provincia:state.province,...counts})}\n const table=sec.querySelector(\'[data37-profile-table]\');table.innerHTML=\'<thead><tr><th>Elección</th><th>Arena</th><th>Cantones encabezados / observados</th><th>Distritos encabezados / observados</th></tr></thead><tbody>\'+records.map(r=>`<tr><th>${r.anio}</th><td>${r.arena}</td><td>${r.cantones} / ${r.cantones_observados}</td><td>${r.distritos} / ${r.distritos_observados}</td></tr>`).join(\'\')+\'</tbody>\';\n sec.querySelectorAll(\'[data-atlas-export]\').forEach(b=>b.addEventListener(\'click\',()=>{const ext=b.dataset.atlasExport;if(ext===\'csv\')return csv(records,\'Atlas_Perfil_Primeros_Lugares.csv\');window.Atlas31TableImage?.(table,\'Primeros lugares · \'+selected+\' · \'+state.from+\'–\'+state.to+\' · \'+state.province,ext,\'Atlas_Perfil_Primeros_Lugares\')}));container.append(sec);\n}\n'
_ATLAS37_PUBLIC_REFINEMENT = '/* Consultas públicas consolidadas: personas, partidos, cartografía y análisis. */\n(function(){\'use strict\';\n if(window.AtlasPublic37||typeof views===\'undefined\'||typeof D===\'undefined\')return;\n const esc=x=>String(x??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\n const norm=x=>String(x??\'\').normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').trim().toUpperCase().replace(/\\s+/g,\' \');\n const ok=x=>x!==null&&x!==undefined&&x!==\'\'&&Number.isFinite(Number(x));\n const fmt=(x,d=0)=>ok(x)?Number(x).toLocaleString(\'es-CR\',{maximumFractionDigits:d}):\'—\';\n const unique=a=>[...new Set(a)];\n const label=k=>(typeof metricLabels!==\'undefined\'&&metricLabels[k])||(typeof labels!==\'undefined\'&&labels[k])||({part_pct:\'Participación (%)\',abst_pct:\'Abstencionismo (%)\',valid_pct:\'Válidos / emitidos (%)\',noval_pct:\'No válidos / emitidos (%)\',validos:\'Votos válidos\',arena:\'Arena\',anio:\'Elección\',partido:\'Partido\',sexo:\'Sexo\',cargo:\'Cargo\',electa:\'Resultado electo\'}[k])||k;\n const color=p=>typeof col===\'function\'?col(p):\'#167e8b\';\n const range=r=>Number(r.anio)>=Number(state.from)&&Number(r.anio)<=Number(state.to);\n const root=()=>document.getElementById(\'content\');\n const field=(k,text,options,value)=>`<label>${esc(text)}<select data37="${k}">${options.map(x=>{const [v,t]=Array.isArray(x)?x:[x,x];return `<option value="${esc(v)}" ${String(v)===String(value)?\'selected\':\'\'}>${esc(t)}</option>`}).join(\'\')}</select></label>`;\n const actions=key=>`<div class="a25-actions">${[\'png\',\'svg\',\'csv\'].map(ext=>`<button type="button" data37-export="${key}:${ext}">${ext.toUpperCase()}</button>`).join(\'\')}</div>`;\n function host(id,titleText,description){const s=document.createElement(\'section\');s.id=id;s.className=\'a25-block a37-workspace\';s.innerHTML=`<div class="a25-header"><h3>${esc(titleText)}</h3><p>${esc(description)}</p></div>`;return s}\n function csv(records,name){if(!records.length)return;const keys=unique(records.flatMap(Object.keys));const quote=v=>\'"\'+String(v??\'\').replace(/"/g,\'""\')+\'"\';const blob=new Blob([\'\\ufeff\'+[keys.map(quote).join(\',\'),...records.map(r=>keys.map(k=>quote(r[k])).join(\',\'))].join(\'\\r\\n\')],{type:\'text/csv;charset=utf-8\'});const u=URL.createObjectURL(blob),a=document.createElement(\'a\');a.href=u;a.download=name+\'.csv\';a.click();setTimeout(()=>URL.revokeObjectURL(u),1000)}\n function chart(node,traces,layout,titleText){if(!window.Plotly)return;node.atlasTitle=titleText;node.atlasScope=`${state.from}–${state.to}`;const rendered=Plotly.react(node,traces,{paper_bgcolor:\'#fff\',plot_bgcolor:\'#fff\',font:{family:\'Inter,Arial,sans-serif\',size:12,color:\'#21485b\'},height:540,margin:{l:100,r:80,t:65,b:80},title:{text:titleText,font:{size:15}},...layout},{responsive:true,displaylogo:false});Promise.resolve(rendered).then(()=>{if(node.isConnected)window.Atlas38Links?.bindPlot(node)});return rendered}\n function exportChart(node,rows,name,ext){if(ext===\'csv\')return csv(rows,name);if(window.Plotly?.downloadImage)return Plotly.downloadImage(node,{format:ext,filename:name,width:1750,height:Math.max(850,node.layout?.height||0),scale:ext===\'png\'?2:1,setBackground:\'white\'})}\n function ranks(a){const sorted=a.map((v,i)=>({v,i})).sort((a,b)=>a.v-b.v),out=[];for(let i=0;i<sorted.length;){let j=i+1;while(j<sorted.length&&sorted[j].v===sorted[i].v)j++;const r=(i+j+1)/2;for(let k=i;k<j;k++)out[sorted[k].i]=r;i=j}return out}\n function stats(x,y){const n=x.length;if(!n)return {n:0,r:null,slope:null,intercept:null,r2:null};const mx=x.reduce((s,v)=>s+v,0)/n,my=y.reduce((s,v)=>s+v,0)/n;let xx=0,yy=0,xy=0;for(let i=0;i<n;i++){xx+=(x[i]-mx)**2;yy+=(y[i]-my)**2;xy+=(x[i]-mx)*(y[i]-my)}const r=xx>0&&yy>0?xy/Math.sqrt(xx*yy):null,slope=xx>0?xy/xx:null;return {n,mx,my,xx,yy,r,slope,intercept:slope===null?null:my-slope*mx,r2:r===null?null:r*r}}\n function cluster(x,y,requested){if(!x.length)return {groups:[],k:0,inertia:0};const s=stats(x,y),sx=Math.sqrt(s.xx/x.length)||1,sy=Math.sqrt(s.yy/y.length)||1,points=x.map((v,i)=>[(v-s.mx)/sx,(y[i]-s.my)/sy]);let centers=[points[0].slice()];const distance=(a,b)=>(a[0]-b[0])**2+(a[1]-b[1])**2;\n  while(centers.length<Math.min(requested,points.length)){let best=null,max=0;for(const p of points){const d=Math.min(...centers.map(c=>distance(p,c)));if(d>max){max=d;best=p}}if(!best||max<1e-12)break;centers.push(best.slice())}\n  let groups=[],iterations=0;for(;iterations<100;iterations++){groups=points.map(p=>centers.reduce((best,c,i)=>{const d=distance(p,c);return d<best.d?{i,d}:best},{i:0,d:Infinity}).i);const next=centers.map((c,k)=>{const items=points.filter((_,i)=>groups[i]===k);return items.length?[items.reduce((s,p)=>s+p[0],0)/items.length,items.reduce((s,p)=>s+p[1],0)/items.length]:c});const change=centers.reduce((s,c,k)=>s+distance(c,next[k]),0);centers=next;if(change<1e-10)break}\n  groups=points.map(p=>centers.reduce((best,c,i)=>{const d=distance(p,c);return d<best.d?{i,d}:best},{i:0,d:Infinity}).i);return {groups,k:new Set(groups).size,inertia:points.reduce((s,p,i)=>s+distance(p,centers[groups[i]]),0),iterations:iterations+1};\n }\n function quantile(arr,p){if(!arr.length)return null;const a=arr.slice().sort((x,y)=>x-y),i=(a.length-1)*p,lo=Math.floor(i);return a[lo]+(a[Math.ceil(i)]-a[lo])*(i-lo)}\n function nominalGroups(rows,pairs=false){const groups=new Map();for(const r of rows){const surnames=[r.apellido1,r.apellido2].map(x=>String(x??\'\').trim());const keys=pairs?(surnames.every(Boolean)?[{key:surnames.map(norm).join(\'|\'),name:surnames.join(\' \')}]:[]):unique(surnames.filter(Boolean).map(norm)).map(key=>({key,name:surnames.find(x=>norm(x)===key)}));for(const item of keys){if(!groups.has(item.key))groups.set(item.key,{...item,rows:[],people:new Map()});const g=groups.get(item.key);g.rows.push(r);const person=r.id_persona||norm(r.nombre);if(!g.people.has(person))g.people.set(person,[]);g.people.get(person).push(r)}}return [...groups.values()].sort((a,b)=>b.people.size-a.people.size||a.name.localeCompare(b.name,\'es\'))}\n function setFocus(year){state.focus=Number(year);state.from=Math.min(Number(state.from),state.focus);state.to=Math.max(Number(state.to),state.focus);for(const id of [\'focus\',\'from\',\'to\']){const element=document.getElementById(id);if(element)element.value=state[id]}render()}\n window.AtlasPublic37={stats,ranks,cluster,quantile,nominalGroups,finite:ok,setFocus};\n\n /* Apellidos: coincidencias documentadas, personas distintas y detalle histórico. */\n views.apellidos=function(){title(\'Apellidos y coincidencias entre personas\',\'Explore coincidencias documentadas para orientar la verificación de posibles familias políticas.\');const c=root(),sec=host(\'atlas37-apellidos\',\'Apellidos en la historia electoral\',\'Los grupos se construyen con los campos de apellidos de las bases; no se deducen de la última palabra del nombre.\');c.append(sec);\n  const source={electas:(D.electas||[]).map(r=>({...r,cargo:\'Diputación\',origen:\'Diputación electa\'})),formulas:(D.formulas||[]).map(r=>({...r,origen:\'Fórmula presidencial\'})),candidaturas:(D.candidaturas||[]).map(r=>({...r,origen:\'Candidatura legislativa\'}))};\n  source.all=[...source.candidaturas,...source.formulas];\n  const referenceRows=[...source.candidaturas,...source.formulas,...source.electas],references=[...new Map(referenceRows.filter(r=>r.nombre&&(r.apellido1||r.apellido2)).map(r=>[norm(r.nombre),r])).values()].sort((a,b)=>a.nombre.localeCompare(b.nombre,\'es\'));\n  sec.innerHTML+=`<div class="a37-toolbar">${field(\'population\',\'Registros\',[[\'all\',\'Todas las candidaturas documentadas\'],[\'ambas\',\'Diputaciones electas y fórmulas\'],[\'electas\',\'Diputaciones electas\'],[\'formulas\',\'Fórmulas presidenciales\'],[\'candidaturas\',\'Candidaturas legislativas\']],\'all\')}${field(\'surname-mode\',\'Coincidencia\',[[\'single\',\'Un apellido documentado\'],[\'pair\',\'Los dos apellidos, en el mismo orden\']],\'single\')}${field(\'minimum\',\'Personas distintas\',[[2,\'Dos o más\'],[1,\'Todas\'],[3,\'Tres o más\']],\'2\')}<label>Buscar apellido<input data37="search" type="search" placeholder="Ej.: Figueres, Calderón…"></label></div><div class="a37-toolbar"><label>Persona de referencia<input type="search" data37="reference" list="a38-reference-names" placeholder="Persona histórica, electa o candidata…"><datalist id="a38-reference-names"></datalist></label>${field(\'party\',\'Partido de las coincidencias\',[[\'all\',\'Todos los partidos\'],...unique(referenceRows.map(r=>r.partido)).sort().map(p=>[p,p])],\'all\')}<label class="a38-check"><input data37="exclude-reference" type="checkbox" checked> Mostrar otras personas</label></div><div class="a38-reference" data37="reference-card" hidden></div><div class="a26-metric-cards" data37="summary"></div><p class="a37-method">Compartir uno o dos apellidos es una coincidencia nominal; no acredita parentesco. Una persona repetida en varias elecciones cuenta una vez en el ranking. El CSV incluye todos los grupos filtrados; la gráfica muestra la página visible.</p><div class="a37-charthead"><h4>Coincidencias por personas distintas</h4>${actions(\'ranking\')}</div><div data37="ranking-plot"></div><div class="a37-toolbar"><button data37="prev" type="button">Anterior</button><span data37="page-note"></span><button data37="next" type="button">Siguiente</button></div><div class="a37-scroll" data37="ranking"></div><section class="a37-detail" data37="detail" hidden><div class="a37-charthead"><h4 data37="detail-title"></h4>${actions(\'detail\')}</div><div data37="person-plot"></div><div data37="person-list"></div></section>`;\n  const q=k=>sec.querySelector(\'[data37="\'+k+\'"]\');const saved=local.a37names||{};for(const k of [\'population\',\'surname-mode\',\'minimum\',\'search\',\'reference\',\'party\'])if(saved[k]!==undefined)q(k).value=saved[k];let all=[],summary=[],detail=[],page=0,selected=saved.selected||null,activeReference=null;\n  function rememberNames(){local.a37names=Object.fromEntries([\'population\',\'surname-mode\',\'minimum\',\'search\',\'reference\',\'party\'].map(k=>[k,q(k).value]));local.a37names.selected=selected}\n\n  function focus(key){selected=key;rememberNames();const g=all.find(x=>x.key===key);if(!g)return;detail=g.rows.map(r=>({apellido_o_par:g.name,nombre:r.nombre,anio:r.anio,partido:r.partido,provincia:r.provincia,cargo:r.cargo,origen:r.origen,id_persona:window.Atlas38Links.identity(r)||\'\',id_evento:r.id_evento||\'\',fuente:r.fuente||r.id_registro||\'\'}));q(\'detail\').hidden=false;q(\'detail-title\').textContent=g.name+\' · \'+g.people.size+\' personas nominales distintas\';\n   q(\'person-list\').innerHTML=[...g.people.values()].map(rs=>`<article class="a37-person" style="border-top:3px solid ${color(rs[0].partido)}"><h4>${window.Atlas38Links.person(rs[0])}</h4><p>${unique(rs.map(x=>x.anio)).sort((a,b)=>a-b).join(\' · \')}</p><small>${unique(rs.map(x=>x.partido)).map(p=>window.Atlas38Links.party(p)).join(\' · \')}</small><p>${esc(unique(rs.map(x=>[x.cargo,x.provincia].filter(Boolean).join(\' · \'))).join(\' / \'))}</p></article>`).join(\'\');\n   const dedup=[...new Map(detail.map(r=>[[r.nombre,r.anio,r.partido,r.cargo,r.provincia].join(\'|\'),r])).values()];const parties=unique(dedup.map(r=>r.partido));chart(q(\'person-plot\'),parties.map(p=>{const rows=dedup.filter(r=>r.partido===p);return {type:\'scatter\',mode:\'markers\',name:typeof labelText===\'function\'?labelText(p):p,x:rows.map(r=>r.anio),y:rows.map(r=>r.nombre),marker:{color:color(p),size:11},text:rows.map(r=>r.cargo+\' · \'+r.provincia),hovertemplate:\'%{y}<br>%{x} · %{text}<extra></extra>\'}}),{height:Math.max(380,g.people.size*35+150),margin:{l:280,r:60,t:60,b:90},xaxis:{title:\'Elección\',tickmode:\'array\',tickvals:Y},yaxis:{automargin:true}},\'Coincidencias documentadas · \'+g.name);\n  }\n  function draw(reset=true){rememberNames();if(reset)page=0;const population=q(\'population\').value,raw=(population===\'ambas\'?[...source.electas,...source.formulas]:source[population]).filter(r=>range(r)&&(state.province===\'Nacional\'||r.origen===\'Fórmula presidencial\'||r.provincia===state.province)),query=norm(q(\'search\').value),minimum=+q(\'minimum\').value;\n   activeReference=references.find(r=>norm(r.nombre)===norm(q(\'reference\').value))||null;const referenceSurnames=activeReference?[activeReference.apellido1,activeReference.apellido2].filter(Boolean).map(norm):[],pairMode=q(\'surname-mode\').value===\'pair\',filtered=raw.filter(r=>(q(\'party\').value===\'all\'||norm(r.partido)===norm(q(\'party\').value))&&(!activeReference||!q(\'exclude-reference\').checked||norm(r.nombre)!==norm(activeReference.nombre)));\n   q(\'reference-card\').hidden=!activeReference;q(\'reference-card\').innerHTML=activeReference?\'<strong>Referencia: \'+window.Atlas38Links.person(activeReference)+\'</strong><p>Apellidos documentados: \'+[activeReference.apellido1,activeReference.apellido2].filter(Boolean).map(esc).join(\' · \')+\'</p><small>Coincidencias nominales para contrastar con fuentes biográficas; no se certifica parentesco.</small>\':\'\';\n   all=nominalGroups(filtered,pairMode).filter(g=>g.people.size>=(activeReference?1:minimum)&&(!query||norm(g.name).includes(query))&&(!activeReference||(pairMode?g.key===referenceSurnames.join(\'|\'):referenceSurnames.includes(g.key))));summary=all.map(g=>({apellido_o_par:g.name,personas_distintas:g.people.size,registros:g.rows.length,primera_eleccion:Math.min(...g.rows.map(r=>r.anio)),ultima_eleccion:Math.max(...g.rows.map(r=>r.anio)),partidos:unique(g.rows.map(r=>r.partido)).join(\' / \')}));\n   q(\'summary\').innerHTML=[[\'Coincidencias\',all.length],[\'Personas en esos grupos\',new Set(all.flatMap(g=>[...g.people.keys()])).size],[\'Registros del universo\',raw.length],[\'Sin apellidos documentados\',raw.filter(r=>!r.apellido1&&!r.apellido2).length]].map(([label,value])=>`<div><small>${label}</small><strong>${fmt(value)}</strong></div>`).join(\'\');\n   page=Math.max(0,Math.min(page,Math.max(0,Math.ceil(all.length/20)-1)));const shown=all.slice(page*20,page*20+20);q(\'prev\').disabled=!page;q(\'next\').disabled=(page+1)*20>=all.length;q(\'page-note\').textContent=all.length?`${page*20+1}–${Math.min(all.length,(page+1)*20)} de ${all.length} coincidencias`:\'Sin coincidencias\';\n   chart(q(\'ranking-plot\'),[{type:\'bar\',orientation:\'h\',x:shown.map(g=>g.people.size).reverse(),y:shown.map(g=>g.name).reverse(),marker:{color:\'#157f8b\'},text:shown.map(g=>g.people.size).reverse(),textposition:\'outside\',cliponaxis:false}],{height:Math.max(350,shown.length*29+150),margin:{l:250,r:90,t:60,b:70},xaxis:{title:\'Personas nominales distintas\',rangemode:\'tozero\'}},\'Apellidos coincidentes · \'+state.from+\'–\'+state.to);\n   q(\'ranking\').innerHTML=\'<table class="a37-table"><thead><tr><th>Apellido o par</th><th>Personas distintas</th><th>Registros</th><th>Período</th><th>Explorar</th></tr></thead><tbody>\'+shown.map(g=>`<tr><th><button type="button" data37-surname="${esc(g.key)}">${esc(g.name)}</button></th><td>${g.people.size}</td><td>${g.rows.length}</td><td>${Math.min(...g.rows.map(r=>r.anio))}–${Math.max(...g.rows.map(r=>r.anio))}</td><td><button type="button" data37-surname="${esc(g.key)}">Ver personas y elecciones</button></td></tr>`).join(\'\')+\'</tbody></table>\';\n   if(selected&&all.some(x=>x.key===selected))focus(selected);else{q(\'detail\').hidden=true;detail=[]}\n  }\n  sec.querySelectorAll(\'select\').forEach(x=>x.addEventListener(\'change\',()=>draw()));q(\'search\').addEventListener(\'input\',()=>draw());q(\'reference\').addEventListener(\'input\',()=>{const query=norm(q(\'reference\').value);const candidates=references.filter(r=>!query||norm(r.nombre).includes(query)).slice(0,80);sec.querySelector(\'#a38-reference-names\').innerHTML=candidates.map(r=>`<option value="${esc(r.nombre)}"></option>`).join(\'\')});q(\'reference\').addEventListener(\'change\',()=>{selected=null;draw()});q(\'exclude-reference\').addEventListener(\'change\',()=>draw());q(\'prev\').onclick=()=>{page--;draw(false)};q(\'next\').onclick=()=>{page++;draw(false)};\n  sec.addEventListener(\'click\',e=>{const surname=e.target.closest(\'[data37-surname]\');if(surname){focus(surname.dataset[\'37Surname\']||surname.getAttribute(\'data37-surname\'));q(\'detail\').scrollIntoView?.({behavior:\'smooth\',block:\'start\'})}const b=e.target.closest(\'[data37-export]\');if(b){const [target,ext]=b.getAttribute(\'data37-export\').split(\':\');exportChart(q(target===\'ranking\'?\'ranking-plot\':\'person-plot\'),target===\'ranking\'?summary:detail,\'Atlas_Apellidos_\'+target,ext)}});draw();\n };\n\n /* Perfiles: un selector y una bandera, con los contenidos históricos conservados. */\n const profilesBase=views.perfiles;\n views.perfiles=function(...args){if(window.AtlasV26SelectedParty){const names=unique([...(D.panel||[]),...(D.candidaturas||[]),...(D.formulas||[])].map(r=>r.partido));const match=names.find(p=>norm(p).replace(/^PARTIDO /,\'\')===norm(window.AtlasV26SelectedParty).replace(/^PARTIDO /,\'\'));if(match)local.profileparty=match;window.AtlasV26SelectedParty=null}return profilesBase.apply(this,args)};\n\n /* Laboratorio único: conservar los cruces originales y reunirlos con las escalas territoriales. */\n views.laboratorio=function(){title(\'Laboratorio de análisis electoral\',\'Un espacio para explorar indicadores, distribuciones y cruces de los datos observados.\');const c=root(),sec=host(\'atlas37-laboratorio\',\'Exploración y contraste de datos\',\'Seleccione el universo y después el método. Las estadísticas y las descargas usan la misma selección completa.\');c.append(sec);\n  const node=document.getElementById(\'atlas-v24-data\'),V=window.Atlas37BasePayload||(window.Atlas37BasePayload=node?JSON.parse(node.textContent):null);const cantonMap=new Map((V?.cantons||[]).map(c=>[String(c[0]),c])),districtMap=new Map((V?.districts||[]).map(d=>[String(d[0]),d]));\n  const territory=window.Atlas37TerritoryRows||(window.Atlas37TerritoryRows=V?(V.metrics||[]).map(r=>Object.fromEntries(V.metrics_fields.map((f,i)=>[f,r[i]]))).map(r=>({...r,anio:r.year,provincia:r.level===\'p\'?r.key:r.level===\'c\'?(cantonMap.get(String(r.key))?.[1]||\'\'):(districtMap.get(String(r.key))?.[2]||\'\'),territorio:r.level===\'p\'?r.key:r.level===\'c\'?(cantonMap.get(String(r.key))?.[2]||r.key):(districtMap.get(String(r.key))?.slice(3,5).reverse().join(\' · \')||r.key)})):[]);\n  const sets={territorios:{name:\'Participación · provincias, cantones y distritos\',rows:territory},metricas:{name:\'Competencia e indicadores provinciales\',rows:(D.metricas||[]).filter(r=>r.provincia!==\'Nacional\'&&Number(r.ronda)===1)},panel:{name:\'Partidos × provincia × elección\',rows:(D.panel||[]).filter(r=>r.provincia!==\'Nacional\')},candidaturas:{name:\'Candidaturas legislativas\',rows:D.candidaturas||[]},electas:{name:\'Diputaciones electas\',rows:D.electas||[]},formulas:{name:\'Fórmulas presidenciales\',rows:D.formulas||[]},eventos:{name:\'Participaciones y trayectorias\',rows:D.eventos||[]}};\n  sec.innerHTML+=`<div class="a37-toolbar">${field(\'source\',\'Universo\',Object.entries(sets).map(([k,v])=>[k,v.name]),\'territorios\')}${field(\'arena\',\'Elección\',[[\'L\',\'Diputaciones\'],[\'P\',\'Presidencia · primera ronda\'],[\'R\',\'Presidencia · segunda ronda\']],\'L\')}${field(\'scale\',\'Unidad territorial\',[[\'p\',\'Provincia × elección\'],[\'c\',\'Cantón × elección\'],[\'d\',\'Distrito × elección\']],\'c\')}${field(\'lab-party\',\'Partido\',[[\'all\',\'Todos los partidos\']],\'all\')}${field(\'task\',\'Método\',[[\'scatter\',\'Dispersión y ajuste lineal\'],[\'trend\',\'Evolución: mediana y cuartiles\'],[\'box\',\'Distribuciones por provincia\'],[\'cluster\',\'Agrupamiento K-means\'],[\'cross\',\'Cruce de categorías\'],[\'correlation\',\'Matriz de correlaciones\'],[\'pca\',\'Componentes principales (PCA)\']],\'scatter\')}</div><div class="a37-toolbar">${field(\'x\',\'Variable X\',[],\'\')}${field(\'y\',\'Variable Y\',[],\'\')}${field(\'row\',\'Filas\',[],\'\')}${field(\'column\',\'Columnas\',[],\'\')}${field(\'cross-unit\',\'Medida del cruce\',[[\'count\',\'Cantidad de registros\'],[\'percent\',\'Porcentaje por fila\']],\'count\')}${field(\'column-page\',\'Bloque de columnas\',[[0,\'1\']],\'0\')}${field(\'color\',\'Color de los puntos\',[[\'provincia\',\'Provincia\'],[\'anio\',\'Elección\'],[\'partido\',\'Partido\']],\'provincia\')}${field(\'k\',\'Número de grupos K\',[2,3,4,5,6],\'3\')}</div><div class="a38-multivariables" data37="multi" hidden><strong>Variables del análisis multivariable (2–8)</strong><div data37="features"></div></div><div class="a26-metric-cards" data37="stats"></div><p class="a37-method" data37="note"></p><div class="a37-charthead"><h4 data37="plot-title"></h4>${actions(\'lab\')}</div><div data37="plot"></div><div data37="additional"></div><details class="a37-records"><summary>Datos de la selección y criterios de cálculo</summary><p data37="coverage"></p><div class="a37-scroll" data37="table"></div><button type="button" data37="more">Mostrar más filas</button><p>Los resultados describen asociaciones y agrupamientos de agregados. No identifican causalidad, conductas individuales ni independencia entre observaciones repetidas. K-means estandariza ambas variables y utiliza una inicialización determinista.</p></details>`;\n  const q=k=>sec.querySelector(\'[data37="\'+k+\'"]\');const saved=local.a37lab||{};for(const k of [\'source\',\'arena\',\'scale\',\'task\',\'cross-unit\',\'k\',\'color\'])if(saved[k]!==undefined)q(k).value=saved[k];let output=[],page=1,restoring=true;\n  function rememberLab(){local.a37lab=Object.fromEntries([\'source\',\'arena\',\'scale\',\'task\',\'x\',\'y\',\'row\',\'column\',\'cross-unit\',\'column-page\',\'k\',\'color\',\'lab-party\'].map(k=>[k,q(k).value]));local.a37lab.features=[...q(\'features\').querySelectorAll(\'input:checked\')].map(x=>x.value)}\n\n  const arenaOf=r=>r.arena===\'Legislativa\'?\'L\':r.arena===\'Presidencial\'?(Number(r.ronda)===2?\'R\':\'P\'):r.arena;\n  function base(){const source=q(\'source\').value,a=q(\'arena\').value;return sets[source].rows.filter(r=>range(r)&&(state.province===\'Nacional\'||r.provincia===state.province||source===\'formulas\')&&(source!==\'territorios\'||r.level===q(\'scale\').value)&&(source===\'panel\'||!r.arena||arenaOf(r)===a)&&(q(\'lab-party\').value===\'all\'||!r.partido||norm(r.partido)===norm(q(\'lab-party\').value)))}\n  function options(node,items,fallback){const prev=node.value;node.innerHTML=items.map(k=>`<option value="${esc(k)}">${esc(label(k))}</option>`).join(\'\');node.value=items.includes(prev)?prev:items.includes(fallback)?fallback:items[0]||\'\'}\n  function configure(){const rows=sets[q(\'source\').value].rows,numeric=unique(rows.slice(0,500).flatMap(Object.keys)).filter(k=>![\'anio\',\'year\',\'ronda\',\'orden\',\'id\',\'key\'].includes(k)&&rows.some(r=>typeof r[k]===\'number\'&&ok(r[k]))),dims=[\'anio\',\'provincia\',\'partido\',\'sexo\',\'tipo\',\'cargo\',\'electa\',\'experiencia_previa\',\'arena\'].filter(k=>rows.some(r=>r[k]!==undefined));\n   const partyNames=unique(rows.map(r=>r.partido).filter(Boolean)).sort((a,b)=>a.localeCompare(b,\'es\'));const formerParty=restoring?saved[\'lab-party\']:q(\'lab-party\').value;q(\'lab-party\').innerHTML=\'<option value="all">Todos los partidos</option>\'+partyNames.map(p=>`<option value="${esc(p)}">${esc(typeof labelText===\'function\'?labelText(p):p)}</option>`).join(\'\');q(\'lab-party\').value=partyNames.includes(formerParty)?formerParty:\'all\';\n   options(q(\'x\'),numeric,q(\'source\').value===\'metricas\'?\'nep\':\'abst_pct\');options(q(\'y\'),numeric,q(\'source\').value===\'metricas\'?\'abstencion_pct\':\'noval_pct\');options(q(\'row\'),dims,\'anio\');options(q(\'column\'),dims,dims.includes(\'sexo\')?\'sexo\':dims.includes(\'partido\')?\'partido\':\'provincia\');const selectedFeatures=restoring&&Array.isArray(saved.features)?saved.features:[q(\'x\').value,q(\'y\').value,...numeric.slice(0,3)];q(\'features\').innerHTML=numeric.map(k=>`<label><input type="checkbox" value="${esc(k)}" ${selectedFeatures.includes(k)?\'checked\':\'\'}> ${esc(label(k))}</label>`).join(\'\');if(restoring){for(const k of [\'x\',\'y\',\'row\',\'column\'])if(saved[k]!==undefined&&[...q(k).options].some(o=>o.value===saved[k]))q(k).value=saved[k];restoring=false}if(numeric.length<2)q(\'task\').value=\'cross\';draw();\n  }\n  function table(){const cols=Object.keys(output[0]||{});q(\'table\').innerHTML=\'<table class="a37-table"><thead><tr>\'+cols.map(k=>\'<th>\'+esc(label(k))+\'</th>\').join(\'\')+\'</tr></thead><tbody>\'+output.slice(0,page*150).map(r=>\'<tr>\'+cols.map(k=>\'<td>\'+esc(typeof r[k]===\'number\'?fmt(r[k],3):r[k]??\'—\')+\'</td>\').join(\'\')+\'</tr>\').join(\'\')+\'</tbody></table>\';q(\'more\').hidden=output.length<=page*150}\n  function draw(){page=1;const rows=base(),task=q(\'task\').value,x=q(\'x\').value,y=q(\'y\').value,source=q(\'source\').value,cross=task===\'cross\',one=task===\'trend\'||task===\'box\',multi=[\'correlation\',\'pca\'].includes(task);q(\'scale\').closest(\'label\').hidden=source!==\'territorios\';q(\'arena\').closest(\'label\').hidden=![\'territorios\',\'metricas\'].includes(source);\n   [\'row\',\'column\',\'cross-unit\',\'column-page\'].forEach(k=>q(k).closest(\'label\').hidden=!cross);q(\'x\').closest(\'label\').hidden=cross||one||multi;q(\'y\').closest(\'label\').hidden=cross||multi;q(\'multi\').hidden=!multi;q(\'k\').closest(\'label\').hidden=task!==\'cluster\';q(\'color\').closest(\'label\').hidden=task!==\'scatter\';q(\'lab-party\').closest(\'label\').hidden=!sets[source].rows.some(r=>r.partido);let traces=[],layout={},cards=[],note=\'\',eligible=0;q(\'additional\').innerHTML=\'\';\n   if(multi){const selected=[...q(\'features\').querySelectorAll(\'input:checked\')].map(x=>x.value).slice(0,8);if(selected.length<2){output=[];note=\'Seleccione al menos dos variables numéricas.\';traces=[];cards=[[\'Variables\',selected.length]]}\n    else if(task===\'correlation\'){output=window.Atlas38Math.correlation(rows,selected);eligible=rows.length;const lookup=new Map(output.map(r=>[JSON.stringify([r.variable_x,r.variable_y]),r]));traces=[{type:\'heatmap\',x:selected.map(label),y:selected.map(label),z:selected.map(a=>selected.map(b=>lookup.get(JSON.stringify([a,b])).coeficiente)),customdata:selected.map(a=>selected.map(b=>lookup.get(JSON.stringify([a,b])).n)),zmin:-1,zmax:1,colorscale:[[0,\'#925789\'],[.5,\'#f6f8fa\'],[1,\'#147f8c\']],hovertemplate:\'%{y} × %{x}<br>r=%{z:.3f}<br>N pares completos=%{customdata}<extra></extra>\',colorbar:{title:{text:\'Pearson r\'}}}];layout={height:Math.max(500,selected.length*55+160),margin:{l:240,r:100,t:65,b:170},xaxis:{tickangle:-35,automargin:true},yaxis:{automargin:true}};cards=[[\'Variables\',selected.length],[\'Registros filtrados\',rows.length],[\'Mínimo N por cruce\',Math.min(...output.map(r=>r.n))]];note=\'Correlaciones de Pearson con exclusión de faltantes por par; cada celda conserva su propio N. Una diagonal constante o sin variación queda sin coeficiente. El CSV incluye todos los pares.\'}\n    else{const result=window.Atlas38Math.pca(rows,selected);eligible=result.n;output=result.rows.map(r=>({anio:r.anio,provincia:r.provincia,territorio:r.territorio||r.provincia,nombre:r.nombre||\'\',partido:r.partido||\'\',pc1:r.pc1,pc2:r.pc2,...Object.fromEntries(selected.map(k=>[k,r[k]]))}));traces=[{type:\'scattergl\',mode:\'markers\',x:result.rows.map(r=>r.pc1),y:result.rows.map(r=>r.pc2),text:result.rows.map(r=>r.nombre||`${r.anio} · ${r.territorio||r.provincia}${r.partido?\' · \'+r.partido:\'\'}`),customdata:result.rows.map(r=>({nombre:r.nombre,id_persona:window.Atlas38Links.identity(r)})),marker:{size:8,opacity:.7,color:result.rows.map(r=>typeof provinceColors!==\'undefined\'?provinceColors[r.provincia]||\'#167e8b\':\'#167e8b\')},hovertemplate:\'%{text}<br>CP1: %{x:.3f}<br>CP2: %{y:.3f}<extra></extra>\'}];layout={xaxis:{title:\'CP1 · \'+fmt(result.variance[0],1)+\' % de varianza\'},yaxis:{title:\'CP2 · \'+fmt(result.variance[1],1)+\' % de varianza\'}};cards=[[\'Casos completos\',result.n],[\'Variables utilizadas\',result.loadings.length],[\'Varianza de CP1 + CP2\',fmt((result.variance[0]||0)+(result.variance[1]||0),1)+\' %\']];note=\'PCA sobre variables estandarizadas, con casos completos de la selección; proyección descriptiva, sin clasificación política ni inferencia causal. \'+(result.omitted.length?\'Variables constantes omitidas: \'+result.omitted.map(label).join(\', \')+\'.\':\'\');\n     q(\'additional\').innerHTML=\'<details class="a38-loadings" open><summary>Coeficientes de las componentes y varianza explicada</summary><table class="a37-table"><thead><tr><th>Variable</th><th>CP1</th><th>CP2</th><th>Media</th><th>Desviación</th></tr></thead><tbody>\'+result.loadings.map(r=>`<tr><th>${esc(label(r.variable))}</th><td>${fmt(r.pc1,4)}</td><td>${fmt(r.pc2,4)}</td><td>${fmt(r.media,3)}</td><td>${fmt(r.desviacion,3)}</td></tr>`).join(\'\')+\'</tbody></table><button type="button" data-a38-loadings>Descargar coeficientes CSV</button></details>\';q(\'additional\').querySelector(\'button\').onclick=()=>csv(result.loadings,\'Atlas_PCA_Coeficientes\');\n    }\n   }\n\n   else if(cross){const row=q(\'row\').value,column=q(\'column\').value,groups=new Map(),categories=unique(rows.map(r=>String(r[column]??\'Sin dato\'))).sort((a,b)=>a.localeCompare(b,\'es\',{numeric:true}));for(const r of rows){const a=String(r[row]??\'Sin dato\'),b=String(r[column]??\'Sin dato\');if(!groups.has(a))groups.set(a,new Map());const g=groups.get(a);g.set(b,(g.get(b)||0)+1)}\n    const pages=Math.max(1,Math.ceil(categories.length/20)),former=Math.min(+(q(\'column-page\').value||0),pages-1);q(\'column-page\').innerHTML=Array.from({length:pages},(_,i)=>`<option value="${i}">${i*20+1}–${Math.min(categories.length,(i+1)*20)}</option>`).join(\'\');q(\'column-page\').value=former;\n    const shown=categories.slice(former*20,former*20+20),keys=[...groups.keys()].sort((a,b)=>a.localeCompare(b,\'es\',{numeric:true})),percent=q(\'cross-unit\').value===\'percent\';output=[];for(const a of keys){const g=groups.get(a),total=[...g.values()].reduce((s,n)=>s+n,0);for(const b of categories)output.push({fila:a,columna:b,registros:g.get(b)||0,total_fila:total,porcentaje:100*(g.get(b)||0)/total})}\n    const lookup=new Map(output.map(r=>[JSON.stringify([r.fila,r.columna]),r]));traces=[{type:\'heatmap\',x:shown,y:keys,z:keys.map(a=>shown.map(b=>lookup.get(JSON.stringify([a,b]))[percent?\'porcentaje\':\'registros\'])),colorscale:\'Teal\',hovertemplate:\'%{y} · %{x}<br>%{z:.2f}\'+(percent?\' %\':\' registros\')+\'<extra></extra>\'}];layout={height:Math.max(450,keys.length*26+180),xaxis:{title:label(column),automargin:true},yaxis:{title:label(row),automargin:true}};const association=window.Atlas38Math.categorical(output);cards=[[\'Registros\',rows.length],[\'Filas\',keys.length],[\'Categorías\',categories.length],[\'V de Cramér\',fmt(association.cramer,3)]];eligible=rows.length;note=\'Los porcentajes usan todas las categorías de cada fila, incluso fuera del bloque visible. El CSV incluye el cruce completo.\';\n   }else{const complete=rows.filter(r=>ok(r[y])&&(one||ok(r[x]))),xx=complete.map(r=>Number(r[x])),yy=complete.map(r=>Number(r[y])),s=stats(xx,yy);eligible=complete.length;output=complete.map((r,i)=>({anio:r.anio,provincia:r.provincia,territorio:r.territorio||r.provincia,partido:r.partido||\'\',arena:arenaOf(r)||\'\',variable_x:one?\'anio\':x,valor_x:one?r.anio:xx[i],variable_y:y,valor_y:yy[i]}));const context=complete.map(r=>`${r.anio} · ${r.territorio||r.provincia||\'\'}${r.partido?\' · \'+r.partido:\'\'}`);\n    if(task===\'trend\'){const ys=unique(complete.map(r=>Number(r.anio))).sort((a,b)=>a-b),sets=ys.map(y0=>complete.filter(r=>Number(r.anio)===y0).map(r=>Number(r[y])));traces=[{type:\'scatter\',x:ys,y:sets.map(a=>quantile(a,.25)),line:{width:0},showlegend:false,hoverinfo:\'skip\'},{type:\'scatter\',x:ys,y:sets.map(a=>quantile(a,.75)),line:{width:0},fill:\'tonexty\',fillcolor:\'rgba(20,133,145,.15)\',name:\'50 % central\'},{type:\'scatter\',mode:\'lines+markers\',x:ys,y:sets.map(a=>quantile(a,.5)),customdata:sets.map(a=>a.length),line:{color:\'#147f8c\'},name:\'Mediana\',hovertemplate:\'%{x}<br>Mediana: %{y:.3f}<br>N=%{customdata}<extra></extra>\'}];layout.xaxis={title:\'Elección\',tickmode:\'array\',tickvals:ys};cards=[[\'Observaciones completas\',eligible],[\'Elecciones\',ys.length]];note=\'Mediana sin ponderar y cuartiles de observaciones territoriales: la banda representa dispersión, no incertidumbre estadística.\'}\n    else if(task===\'box\'){traces=unique(complete.map(r=>r.provincia)).map((p,i)=>{const g=complete.filter(r=>r.provincia===p);return {type:\'box\',name:p,y:g.map(r=>Number(r[y])),text:g.map(r=>String(r.anio)),boxpoints:\'all\',jitter:.2,marker:{color:typeof provinceColors!==\'undefined\'?provinceColors[p]:\'#167e8b\'},hovertemplate:\'%{text}<br>%{y:.3f}<extra>\'+esc(p)+\'</extra>\'}});cards=[[\'Observaciones completas\',eligible],[\'Provincias\',traces.length]];note=\'Cada punto conserva su año. Las cajas resumen mediana y cuartiles; los bigotes describen dispersión.\'}\n    else{let colors=complete.map(r=>q(\'color\').value===\'anio\'?`hsl(${210-170*(Number(r.anio)-1953)/73} 60% 42%)`:q(\'color\').value===\'partido\'&&r.partido?color(r.partido):typeof provinceColors!==\'undefined\'?provinceColors[r.provincia]||\'#167e8b\':\'#167e8b\');if(task===\'cluster\'){const k=cluster(xx,yy,+q(\'k\').value);colors=k.groups.map(i=>[\'#147f8c\',\'#b87945\',\'#7957a1\',\'#269672\',\'#b5547a\',\'#7e8d36\'][i]);output.forEach((r,i)=>r.grupo=k.groups[i]+1);cards=[[\'Observaciones completas\',eligible],[\'Grupos efectivos\',k.k],[\'Inercia estandarizada\',fmt(k.inertia,3)],[\'Iteraciones\',k.iterations]];note=\'K-means sobre variables estandarizadas. Se informa el número efectivo de grupos; no se descartan filas para dibujar.\'}else{const spearman=s.n?stats(ranks(xx),ranks(yy)).r:null;cards=[[\'Pares completos\',eligible],[\'Pearson r\',fmt(s.r,3)],[\'Spearman ρ\',fmt(spearman,3)],[\'R² descriptivo\',fmt(s.r2,3)],[\'Pendiente\',fmt(s.slope,3)]];note=\'Los ceros observados se conservan; los valores ausentes se excluyen. Correlación y ajuste descriptivos, con observaciones repetidas por territorio.\'}\n     traces=[{type:\'scattergl\',mode:\'markers\',x:xx,y:yy,text:context,marker:{color:colors,size:7,opacity:.7},name:\'Observaciones\',hovertemplate:\'%{text}<br>X: %{x:.3f}<br>Y: %{y:.3f}<extra></extra>\'}];if(task===\'scatter\'&&s.slope!==null){const lo=Math.min(...xx),hi=Math.max(...xx);traces.push({type:\'scatter\',mode:\'lines\',x:[lo,hi],y:[s.intercept+s.slope*lo,s.intercept+s.slope*hi],name:\'Ajuste lineal\',line:{color:\'#b87945\',dash:\'dash\'}})}layout.xaxis={title:label(x)};\n    }layout.yaxis={title:label(y),automargin:true};\n   }\n   const method=q(\'task\').selectedOptions[0].textContent,titleText=method+\' · \'+sets[source].name+\' · \'+state.from+\'–\'+state.to;q(\'plot-title\').textContent=method;q(\'stats\').innerHTML=cards.map(([k,v])=>`<div><small>${esc(k)}</small><strong>${esc(v)}</strong></div>`).join(\'\');q(\'note\').textContent=eligible?note:\'Sin observaciones completas para esta selección. Cambie el universo o las variables.\';q(\'coverage\').textContent=`${eligible} observaciones analizadas de ${rows.length} registros filtrados. Tabla paginada; el CSV conserva la selección completa.`;chart(q(\'plot\'),traces,layout,titleText);table();rememberLab();\n  }\n  q(\'features\').addEventListener(\'change\',e=>{if(q(\'features\').querySelectorAll(\'input:checked\').length>8)e.target.checked=false;draw()});q(\'source\').addEventListener(\'change\',configure);sec.querySelectorAll(\'select:not([data37="source"])\').forEach(s=>s.addEventListener(\'change\',draw));q(\'more\').onclick=()=>{page++;table()};sec.querySelectorAll(\'[data37-export]\').forEach(b=>b.addEventListener(\'click\',()=>exportChart(q(\'plot\'),output,\'Atlas_Laboratorio_\'+q(\'source\').value+\'_\'+q(\'task\').value,b.getAttribute(\'data37-export\').split(\':\')[1])));configure();\n };\n})();\n\n/* Navegación contextual sobre identidades documentadas. */\n(function(){\'use strict\';\n const esc=x=>String(x??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\n const norm=x=>String(x??\'\').normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toUpperCase().replace(/\\s+/g,\' \').trim();\n const people=new Map(),events=new Map(),parties=new Map();\n for(const r of [...(D.resultados_personas||[]),...(D.personas||[]),...(D.resultados_candidaturas||[]),...(D.eventos||[])]){if(!r.nombre)continue;const k=norm(r.nombre);if(!people.has(k))people.set(k,new Map());if(r.id_persona)people.get(k).set(r.id_persona,r);if(r.id_evento&&r.id_persona)events.set(r.id_evento,r.id_persona)}\n for(const r of [...(D.panel||[]),...(D.candidaturas||[]),...(D.formulas||[])])if(r.partido)parties.set(norm(r.partido),r.partido);\n const identity=r=>r.id_persona||events.get(r.id_evento)||(people.get(norm(r.nombre))?.size===1?[...people.get(norm(r.nombre)).keys()][0]:null);\n function openPerson(r){const pid=identity(r);for(const k of [\'personquery\',\'credyear\',\'credparty\',\'credrole\',\'credresult\',\'credprovince\',\'credentialpage\'])delete local[k];local.credentialscope=\'Todas\';if(pid){local.credentialperson=pid;if(typeof atlasOpenPerson===\'function\')atlasOpenPerson(pid);else gotoTab(\'credenciales\')}else{local.credentialperson=\'\';local.personquery=r.nombre||\'\';gotoTab(\'credenciales\')}}\n function route(tab,party,province,year){if(Number.isFinite(+year)&&+year>=1953&&+year<=2026){state.focus=+year;state.from=Math.min(+state.from,+year);state.to=Math.max(+state.to,+year)}if(province&&P.includes(province))state.province=province;\n  const selected=party&&parties.get(norm(party));if(selected){const fields={perfiles:[\'profileparty\'],dualidad:[\'dp\'],candidaturas:[\'cp\'],genero:[\'gp\'],formulas:[\'fp\'],partidos:[\'pt\'],mapas:[\'mparty\'],trayectorias:[\'tp\'],credenciales:[\'credparty\'],outcomes:[\'oparty\'],cancelaciones:[\'reuse\']};for(const key of fields[tab]||[])local[key]=selected;if(tab===\'mapas\')local.a38map={...(local.a38map||{}),party:selected,province:province||state.province,mode:\'value\',measure:\'share\'};if(tab===\'laboratorio\')local.a37lab={...(local.a37lab||{}),source:\'panel\',\'lab-party\':selected,task:\'scatter\',x:\'porcentaje_Legislativa\',y:\'porcentaje_Presidencial\'};if(tab===\'apellidos\')local.a37names={...(local.a37names||{}),party:selected};}\n  for(const id of [\'focus\',\'from\',\'to\',\'province\']){const node=document.getElementById(id);if(node)node.value=state[id]}gotoTab(tab);if(selected)setTimeout(()=>{const selector=document.getElementById(\'a26-n-party\');if(selector){const option=[...selector.options].find(o=>norm(o.value)===norm(selected));if(option){selector.value=option.value;selector.dispatchEvent(new Event(\'change\',{bubbles:true}))}}},0);\n }\n const button=(text,attrs)=>`<button type="button" class="a38-link" ${attrs}>${esc(text)}</button>`;\n const person=r=>button(r.nombre||\'\',`data-a38-person="${esc(identity(r)||\'\')}" data-a38-name="${esc(r.nombre||\'\')}"`);\n const party=(name,tab=\'perfiles\',year=\'\',province=\'\')=>button(typeof labelText===\'function\'?labelText(name):name,`data-a38-tab="${tab}" data-a38-party="${esc(name)}" data-a38-year="${esc(year)}" data-a38-province="${esc(province)}"`);\n const link=(tab,text,partyName=\'\',province=\'\',year=\'\')=>button(text,`data-a38-tab="${esc(tab)}" data-a38-party="${esc(partyName)}" data-a38-province="${esc(province)}" data-a38-year="${esc(year)}"`);\n function bindPlot(node){if(!node?.on)return;if(node.__a38Link)node.removeListener?.(\'plotly_click\',node.__a38Link);node.__a38Link=e=>{const p=e.points?.[0];if(!p)return;const candidates=[p.customdata?.nombre,p.y,p.text,p.data?.text?.[p.pointIndex]];for(const name of candidates){if(typeof name!==\'string\'||!people.has(norm(name)))continue;openPerson({nombre:name,id_persona:p.customdata?.id_persona});break}};node.on(\'plotly_click\',node.__a38Link);for(const text of node.querySelectorAll?.(\'.ytick text,.xtick text\')||[]){const name=text.textContent?.trim();if(!people.has(norm(name))||text.hasAttribute(\'data-a38-bound\'))continue;text.setAttribute(\'data-a38-bound\',\'1\');text.setAttribute(\'role\',\'link\');text.setAttribute(\'tabindex\',\'0\');text.style.cursor=\'pointer\';const go=e=>{e.preventDefault();e.stopPropagation();openPerson({nombre:name})};text.addEventListener(\'click\',go);text.addEventListener(\'keydown\',e=>{if(e.key===\'Enter\'||e.key===\' \')go(e)})}}\n function decorate(container){if(state.tab===\'gobierno\'||!container)return;for(const table of container.querySelectorAll(\'table\'))for(const cell of table.querySelectorAll(\'tbody td,tbody th\')){if(cell.querySelector(\'button,a,input,select\'))continue;const text=cell.textContent.trim();if(!text)continue;if(people.has(norm(text)))cell.innerHTML=person({nombre:text});else if(parties.has(norm(text)))cell.innerHTML=party(parties.get(norm(text)));}}\n const originalPerson=typeof atlasPersonButton===\'function\'?atlasPersonButton:null;\n if(originalPerson)atlasPersonButton=function(r){return state.tab===\'gobierno\'?originalPerson(r):person(r)};\n document.addEventListener(\'click\',e=>{const p=e.target.closest(\'[data-a38-person]\');if(p){e.preventDefault();openPerson({id_persona:p.getAttribute(\'data-a38-person\')||null,nombre:p.getAttribute(\'data-a38-name\')});return}const b=e.target.closest(\'[data-a38-tab]\');if(b){e.preventDefault();route(b.getAttribute(\'data-a38-tab\'),b.getAttribute(\'data-a38-party\'),b.getAttribute(\'data-a38-province\'),b.getAttribute(\'data-a38-year\'))}});\n window.Atlas38Links={identity,openPerson,route,person,party,link,bindPlot,decorate,norm,knownPerson:name=>people.has(norm(name))};\n})();\n\n(function(){\'use strict\';const esc=x=>String(x??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));const num=(x,d=1)=>x!==null&&x!==undefined&&Number.isFinite(Number(x))?Number(x).toLocaleString(\'es-CR\',{maximumFractionDigits:d}):\'—\';\n function csv(rows,name){if(!rows.length)return;const keys=[...new Set(rows.flatMap(Object.keys))],quote=v=>\'"\'+String(v??\'\').replace(/"/g,\'""\')+\'"\',blob=new Blob([\'\\ufeff\'+[keys.map(quote).join(\',\'),...rows.map(r=>keys.map(k=>quote(r[k])).join(\',\'))].join(\'\\r\\n\')],{type:\'text/csv;charset=utf-8\'}),url=URL.createObjectURL(blob),a=document.createElement(\'a\');a.href=url;a.download=name+\'.csv\';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000)}\n const actions=id=>\'<div class="a25-actions">\'+[\'png\',\'svg\',\'csv\'].map(ext=>`<button type="button" data-a38-export="${id}:${ext}">${ext.toUpperCase()}</button>`).join(\'\')+\'</div>\';\n function card(id,title,description=\'\'){const e=document.createElement(\'section\');e.id=id;e.className=\'a25-block a38-card\';e.innerHTML=`<div class="a38-head"><div><h3>${esc(title)}</h3><p>${esc(description)}</p></div>${actions(id)}</div><div data-a38-plot></div><div data-a38-note></div>`;return e}\n function plot(node,traces,layout,title){node.atlasTitle=title;node.atlasScope=state.from+\'–\'+state.to;const p=Plotly.react(node,traces,{height:480,margin:{l:110,r:100,t:70,b:90},font:{family:\'Inter,Arial,sans-serif\',color:\'#234a5d\',size:12},paper_bgcolor:\'#fff\',plot_bgcolor:\'#fff\',title:{text:title,font:{size:16}},...layout},{responsive:true,displaylogo:false});return Promise.resolve(p).then(()=>{if(node.isConnected)Atlas38Links.bindPlot(node);return node})}\n function bind(card,rows,name){card.querySelectorAll(\'[data-a38-export]\').forEach(b=>b.onclick=()=>{const ext=b.getAttribute(\'data-a38-export\').split(\':\')[1],data=typeof rows===\'function\'?rows():rows;if(ext===\'csv\')csv(data,name);else Plotly.downloadImage(card.querySelector(\'[data-a38-plot]\'),{format:ext,filename:name,width:1800,height:Math.max(850,card.querySelector(\'[data-a38-plot]\').layout?.height||0),scale:ext===\'png\'?2:1})})}\n window.Atlas38UI={esc,num,csv,actions,card,plot,bind};})();\n\n/* Cálculos descriptivos del laboratorio; sin pruebas que supongan independencia. */\n(function(){\'use strict\';const valid=x=>x!==null&&x!==undefined&&x!==\'\'&&Number.isFinite(Number(x));\n function correlation(rows,keys,kind=\'pearson\'){const records=[];for(const a of keys)for(const b of keys){const pair=rows.filter(r=>valid(r[a])&&valid(r[b])),x=pair.map(r=>+r[a]),y=pair.map(r=>+r[b]),r=kind===\'spearman\'?AtlasPublic37.stats(AtlasPublic37.ranks(x),AtlasPublic37.ranks(y)).r:AtlasPublic37.stats(x,y).r;records.push({variable_x:a,variable_y:b,n:pair.length,coeficiente:r,metodo:kind})}return records}\n function pca(rows,keys){const complete=rows.filter(r=>keys.every(k=>valid(r[k])));if(complete.length<2)return {rows:[],loadings:[],variance:[],omitted:keys,n:complete.length};const n=complete.length,means=keys.map(k=>complete.reduce((s,r)=>s+Number(r[k]),0)/n),sd=keys.map((k,j)=>Math.sqrt(complete.reduce((s,r)=>s+(Number(r[k])-means[j])**2,0)/(n-1))),active=keys.map((k,i)=>i).filter(i=>sd[i]>1e-12),omitted=keys.filter((_,i)=>!active.includes(i));if(active.length<2)return {rows:[],loadings:[],variance:[],omitted,n};\n  const z=complete.map(r=>active.map(i=>(Number(r[keys[i]])-means[i])/sd[i])),m=active.length,a=Array.from({length:m},(_,i)=>Array.from({length:m},(_,j)=>z.reduce((s,p)=>s+p[i]*p[j],0)/(n-1))),vectors=Array.from({length:m},(_,i)=>Array.from({length:m},(_,j)=>i===j?1:0));\n  for(let t=0;t<100*m*m;t++){let p=0,q=1,max=0;for(let i=0;i<m;i++)for(let j=i+1;j<m;j++)if(Math.abs(a[i][j])>max){max=Math.abs(a[i][j]);p=i;q=j}if(max<1e-11)break;const theta=.5*Math.atan2(2*a[p][q],a[q][q]-a[p][p]),c=Math.cos(theta),s=Math.sin(theta),ap=a[p][p],aq=a[q][q],apq=a[p][q];for(let k=0;k<m;k++)if(k!==p&&k!==q){const x=a[k][p],y=a[k][q];a[k][p]=a[p][k]=c*x-s*y;a[k][q]=a[q][k]=s*x+c*y}a[p][p]=c*c*ap-2*s*c*apq+s*s*aq;a[q][q]=s*s*ap+2*s*c*apq+c*c*aq;a[p][q]=a[q][p]=0;for(let k=0;k<m;k++){const x=vectors[k][p],y=vectors[k][q];vectors[k][p]=c*x-s*y;vectors[k][q]=s*x+c*y}}\n  const order=Array.from({length:m},(_,i)=>i).sort((i,j)=>a[j][j]-a[i][i]),total=order.reduce((s,i)=>s+Math.max(0,a[i][i]),0),eigen=order.map(i=>Math.max(0,a[i][i]));for(const i of order){let k=0;for(let j=1;j<m;j++)if(Math.abs(vectors[j][i])>Math.abs(vectors[k][i]))k=j;if(vectors[k][i]<0)for(let j=0;j<m;j++)vectors[j][i]*=-1}\n  return {n,rows:complete.map((r,i)=>({...r,pc1:z[i].reduce((s,v,j)=>s+v*vectors[j][order[0]],0),pc2:z[i].reduce((s,v,j)=>s+v*vectors[j][order[1]],0)})),loadings:active.map((k,j)=>({variable:keys[k],pc1:vectors[j][order[0]],pc2:vectors[j][order[1]],media:means[k],desviacion:sd[k]})),variance:eigen.map(v=>100*v/total),eigen,omitted};\n }\n function categorical(records){const rs=[...new Set(records.map(r=>r.fila))],cs=[...new Set(records.map(r=>r.columna))],total=records.reduce((s,r)=>s+r.registros,0);if(!total)return {n:0,chi2:null,cramer:null};const row=new Map(rs.map(k=>[k,records.filter(r=>r.fila===k).reduce((s,r)=>s+r.registros,0)])),col=new Map(cs.map(k=>[k,records.filter(r=>r.columna===k).reduce((s,r)=>s+r.registros,0)]));let chi2=0;for(const r of records){const e=row.get(r.fila)*col.get(r.columna)/total;if(e>0)chi2+=(r.registros-e)**2/e}const dimension=Math.min(rs.filter(k=>row.get(k)>0).length-1,cs.filter(k=>col.get(k)>0).length-1);return {n:total,chi2,cramer:dimension>0?Math.sqrt(chi2/(total*dimension)):null}}\n window.Atlas38Math={correlation,pca,categorical,valid};\n})();\n\n/* Un cálculo compartido por cartografía, ranking y comparación histórica. */\n(function(){\'use strict\';const U=Atlas38UI,L=Atlas38Links,ok=Atlas38Math.valid,N=L.norm;\n const provinceIndex=new Map(),partyIndex=new Map(),representation=new Map();for(const r of D.metricas||[]){if(Number(r.ronda)!==1)continue;provinceIndex.set([r.anio,r.provincia,r.arena].join(\'|\'),r)}for(const r of D.panel||[]){const key=[r.anio,r.provincia].join(\'|\');if(!partyIndex.has(key))partyIndex.set(key,[]);partyIndex.get(key).push(r)}for(const r of D.representacion_territorial||[])representation.set([r.anio,r.provincia].join(\'|\'),r);\n const measureLabel={participacion_pct:\'Participación (%)\',abstencion_pct:\'Abstencionismo (%)\',validshare:\'Válidos / emitidos (%)\',nep:\'Número efectivo de partidos\',margen:\'Margen entre primero y segundo (pp)\',padron:\'Personas empadronadas\',magnitud:\'Escaños\',indice_representacion:\'Representación relativa al padrón\',mujeres_electas_pct:\'Mujeres entre diputaciones electas (%)\',concentracion_1:\'Cuota del primer partido (%)\',concentracion_2:\'Concentración de los dos primeros (%)\',volatilidad_nominal:\'Volatilidad nominal (pp)\',gallagher:\'Índice de Gallagher\',share:\'Voto válido del partido (%)\',votes:\'Votos del partido\',escanos:\'Escaños del partido\',candidaturas:\'Candidaturas del partido\'};\n function value(year,province,arena,party,measure){if(party!==\'Todos\'){const r=(partyIndex.get([year,province].join(\'|\'))||[]).find(r=>N(r.partido)===N(party));if(!r)return null;const key=measure===\'share\'?\'porcentaje_\'+arena:measure===\'votes\'?\'votos_\'+arena:measure;return ok(r[key])?+r[key]:null}const r=[\'magnitud\',\'indice_representacion\'].includes(measure)?representation.get([year,province].join(\'|\')):provinceIndex.get([year,province,arena].join(\'|\'));if(!r)return null;if(measure===\'validshare\')return ok(r.validos)&&ok(r.emitidos)&&+r.emitidos>0?100*r.validos/r.emitidos:null;return ok(r[measure])?+r[measure]:null}\n function gap(year,province,party){const rows=partyIndex.get([year,province].join(\'|\'))||[];if(party!==\'Todos\'){const row=rows.find(r=>N(r.partido)===N(party));return row&&ok(row.porcentaje_Presidencial)&&ok(row.porcentaje_Legislativa)?+row.porcentaje_Presidencial-row.porcentaje_Legislativa:null}const pv=rows.reduce((s,r)=>s+(ok(r.votos_Presidencial)?+r.votos_Presidencial:0),0),lv=rows.reduce((s,r)=>s+(ok(r.votos_Legislativa)?+r.votos_Legislativa:0),0);if(!pv||!lv)return null;return 50*rows.reduce((s,r)=>s+Math.abs((ok(r.votos_Presidencial)?+r.votos_Presidencial:0)/pv-(ok(r.votos_Legislativa)?+r.votos_Legislativa:0)/lv),0)}\n function rows(selection){const {year,reference,arena,party,measure,mode}=selection;return P.map(provincia=>{const current=mode===\'gap\'?gap(year,provincia,party):value(year,provincia,arena,mode===\'representation\'?\'Todos\':party,mode===\'representation\'?\'indice_representacion\':measure),prior=reference==null?null:value(reference,provincia,arena,party,measure),delta=[\'change\',\'compare\'].includes(mode);return {anio:year,anio_referencia:delta?reference:null,provincia,arena,partido:mode===\'representation\'?\'Todos\':party,indicador:mode===\'gap\'?(party===\'Todos\'?\'Distancia entre distribuciones de votos (pp)\':\'% presidencial − % legislativo (pp)\'):measureLabel[mode===\'representation\'?\'indice_representacion\':measure],valor_actual:current,valor_referencia:delta?prior:null,valor:delta?(ok(current)&&ok(prior)?current-prior:null):current}})}\n window.Atlas38Maps={value,gap,rows};\n function openTotal(province,selection){const code=String(P.indexOf(province)+1);state.province=province;state.focus=selection.year;for(const k of [\'province\',\'focus\']){const el=document.getElementById(k);if(el)el.value=state[k]}gotoTab(\'gobierno\');setTimeout(()=>{window.Atlas31Results?.setSelection({year:selection.year,arena:selection.arena===\'Presidencial\'?0:1,round:1,scope:\'p\',level:\'p\',province:code,canton:\'\',district:\'\',search:\'\'});document.querySelector(\'#atlas-v2317-seccion07 [data-a33-detail="total"]\')?.click()},0)}\n const option=(id,title,choices,selected)=>`<label>${title}<select data-a38-map="${id}">${choices.map(x=>{const [v,t]=Array.isArray(x)?x:[x,x];return `<option value="${U.esc(v)}" ${String(v)===String(selected)?\'selected\':\'\'}>${U.esc(t)}</option>`}).join(\'\')}</select></label>`;\n views.mapas=function(){title(\'Atlas provincial comparado\',\'Indicadores, distribución del voto y cambios históricos de las siete provincias.\');const c=document.getElementById(\'content\'),cfg=local.a38map||{arena:local.marena||\'Legislativa\',party:local.mparty||\'Todos\',mode:\'value\',measure:\'participacion_pct\',reference:Y[Math.max(0,Y.indexOf(+state.focus)-1)],province:\'San José\'};local.a38map=cfg;if(![\'Legislativa\',\'Presidencial\'].includes(cfg.arena))cfg.arena=\'Legislativa\';if(!Y.includes(Number(cfg.reference)))cfg.reference=Y[Math.max(0,Y.indexOf(+state.focus)-1)];if(!P.includes(cfg.province))cfg.province=P.includes(state.province)?state.province:P[0];let year=Number(state.focus);const known=[\'Todos\',...uniq(D.panel.map(r=>r.partido)).sort()];if(!known.includes(cfg.party))cfg.party=\'Todos\';\n  const panel=document.createElement(\'section\');panel.className=\'a38-map-controls\';panel.innerHTML=option(\'year\',\'Elección mostrada\',Y,year)+option(\'mode\',\'Lectura\',[[\'value\',\'Valor observado\'],[\'change\',\'Cambio desde la elección anterior\'],[\'compare\',\'Comparar dos elecciones\'],[\'gap\',\'Diferencias entre Presidencia y diputaciones\'],[\'representation\',\'Representación relativa al padrón\']],cfg.mode)+option(\'arena\',\'Arena\',[\'Legislativa\',\'Presidencial\'],cfg.arena)+option(\'party\',\'Partido\',known.map(p=>[p,p===\'Todos\'?\'General · todos los partidos\':labelText(p)]),cfg.party)+option(\'measure\',\'Indicador\',[],cfg.measure)+option(\'reference\',\'Elección de referencia\',Y,cfg.reference);c.append(panel);const q=k=>panel.querySelector(\'[data-a38-map="\'+k+\'"]\');\n  const map=U.card(\'atlas38-map\',\'Cartografía provincial\',\'Las siete provincias permanecen visibles; seleccione una para explorar su contexto.\'),ranking=U.card(\'atlas38-map-ranking\',\'Comparación de las siete provincias\'),history=U.card(\'atlas38-map-history\',\'Evolución provincial\'),focus=document.createElement(\'section\');focus.className=\'a38-province-focus\';focus.id=\'atlas38-province-focus\';c.append(map,ranking,focus,history);let data=[],selection=null;\n  function configure(){const party=q(\'party\').value,choices=party===\'Todos\'?[\'participacion_pct\',\'abstencion_pct\',\'validshare\',\'nep\',\'margen\',\'padron\',\'magnitud\',\'indice_representacion\',\'mujeres_electas_pct\',\'concentracion_1\',\'concentracion_2\',\'volatilidad_nominal\',\'gallagher\']:q(\'arena\').value===\'Legislativa\'?[\'share\',\'votes\',\'escanos\',\'candidaturas\']:[\'share\',\'votes\'];const former=q(\'measure\').value||cfg.measure;q(\'measure\').innerHTML=choices.map(k=>`<option value="${k}">${measureLabel[k]}</option>`).join(\'\');q(\'measure\').value=choices.includes(former)?former:choices[0]}\n  function draw(){cfg.mode=q(\'mode\').value;cfg.arena=q(\'arena\').value;cfg.party=q(\'party\').value;cfg.measure=q(\'measure\').value;cfg.reference=Number(q(\'reference\').value);year=Number(q(\'year\').value);selection={year,reference:cfg.mode===\'change\'?(Y[Y.indexOf(year)-1]??null):cfg.reference,arena:cfg.arena,party:cfg.party,measure:cfg.measure,mode:cfg.mode};data=rows(selection);q(\'reference\').closest(\'label\').hidden=cfg.mode!==\'compare\';q(\'measure\').closest(\'label\').hidden=[\'gap\',\'representation\'].includes(cfg.mode);q(\'party\').closest(\'label\').hidden=cfg.mode===\'representation\';q(\'arena\').closest(\'label\').hidden=[\'gap\',\'representation\'].includes(cfg.mode);const valid=data.filter(r=>ok(r.valor)),isDelta=[\'change\',\'compare\'].includes(cfg.mode)||(cfg.mode===\'gap\'&&cfg.party!==\'Todos\'),min=isDelta?-Math.max(1,...valid.map(r=>Math.abs(r.valor))):Math.min(0,...valid.map(r=>r.valor)),max=isDelta?-min:Math.max(1,...valid.map(r=>r.valor));\n   const mix=(a,b,t)=>{const rgb=h=>[1,3,5].map(i=>parseInt(h.slice(i,i+2),16));return \'rgb(\'+rgb(a).map((v,i)=>Math.round(v+(rgb(b)[i]-v)*Math.max(0,Math.min(1,t)))).join(\',\')+\')\'},paint=v=>!ok(v)?\'#dfe8ed\':isDelta?v<0?mix(\'#f7f5f8\',\'#95618b\',Math.abs(v)/max):mix(\'#f3f8f8\',\'#0b8391\',v/max):mix(\'#eaf5f6\',\'#0b8391\',(v-min)/(max-min));\n   const traces=[];for(const feature of D.geografia?.features||[]){const province=P.find(p=>N(p)===N(feature.properties.NAME_1||feature.properties.name)),record=data.find(r=>r.provincia===province);if(!record)continue;const polys=feature.geometry.type===\'MultiPolygon\'?feature.geometry.coordinates:[feature.geometry.coordinates];for(const polygon of polys){const ring=polygon[0];if(!ring?.length)continue;traces.push({type:\'scatter\',mode:\'lines\',x:ring.map(p=>p[0]),y:ring.map(p=>p[1]),fill:\'toself\',fillcolor:paint(record.valor),line:{color:\'#fff\',width:1.5},showlegend:false,meta:{provincia:province},text:`${province} · ${year}<br>${record.indicador}: ${U.num(record.valor,2)}${ok(record.valor)?\'\':\' · sin dato comparable\'}`,hoverinfo:\'text\',hoveron:\'fills\'})}}\n   traces.push({type:\'scatter\',mode:\'markers\',x:[null,null],y:[null,null],marker:{size:0,color:[min,max],cmin:min,cmax:max,colorscale:isDelta?[[0,\'#95618b\'],[.5,\'#f8fafb\'],[1,\'#0b8391\']]:[[0,\'#eaf5f6\'],[1,\'#0b8391\']],showscale:true,colorbar:{title:{text:\'Valor\'},thickness:14}},showlegend:false,hoverinfo:\'skip\'});\n   const context=year+\' · \'+data[0].indicador+(cfg.party===\'Todos\'||cfg.mode===\'representation\'?\'\':\' · \'+labelText(cfg.party))+(selection.reference!==null&&[\'change\',\'compare\'].includes(cfg.mode)?\' · referencia \'+selection.reference:\'\');U.plot(map.querySelector(\'[data-a38-plot]\'),traces,{height:640,margin:{l:20,r:100,t:75,b:20},xaxis:{visible:false,range:[-86.2,-82.4],constrain:\'domain\'},yaxis:{visible:false,range:[8,11.35],scaleanchor:\'x\',scaleratio:1,constrain:\'domain\'},showlegend:false},context).then(node=>{if(!node.isConnected)return;if(node.__a38MapListener)node.removeListener?.(\'plotly_click\',node.__a38MapListener);node.__a38MapListener=e=>{const province=e.points?.[0]?.data?.meta?.provincia;if(P.includes(province)){cfg.province=province;drawFocus()}};node.on(\'plotly_click\',node.__a38MapListener)});\n   map.querySelector(\'[data-a38-note]\').innerHTML=\'<p class="a37-method">\'+(cfg.mode===\'gap\'?(cfg.party===\'Todos\'?\'Distancia general: ½ × suma de diferencias absolutas entre las cuotas presidenciales y legislativas de todos los partidos, por provincia. Escala 0–100 pp. No identifica electores que dividieron su voto.\':\'Brecha firmada del partido: porcentaje presidencial menos porcentaje legislativo. Positivo favorece Presidencia; negativo, diputaciones.\'):(cfg.mode===\'change\'&&selection.reference===null?\'El primer proceso no tiene elección anterior. No se fabrica un cambio igual a cero.\':cfg.mode===\'representation\'?\'Índice = porcentaje de escaños de la provincia / porcentaje del padrón nacional. 1 indica igualdad proporcional.\':\'Valores observados y denominadores de la arena seleccionada. Un dato ausente no se convierte en cero.\'))+\'</p>\';\n   const ordered=data.filter(r=>ok(r.valor)).slice().sort((a,b)=>b.valor-a.valor);const comparison=[\'compare\',\'change\'].includes(cfg.mode)&&selection.reference!==null;U.plot(ranking.querySelector(\'[data-a38-plot]\'),comparison?[{type:\'scatter\',mode:\'markers\',name:String(selection.reference),x:ordered.map(r=>r.valor_referencia),y:ordered.map(r=>r.provincia),marker:{size:11,color:\'#899cab\'}},{type:\'scatter\',mode:\'markers\',name:String(year),x:ordered.map(r=>r.valor_actual),y:ordered.map(r=>r.provincia),marker:{size:11,color:\'#0b8391\'}}]:[{type:\'bar\',orientation:\'h\',x:ordered.map(r=>r.valor).reverse(),y:ordered.map(r=>r.provincia).reverse(),marker:{color:ordered.map(r=>paint(r.valor)).reverse()},text:ordered.map(r=>U.num(r.valor,2)).reverse(),textposition:\'outside\',cliponaxis:false}],{height:420,margin:{l:150,r:120,t:70,b:90},xaxis:{title:comparison?measureLabel[cfg.measure]:data[0].indicador,zeroline:true}},comparison?\'Valores de ambas elecciones · \'+context:\'Ranking · \'+context);\n   drawFocus();U.bind(map,()=>data,\'Atlas_Mapa_Provincial_\'+year);U.bind(ranking,()=>data,\'Atlas_Comparacion_Provincial_\'+year);\n  }\n  function drawFocus(){const province=cfg.province||P[0],r=data.find(r=>r.provincia===province),ys=Y.filter(y=>y>=state.from&&y<=state.to),series=ys.flatMap(y=>rows({...selection,year:y,mode:selection.mode===\'gap\'?\'gap\':selection.mode===\'representation\'?\'representation\':\'value\'}));focus.innerHTML=\'<div class="a38-head"><div><h3>Explorar \'+U.esc(province)+\'</h3><p>El mapa conserva las siete provincias; este bloque se centra en la seleccionada.</p></div><div class="a38-province-tabs">\'+P.map(p=>`<button type="button" data-a38-province="${U.esc(p)}" aria-pressed="${p===province}">${U.esc(p)}</button>`).join(\'\')+\'</div></div><div class="a26-metric-cards">\'+[[\'Valor mostrado\',U.num(r?.valor,2)],[\'Valor de \'+selection.year,U.num(r?.valor_actual,2)],[\'Referencia \'+(selection.reference??\'—\'),U.num(r?.valor_referencia,2)]].map(([label,value])=>`<div><small>${label}</small><strong>${value}</strong></div>`).join(\'\')+\'</div><div class="a38-context-links">\'+L.link(\'participacion\',\'Participación\',null,province,selection.year)+L.link(\'competencia\',\'Competencia\',null,province,selection.year)+L.link(\'laboratorio\',\'Analizar en laboratorio\',cfg.party===\'Todos\'?null:cfg.party,province,selection.year)+(cfg.party!==\'Todos\'?L.link(\'perfiles\',\'Perfil del partido\',cfg.party,province,selection.year):\'\')+\'<button type="button" data-a38-total>Consultar resultado provincial</button></div>\';\n   focus.querySelectorAll(\'[data-a38-province]\').forEach(b=>b.onclick=()=>{cfg.province=b.getAttribute(\'data-a38-province\');drawFocus()});focus.querySelector(\'[data-a38-total]\').onclick=()=>openTotal(province,selection);\n   const tr=P.map(p=>({type:\'scatter\',mode:\'lines+markers\',name:p,x:ys,y:ys.map(y=>series.find(r=>r.anio===y&&r.provincia===p)?.valor??null),connectgaps:false,line:{color:provinceColors[p],width:p===province?3.5:1.3},marker:{size:p===province?7:4},opacity:p===province?1:.4,hovertemplate:p+\' · %{x}<br>%{y:.3f}<extra></extra>\'}));U.plot(history.querySelector(\'[data-a38-plot]\'),tr,{height:490,xaxis:{title:\'Elección\',tickmode:\'array\',tickvals:ys},yaxis:{title:series[0]?.indicador||r?.indicador}},\'Evolución provincial · \'+(series[0]?.indicador||\'indicador\')+\' · \'+state.from+\'–\'+state.to);U.bind(history,()=>series,\'Atlas_Trayectoria_Provincial\');\n  }\n  panel.querySelectorAll(\'select\').forEach(s=>s.onchange=()=>{if(s===q(\'year\')){state.focus=+s.value;state.from=Math.min(state.from,state.focus);state.to=Math.max(state.to,state.focus);for(const k of [\'from\',\'to\',\'focus\']){const el=document.getElementById(k);if(el)el.value=state[k]}}if([\'party\',\'arena\'].includes(s.getAttribute(\'data-a38-map\')))configure();draw()});configure();draw();\n };\n})();\n\n/* Profundización del perfil único: territorio, representación y personas. */\n(function(){\'use strict\';const U=Atlas38UI,L=Atlas38Links,base=views.perfiles,normal=x=>L.norm(x).replace(/^PARTIDO /,\'\'),observed=r=>Number(r.anio)>=state.from&&Number(r.anio)<=state.to;\n views.perfiles=function(...args){const result=base.apply(this,args),c=document.getElementById(\'content\'),party=local.profileparty||\'LIBERACION NACIONAL\',match=r=>normal(r.partido)===normal(party),years=Y.filter(y=>y>=state.from&&y<=state.to),events=(D.resultados_candidaturas||[]).filter(r=>observed(r)&&match(r)),leg=events.filter(r=>r.cargo===\'Diputación\'),elected=(D.electas||[]).filter(r=>observed(r)&&match(r));\n  const shell=document.createElement(\'section\');shell.id=\'atlas38-party-analysis\';shell.className=\'a38-party-analysis\';shell.innerHTML=\'<div class="a38-head"><div><h3>Explorar el partido en el Atlas</h3><p>Los vínculos conservan la organización seleccionada, la elección y el territorio cuando corresponda.</p></div></div><nav class="a38-context-links">\'+[[\'partidos\',\'Liderazgos y territorio\'],[\'dualidad\',\'Voto presidencial / legislativo\'],[\'candidaturas\',\'Candidaturas\'],[\'genero\',\'Sexo y representación\'],[\'formulas\',\'Fórmulas presidenciales\'],[\'credenciales\',\'Explorador de personas\'],[\'outcomes\',\'Éxito electoral\'],[\'apellidos\',\'Coincidencias de apellidos\'],[\'mapas\',\'Mapas provinciales\'],[\'laboratorio\',\'Laboratorio\'],[\'cancelaciones\',\'Registros partidarios\']].map(([tab,text])=>L.link(tab,text,party,state.province,state.focus)).join(\'\')+\'</nav><div class="a38-subtabs" role="group" aria-label="Análisis del partido">\'+[[\'territory\',\'Territorios encabezados\'],[\'representation\',\'Candidaturas y representación\'],[\'people\',\'Personas del partido\']].map(([id,text],i)=>`<button type="button" data-a38-party-tab="${id}" aria-pressed="${i===0}">${text}</button>`).join(\'\')+\'</div><div data-a38-party-panel="territory"></div><div data-a38-party-panel="representation" hidden></div><div data-a38-party-panel="people" hidden></div>\';\n  const controls=c.querySelector(\'#profileparty\')?.closest(\'.controls\'),identity=controls?.nextElementSibling,anchor=identity?.matches(\'.card\')?identity:controls||c.querySelector(\'.scope\')||c.querySelector(\'.section-head\');if(anchor)anchor.after(shell);else c.prepend(shell);const panel=id=>shell.querySelector(\'[data-a38-party-panel="\'+id+\'"]\');let territoryRows=[];\n  const territory=U.card(\'atlas38-party-territory\',\'Dónde encabeza cantones y distritos\',party+\' · conteos y denominadores históricos por provincia.\');territory.querySelector(\'.a38-head\').insertAdjacentHTML(\'afterend\',\'<div class="a37-toolbar"><label>Arena<select data-a38-party-arena><option value="1">Diputaciones</option><option value="0">Presidencia · primera ronda</option></select></label><label>Unidades<select data-a38-party-level><option value="c">Cantones</option><option value="d">Distritos</option></select></label></div>\');panel(\'territory\').append(territory);\n  function drawTerritory(){const arena=+territory.querySelector(\'[data-a38-party-arena]\').value,level=territory.querySelector(\'[data-a38-party-level]\').value,api=window.Atlas32Territory;territoryRows=[];for(const year of years)for(const province of P){const code=String(P.indexOf(province)+1),all=api?.leaders(year,arena,1,level).filter(g=>g.id[0]===code)||[],wins=all.filter(g=>!g.tie&&normal(api.name(g.winner))===normal(party)).length;territoryRows.push({anio:year,provincia:province,arena:arena?\'Diputaciones\':\'Presidencia\',nivel:level,partido:party,ganados:wins,observados:all.length,porcentaje:all.length?100*wins/all.length:null})}const find=(p,y)=>territoryRows.find(r=>r.provincia===p&&r.anio===y);U.plot(territory.querySelector(\'[data-a38-plot]\'),[{type:\'heatmap\',x:years,y:P,z:P.map(p=>years.map(y=>find(p,y).porcentaje)),customdata:P.map(p=>years.map(y=>[find(p,y).ganados,find(p,y).observados])),zmin:0,zmax:100,colorscale:[[0,\'#f3f8fa\'],[1,typeof col===\'function\'?col(party):\'#078b88\']],colorbar:{title:{text:\'% de unidades\'}},hovertemplate:\'%{y} · %{x}<br>%{customdata[0]} de %{customdata[1]} unidades<br>%{z:.1f}%<extra></extra>\'}],{height:460,margin:{l:140,r:100,t:70,b:95},xaxis:{tickmode:\'array\',tickvals:years},yaxis:{automargin:true}},labelText(party)+\' · \'+(arena?\'Diputaciones\':\'Presidencia\')+\' · \'+(level===\'c\'?\'cantones\':\'distritos\')).then(node=>{if(!node.isConnected)return;if(node.__a38Territory)node.removeListener?.(\'plotly_click\',node.__a38Territory);node.__a38Territory=e=>{const p=e.points?.[0];if(!p)return;L.route(\'mapas\',party,p.y,Number(p.x))};node.on(\'plotly_click\',node.__a38Territory)});territory.querySelector(\'[data-a38-note]\').innerHTML=\'<p class="a37-method">Cada celda usa las unidades con resultados de esa provincia y año. Los empates no se adjudican al partido. Toque una celda para abrir el contexto provincial y electoral.</p>\';U.bind(territory,()=>territoryRows,\'Atlas_Partido_Territorios\')}\n  territory.querySelectorAll(\'select\').forEach(x=>x.onchange=drawTerritory);drawTerritory();\n  const conversion=U.card(\'atlas38-party-conversion\',\'Candidaturas legislativas y diputaciones electas\',\'Ámbito nacional. Conteos por elección; candidaturas y personas electas son unidades diferentes.\');panel(\'representation\').append(conversion);const byYear=years.map(anio=>({anio,partido:party,candidaturas:leg.filter(r=>r.anio===anio).length,electas:elected.filter(r=>r.anio===anio).length}));U.plot(conversion.querySelector(\'[data-a38-plot]\'),[{type:\'bar\',name:\'Candidaturas legislativas\',x:years,y:byYear.map(r=>r.candidaturas),marker:{color:\'#91b7c6\'}},{type:\'bar\',name:\'Diputaciones electas\',x:years,y:byYear.map(r=>r.electas),marker:{color:typeof col===\'function\'?col(party):\'#078b88\'}}],{barmode:\'group\',xaxis:{tickmode:\'array\',tickvals:years},yaxis:{title:\'Registros\'}},\'Candidaturas y escaños · \'+labelText(party));U.bind(conversion,byYear,\'Atlas_Partido_Candidaturas_Electas\');\n  const sex=U.card(\'atlas38-party-sex\',\'Mujeres entre candidaturas y diputaciones electas\',\'Ámbito nacional. Porcentajes con denominadores propios, entre registros con sexo documentado Mujer/Hombre.\');panel(\'representation\').append(sex);const rates=years.map(anio=>{const a=leg.filter(r=>r.anio===anio&&[\'Mujer\',\'Hombre\'].includes(r.sexo)),b=elected.filter(r=>r.anio===anio&&[\'Mujer\',\'Hombre\'].includes(r.sexo)),w=a.filter(r=>r.sexo===\'Mujer\').length,v=b.filter(r=>r.sexo===\'Mujer\').length;return {anio,partido:party,mujeres_candidatas:w,total_candidaturas:a.length,mujeres_electas:v,total_electas:b.length,candidatas_pct:a.length?100*w/a.length:null,electas_pct:b.length?100*v/b.length:null}});U.plot(sex.querySelector(\'[data-a38-plot]\'),[{type:\'scatter\',mode:\'lines+markers\',name:\'Candidaturas\',x:years,y:rates.map(r=>r.candidatas_pct),customdata:rates.map(r=>[r.mujeres_candidatas,r.total_candidaturas]),line:{color:\'#ae7096\'},hovertemplate:\'%{x}<br>%{customdata[0]} de %{customdata[1]}<br>%{y:.1f}%<extra></extra>\'},{type:\'scatter\',mode:\'lines+markers\',name:\'Diputaciones electas\',x:years,y:rates.map(r=>r.electas_pct),customdata:rates.map(r=>[r.mujeres_electas,r.total_electas]),line:{color:\'#167f8d\'},hovertemplate:\'%{x}<br>%{customdata[0]} de %{customdata[1]}<br>%{y:.1f}%<extra></extra>\'}],{xaxis:{tickmode:\'array\',tickvals:years},yaxis:{title:\'%\',range:[0,100]}},\'Representación por sexo · \'+labelText(party));U.bind(sex,rates,\'Atlas_Partido_Representacion_Sexo\');\n  const personPanel=panel(\'people\');personPanel.innerHTML=\'<div class="a37-toolbar"><label>Elección<select data-a38-party-year>\'+years.map(y=>`<option value="${y}" ${y===+state.focus?\'selected\':\'\'}>${y}</option>`).join(\'\')+\'</select></label><label>Registros<select data-a38-party-source><option value="all">Todas las candidaturas del partido</option><option value="won">Candidaturas electas</option><option value="pres">Fórmulas presidenciales</option></select></label></div><div data-a38-party-persons></div>\';\n  function drawPeople(){const year=+personPanel.querySelector(\'select\').value,source=personPanel.querySelector(\'[data-a38-party-source]\').value,rows=events.filter(r=>r.anio===year&&(source!==\'won\'||r.electa)&&(source!==\'pres\'||r.cargo!==\'Diputación\'));personPanel.querySelector(\'[data-a38-party-persons]\').innerHTML=\'<p class="a37-method">\'+rows.length+\' candidaturas; cada nombre abre su trayectoria nominal completa.</p><div class="a38-person-grid">\'+rows.map(r=>`<article class="a37-person"><h4>${L.person(r)}</h4><p>${U.esc(r.cargo)} · ${U.esc(r.provincia)}</p><span class="a38-result ${r.electa?\'won\':\'\'}">${r.electa?\'Electa\':\'Candidatura no electa\'}</span><div>${L.link(\'apellidos\',\'Coincidencias de apellidos\',\'\',r.provincia,r.anio)}</div></article>`).join(\'\')+\'</div>\';personPanel.querySelectorAll(\'[data-a38-tab="apellidos"]\').forEach(b=>b.addEventListener(\'click\',()=>{const card=b.closest(\'.a37-person\'),name=card.querySelector(\'[data-a38-name]\')?.getAttribute(\'data-a38-name\');local.a37names={...(local.a37names||{}),reference:name,party:\'all\',population:\'all\'}}))}\n  personPanel.querySelectorAll(\'select\').forEach(x=>x.onchange=drawPeople);drawPeople();\n  shell.querySelectorAll(\'[data-a38-party-tab]\').forEach(b=>b.onclick=()=>{const key=b.getAttribute(\'data-a38-party-tab\');shell.querySelectorAll(\'[data-a38-party-panel]\').forEach(p=>p.hidden=p.getAttribute(\'data-a38-party-panel\')!==key);shell.querySelectorAll(\'[data-a38-party-tab]\').forEach(x=>x.setAttribute(\'aria-pressed\',String(x===b)));shell.querySelectorAll(\'[data-a38-plot]\').forEach(node=>{if(node.data)Plotly.Plots.resize(node)})});L.decorate(c);return result;\n };\n})();\n\n/* Accesos entre preguntas analíticas; no reemplaza el visor de resultados de la sección 7. */\n(function(){\'use strict\';const L=Atlas38Links;\n const originalPersonOpen=typeof atlasOpenPerson===\'function\'?atlasOpenPerson:null;if(originalPersonOpen){atlasOpenPerson=function(pid){if(state.tab!==\'gobierno\'){for(const key of [\'personquery\',\'credyear\',\'credparty\',\'credrole\',\'credresult\',\'credprovince\',\'credentialpage\'])delete local[key];local.credentialscope=\'Todas\'}return originalPersonOpen(pid)};window.atlasOpenPerson=atlasOpenPerson}\n for(const key of Object.keys(views)){if(key===\'gobierno\')continue;const previous=views[key];views[key]=function(...args){const result=previous.apply(this,args),c=document.getElementById(\'content\');L.decorate(c);if([\'participacion\',\'competencia\'].includes(key)&&!c.querySelector(\'#atlas38-analysis-links\')){const nav=document.createElement(\'nav\');nav.id=\'atlas38-analysis-links\';nav.className=\'a38-context-links\';nav.innerHTML=L.link(\'laboratorio\',\'Contrastar estos indicadores en el laboratorio\')+L.link(\'mapas\',\'Comparar las siete provincias\')+L.link(\'perfiles\',\'Explorar un partido\');const header=c.querySelector(\'.section-head\');if(header)header.after(nav);else c.prepend(nav);nav.querySelector(\'[data-a38-tab="laboratorio"]\').onclick=()=>{if(key===\'participacion\'){const metric=c.querySelector(\'#a27-p-indicator\')?.value||\'abst_pct\';local.a37lab={source:\'territorios\',arena:c.querySelector(\'#a27-p-arena\')?.value||\'L\',scale:c.querySelector(\'#a27-p-scale\')?.value||\'c\',task:\'scatter\',x:metric,y:metric===\'abst_pct\'?\'noval_pct\':\'abst_pct\'}}else local.a37lab={source:\'metricas\',arena:\'L\',task:\'scatter\',x:local.cm||\'nep\',y:\'abstencion_pct\'}};nav.querySelector(\'[data-a38-tab="mapas"]\').onclick=()=>{local.a38map={...(local.a38map||{}),party:\'Todos\',mode:\'value\',arena:\'Legislativa\',measure:key===\'competencia\'?(local.cm||\'nep\'):\'abstencion_pct\'}}}return result}}\n})();\n'
_ATLAS_V31_RESULTS = '/* Atlas Electoral · Exploración territorial, resultados y mapas. */\n(function(){\n \'use strict\';\n if(window.ATLAS_V29_RESULTS)return;\n window.ATLAS_V29_RESULTS=true;\n const PROV=[\'San José\',\'Alajuela\',\'Cartago\',\'Heredia\',\'Guanacaste\',\'Puntarenas\',\'Limón\'];\n const PNUM=Object.fromEntries(PROV.map((p,i)=>[p,String(i+1)]));\n const CODE_TO_PROV=Object.fromEntries(PROV.map((p,i)=>[String(i+1),p]));\n const nf=new Intl.NumberFormat(\'es-CR\',{maximumFractionDigits:0});\n const df=new Intl.NumberFormat(\'es-CR\',{minimumFractionDigits:1,maximumFractionDigits:2});\n const fmt=n=>n!==null&&n!==undefined&&n!==\'\'&&n!==null&&n!==undefined&&n!==\'\'&&n!==null&&n!==undefined&&n!==\'\'&&Number.isFinite(Number(n))?nf.format(Number(n)):\'—\';\n const pct=n=>n!==null&&n!==undefined&&n!==\'\'&&n!==null&&n!==undefined&&n!==\'\'&&n!==null&&n!==undefined&&n!==\'\'&&Number.isFinite(Number(n))?df.format(Number(n))+\' %\':\'—\';\n const esc=s=>String(s??\'\').replace(/[&<>"\']/g,x=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[x]));\n const N=s=>String(s||\'\').normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toUpperCase().replace(/[^A-Z0-9]/g,\'\');\n const partyColors={\'LIBERACIONNACIONAL\':\'#079648\',\'UNIDADSOCIALCRISTIANA\':\'#0876B6\',\'ACCIONCIUDADANA\':\'#E9BA32\',\'FRENTEAMPLIO\':\'#CDA02A\',\'MOVIMIENTOLIBERTARIO\':\'#BA9A3E\',\'PUEBLOSOBERANO\':\'#14A3AF\',\'NUEVAREPUBLICA\':\'#2C8CAD\',\'LIBERALPROGRESISTA\':\'#EB714E\',\'PROGRESOSOCIALDEMOCRATICO\':\'#303D91\'};\n let db=null, index=new Map(), placeKeys=new Map(), originalRoot=null, page=0, viewRows=[], currentParties=[], currentBreakdown=\'total\';\n const S={year:2026,arena:0,round:1,scope:\'n\',province:\'\',canton:\'\',district:\'\',level:\'n\',unit:\'pct\',composition:\'valid\',search:\'\'};\n function dataset(){\n  if(db)return db;\n  const node=document.getElementById(\'atlas-v28-dataset\');\n  if(!node)return null;\n  db=(window.AtlasTerritorialData||(window.AtlasTerritorialData=JSON.parse(node.textContent)));\n  if(!Array.isArray(db.rows)||!db.places)throw Error(\'No hay resultados territoriales documentados\');\n  // Indexar una sola vez para evitar 200.000 recorridos por cada cambio de filtro.\n  for(const r of db.rows){const k=r[0]+\'|\'+r[1]+\'|\'+r[2];if(!index.has(k))index.set(k,[]);index.get(k).push(r)}\n  return db;\n }\n function color(id){let name=db.parties[id]||\'\',k=N(name);if(partyColors[k])return partyColors[k];let c=db.colors?.[name]||db.colors?.[k];return /^#[0-9a-f]{6}$/i.test(c||\'\')?c:\'#718B9B\'}\n function identity(code){let parts=db.places[S.year+\'|\'+code];return parts||[CODE_TO_PROV[code[0]]||\'Provincia no identificada\',\'Cantón sin identificar\',\'Distrito sin identificar\']}\n function tally(){const key=S.year+\'|\'+S.arena+\'|\'+S.round, rows=index.get(key)||[];\n  const groups=new Map();\n  for(const r of rows){const code=String(r[3]).padStart(5,\'0\');if(S.scope===\'p\'&&code[0]!==S.province)continue;\n   if(S.scope===\'c\'&&code.slice(0,3)!==S.canton)continue;\n   if(S.scope===\'d\'&&code!==S.district)continue;\n   const territory=S.level===\'n\'?\'CR\':S.level===\'p\'?code[0]:S.level===\'c\'?code.slice(0,3):code;\n   if(!groups.has(territory)){\n    const parts=identity(code);\n    groups.set(territory,{id:territory,code,province:parts[0],canton:parts[1],district:parts[2],label:S.level===\'n\'?\'Costa Rica\':S.level===\'p\'?parts[0]:S.level===\'c\'?parts[1]:parts[2],parties:new Map(),valid:0,blank:0,null:0,abst:0});\n   }\n   const g=groups.get(territory),kind=Number(r[4]),votes=Number(r[5]);if(!Number.isFinite(votes)||votes<0)continue;\n   if(kind>=0){g.parties.set(kind,(g.parties.get(kind)||0)+votes);g.valid+=votes}\n   else if(kind===-1)g.blank+=votes;\n   else if(kind===-2)g.null+=votes;\n   else if(kind===-3)g.abst+=votes;\n  }\n  return [...groups.values()].sort((a,b)=>{\n   let x=PROV.indexOf(a.province)-PROV.indexOf(b.province);if(x)return x;\n   x=a.canton.localeCompare(b.canton,\'es\');return x||a.district.localeCompare(b.district,\'es\');\n  });\n }\n function globalOptions(){let d=dataset(),subset=index.get(S.year+\'|\'+S.arena+\'|\'+S.round)||[];\n  let provinces=new Set(), cantons=new Map(),districts=new Map();\n  for(const row of subset){let code=String(row[3]).padStart(5,\'0\'),pr=code[0],ca=code.slice(0,3);provinces.add(pr);let names=identity(code);\n   if(!cantons.has(ca))cantons.set(ca,{id:ca,province:pr,label:names[1]});\n   if(!districts.has(code))districts.set(code,{id:code,canton:ca,province:pr,label:names[2]});\n  }\n  return {provinces:[...provinces].sort(),cantons:[...cantons.values()].sort((a,b)=>a.id.localeCompare(b.id,\'es\',{numeric:true})),districts:[...districts.values()].sort((a,b)=>a.id.localeCompare(b.id,\'es\',{numeric:true}))};\n }\n function selectOptions(entries,value){return entries.map(([k,text])=>\'<option value="\'+esc(k)+\'" \'+(String(k)===String(value)?\'selected\':\'\')+\'>\'+esc(text)+\'</option>\').join(\'\')}\n function scope(){const all=globalOptions();\n  if(S.scope!==\'n\'&&!all.provinces.includes(S.province))S.province=all.provinces[0]||\'\';\n  const matchingCantons=all.cantons.filter(c=>c.province===S.province);\n  if((S.scope===\'c\'||S.scope===\'d\')&&!matchingCantons.some(x=>x.id===S.canton))S.canton=matchingCantons[0]?.id||\'\';\n  const matchingDistricts=all.districts.filter(d=>d.canton===S.canton);\n  if(S.scope===\'d\'&&!matchingDistricts.some(x=>x.id===S.district))S.district=matchingDistricts[0]?.id||\'\';\n  const levels={n:[\'n\',\'p\',\'c\',\'d\'],p:[\'p\',\'c\',\'d\'],c:[\'c\',\'d\'],d:[\'d\']}[S.scope];if(!levels.includes(S.level))S.level=levels[0];\n  return {all,matchingCantons,matchingDistricts,levels};\n }\n function updateControls(){const root=originalRoot,q=id=>root.querySelector(\'[data-atlas29="\'+id+\'"]\'), data=scope();\n  q(\'year\').innerHTML=selectOptions(db.years.map(y=>[y,y]),S.year);\n  q(\'round\').innerHTML=selectOptions(S.arena===0&&availableRounds().includes(2)?[[1,\'Primera ronda\'],[2,\'Segunda ronda\']]:[[1,S.arena===0?\'Primera ronda\':\'Diputaciones\']],S.round);\n  q(\'arena\').value=String(S.arena);q(\'scope\').value=S.scope;q(\'round\').closest(\'.a29-input\').hidden=S.arena!==0;\n  q(\'province\').closest(\'.a29-input\').hidden=S.scope===\'n\';q(\'province\').innerHTML=selectOptions(data.all.provinces.map(x=>[x,CODE_TO_PROV[x]]),S.province);\n  q(\'canton\').closest(\'.a29-input\').hidden=!(S.scope===\'c\'||S.scope===\'d\');q(\'canton\').innerHTML=selectOptions(data.matchingCantons.map(x=>[x.id,x.label]),S.canton);\n  q(\'district\').closest(\'.a29-input\').hidden=S.scope!==\'d\';q(\'district\').innerHTML=selectOptions(data.matchingDistricts.map(x=>[x.id,x.label]),S.district);\n  const labels={n:\'Resultado total\',p:\'Por provincias\',c:\'Por cantones\',d:\'Por distritos\'};\n  // Total del ámbito seleccionado = una sola fila agregada, sin confundirlo con provincia.\n  const choices=[[\'total\',\'Resultado total\'],...data.levels.filter(l=>l!==S.scope).map(l=>[l,labels[l]])];\n  q(\'detail\').innerHTML=selectOptions(choices,S.level===S.scope?\'total\':S.level);\n  const detail=q(\'detail\').value;if(detail===\'total\')currentBreakdown=\'total\';else currentBreakdown=detail;\n  q(\'unit\').value=S.unit;q(\'composition\').value=S.composition;\n  q(\'crumb\').innerHTML=[[\'Costa Rica\',\'n\'],S.scope!==\'n\'?[CODE_TO_PROV[S.province],\'p\']:null,(S.scope===\'c\'||S.scope===\'d\')?[data.matchingCantons.find(x=>x.id===S.canton)?.label||\'Cantón\',\'c\']:null,S.scope===\'d\'?[data.matchingDistricts.find(x=>x.id===S.district)?.label||\'Distrito\',\'d\']:null].filter(Boolean).map(([label,l],i)=>`<button type="button" data-a29-crumb="${l}">${esc(label)}</button>`).join(\'<span>›</span>\');\n }\n function buildRows(){const l=currentBreakdown===\'total\'?S.scope:currentBreakdown;\n  const former=S.level;S.level=l;const rows=tally();S.level=former;\n  if(currentBreakdown===\'total\'&&rows.length!==1){const g={id:\'TOTAL\',label:S.scope===\'n\'?\'Costa Rica\':S.scope===\'p\'?CODE_TO_PROV[S.province]:S.scope===\'c\'?(globalOptions().cantons.find(x=>x.id===S.canton)?.label||\'Cantón\'):\'Distrito\',province:CODE_TO_PROV[S.province]||\'Costa Rica\',canton:\'\',district:\'\',parties:new Map(),valid:0,blank:0,null:0,abst:0};\n   for(const r of rows){g.valid+=r.valid;g.blank+=r.blank;g.null+=r.null;g.abst+=r.abst;r.parties.forEach((v,k)=>g.parties.set(k,(g.parties.get(k)||0)+v))}return [g]}\n  return rows;\n }\n function total(rows){const agg={valid:0,blank:0,null:0,abst:0,parties:new Map()};for(const r of rows){agg.valid+=r.valid;agg.blank+=r.blank;agg.null+=r.null;agg.abst+=r.abst;r.parties.forEach((v,k)=>agg.parties.set(k,(agg.parties.get(k)||0)+v))}return agg}\n function percentDen(g){return S.composition===\'valid\'?g.valid:S.composition===\'emitidos\'?g.valid+g.blank+g.null:g.valid+g.blank+g.null+g.abst}\n function val(v,g){return S.unit===\'votes\'?fmt(v):S.unit===\'pct\'?pct(percentDen(g)>0?100*v/percentDen(g):NaN):fmt(v)+\' · \'+pct(percentDen(g)>0?100*v/percentDen(g):NaN)}\n function getSortedParty(rows){const g=total(rows);return [...g.parties.entries()].sort((a,b)=>b[1]-a[1]).map(x=>x[0])}\n function renderMatrix(rows){const host=originalRoot.querySelector(\'[data-atlas29="table-panel"]\'),parties=getSortedParty(rows),gall=total(rows);currentParties=parties;viewRows=rows;\n  const visible=rows.filter(r=>(r.label+\' \'+r.canton+\' \'+r.province+\' \'+r.district).toLocaleLowerCase(\'es\').includes(S.search.toLocaleLowerCase(\'es\')));\n  const maxPage=0;page=0;const windowRows=visible;let lastProvince=\'\',lastCanton=\'\';const colCount=parties.length+2+(S.composition!==\'valid\'?2:0)+(S.composition===\'padron\'?1:0); // Un solo cuadro, sin paginación territorial.\n  host.querySelector(\'.a29-table-box\').innerHTML=`<table class="a29-matrix"><thead><tr><th class="a29-geo">Territorio</th>${parties.map(p=>`<th class="a29-party" style="--a29-color:${color(p)}"><span>${esc(db.parties[p])}</span></th>`).join(\'\')}${S.composition!==\'valid\'?\'<th>Blancos</th><th>Nulos</th>\':\'\'}${S.composition===\'padron\'?\'<th>Abstención</th>\':\'\'}<th>Total de votos válidos</th></tr></thead><tbody>${windowRows.map((g,i)=>{\n   const maxVote=Math.max(0,...g.parties.values());\n   const cells=parties.map(p=>`<td style="${cellStyle(color(p),maxVote>0&&g.parties.get(p)===maxVote)}" title="${esc(db.parties[p])} · ${fmt(g.parties.get(p)||0)} votos">${val(g.parties.get(p)||0,g)}</td>`).join(\'\');\n   const xtra=S.composition!==\'valid\'?`<td>${val(g.blank,g)}</td><td>${val(g.null,g)}</td>`:\'\';\n   const ex2=S.composition===\'padron\'?`<td>${val(g.abst,g)}</td>`:\'\';\n   const geo=currentBreakdown===\'p\'?g.label:currentBreakdown===\'c\'?g.canton:currentBreakdown===\'d\'?[g.canton,g.district].join(\' · \'):g.label;\n   const drill=currentBreakdown===\'p\'?`<button data-a29-drill="p" data-code="${g.id}">Ver cantones</button>`:currentBreakdown===\'c\'?`<button data-a29-drill="c" data-code="${g.id}">Ver distritos</button>`:currentBreakdown===\'d\'?`<button data-a29-drill="d" data-code="${g.id}">Resultado</button>`:\'\';\n   const open=currentBreakdown===\'p\'||currentBreakdown===\'c\'?`<button data-a33-open="${currentBreakdown}" data-code="${g.id}">Ver resultado total</button>`:\'\';\n   const heading=`<th class="a29-geo"><strong>${esc(g.label)}</strong>${currentBreakdown===\'d\'?`<small>${esc(g.canton)} · ${esc(g.province)}</small>`:currentBreakdown===\'c\'?`<small>${esc(g.province)}</small>`:\'\'}${open}${drill}</th>`;\n   let group=\'\';if([\'c\',\'d\'].includes(currentBreakdown)&&lastProvince!==g.province){lastProvince=g.province;lastCanton=\'\';group+=`<tr class="a33-province-divider"><th colspan="${colCount}">${esc(g.province)}</th></tr>`}if(currentBreakdown===\'d\'&&lastCanton!==g.code.slice(0,3)){lastCanton=g.code.slice(0,3);group+=`<tr class="a33-canton-divider"><th colspan="${colCount}">Cantón de ${esc(g.canton)}</th></tr>`}return group+`<tr data-a29-province="${esc(g.province)}">${heading}${cells}${xtra}${ex2}<td class="a29-valid">${fmt(g.valid)}</td></tr>`;\n  }).join(\'\')}</tbody></table>`;\n  const summary=originalRoot.querySelector(\'[data-atlas29="count"]\');summary.textContent=`${fmt(visible.length)} territorios · ${fmt(parties.length)} organizaciones · ${S.year}`;\n  originalRoot.querySelector(\'[data-atlas29="page"]\') .textContent=`Página ${page+1} de ${maxPage+1}`;\n  originalRoot.querySelector(\'[data-atlas29="prev"]\').disabled=page===0;originalRoot.querySelector(\'[data-atlas29="next"]\').disabled=page===maxPage;\n  const note=originalRoot.querySelector(\'[data-atlas29="base"]\');note.textContent=S.composition===\'valid\'?\'Porcentaje de los votos válidos de cada territorio. Todas las columnas partidarias suman 100 % cuando hay votos válidos.\':S.composition===\'emitidos\'?\'Porcentaje de las papeletas emitidas: votos partidarios + blancos + nulos.\':\'Porcentaje del padrón analítico: votos partidarios + blancos + nulos + abstención. El padrón se calcula a partir de los registros territoriales disponibles.\';\n }\n function renderTotals(rows){const root=originalRoot,kpi=root.querySelector(\'[data-atlas29="kpis"]\'),g=total(rows),emit=g.valid+g.blank+g.null,pad=emit+g.abst;\n  kpi.innerHTML=[[\'Votos válidos\',fmt(g.valid)],[\'Votos emitidos\',fmt(emit)],[\'Participación\',pct(pad?100*emit/pad:NaN)],[\'Abstencionismo\',pct(pad?100*g.abst/pad:NaN)],[\'Blancos\',fmt(g.blank)],[\'Nulos\',fmt(g.null)]].map(([title,value])=>`<div class="a29-kpi"><small>${title}</small><strong>${value}</strong></div>`).join(\'\');\n  const graph=root.querySelector(\'[data-atlas29="bars"]\'),box=root.querySelector(\'[data-atlas29="bar-panel"]\');box.hidden=currentBreakdown!==\'total\';\n  if(box.hidden)return;\n  const ids=getSortedParty(rows),values=ids.map(id=>g.parties.get(id)||0),labels=ids.map(id=>db.parties[id]),v=ids.map(x=>color(x)),den=percentDen(g);\n  if(window.Plotly)Plotly.react(graph,[{type:\'bar\',orientation:\'h\',x:values.slice().reverse(),y:labels.slice().reverse(),marker:{color:v.slice().reverse()},text:values.slice().reverse().map(n=>S.unit===\'votes\'?fmt(n):S.unit===\'pct\'?pct(den?100*n/den:NaN):fmt(n)+\' · \'+pct(den?100*n/den:NaN)),textposition:\'outside\',cliponaxis:false,customdata:values.slice().reverse().map(n=>den?100*n/den:null),hovertemplate:\'%{y}<br>%{x:,.0f} votos<br>%{customdata:.2f}%<extra></extra>\'}],{height:Math.max(480,ids.length*37+165),margin:{l:Math.min(340,Math.max(180,labels.reduce((m,s)=>Math.max(m,s.length*7),0))),r:210,t:100,b:55},paper_bgcolor:\'#ffffff\',plot_bgcolor:\'#ffffff\',title:{text:contextTitle(),font:{size:16}},xaxis:{title:\'Votos registrados\',range:[0,Math.max(1,...values)*1.22],automargin:true},yaxis:{automargin:true},font:{family:\'Inter, Arial, sans-serif\',size:11,color:\'#2b5164\'}},{displaylogo:false,responsive:true});\n }\n\n function availableRounds(){return [1,2].filter(round=>index.has(S.year+\'|\'+S.arena+\'|\'+round))}\n function contextTitle(){const opt=globalOptions();const place=S.scope===\'n\'?\'Costa Rica\':S.scope===\'p\'?CODE_TO_PROV[S.province]:S.scope===\'c\'?opt.cantons.find(x=>x.id===S.canton)?.label:opt.districts.find(x=>x.id===S.district)?.label;return `${S.arena===0?\'Presidencia\':\'Diputaciones\'} · ${S.year}${S.arena===0?\' · \'+(S.round===2?\'Segunda ronda\':\'Primera ronda\'):\'\'} · ${place||\'Sin territorio observado\'}`}\n function cellStyle(c,winner){const rgb=[1,3,5].map(i=>parseInt(c.slice(i,i+2),16));const dark=\'#\'+rgb.map(v=>Math.round(v*.54).toString(16).padStart(2,\'0\')).join(\'\');return `background-color:${winner?dark:\'rgb(\'+rgb.map(v=>Math.round(v*.12+255*.88)).join(\',\')+\')\'};color:${winner?\'#ffffff\':\'#17394d\'};font-weight:${winner?800:400}`}\n let mapVersion=0,mapDrawing=false,mapPending=false,lastMapKey=\'\';\n function mapRecords(){const saved={scope:S.scope,level:S.level};S.scope=\'n\';S.level=\'p\';const rows=tally();Object.assign(S,saved);return rows.map(g=>{const sorted=[...g.parties].sort((a,b)=>b[1]-a[1]);return {provincia:g.province,partido:sorted.length?db.parties[sorted[0][0]]:null,votos:sorted[0]?.[1]??null,porcentaje:g.valid?100*(sorted[0]?.[1]||0)/g.valid:null,color:sorted.length?color(sorted[0][0]):\'#e2e8ec\',ronda:S.round,eleccion:S.year,arena:S.arena===0?\'Presidencia\':\'Diputaciones\'}})}\n function renderMap(){const key=[S.year,S.arena,S.round,S.province,S.scope===\'n\'].join(\'|\');if(mapDrawing){mapPending=true;return}if(key===lastMapKey)return;const gd=originalRoot.querySelector(\'[data-atlas29="map"]\');if(!gd||!window.Plotly)return;\n  const payload=document.getElementById(\'atlas-v2317-json\');const geometry=payload?JSON.parse(payload.textContent).geometry:(typeof D!==\'undefined\'?D.geografia:null);\n  if(!geometry?.features?.length)throw Error(\'Falta la cartografía provincial incorporada\');\n  const rows=mapRecords(),traces=[],version=++mapVersion;\n  for(const f of geometry.features){const props=f.properties||{},raw=props.NAME_1||props.shapeName||props.name||props.provincia;const prov=PROV.find(p=>N(p)===N(String(raw).replace(/^Provincia\\s+/i,\'\')));if(!prov)continue;const r=rows.find(x=>x.provincia===prov);const polygons=f.geometry.type===\'MultiPolygon\'?f.geometry.coordinates:[f.geometry.coordinates];\n   for(const poly of polygons){const ring=poly[0];if(!ring?.length)continue;traces.push({type:\'scatter\',mode:\'lines\',x:ring.map(p=>p[0]),y:ring.map(p=>p[1]),fill:\'toself\',fillcolor:r?.color||\'#e2e8ec\',line:{color:S.scope!==\'n\'&&CODE_TO_PROV[S.province]===prov?\'#163b50\':\'#ffffff\',width:S.scope!==\'n\'&&CODE_TO_PROV[S.province]===prov?3:1.3},meta:{provincia:prov},customdata:ring.map(()=>prov),hoveron:\'fills\',hoverinfo:\'text\',text:`${esc(prov)}<br>${esc(r?.partido||\'Sin datos\')}<br>${fmt(r?.votos)} votos · ${pct(r?.porcentaje)}`,showlegend:false})}}\n  const title=`Mapa provincial · ${S.arena===0?\'Presidencia\':\'Diputaciones\'} · ${S.year}${S.arena===0?\' · \'+(S.round===2?\'Segunda ronda\':\'Primera ronda\'):\'\'}${S.scope!==\'n\'?\' · \'+CODE_TO_PROV[S.province]+\' seleccionada\':\'\'}`;\n  originalRoot.querySelector(\'[data-atlas29="map-title"]\').textContent=title;\n  mapDrawing=true;lastMapKey=key;Promise.resolve(Plotly.react(gd,traces,{title:{text:title,font:{size:14}},height:650,paper_bgcolor:\'#ffffff\',plot_bgcolor:\'#ffffff\',showlegend:false,margin:{l:15,r:15,t:90,b:25},xaxis:{visible:false,range:[-86.3,-82.1],fixedrange:true},yaxis:{visible:false,range:[7.8,11.4],scaleanchor:\'x\',fixedrange:true},dragmode:false},{responsive:true,displaylogo:false})).then(()=>{\n   if(version!==mapVersion||!gd.isConnected)return;gd.removeAllListeners?.(\'plotly_click\');gd.on(\'plotly_click\',ev=>{const point=ev?.points?.[0],raw=point?.data?.meta?.provincia||point?.customdata||point?.location;const prov=PROV.find(p=>N(p)===N(String(raw).replace(/^Provincia\\s+/i,\'\')));if(!prov)return;S.scope=\'p\';S.province=PNUM[prov];S.canton=\'\';S.district=\'\';S.level=\'p\';currentBreakdown=\'total\';render()});\n  }).catch(e=>{lastMapKey=\'\';console.error(\'Mapa provincial\',e)}).finally(()=>{mapDrawing=false;if(mapPending){mapPending=false;renderMap()}});\n }\n window.Atlas31Results={getSelection:()=>({...S}),setSelection:patch=>{Object.assign(S,patch);render()},mapRecords:()=>mapRecords()};\n\n\n function renderNavigation(){const all=globalOptions(),place=S.scope===\'n\'?\'Costa Rica\':S.scope===\'p\'?CODE_TO_PROV[S.province]:S.scope===\'c\'?all.cantons.find(c=>c.id===S.canton)?.label:all.districts.find(d=>d.id===S.district)?.label;\n  const levels={n:[\'p\',\'c\',\'d\'],p:[\'c\',\'d\'],c:[\'d\'],d:[]}[S.scope],label={p:\'Ver por provincias\',c:\'Ver por cantones\',d:\'Ver por distritos\'},base=currentBreakdown===\'total\'?\'Resultado total de \'+place:\'Resultados de \'+place+\' · \'+label[currentBreakdown].replace(\'Ver por \',\'por \');\n  const html=`<div class="a33-view-context"><strong>${esc(base)}</strong><span>${S.year} · ${S.arena===0?\'Presidencia\':\'Diputaciones\'}${S.arena===0?\' · \'+(S.round===2?\'Segunda ronda\':\'Primera ronda\'):\'\'}</span></div><div class="a33-territory-actions"><button type="button" data-a33-home ${S.scope===\'n\'&&currentBreakdown===\'total\'?\'disabled\':\'\'}>Volver al total de Costa Rica</button>${S.scope===\'c\'||S.scope===\'d\'?`<button type="button" data-a33-parent="p">Volver al total de ${esc(CODE_TO_PROV[S.province])}</button>`:\'\'}${S.scope===\'d\'?\'<button type="button" data-a33-parent="c">Volver al total del cantón</button>\':\'\'}<div class="a33-breakdowns" role="group" aria-label="Desglosar resultados"><button type="button" data-a33-detail="total" aria-pressed="${currentBreakdown===\'total\'}">${S.scope===\'n\'?\'Total de Costa Rica\':\'Total de \'+esc(place)}</button>${levels.map(l=>`<button type="button" data-a33-detail="${l}" aria-pressed="${currentBreakdown===l}">${label[l]}</button>`).join(\'\')}</div></div>`;\n  for(const el of originalRoot.querySelectorAll(\'[data-atlas29="near-navigation"]\'))el.innerHTML=html;\n }\n function navigateNear(button){\n  if(button.hasAttribute(\'data-a33-home\')){S.scope=\'n\';S.level=\'n\';S.province=\'\';S.canton=\'\';S.district=\'\';currentBreakdown=\'total\'}\n  else if(button.dataset.a33Parent){S.scope=button.dataset.a33Parent;S.level=S.scope;if(S.scope===\'p\'){S.canton=\'\';S.district=\'\'}else S.district=\'\';currentBreakdown=\'total\'}\n  else if(button.dataset.a33Detail){currentBreakdown=button.dataset.a33Detail;S.level=currentBreakdown===\'total\'?S.scope:currentBreakdown}\n  else if(button.dataset.a33Open){S.scope=button.dataset.a33Open;S.level=S.scope;const id=button.dataset.code;S.province=id[0];S.canton=S.scope===\'c\'?id:\'\';S.district=\'\';currentBreakdown=\'total\'}else return false;\n  S.search=\'\';page=0;render();return true;\n }\n\n function render(){if(!originalRoot?.isConnected)return;\n  if(S.arena!==0||!availableRounds().includes(2))S.round=1;\n  updateControls();renderNavigation();const rows=buildRows();renderMap();renderTotals(rows);\n  const grid=originalRoot.querySelector(\'[data-atlas29="table-panel"]\'),bars=originalRoot.querySelector(\'[data-atlas29="bar-panel"]\');\n  grid.hidden=currentBreakdown===\'total\';bars.hidden=currentBreakdown!==\'total\';\n  if(!grid.hidden)renderMatrix(rows);\n  const title=originalRoot.querySelector(\'[data-atlas29="title"]\');title.textContent=contextTitle();\n  originalRoot.querySelector(\'[data-atlas29="subtitle"]\').textContent=currentBreakdown===\'total\'?\'Resultado por partido del territorio seleccionado.\':`Desglose por ${currentBreakdown===\'p\'?\'provincia\':currentBreakdown===\'c\'?\'cantón\':\'distrito\'} · filas territoriales y partidos en columnas.`;\n }\n function changeRoot(e){const el=e.target;const field=el.dataset.atlas29;if(!field)return;\n  if(field===\'year\'){S.year=Number(el.value);}\n  else if(field===\'arena\'){S.arena=Number(el.value);S.round=1;}\n  else if(field===\'round\'){S.round=Number(el.value);}\n  else if(field===\'scope\'){S.scope=el.value;S.level=S.scope;}\n  else if(field===\'province\'){S.province=el.value;S.canton=\'\';S.district=\'\';S.level=S.scope;}\n  else if(field===\'canton\'){S.canton=el.value;S.district=\'\';S.level=S.scope;}\n  else if(field===\'district\'){S.district=el.value;S.level=S.scope;}\n  else if(field===\'detail\'){currentBreakdown=el.value;S.level=currentBreakdown===\'total\'?S.scope:currentBreakdown;}\n  else if(field===\'unit\'){S.unit=el.value;}\n  else if(field===\'composition\'){S.composition=el.value;}\n  else if(field===\'search\'){S.search=el.value;page=0;}\n  render();\n }\n function exportRows(){const all=buildRows(),partyIds=getSortedParty(all);return all.map(g=>{\n  const record={eleccion:S.year,arena:S.arena===0?\'Presidencia\':\'Diputaciones\',ronda:S.round,ambito:S.scope,detalle:currentBreakdown,territorio:g.label,provincia:g.province,canton:g.canton,distrito:g.district,codigo:g.id,votos_validos:g.valid,votos_blancos:g.blank,votos_nulos:g.null,abstenciones:g.abst};\n  for(const p of partyIds){const nom=db.parties[p];record[nom+\' · votos\']=g.parties.get(p)||0;record[nom+\' · porcentaje\']=percentDen(g)?Number((100*(g.parties.get(p)||0)/percentDen(g)).toFixed(5)):null}return record;\n })}\n function download(name,text,mime){const blob=text instanceof Blob?text:new Blob([text],{type:mime});const a=document.createElement(\'a\');a.href=URL.createObjectURL(blob);a.download=name;document.body.appendChild(a);a.click();setTimeout(()=>{URL.revokeObjectURL(a.href);a.remove()},1000)}\n function csv(rows){if(!rows.length)return \'\';const cols=Object.keys(rows[0]);return \'\\ufeff\'+cols.join(\',\')+\'\\r\\n\'+rows.map(r=>cols.map(k=>\'"\'+String(r[k]??\'\').replace(/"/g,\'""\')+\'"\').join(\',\')).join(\'\\r\\n\')}\n function matrixSvg(){const table=originalRoot.querySelector(\'.a29-matrix\');if(!table)return \'<svg xmlns="http://www.w3.org/2000/svg"/>\';\n  const head=[...table.tHead.rows[0].cells].map(c=>c.textContent.trim()),data=[...table.tBodies[0].rows].map(r=>[...r.cells].map(c=>c.innerText.replace(/\\n+/g,\' · \').trim()));\n  const cols=head.length,w=Math.min(3200,180+cols*122),rowHeight=34,h=155+(data.length+1)*rowHeight;let out=`<svg xmlns="http://www.w3.org/2000/svg" width="${w}" height="${h}" viewBox="0 0 ${w} ${h}"><rect width="100%" height="100%" fill="#fff"/>`;\n  out+=`<text x="14" y="26" font-size="18" fill="#17394d" font-family="Arial">${esc(contextTitle())}</text><text x="14" y="50" font-size="12" fill="#17394d" font-family="Arial">${esc(originalRoot.querySelector(\'[data-atlas29="subtitle"]\').textContent)} · ${esc(S.unit)} · ${esc(S.composition)}</text><g transform="translate(0,80)">`;\n  const widths=Array(cols).fill((w-190)/(cols-1));widths[0]=190;\n  const x=Array.from({length:cols},(_,i)=>widths.slice(0,i).reduce((a,b)=>a+b,0));\n  for(let i=0;i<cols;i++){out+=`<rect x="${x[i]}" y="0" width="${widths[i]}" height="58" fill="${i===0?\'#163e50\':i<=currentParties.length?color(currentParties[i-1]):\'#476b7d\'}"/><text x="${x[i]+6}" y="20" font-size="10" fill="#fff" font-family="Arial">${esc(head[i].slice(0,19))}</text>`}\n  data.forEach((row,i)=>row.forEach((text,j)=>{let y=58+i*rowHeight;out+=`<rect x="${x[j]}" y="${y}" width="${widths[j]}" height="${rowHeight}" fill="${j>0&&j<=currentParties.length?table.tBodies[0].rows[i].cells[j].style.backgroundColor:(i%2?\'#eef6f9\':\'#fff\')}" stroke="#d5e6eb" stroke-width=".6"/><text x="${x[j]+5}" y="${y+20}" font-size="${j===0?10:9}" fill="${j>0&&j<=currentParties.length?table.tBodies[0].rows[i].cells[j].style.color:\'#244b5b\'}" font-weight="${j>0&&j<=currentParties.length?table.tBodies[0].rows[i].cells[j].style.fontWeight:400}" font-family="Arial">${esc(text.slice(0,j===0?26:18))}</text>`}));return out+\'</g></svg>\';\n }\n function exportVisual(extension){const name=`Atlas_${S.arena===0?\'Presidencia\':\'Diputaciones\'}_${S.year}_R${S.round}_${S.scope}_${S.province}_${S.canton}_${S.district}_${currentBreakdown}`;\n  if(extension===\'csv\')return download(name+\'.csv\',csv(exportRows()),\'text/csv;charset=utf-8\');\n  if(currentBreakdown===\'total\')return Plotly.downloadImage(originalRoot.querySelector(\'[data-atlas29="bars"]\'),{format:extension,width:1700,height:Math.max(650,(originalRoot.querySelector(\'[data-atlas29="bars"]\').layout?.height||950)),scale:extension===\'png\'?2:1,filename:name});\n  if(window.Atlas31TableImage)return window.Atlas31TableImage(originalRoot.querySelector(\'.a29-matrix\'),contextTitle()+\' · \'+originalRoot.querySelector(\'[data-atlas29="subtitle"]\').textContent+\' · \'+S.unit+\' · \'+S.composition,extension,name);\n  if(window.Atlas31TableImage)return window.Atlas31TableImage(originalRoot.querySelector(\'.a29-matrix\'),contextTitle()+\' · \'+originalRoot.querySelector(\'[data-atlas29="subtitle"]\').textContent+\' · \'+S.unit+\' · \'+S.composition,extension,name);\n  if(window.Atlas31TableImage)return window.Atlas31TableImage(originalRoot.querySelector(\'.a29-matrix\'),contextTitle()+\' · \'+originalRoot.querySelector(\'[data-atlas29="subtitle"]\').textContent+\' · \'+S.unit+\' · \'+S.composition,extension,name);\n  const xml=matrixSvg();if(extension===\'svg\')return download(name+\'.svg\',xml,\'image/svg+xml\');\n  const url=URL.createObjectURL(new Blob([xml],{type:\'image/svg+xml\'})),image=new Image();image.onload=()=>{const cvs=document.createElement(\'canvas\');cvs.width=image.naturalWidth;cvs.height=image.naturalHeight;const ctx=cvs.getContext(\'2d\');ctx.drawImage(image,0,0);cvs.toBlob(x=>{if(x)download(name+\'.png\',x,\'image/png\');URL.revokeObjectURL(url)},\'image/png\')};image.onerror=()=>URL.revokeObjectURL(url);image.src=url;\n }\n function sourceSelect(id,items){return `<label class="a29-input"><span>${esc(id)}</span><select data-atlas29="${items}"></select></label>`}\n function mount(){const container=document.getElementById(\'atlas-v2317-seccion07\');if(!container||!container.querySelector(\'.ar-shell\'))return;\n  if(container.querySelector(\'#atlas-v29-main\')){originalRoot=container;return}\n  const d=dataset();if(!d)return;\n  lastMapKey=\'\';originalRoot=container;let shell=container.querySelector(\'.ar-shell\');\n  const head=shell.querySelector(\'.ar-top\');if(head){const title=head.querySelector(\'h2\');if(title)title.textContent=\'Resultados electorales · Atlas de Costa Rica\';const info=head.querySelector(\'p:not(.ar-eyebrow)\');if(info)info.textContent=\'Explore las elecciones por ámbito territorial y desglose de votos; seleccione una provincia en el mapa para profundizar.\';}\n  // Todas las pantallas se construyen dentro de la raíz ya existente: ningún explorador paralelo.\n  const top=document.createElement(\'section\');top.id=\'atlas-v29-main\';top.innerHTML=`<div class="a29-header"><div><span class="a29-kicker">RESULTADOS ELECTORALES · 1953–2026</span><h3 data-atlas29="title"></h3><p data-atlas29="subtitle"></p></div><span class="a29-tag">Resultados históricos</span></div><div class="a29-toolbar" aria-label="1. Elección y territorio"><label class="a29-input"><span>Elección</span><select data-atlas29="year"></select></label><label class="a29-input"><span>Elección de</span><select data-atlas29="arena"><option value="0">Presidencia</option><option value="1">Diputaciones</option></select></label><label class="a29-input"><span>Ronda</span><select data-atlas29="round"></select></label><div class="a31-control-divider">2. Presentación de los resultados</div><label class="a29-input"><span>Ver valores</span><select data-atlas29="unit"><option value="pct">Porcentajes</option><option value="votes">Votos</option><option value="both">Ambos</option></select></label><label class="a29-input"><span>Composición</span><select data-atlas29="composition"><option value="valid">Votos válidos</option><option value="emitidos">Incluye blancos y nulos</option><option value="padron">Incluye abstención</option></select></label></div><details class="a33-exact"><summary>Buscar un territorio específico</summary><div class="a29-toolbar"><label class="a29-input"><span>Territorio</span><select data-atlas29="scope"><option value="n">Costa Rica</option><option value="p">Provincia</option><option value="c">Cantón</option><option value="d">Distrito</option></select></label><label class="a29-input"><span>Provincia</span><select data-atlas29="province"></select></label><label class="a29-input"><span>Cantón</span><select data-atlas29="canton"></select></label><label class="a29-input"><span>Distrito</span><select data-atlas29="district"></select></label><label class="a29-input"><span>Mostrar</span><select data-atlas29="detail"></select></label></div></details><div class="a29-breadcrumb" data-atlas29="crumb"></div><div class="a29-kpis" data-atlas29="kpis"></div><div class="a29-visuals"><div data-atlas29="map-host"></div><nav class="a33-near-navigation" data-atlas29="near-navigation" aria-label="Exploración territorial"></nav><div data-atlas29="bar-panel" class="a29-bar-panel"><header><h4>Votos por organización política</h4><div class="a29-export"><button data-a29-export="png">PNG</button><button data-a29-export="svg">SVG</button><button data-a29-export="csv">CSV</button></div></header><div data-atlas29="bars" class="a29-bars"></div><nav class="a33-near-navigation a33-result-footer" data-atlas29="near-navigation" aria-label="Volver o desglosar el resultado"></nav></div></div><section data-atlas29="table-panel" class="a29-matrix-panel" hidden><header><div><h4>Resultados por territorio y organización política</h4><p data-atlas29="base"></p></div><div class="a29-export"><button data-a29-export="png">PNG</button><button data-a29-export="svg">SVG</button><button data-a29-export="csv">CSV</button></div></header><div class="a29-matrix-tools"><label>Buscar territorio <input data-atlas29="search" placeholder="Provincia, cantón o distrito" type="search"/></label><span data-atlas29="count"></span></div><div class="a29-table-box"></div><nav class="a33-near-navigation" data-atlas29="near-navigation" aria-label="Volver o cambiar el desglose"></nav><div class="a29-pagination"><button data-atlas29="prev">Anterior</button><span data-atlas29="page"></span><button data-atlas29="next">Siguiente</button></div></section></div>`;\n  let anchor=shell.querySelector(\'.ar-top\');anchor.after(top);\n  const mainGrid=shell.querySelector(\'.ar-grid\'),mapPanel=mainGrid?.querySelector(\'.ar-panel:has([data-role="map"])\');if(mapPanel){\n   // Never share a Plotly node or map-title with the historical asynchronous renderer.\n   const panel=document.createElement(\'section\');panel.className=\'ar-panel a29-map-panel\';\n   panel.innerHTML=\'<div class="ar-panel-head"><div><h3 data-atlas29="map-title">Mapa provincial</h3><p>Pulse una provincia para consultar su resultado total.</p></div><div class="ar-panel-controls"><button data-export="map-png">PNG</button><button data-export="map-svg">SVG</button><button data-export="map-csv">CSV</button></div></div><div class="ar-map" data-atlas29="map"></div>\';\n   top.querySelector(\'[data-atlas29="map-host"]\').append(panel);mapPanel.hidden=true;\n  }\n  const older=[shell.querySelector(\'.ar-tabs\'),shell.querySelector(\'.ar-controls\'),shell.querySelector(\'.ar-kpis\')];older.forEach(e=>{if(e)e.hidden=true});\n  // Los análisis previos continúan accesibles, pero no interrumpen el escrutinio principal.\n  const extra=document.createElement(\'details\');extra.className=\'a29-advanced\';extra.innerHTML=\'<summary>Comparaciones, composición legislativa y detalles complementarios</summary>\';\n  for(const selector of [\'.ar-integration\',\'.ar-section-grid\']){const element=shell.querySelector(selector);if(element)extra.append(element)}\n  if(mainGrid){for(const element of [...mainGrid.children])extra.append(element);mainGrid.remove()}\n  shell.append(extra);\n  const oldNames=extra.querySelector(\'[data-role="names"]\')?.closest(\'.ar-panel\');if(oldNames){const namesDetails=document.createElement(\'details\');namesDetails.className=\'a29-advanced\';namesDetails.innerHTML=\'<summary>Diputaciones electas y consulta nominal</summary>\';namesDetails.append(oldNames);shell.append(namesDetails)}\n  const comment=shell.querySelector(\'.ar-comment\');if(comment)comment.hidden=true;\n  top.addEventListener(\'change\',changeRoot);\n  top.addEventListener(\'click\',event=>{const b=event.target.closest(\'[data-export]\');if(b&&b.dataset.export.startsWith(\'map-\')){event.preventDefault();event.stopImmediatePropagation();const ext=b.dataset.export.slice(4);if(ext===\'csv\')download(\'Atlas_Mapa_\'+S.year+\'.csv\',csv(mapRecords()),\'text/csv\');else Plotly.downloadImage(top.querySelector(\'[data-atlas29="map"]\'),{format:ext,filename:\'Atlas_Mapa_\'+S.year+\'_ronda_\'+S.round,width:1700,height:1100,scale:ext===\'png\'?2:1});}},true);\n  top.addEventListener(\'input\',e=>{if(e.target.dataset.atlas29===\'search\'){S.search=e.target.value;page=0;renderMatrix(viewRows)}});\n  top.addEventListener(\'click\',e=>{const button=e.target.closest(\'button\');if(!button)return;\n   if(navigateNear(button))return;\n   if(button.dataset.a29Export)return exportVisual(button.dataset.a29Export);\n   if(button.dataset.atlas29===\'prev\'||button.dataset.atlas29===\'next\'){page+=button.dataset.atlas29===\'next\'?1:-1;renderMatrix(buildRows());return}\n   if(button.dataset.a29Crumb){const level=button.dataset.a29Crumb;S.scope=level;S.level=level;S.province=level===\'n\'?\'\':S.province;S.canton=level===\'p\'?\'\':S.canton;S.district=level===\'c\'?\'\':S.district;page=0;render();return}\n   if(button.dataset.a29Drill){let level=button.dataset.a29Drill,id=button.dataset.code;S.scope=level;S.level=level;if(level===\'p\'){S.province=id;S.canton=\'\';S.district=\'\';S.level=\'c\'}else if(level===\'c\'){S.canton=id;S.province=id[0];S.district=\'\';S.level=\'d\'}else {S.district=id;S.province=id[0];S.canton=id.slice(0,3)}page=0;render()}\n  });\n  const focus=typeof state!==\'undefined\'?Number(state.focus):NaN;S.year=db.years.includes(focus)?focus:db.years[db.years.length-1];render();\n }\n function attach(){const old=typeof views===\'object\'&&views?.gobierno;if(typeof old!==\'function\')return;\n  views.gobierno=function(...args){const out=old.apply(this,args);try{mount()}catch(e){console.error(\'Error al cargar los resultados territoriales\',e)}return out};\n  if(typeof state!==\'undefined\'&&state.tab===\'gobierno\')mount();\n }\n if(document.readyState===\'loading\')document.addEventListener(\'DOMContentLoaded\',attach);else attach();\n})();\n'

_ATLAS_V31_ANALYSIS = '/* Consulta de segunda ronda separada; solo elecciones documentadas. */\n(function(){const n=document.getElementById(\'atlas-v28-dataset\');if(!n)return;const db=JSON.parse(n.textContent),available=[...new Set(db.rows.filter(r=>r[1]===0&&r[2]===2).map(r=>r[0]))].sort((a,b)=>a-b);\n window.Atlas33RoundSelection=function(root,arena,ordinaryYears){root.hidden=arena!==0;const enabled=root.querySelector(\'[data33-runoff]\'),list=root.querySelector(\'[data33-years]\');if(arena!==0)enabled.checked=false;\n  if(!list.dataset.ready){list.innerHTML=available.map(y=>`<label><input type="checkbox" value="${y}" checked> ${y}</label>`).join(\'\');list.dataset.ready=\'1\'}\n  list.hidden=!enabled.checked;return enabled.checked?{round:2,years:[...list.querySelectorAll(\'input:checked\')].map(i=>+i.value)}:{round:1,years:ordinaryYears};\n };\n window.Atlas33RoundMarkup=\'<details class="a33-runoff"><summary>Segunda ronda presidencial</summary><label class="a33-runoff-toggle"><input type="checkbox" data33-runoff> Consultar únicamente segundas rondas</label><div class="a33-year-choices" data33-years hidden></div></details>\';\n})();\n\n/* Liderazgos: voto provincial, distribución territorial y diferencias. */\n(function(){\'use strict\';\n const node=document.getElementById(\'atlas-v28-dataset\');if(!node||typeof views===\'undefined\')return;\n const db=(window.AtlasTerritorialData||(window.AtlasTerritorialData=JSON.parse(node.textContent))),provinces=[\'San José\',\'Alajuela\',\'Cartago\',\'Heredia\',\'Guanacaste\',\'Puntarenas\',\'Limón\'];\n const pal=[\'#278e96\',\'#346ab2\',\'#7859ad\',\'#308c76\',\'#c47b46\',\'#297eaa\',\'#a65d87\'];\n const esc=s=>String(s??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\n const norm=s=>String(s||\'\').normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toUpperCase().replace(/[^A-Z0-9]/g,\'\');\n const colors={LIBERACIONNACIONAL:\'#079648\',UNIDADSOCIALCRISTIANA:\'#0876B6\',PUEBLOSOBERANO:\'#14A3AF\',PROGRESOSOCIALDEMOCRATICO:\'#303D91\',FRENTEAMPLIO:\'#CDA02A\',ACCIONCIUDADANA:\'#e0b239\'};\n const color=id=>id===-99?\'#7b8792\':colors[norm(db.parties[id])]||db.colors?.[db.parties[id]]||\'#718B9B\';\n const name=id=>id===-99?\'Empate\':db.parties[id]||\'Sin datos\';\n const ink=c=>{const rgb=[1,3,5].map(i=>parseInt(c.slice(i,i+2),16)/255).map(v=>v<=.04045?v/12.92:((v+.055)/1.055)**2.4);return .2126*rgb[0]+.7152*rgb[1]+.0722*rgb[2]>.179?\'#123546\':\'#fff\'};\n const partyShort=id=>{if(id===-99)return \'Empate\';const full=name(id),known={LIBERACIONNACIONAL:\'PLN\',UNIDADSOCIALCRISTIANA:\'PUSC\',ACCIONCIUDADANA:\'PAC\',FRENTEAMPLIO:\'FA\',PUEBLOSOBERANO:\'PPSO\',COALICIONUNIDAD:\'Coalición Unidad\',UNIFICACIONNACIONAL:\'Unificación Nacional\',PROGRESOSOCIALDEMOCRATICO:\'PPSD\',RESTAURACIONNACIONAL:\'PRN\'};return known[norm(full)]||full};\n const byKey=new Map(),cache=new Map();for(const r of db.rows){const k=[r[0],r[1],r[2]].join(\'|\');if(!byKey.has(k))byKey.set(k,[]);byKey.get(k).push(r)}\n function leaders(y,a,round,l){const key=[y,a,round,l].join(\'|\');if(cache.has(key))return cache.get(key);const groups=new Map();\n  for(const r of byKey.get([y,a,round].join(\'|\'))||[]){if(r[4]<0)continue;const code=String(r[3]).padStart(5,\'0\'),id=l===\'p\'?code[0]:l===\'c\'?code.slice(0,3):code;if(!groups.has(id))groups.set(id,{id,code,parties:new Map()});const g=groups.get(id);g.parties.set(r[4],(g.parties.get(r[4])||0)+Number(r[5]))}\n  const out=[...groups.values()].map(g=>{const max=Math.max(...g.parties.values()),winners=max>0?[...g.parties].filter(x=>x[1]===max).map(x=>x[0]):[];return {...g,winners,winner:winners.length===1?winners[0]:-99,tie:winners.length>1}}).filter(g=>g.winners.length);cache.set(key,out);return out;\n }\n function composition(y,a,round,l){const out=provinces.map((p,i)=>({province:p,id:String(i+1),counts:new Map(),total:0}));for(const g of leaders(y,a,round,l)){const row=out[+g.id[0]-1];row.total++;row.counts.set(g.winner,(row.counts.get(g.winner)||0)+1)}return out}\n function dominance(row){const exclusive=[...row.counts].filter(x=>x[0]>=0),max=Math.max(0,...exclusive.map(x=>x[1]));return {...row,max,winners:max?exclusive.filter(x=>x[1]===max).map(x=>x[0]):[]}}\n function loc(y,g,l){const p=db.places[y+\'|\'+g.code]||[provinces[+g.code[0]-1],\'Código \'+g.id,\'Código \'+g.id];return {province:p[0],canton:p[1],label:l===\'p\'?p[0]:l===\'c\'?p[1]:p[2]}}\n window.Atlas32Territory={leaders,composition,dominance,name,color};\n function save(name,data,type){const u=URL.createObjectURL(data instanceof Blob?data:new Blob([data],{type})),a=document.createElement(\'a\');a.href=u;a.download=name;a.click();setTimeout(()=>URL.revokeObjectURL(u),1000)}\n function tableImage(table,title,ext,name){\n  if(!table)return;\n  const rows=[...table.rows],cols=Math.max(...rows.map(r=>[...r.cells].reduce((s,c)=>s+c.colSpan,0))),widths=Array.from({length:cols},(_,i)=>i?190:230),w=widths.reduce((a,b)=>a+b,0);\n  const prepared=rows.map(row=>{let col=0;return [...row.cells].map(cell=>{const x=widths.slice(0,col).reduce((a,b)=>a+b,0),cw=widths.slice(col,col+cell.colSpan).reduce((a,b)=>a+b,0),cs=getComputedStyle(cell),bg=cs.backgroundColor===\'rgba(0, 0, 0, 0)\'?\'#fff\':cs.backgroundColor;let lines=[];\n   for(const block of (cell.innerText||cell.textContent).split(\'\\n\').filter(Boolean)){let line=\'\';for(const word of block.split(/\\s+/)){if((line+\' \'+word).length>Math.floor(cw/6)-2){if(line)lines.push(line);line=word}else line+=(line?\' \':\'\')+word}if(line)lines.push(line)}col+=cell.colSpan;return {x,cw,lines,cs,bg}})});\n  const heights=prepared.map(cells=>Math.max(32,Math.max(...cells.map(c=>c.lines.length))*14+20)),h=80+heights.reduce((a,b)=>a+b,0);\n  let svg=`<svg xmlns="http://www.w3.org/2000/svg" width="${w}" height="${h}"><rect width="100%" height="100%" fill="white"/><text x="14" y="30" font-family="Arial" font-size="17" fill="#17394d">${esc(title)}</text>`,y=65;\n  prepared.forEach((cells,i)=>{for(const c of cells){svg+=`<rect x="${c.x}" y="${y}" width="${c.cw-1}" height="${heights[i]-1}" fill="${c.bg}"/><text x="${c.x+9}" y="${y+20}" fill="${c.cs.color}" font-family="Arial" font-size="11" font-weight="${c.cs.fontWeight}">${c.lines.map((s,j)=>`<tspan x="${c.x+9}" dy="${j?14:0}">${esc(s)}</tspan>`).join(\'\')}</text>`}y+=heights[i]});svg+=\'</svg>\';\n  if(ext===\'svg\')return save(name+\'.svg\',svg,\'image/svg+xml\');const u=URL.createObjectURL(new Blob([svg],{type:\'image/svg+xml\'})),im=new Image();im.onload=()=>{const c=document.createElement(\'canvas\');c.width=w;c.height=h;const ctx=c.getContext(\'2d\');ctx.fillStyle=\'white\';ctx.fillRect(0,0,w,h);ctx.drawImage(im,0,0);c.toBlob(b=>{if(b)save(name+\'.png\',b,\'image/png\');URL.revokeObjectURL(u)})};im.src=u;\n }\n window.Atlas31TableImage=tableImage;\n function buttons(target){return \'<div class="a25-actions a32-actions">\'+[\'csv\',\'svg\',\'png\'].map(e=>`<button type="button" data-l-export="${e}" data-target="${target}">${e.toUpperCase()}</button>`).join(\'\')+\'</div>\'}\n function mount(){const content=document.getElementById(\'content\');if(!content||content.querySelector(\'#atlas-v31-leadership\'))return;\n  let before=null;for(const h of [...content.querySelectorAll(\'h2,h3,h4\')])if(/partido más votado por provincia|liderazgos por provincia y arena|liderazgos provinciales y cantonales|liderazgos por cantón y elección/i.test(h.textContent)){const card=h.closest(\'.card,article,section\');if(card&&!card.querySelector(\'#atlas-v30-bastiones,#atlas-v29-composicion\')){if(!before){before=document.createElement(\'div\');card.before(before)}card.remove()}}\n  const sec=document.createElement(\'section\');sec.id=\'atlas-v31-leadership\';sec.className=\'a25-block\';sec.innerHTML=`<div class="a25-header"><h3>Liderazgos provinciales y territoriales</h3><p>Distinga el primer lugar por votos del predominio en cantidad de cantones o distritos.</p></div><div class="a29-compare-controls"><label>División territorial<select data-l="level"><option value="p">Provincias</option><option value="c">Cantones</option><option value="d">Distritos</option></select></label><label>Vista de resultados<select data-l="arena"><option value="1">Legislativa</option><option value="0">Presidencial</option></select></label><label>Comparación dentro de cada provincia<select data-l="basis"><option value="c">Cantones encabezados</option><option value="d">Distritos encabezados</option></select></label></div><div data-l="round-tools">${window.Atlas33RoundMarkup}</div><p data-l="note"></p><div class="a32-head"><h4>Partido más votado</h4>${buttons(\'votes\')}</div><div class="a32-scroll" data-l="votes"></div><div class="a32-head"><h4 data-l="dominance-title"></h4>${buttons(\'dominance\')}</div><div class="a32-scroll" data-l="dominance"></div><div class="a32-head"><h4>Diferencias: votos provinciales y predominio territorial</h4>${buttons(\'differences\')}</div><p class="a32-method">Se compara el partido más votado en cada provincia con el que encabeza más cantones o distritos en esa misma provincia. n/N indica territorios encabezados / territorios con resultados en cada año. Empates explícitos; predominio no implica mayoría absoluta. Las siglas mantienen el nombre completo al pasar el cursor y en el CSV.</p><div class="a32-scroll" data-l="differences"></div>`;\n  if(before)before.replaceWith(sec);else content.append(sec);const q=k=>sec.querySelector(\'[data-l="\'+k+\'"]\');let exports={};\n  function table(years,rows,title,cell,grouped=false){let last=\'\';return `<table class="a31-leader-table"><caption>${esc(title)}</caption><thead><tr><th>Territorio</th>${years.map(()=>\'<th aria-hidden="true"></th>\').join(\'\')}</tr></thead><tbody>`+rows.map(u=>{let group=\'\';if(grouped&&last!==u.province){last=u.province;group=`<tr class="a32-group"><th colspan="${years.length+1}" style="border-left:6px solid ${pal[provinces.indexOf(last)]}">${esc(last)}</th></tr>`}return group+\'<tr><th>\'+esc(u.label)+(u.sub?\'<small>\'+esc(u.sub)+\'</small>\':\'\')+\'</th>\'+years.map(y=>cell(y,u)).join(\'\')+\'</tr>\'}).join(\'\')+\'</tbody><tfoot><tr><th>Año</th>\'+years.map(y=>\'<th>\'+y+\'</th>\').join(\'\')+\'</tr></tfoot></table>\'}\n  function draw(){const a=+q(\'arena\').value,l=q(\'level\').value,basis=q(\'basis\').value;sec.dataset.scale=l;const from=typeof state!==\'undefined\'?+state.from:1953,to=typeof state!==\'undefined\'?+state.to:2026,mode=window.Atlas33RoundSelection(q(\'round-tools\').firstElementChild,a,db.years.filter(y=>y>=from&&y<=to)),round=mode.round,years=mode.years.filter(y=>byKey.has([y,a,round].join(\'|\'))),hist=new Map(),names=new Map();exports={votes:[],dominance:[],differences:[]};\n   for(const y of years)for(const g of leaders(y,a,round,l)){hist.set(y+\'|\'+g.id,g);names.set(g.id,{id:g.id,...loc(y,g,l)})}\n   const units=[...names.values()].sort((x,z)=>provinces.indexOf(x.province)-provinces.indexOf(z.province)||x.id.localeCompare(z.id)),context=`${a===0?\'Presidencia\':\'Diputaciones\'} · ${round===2?\'Segunda ronda · \'+(years.join(\', \')||\'sin elecciones seleccionadas\'):from+\'–\'+to}`,unitName=basis===\'c\'?\'cantones\':\'distritos\';q(\'note\').textContent=context+\' · Los años aparecen únicamente abajo. Las celdas vacías indican ausencia de resultados observados.\';\n   q(\'votes\').innerHTML=table(years,units,\'Partido más votado · \'+context,(y,u)=>{const g=hist.get(y+\'|\'+u.id);if(!g)return \'<td class="a32-na">—</td>\';const label=g.winners.map(name).join(\' / \'),c=g.tie?\'#7b8792\':color(g.winner);exports.votes.push({anio:y,arena:a,ronda:round,provincia:u.province,territorio:u.label,partido:label,empate:g.tie});return `<td style="background:${c};color:${ink(c)}"><strong>${esc(label)}</strong>${g.tie?\'<small>Empate en votos</small>\':\'\'}</td>`},l!==\'p\');\n   const provinceRows=provinces.map((p,i)=>({label:p,province:p,id:String(i+1)})),dom=new Map(),voteHist=new Map();for(const y of years){for(const row of composition(y,a,round,basis))dom.set(y+\'|\'+row.id,dominance(row));for(const g of leaders(y,a,round,\'p\'))voteHist.set(y+\'|\'+g.id,g)}\n   q(\'dominance-title\').textContent=\'Partido que encabeza más \'+unitName+\' dentro de cada provincia\';\n   q(\'dominance\').innerHTML=table(years,provinceRows,\'Predominio en \'+unitName+\' · \'+context,(y,u)=>{const d=dom.get(y+\'|\'+u.id);if(!d?.total)return \'<td class="a32-na">—</td>\';const label=d.winners.map(name).join(\' / \'),c=d.winners.length===1?color(d.winners[0]):\'#7b8792\',share=100*d.max/d.total;exports.dominance.push({anio:y,arena:a,ronda:round,provincia:u.label,nivel:basis,partido:label,ganados:d.max,observados:d.total,porcentaje:share,empate:d.winners.length>1});return `<td style="background:${c};color:${ink(c)}"><strong>${esc(label)}</strong><small>${d.max} de ${d.total} ${unitName}</small><b>${share.toFixed(1)} %</b></td>`});\n   q(\'differences\').innerHTML=table(years,provinceRows,\'Diferencias · votos frente a \'+unitName+\' · \'+context,(y,u)=>{const d=dom.get(y+\'|\'+u.id),v=voteHist.get(y+\'|\'+u.id);if(!d?.total||!v)return \'<td class="a32-na">—</td>\';const tied=v.tie||d.winners.length!==1||d.winners[0]===-99,agree=!tied&&v.winner===d.winners[0],status=tied?\'Empate\':agree?\'Coincide\':\'Difiere\',pv=v.winners.map(name).join(\' / \'),pt=d.winners.map(name).join(\' / \');exports.differences.push({anio:y,arena:a,ronda:round,provincia:u.label,nivel:basis,estado:status,primero_votos:pv,primero_territorios:pt,ganados:d.max,observados:d.total});return `<td title="${esc(pv)} · ${esc(pt)}" class="a32-diff ${agree?\'a32-agree\':tied?\'a32-tie\':\'a32-different\'}"><strong>${status}</strong><span style="border-left:4px solid ${color(v.winner)}">Votos: ${esc(v.winners.map(partyShort).join(" / "))}</span><span style="border-left:4px solid ${d.winners.length===1?color(d.winners[0]):\'#7b8792\'}">${basis===\'c\'?\'Cantones\':\'Distritos\'}: ${esc(d.winners.map(partyShort).join(" / "))}</span><small>${d.max} de ${d.total} · ${(100*d.max/d.total).toFixed(1)} %</small></td>`});\n  }\n  sec.addEventListener(\'change\',draw);sec.addEventListener(\'click\',e=>{const b=e.target.closest(\'[data-l-export]\');if(!b)return;const ext=b.dataset.lExport,target=b.dataset.target,rows=exports[target];if(ext===\'csv\'){if(!rows.length)return;const cols=Object.keys(rows[0]);return save(\'Atlas_\'+target+\'.csv\',\'\\ufeff\'+cols.join(\',\')+\'\\n\'+rows.map(r=>cols.map(k=>\'"\'+String(r[k]??\'\').replace(/"/g,\'""\')+\'"\').join(\',\')).join(\'\\n\'),\'text/csv\')}tableImage(q(target).querySelector(\'table\'),q(\'note\').textContent+\' · \'+target,ext,\'Atlas_\'+target)});draw();\n }\n const old=views.partidos;if(typeof old===\'function\')views.partidos=function(...args){const out=old.apply(this,args);mount();return out};if(typeof state!==\'undefined\'&&state.tab===\'partidos\')mount();\n})();\n\n/* Every Plotly download carries an opaque background and an embedded contextual title. */\n(function(){if(!window.Plotly||Plotly.__atlas31Export)return;Plotly.__atlas31Export=true;const original=Plotly.downloadImage;Plotly.downloadImage=async function(gd,opts={}){if(typeof gd===\'string\')gd=document.getElementById(gd);const card=gd.closest?.(\'.card,section,article,.ar-panel\');let title=gd.layout?.title?.text;if(!title){const heading=card?.querySelector(\'h3,h4,h2\');title=heading?.textContent||\'Atlas Electoral de Costa Rica\';const controls=[...(card?.querySelectorAll(\'select\')||[])].map(s=>s.selectedOptions?.[0]?.textContent).filter(Boolean);if(controls.length)title+=\' · \'+controls.join(\' · \');else if(typeof state!==\'undefined\')title+=\' · \'+state.focus}await Plotly.relayout(gd,{paper_bgcolor:\'#ffffff\',plot_bgcolor:\'#ffffff\',\'font.color\':\'#17394d\',\'title.text\':title,\'title.font.size\':16,\'margin.t\':100,\'margin.r\':200});return original.call(this,gd,{...opts,height:Math.max(opts.height||0,gd.layout?.height||650)})}})();\n\n/* Figuras longitudinales y clasificación temporal dentro de las secciones existentes. */\n(function(){\n \'use strict\';\n if(window.ATLAS_V29_ANALYSIS)return;\n window.ATLAS_V29_ANALYSIS=true;\n const el=id=>document.getElementById(id);\n const data=el(\'atlas-v28-dataset\');if(!data)return;\n let db;try{db=(window.AtlasTerritorialData||(window.AtlasTerritorialData=JSON.parse(data.textContent)))}catch(e){return}\n const norm=s=>String(s||\'\').normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toUpperCase().replace(/[^A-Z0-9]/g,\'\');\n const safe=s=>String(s??\'\').replace(/[&<>"\']/g,x=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[x]));\n const pf=new Intl.NumberFormat(\'es-CR\',{maximumFractionDigits:1,minimumFractionDigits:1});\n const nf=new Intl.NumberFormat(\'es-CR\',{maximumFractionDigits:0});\n const fmt=n=>nf.format(n),pct=n=>pf.format(n)+\' %\';\n const palette={\'PUEBLOSOBERANO\':\'#14A3AF\',\'PROGRESOSOCIALDEMOCRATICO\':\'#303D91\',\'LIBERACIONNACIONAL\':\'#078c4f\',\'UNIDADSOCIALCRISTIANA\':\'#0875b7\',\'ACCIONCIUDADANA\':\'#e0b239\',\'FRENTEAMPLIO\':\'#d8b02c\'};\n function color(id){if(id===-99)return \'#7b8792\';let s=db.parties[id]||\'\';return palette[norm(s)]||db.colors?.[s]||\'#8296A5\'}\n const yrs=db.years;\n // indexar los registros de votos por elección y arena sin bloquear cada navegación.\n const cache=new Map();\n function leader(year,arena,level,round=1){return window.Atlas32Territory.leaders(year,arena,round,level).map(g=>({territory:g.id,party:g.winner,ties:g.tie}))}\n function attachComposition(){const content=el(\'content\');if(!content||el(\'atlas-v29-composicion\'))return;\n  const anchor=content.querySelector(\'#atlas-v31-leadership\');if(!anchor)return;\n  const actions=(target,label)=>`<div class="a33-downloads"><span>${label}</span><div class="a25-actions a32-actions">${[\'png\',\'svg\',\'csv\'].map(ext=>`<button type="button" data-v33-save="${target}:${ext}" aria-label="${label} ${ext.toUpperCase()}">${ext.toUpperCase()}</button>`).join(\'\')}</div></div>`;\n  const box=document.createElement(\'section\');box.id=\'atlas-v29-composicion\';box.className=\'a29-compare\';box.innerHTML=`<header><div><span class="a29-kicker">COMPETENCIA TERRITORIAL</span><h3>Composición de los primeros lugares electorales</h3><p>Resumen nacional y distribución dentro de las siete provincias.</p></div>${actions(\'both\',\'Descargar ambas gráficas\')}</header><div class="a29-compare-controls"><label>Desde<select data-v29-c="start"></select></label><label>Hasta<select data-v29-c="end"></select></label><label>Vista de resultados<select data-v29-c="arena"><option value="1">Diputaciones</option><option value="0">Presidencia</option></select></label><label>División territorial<select data-v29-c="level"><option value="c">Cantones</option><option value="p">Provincias</option><option value="d">Distritos</option></select></label></div><div data-v29-c="round-tools">${window.Atlas33RoundMarkup}</div><div class="a29-compare-stats" data-v29-c="stats"></div><section class="a33-composition-panel"><div class="a32-head"><h4>Resumen nacional</h4>${actions(\'national\',\'Descargar resumen nacional\')}</div><div class="a29-compare-fig" data-v29-c="graph"></div></section><section class="a33-composition-panel" data-v29-c="prov-panel"><div class="a32-head"><h4>Composición por provincia</h4>${actions(\'provinces\',\'Descargar composición provincial\')}</div><p>Primeros lugares acumulados en el período, como territorios × elección. Cada provincia suma 100 %.</p><div class="a29-compare-fig" data-v29-c="prov-graph"></div><details class="a33-data"><summary>Cantidades y porcentajes por partido y provincia</summary><div data-v29-c="prov-table"></div></details></section><footer class="a29-compare-footer"><span data-v29-c="note"></span></footer>`;anchor.after(box);\n  const q=x=>box.querySelector(\'[data-v29-c="\'+x+\'"]\');for(const key of [\'start\',\'end\'])q(key).innerHTML=yrs.map(y=>`<option value="${y}">${y}</option>`).join(\'\');q(\'start\').value=yrs[0];q(\'end\').value=yrs.at(-1);\n  let records=[],provinceRecords=[],ctx=\'\',ready=Promise.resolve();\n  function draw(){const a=+q(\'arena\').value,l=q(\'level\').value;let y0=+q(\'start\').value,y1=+q(\'end\').value;if(y0>y1){q(\'end\').value=y0;y1=y0}\n   const mode=window.Atlas33RoundSelection(q(\'round-tools\').firstElementChild,a,yrs.filter(y=>y>=y0&&y<=y1)),round=mode.round,years=mode.years.filter(y=>db.rows.some(r=>r[0]===y&&r[1]===a&&r[2]===round));for(const key of [\'start\',\'end\'])q(key).closest(\'label\').hidden=round===2;\n   const api=window.Atlas32Territory,units={p:\'provinciales\',c:\'cantonales\',d:\'distritales\'}[l];ctx=`${a===0?\'Presidencia\':\'Diputaciones\'} · ${round===2?\'Segunda ronda · \'+(years.join(\', \')||\'sin elecciones seleccionadas\'):y0+\'–\'+y1} · primeros lugares ${units}`;\n   const count=new Map(),distinct=new Map();let nt=0,ties=0;for(const y of years)for(const g of api.leaders(y,a,round,l)){count.set(g.winner,(count.get(g.winner)||0)+1);if(!distinct.has(g.winner))distinct.set(g.winner,new Set());distinct.get(g.winner).add(g.id);nt++;if(g.tie)ties++}\n   records=[...count].sort((a,b)=>b[1]-a[1]).map(([party,n])=>({partido:api.name(party),cantidad_primeros_lugares:n,porcentaje_observaciones:nt?100*n/nt:null,territorios_distintos:distinct.get(party).size,partyId:party,arena:a===0?\'Presidencia\':\'Diputaciones\',nivel:l,ronda:round,elecciones:years.join(\';\')}));\n   q(\'stats\').innerHTML=[[\'Elecciones incluidas\',years.length],[\'Territorios × elección\',fmt(nt)],[\'Partidos y empates\',records.length]].map(([label,n])=>`<div><span>${label}</span><strong>${n}</strong></div>`).join(\'\');q(\'note\').textContent=`${ctx}. Cada primer lugar territorial en una elección cuenta una vez; no son territorios únicos.${ties?\' Empates en categoría separada.\':\'\'}`;\n   const arr=records.slice().reverse(),h=Math.max(430,arr.length*36+180),general=Plotly.react(q(\'graph\'),[{type:\'bar\',orientation:\'h\',x:arr.map(r=>r.cantidad_primeros_lugares),y:arr.map(r=>r.partido),marker:{color:arr.map(r=>api.color(r.partyId))},text:arr.map(r=>fmt(r.cantidad_primeros_lugares)+\' · \'+pct(r.porcentaje_observaciones)),textposition:\'outside\',cliponaxis:false,customdata:arr.map(r=>r.porcentaje_observaciones),hovertemplate:\'%{y}<br>%{x} primeros lugares · %{customdata:.1f}%<extra></extra>\'}],{height:h,title:{text:ctx,font:{size:16}},margin:{l:280,r:110,t:95,b:75},font:{family:\'Inter,Arial,sans-serif\',size:12,color:\'#21485c\'},xaxis:{title:\'Primeros lugares territoriales × elección\',range:[0,Math.max(1,...arr.map(r=>r.cantidad_primeros_lugares))*1.23]},yaxis:{automargin:true},paper_bgcolor:\'#fff\',plot_bgcolor:\'#fff\'},{responsive:true,displaylogo:false});\n   q(\'prov-panel\').hidden=l===\'p\';box.querySelector(\'[data-v33-save="both:png"]\').closest(\'.a33-downloads\').hidden=l===\'p\';provinceRecords=[];let provincial=Promise.resolve();\n   if(l!==\'p\'){\n    const provs=[\'San José\',\'Alajuela\',\'Cartago\',\'Heredia\',\'Guanacaste\',\'Puntarenas\',\'Limón\'],totals=provs.map(()=>new Map()),den=provs.map(()=>0);for(const y of years)for(const row of api.composition(y,a,round,l)){const i=provs.indexOf(row.province);den[i]+=row.total;for(const [p,n]of row.counts)totals[i].set(p,(totals[i].get(p)||0)+n)}\n    const parties=[...new Set(totals.flatMap(m=>[...m.keys()]))].sort((p,q)=>totals.reduce((s,m)=>s+(m.get(q)||0)-(m.get(p)||0),0));\n    for(let i=0;i<7;i++)for(const p of parties)provinceRecords.push({provincia:provs[i],partido:api.name(p),primeros_lugares:totals[i].get(p)||0,observaciones:den[i],porcentaje:den[i]?100*(totals[i].get(p)||0)/den[i]:null,arena:a===0?\'Presidencia\':\'Diputaciones\',nivel:l,ronda:round,elecciones:years.join(\';\')});\n    const traces=parties.map(p=>({type:\'bar\',orientation:\'h\',name:api.name(p),y:provs,x:totals.map((m,i)=>den[i]?100*(m.get(p)||0)/den[i]:null),marker:{color:api.color(p)},customdata:totals.map((m,i)=>[m.get(p)||0,den[i]]),text:totals.map((m,i)=>den[i]&&100*(m.get(p)||0)/den[i]>=9?(100*(m.get(p)||0)/den[i]).toFixed(1)+\' %\':\'\'),textposition:\'inside\',hovertemplate:\'%{y}<br>%{fullData.name}<br>%{customdata[0]} de %{customdata[1]} · %{x:.1f}%<extra></extra>\'}));\n    provincial=Plotly.react(q(\'prov-graph\'),traces,{height:Math.max(620,520+Math.ceil(parties.length/5)*24),barmode:\'stack\',title:{text:\'Composición por provincia · \'+ctx,font:{size:15}},margin:{l:125,r:40,t:100,b:160},font:{family:\'Inter,Arial,sans-serif\',size:12,color:\'#21485c\'},xaxis:{range:[0,100],ticksuffix:\' %\'},yaxis:{autorange:\'reversed\',automargin:true},legend:{orientation:\'h\',y:-.2,font:{size:11}},paper_bgcolor:\'#fff\',plot_bgcolor:\'#fff\'},{responsive:true,displaylogo:false});\n    q(\'prov-table\').innerHTML=\'<table class="a32-summary-table"><thead><tr><th>Provincia</th><th>Partido</th><th>Primeros lugares</th><th>Total observado</th><th>%</th></tr></thead><tbody>\'+provinceRecords.filter(r=>r.primeros_lugares).map(r=>`<tr><th>${safe(r.provincia)}</th><td>${safe(r.partido)}</td><td>${fmt(r.primeros_lugares)}</td><td>${fmt(r.observaciones)}</td><td>${pct(r.porcentaje)}</td></tr>`).join(\'\')+\'</tbody></table>\';\n   }\n   ready=Promise.all([general,provincial]);\n  }\n  function csv(rows){const cols=[...new Set(rows.flatMap(Object.keys))];return \'\\ufeff\'+cols.join(\',\')+\'\\r\\n\'+rows.map(r=>cols.map(k=>\'"\'+String(r[k]??\'\').replace(/"/g,\'""\')+\'"\').join(\',\')).join(\'\\r\\n\')}\n  async function exportBoth(ext){const top=q(\'graph\'),bottom=q(\'prov-graph\');const g=top.layout,p=bottom.layout,h1=g.height||800,h2=p.height||650,h=h1+h2+140,gap=.09,cut=h2/(h1+h2),clone=x=>JSON.parse(JSON.stringify(x));\n   const traces=[...clone(top.data).map(t=>({...t,xaxis:\'x\',yaxis:\'y\',showlegend:false})),...clone(bottom.data).map(t=>({...t,xaxis:\'x2\',yaxis:\'y2\'}))],host=document.createElement(\'div\');host.style.cssText=\'position:fixed;left:-20000px;top:0;width:1800px;\';box.append(host);\n   try{await Plotly.newPlot(host,traces,{height:h,width:1800,title:{text:ctx,font:{size:20}},paper_bgcolor:\'#fff\',plot_bgcolor:\'#fff\',font:{family:\'Arial\',size:13,color:\'#21485c\'},margin:{l:300,r:130,t:140,b:180},barmode:\'stack\',xaxis:{...clone(g.xaxis),domain:[0,1],anchor:\'y\'},yaxis:{...clone(g.yaxis),domain:[cut+gap,1],anchor:\'x\'},xaxis2:{...clone(p.xaxis),domain:[0,1],anchor:\'y2\'},yaxis2:{...clone(p.yaxis),domain:[0,cut-gap/2],anchor:\'x2\'},legend:{orientation:\'h\',y:-.08,font:{size:12}},annotations:[{xref:\'paper\',yref:\'paper\',x:0,y:1.025,text:\'<b>Resumen nacional</b>\',showarrow:false,xanchor:\'left\'},{xref:\'paper\',yref:\'paper\',x:0,y:cut+.018,text:\'<b>Composición de las siete provincias</b>\',showarrow:false,xanchor:\'left\'}]},{displaylogo:false});await Plotly.downloadImage(host,{format:ext,filename:\'Atlas_Composicion_Nacional_y_Provincias\',width:1800,height:h,scale:ext===\'png\'?1.5:1})}finally{Plotly.purge(host);host.remove()}\n  }\n  box.addEventListener(\'change\',draw);box.addEventListener(\'click\',async e=>{const b=e.target.closest(\'[data-v33-save]\');if(!b)return;const [target,ext]=b.dataset.v33Save.split(\':\');b.disabled=true;\n   try{await ready;if(ext===\'csv\'){const rows=target===\'national\'?records.map(({partyId,...r})=>({vista:\'Resumen nacional\',...r})):target===\'provinces\'?provinceRecords.map(r=>({vista:\'Composición provincial\',...r})):[...records.map(({partyId,...r})=>({vista:\'Resumen nacional\',...r})),...provinceRecords.map(r=>({vista:\'Composición provincial\',...r}))];const u=URL.createObjectURL(new Blob([csv(rows)],{type:\'text/csv;charset=utf-8\'})),a=document.createElement(\'a\');a.href=u;a.download=\'Atlas_Composicion_\'+target+\'.csv\';a.click();setTimeout(()=>URL.revokeObjectURL(u),1000)}else if(target===\'both\')await exportBoth(ext);else{const gd=q(target===\'national\'?\'graph\':\'prov-graph\');await Plotly.downloadImage(gd,{format:ext,filename:\'Atlas_Composicion_\'+target,width:1800,height:gd.layout.height,scale:ext===\'png\'?1.5:1})}}catch(err){console.error(\'Descarga de composición\',err)}finally{b.disabled=false}});draw();\n }\n\n // La anotación se dibuja EN la figura; no depende de una tabla HTML externa.\n function annotateFirst(){const sec=el(\'atlas-v27-participacion\');if(!sec)return;\n  const mode=sec.querySelector(\'#a27-p-mode\');if(!mode||mode.value!==\'first\')return;\n  const chart=sec.querySelector(\'#a27-p-plot\');if(!chart?.data?.[0]?.customdata||!window.Plotly)return;\n  const values=chart.data[0].customdata;const labels=values.map(row=>String(row?.[0]||\'\'));\n  if(chart.__v29LabelKey===JSON.stringify(labels))return;\n  chart.__v29LabelKey=JSON.stringify(labels);\n  Plotly.restyle(chart,{mode:\'lines+markers+text\',text:[labels],textposition:\'top center\',textfont:{size:9,color:\'#235668\'}},[0]).catch(()=>{});\n }\n function participationLabels(){const sec=el(\'atlas-v27-participacion\');if(!sec)return;\n  const t=sec.querySelector(\'#a27-p-table-region table\');if(t?.tHead&&t?.tFoot){const h=t.tHead.rows[0],f=t.tFoot.rows[0];if(h&&f){for(let i=1;i<h.cells.length;i++)if(/^\\d{4}$/.test(h.cells[i].textContent.trim()))h.cells[i].textContent=\'\';if(f.cells[0])f.cells[0].textContent=\'Año\'}}\n  const scale=sec.querySelector(\'#a27-p-scale\'),mode=sec.querySelector(\'#a27-p-mode\');if(scale&&mode){for(const [value,invalid] of [[\'top10\',scale.value===\'p\'],[\'matrix\',false]]){const option=mode.querySelector(`option[value="${value}"]`);if(option){option.disabled=invalid;option.hidden=invalid}}}\n  annotateFirst();\n }\n function integrate(){const prev=typeof views===\'object\'?views.partidos:null;if(typeof prev===\'function\')views.partidos=function(...args){const out=prev.apply(this,args);try{attachComposition()}catch(e){console.error(\'Composición territorial\',e)}return out};\n  const orig=views?.participacion;if(typeof orig===\'function\')views.participacion=function(...args){const result=orig.apply(this,args);setTimeout(participationLabels,300);return result};\n  if(typeof state!==\'undefined\'&&state.tab===\'partidos\')attachComposition();\n  if(typeof state!==\'undefined\'&&state.tab===\'participacion\')participationLabels();\n  document.addEventListener(\'change\',e=>{if(e.target?.closest?.(\'#atlas-v27-participacion\'))setTimeout(participationLabels,150)},true);\n }\n if(document.readyState===\'loading\')document.addEventListener(\'DOMContentLoaded\',integrate);else integrate();\n})();\n\n/* Atlas Electoral · análisis territorial comparado y persistencia partidaria. */\n(function(){\n \'use strict\';\n if(window.__ATLAS_V30_BASTIONS__)return;\n window.__ATLAS_V30_BASTIONS__=true;\n const DATA_ID=\'atlas-v28-dataset\', ROOT_ID=\'atlas-v30-bastiones\';\n const YEARS=[1953,1958,1962,1966,1970,1974,1978,1982,1986,1990,1994,1998,2002,2006,2010,2014,2018,2022,2026];\n const PROVINCES=[\'San José\',\'Alajuela\',\'Cartago\',\'Heredia\',\'Guanacaste\',\'Puntarenas\',\'Limón\'];\n const PCOL=[\'#087e9e\',\'#3684c0\',\'#8b61a8\',\'#31977d\',\'#cf8646\',\'#2771a7\',\'#b66381\'];\n const fmtInt=new Intl.NumberFormat(\'es-CR\',{maximumFractionDigits:0});\n const fmtDec=new Intl.NumberFormat(\'es-CR\',{minimumFractionDigits:1,maximumFractionDigits:2});\n const fmt=n=>Number.isFinite(+n)?fmtInt.format(+n):\'—\';\n const pct=n=>Number.isFinite(+n)?fmtDec.format(+n)+\' %\':\'—\';\n const pp=n=>Number.isFinite(+n)?(n>0?\'+\':\'\')+fmtDec.format(+n)+\' pp\':\'—\';\n const safe=s=>String(s??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\n const norm=s=>String(s||\'\').normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toUpperCase().replace(/[^A-Z0-9]/g,\'\').replace(/^PARTIDO/,\'\');\n let db,rawMap,cache=new Map(),nationCache=new Map(),meta={},mounted=null;\n const st={year:2026,arena:1,level:\'c\',party:null,mode:\'advantage\',from:1953,to:2026,filter:\'positive\',allParties:false,focusTerritory:null,display:25};\n const countryCode=c=>String(c||\'\').padStart(5,\'0\');\n function prepare(){\n  if(db)return true;\n  const node=document.getElementById(DATA_ID);if(!node)return false;\n  try{db=(window.AtlasTerritorialData||(window.AtlasTerritorialData=JSON.parse(node.textContent)))}catch(e){console.error(\'Resultados territoriales ilegibles\',e);return false}\n  rawMap=new Map();\n  if(!Array.isArray(db.rows)||!Array.isArray(db.parties))return false;\n  for(const r of db.rows){if(r[2]!==1||r[4]<0||r[5]<0)continue;\n   const k=r[0]+\'|\'+r[1];if(!rawMap.has(k))rawMap.set(k,[]);rawMap.get(k).push(r);\n  }\n  meta={years:db.years||YEARS,partyNames:db.parties};if(!meta.years.includes(st.year))st.year=meta.years[meta.years.length-1];return true;\n }\n function color(id){const name=db.parties[id]||\'\';const ref={LIBERACIONNACIONAL:\'#079648\',UNIDADSOCIALCRISTIANA:\'#0875B6\',ACCIONCIUDADANA:\'#E0B239\',FRENTEAMPLIO:\'#D7AF2D\',PUEBLOSOBERANO:\'#10A6B0\',NUEVAREPUBLICA:\'#2D8DAD\',LIBERALPROGRESISTA:\'#DE7451\'};\n  const known=db.colors?.[name]||db.colors?.[norm(name)]||ref[norm(name)];if(/^#[0-9a-f]{6}$/i.test(known||\'\'))return known;\n  let hash=0;for(const c of norm(name))hash=(Math.imul(hash,31)+c.charCodeAt(0))|0;return `hsl(${Math.abs(hash)%360} 58% 43%)`;\n }\n function place(year,code){return db.places?.[year+\'|\'+code]||[PROVINCES[(+code[0])-1]||\'Provincia\', \'Cantón\', \'Distrito\']}\n function geoName(year,code,level){const p=place(year,code),units=level===\'p\'?[p[0]]:level===\'c\'?[p[1],p[0]]:[p[2],p[1],p[0]];return units.filter((x,i)=>i===0||norm(x)!==norm(units[i-1])).join(\' · \')}\n function indexGroups(year,arena,level){const key=[year,arena,level].join(\'|\');if(cache.has(key))return cache.get(key);\n  const source=rawMap.get(year+\'|\'+arena)||[],results=new Map();\n  for(const r of source){const code=countryCode(r[3]),gcode=level===\'p\'?code.slice(0,1):level===\'c\'?code.slice(0,3):code,id=+r[4],votes=+r[5];\n   if(!results.has(gcode))results.set(gcode,{id:gcode,code,province:PROVINCES[+code[0]-1]||\'\',canton:place(year,code)[1],label:geoName(year,code,level),valid:0,parties:new Map()});\n   const g=results.get(gcode);g.valid+=votes;g.parties.set(id,(g.parties.get(id)||0)+votes);\n  }\n  const items=[...results.values()].filter(g=>g.valid>0).sort((a,b)=>+a.id- +b.id);\n  for(const g of items){const ranked=[...g.parties].sort((a,b)=>b[1]-a[1]||a[0]-b[0]);g.winner=ranked[0]?.[0]??null;g.tie=ranked.length>1&&ranked[0][1]===ranked[1][1];}\n  cache.set(key,items);return items;\n }\n function nation(year,arena){const key=year+\'|\'+arena;if(nationCache.has(key))return nationCache.get(key);\n  const d=rawMap.get(key)||[],votes=new Map();let valid=0;\n  for(const r of d){const p=+r[4],v=+r[5];valid+=v;votes.set(p,(votes.get(p)||0)+v)}\n  const out={valid,votes};nationCache.set(key,out);return out;\n }\n function share(g,party){return g.parties.has(party)&&g.valid?100*g.parties.get(party)/g.valid:null}\n function nationalShare(y,a,party){const n=nation(y,a);return n.votes.has(party)&&n.valid?100*n.votes.get(party)/n.valid:null}\n function metric(g,y,a,p){const local=share(g,p),nat=nationalShare(y,a,p);return {year:y,arena:a,level:st.level,id:g.id,territory:g.label,province:g.province,local,national:nat,gap:local===null||nat===null?null:local-nat,votes:g.parties.has(p)?g.parties.get(p):null,first:g.winner===p,tie:g.tie}}\n function eligibleParties(){if(st.mode===\'history\')return [...new Set(historicalRows().ranked.map(p=>p.party))].sort((a,b)=>db.parties[a].localeCompare(db.parties[b],\'es\'));const ns=nation(st.year,st.arena),current=[...ns.votes.keys()];\n  const leaders=new Set(indexGroups(st.year,st.arena,st.level).map(g=>g.winner));\n  let relevant=current.filter(pid=>ns.valid&&100*ns.votes.get(pid)/ns.valid>=3||leaders.has(pid));\n  if(st.allParties)relevant=current;\n  relevant.sort((a,b)=>(ns.votes.get(b)||0)-(ns.votes.get(a)||0));return relevant;\n }\n function allTerritoryRecords(){return indexGroups(st.year,st.arena,st.level).map(g=>metric(g,st.year,st.arena,st.party))}\n function shownParties(){const list=eligibleParties();if(st.mode!==\'history\'&&st.party!==null&&!list.includes(st.party)&&nation(st.year,st.arena).votes.has(st.party))list.push(st.party);return list}\n function smallLabel(level){return {p:\'provincias\',c:\'cantones\',d:\'distritos\'}[level]}\n function optionYear(value){return (meta.years||YEARS).map(y=>`<option value="${y}" ${+value===+y?\'selected\':\'\'}>${y}</option>`).join(\'\')}\n function ui(){return `<header class="a30-hero"><div><span class="a30-overline">GEOGRAFÍA ELECTORAL · FORTALEZAS TERRITORIALES</span><h3>Bastiones y concentración del voto</h3><p>Compare la ventaja relativa de una organización, sus primeros lugares y la persistencia de su apoyo electoral. Una ventaja puntual no implica por sí sola un bastión histórico.</p></div><div class="a30-hero-mark">1953—2026</div></header>\n  <div class="a30-toolbar"><label>Elección<select data-a30="year">${optionYear(st.year)}</select></label><label>Arena<select data-a30="arena"><option value="1">Diputaciones</option><option value="0">Presidencia</option></select></label><label>Escala<select data-a30="level"><option value="p">Provincia</option><option value="c">Cantón</option><option value="d">Distrito</option></select></label><label>Organización política<select data-a30="party"></select></label><label class="a30-include"><input type="checkbox" data-a30="all"/>Mostrar todos los partidos</label></div>\n  <div class="a30-kpis" data-a30="kpis"></div>\n  <nav class="a30-tabs" aria-label="Análisis territorial"><button data-a30-mode="advantage" type="button">Ventaja territorial</button><button data-a30-mode="competition" type="button">Competencia electoral</button><button data-a30-mode="persistence" type="button">Persistencia histórica</button><button data-a30-mode="history" type="button">Resumen histórico</button></nav>\n  <div class="a30-panel" data-a30-panel="advantage"><div class="a30-panel-top"><div><h4>¿Dónde se concentra el apoyo?</h4><p>Porcentaje local menos porcentaje nacional del mismo partido y elección (puntos porcentuales).</p></div><label>Mostrar<select data-a30="filter"><option value="positive">Ventaja positiva</option><option value="first">Solo primeros lugares</option><option value="all">Todos los territorios</option></select></label></div><div class="a30-figure" data-a30="advantage"></div><div class="a30-scrolldata" data-a30="advtable"></div><div class="a30-export"><span>Ventaja positiva ≠ victoria electoral. Las dos condiciones se señalan por separado.</span><button data-a30-export="advantage:csv">CSV</button><button data-a30-export="advantage:png">PNG</button><button data-a30-export="advantage:svg">SVG</button></div></div>\n  <div class="a30-panel" data-a30-panel="competition" hidden><div class="a30-panel-top"><div><h4>Matriz de competencia territorial</h4><p>Cada fila representa una unidad territorial; cada columna, un partido. Los encabezados y la intensidad de las celdas identifican la organización.</p></div></div><div class="a30-scrolldata a30-competition-table" data-a30="competition"></div><div class="a30-export"><span>Cuotas sobre votos válidos de cada territorio. — indica sin candidatura o dato partidario registrado.</span><button data-a30-export="competition:csv">CSV</button><button data-a30-export="competition:png">PNG</button><button data-a30-export="competition:svg">SVG</button></div></div>\n  <div class="a30-panel" data-a30-panel="persistence" hidden><div class="a30-panel-top"><div><h4>¿Dónde persiste la fortaleza?</h4><p>Años electorales en columnas; territorios en filas. Se distingue presencia, ventaja sobre el país y primeros lugares.</p></div><div class="a30-period"><label>Desde<select data-a30="from">${optionYear(st.from)}</select></label><label>Hasta<select data-a30="to">${optionYear(st.to)}</select></label><label>Filas<select data-a30="count"><option value="12">12</option><option value="25">25</option><option value="50">50</option><option value="0">Todas</option></select></label></div></div><div class="a30-scrolldata a30-persistence-table" data-a30="persistence"></div><div class="a30-export"><span>Se cuentan únicamente elecciones con observación válida para esa organización y territorio.</span><button data-a30-export="persistence:csv">CSV</button><button data-a30-export="persistence:png">PNG</button><button data-a30-export="persistence:svg">SVG</button></div></div>\n  <div class="a30-panel" data-a30-panel="history" hidden><div class="a30-panel-top"><div><h4>Primeros lugares a lo largo de la historia</h4><p>Compare partidos y territorios por victorias repetidas; no se exige presencia en la elección de detalle.</p></div><div class="a30-period"><label>Desde<select data-a30="hfrom">${optionYear(st.from)}</select></label><label>Hasta<select data-a30="hto">${optionYear(st.to)}</select></label><label>Mínimo de victorias<select data-a30="hist-min"><option value="1">1</option><option value="2">2</option><option value="3" selected>3</option><option value="5">5</option></select></label></div></div><p data-a30="history-note"></p><div class="a30-scrolldata" data-a30="history"></div><div class="a30-export"><span>● Primer lugar exclusivo · ○ Otra organización · E Empate · — Sin resultado territorial. Hasta 25 pares por vista.</span><button data-a30-export="history:csv">CSV</button><button data-a30-export="history:png">PNG</button><button data-a30-export="history:svg">SVG</button></div></div>\n  <section class="a30-profile" data-a30="profile" hidden><header><div><span class="a30-overline">HISTORIAL ELECTORAL DEL TERRITORIO</span><h4 data-a30="profile-title"></h4></div><button type="button" data-a30="close">Cerrar</button></header><div class="a30-profile-plot" data-a30="profile-plot"></div><div class="a30-export"><span>Serie local y nacional, misma arena y partido.</span><button data-a30-export="profile:csv">CSV</button><button data-a30-export="profile:png">PNG</button><button data-a30-export="profile:svg">SVG</button></div></section>\n  <footer class="a30-method">Se distinguen tres conceptos: <strong>sobrerrepresentación territorial</strong> (diferencia en pp), <strong>primer lugar</strong> (máximo de votos locales) y <strong>persistencia</strong> (número de elecciones con ventaja registrada). No se infieren causas ni arraigo organizativo sin evidencia adicional.</footer>`}\n const uiGet=(k)=>mounted?.querySelector(\'[data-a30="\'+k+\'"]\');\n function buildOptions(){const select=uiGet(\'party\'),parties=shownParties();\n  if(st.mode===\'history\'&&st.party===null)st.party=-1;if(!(st.mode===\'history\'&&st.party===-1)&&!parties.includes(st.party))st.party=parties[0]??null;\n  select.innerHTML=(st.mode===\'history\'?\'<option value="-1" \'+(st.party===-1?\'selected\':\'\')+\'>Todos los partidos con primeros lugares</option>\':\'\')+parties.map(p=>`<option value="${p}" ${p===st.party?\'selected\':\'\'}>${safe(db.parties[p])}</option>`).join(\'\');\n  uiGet(\'year\').value=String(st.year);uiGet(\'arena\').value=String(st.arena);uiGet(\'level\').value=st.level;uiGet(\'all\').checked=st.allParties;uiGet(\'filter\').value=st.filter;uiGet(\'from\').value=st.from;uiGet(\'to\').value=st.to;uiGet(\'count\').value=String(st.display);uiGet(\'hfrom\').value=st.from;uiGet(\'hto\').value=st.to;uiGet(\'year\').closest(\'label\').hidden=st.mode===\'history\';uiGet(\'all\').closest(\'label\').hidden=st.mode===\'history\';\n }\n function kpis(rows){const local=rows.filter(r=>r.gap!==null),win=local.filter(r=>r.first),pos=local.filter(r=>r.gap>0),max=pos.length?Math.max(...pos.map(r=>r.gap)):null,base=st.party===null?null:nationalShare(st.year,st.arena,st.party);\n  const cards=[[\'Cuota nacional del partido\',base===null?\'—\':pct(base)],[\'Territorios con resultados\',fmt(local.length)],[\'Territorios por encima del país\',fmt(pos.length)],[\'Primer lugar territorial\',fmt(win.length)],[\'Mayor ventaja observada\',max===null?\'—\':pp(max)]];\n  uiGet(\'kpis\').innerHTML=cards.map(([label,value])=>`<div class="a30-kpi"><small>${label}</small><strong>${value}</strong></div>`).join(\'\');\n }\n function drawAdvantage(rows){let sorted=rows.filter(r=>r.gap!==null);if(st.filter===\'positive\')sorted=sorted.filter(r=>r.gap>0);if(st.filter===\'first\')sorted=sorted.filter(r=>r.first);\n  sorted.sort((a,b)=>b.gap-a.gap||a.territory.localeCompare(b.territory,\'es\'));\n  const summary=sorted.slice(0,32);const host=uiGet(\'advantage\');\n  if(window.Plotly){const arr=summary.slice().reverse();Plotly.react(host,[{type:\'bar\',orientation:\'h\',x:arr.map(r=>r.gap),y:arr.map(r=>r.territory),marker:{color:arr.map(r=>PCOL[PROVINCES.indexOf(r.province)]||color(st.party)),line:{color:color(st.party),width:1.1}},customdata:arr.map(r=>[r.local,r.national,r.first?\'Primer lugar\':\'No fue primero\',r.votes]),text:arr.map(r=>pp(r.gap)),textposition:\'outside\',hovertemplate:\'%{y}<br>Cuota local: %{customdata[0]:.2f}%<br>Cuota nacional: %{customdata[1]:.2f}%<br>Ventaja: %{x:+.2f} pp<br>%{customdata[2]} · %{customdata[3]:,} votos<extra></extra>\'}],{height:Math.max(330,arr.length*25+90),margin:{l:Math.min(430,Math.max(210,...arr.map(r=>r.territory.length*5.5))),r:95,t:10,b:45},xaxis:{title:\'Diferencia respecto a la cuota nacional · pp\',zeroline:true,zerolinecolor:\'#acc4cd\'},paper_bgcolor:\'#ffffff\',plot_bgcolor:\'#ffffff\',font:{color:\'#244a5d\',family:\'Inter, Arial, sans-serif\',size:11}},{responsive:true,displaylogo:false})}\n  else host.textContent=summary.map(r=>`${r.territory}: ${pp(r.gap)}`).join(\' · \');\n  let table=\'<table class="a30-data-table"><thead><tr><th>Territorio</th><th>% local</th><th>% nacional</th><th>Ventaja</th><th>Primer lugar</th><th>Historial</th></tr></thead><tbody>\';\n  for(const r of sorted){table+=`<tr><td><i style="background:${PCOL[PROVINCES.indexOf(r.province)]||\'#5f8aa0\'}"></i>${safe(r.territory)}</td><td>${pct(r.local)}</td><td>${pct(r.national)}</td><td class="${r.gap>=0?\'a30-pos\':\'a30-neg\'}">${pp(r.gap)}</td><td>${r.first?(r.tie?\'Empate\':\'Sí\'):\'—\'}</td><td><button type="button" data-a30-territory="${safe(r.id)}">Explorar</button></td></tr>`}\n  table+=\'</tbody></table>\';uiGet(\'advtable\').innerHTML=table;\n  return sorted.map(r=>({anio:r.year,arena:r.arena===0?\'Presidencia\':\'Diputaciones\',nivel:st.level,territorio:r.territory,codigo:r.id,provincia:r.province,partido:db.parties[st.party],votos:r.votes,porcentaje_local:r.local,porcentaje_nacional:r.national,ventaja_pp:r.gap,primer_lugar:r.first,empate:r.tie}));\n }\n function competition(rows){const host=uiGet(\'competition\'),parties=shownParties(),units=indexGroups(st.year,st.arena,st.level);let body=\'\',csv=[];\n  const domain=parties.filter(pid=>nation(st.year,st.arena).votes.has(pid));\n  for(const g of units){const row=[g.label];body+=`<tr><th title="${safe(g.label)}">${safe(g.label)}</th>`;\n   for(const p of domain){const v=share(g,p),cell=v===null?\'—\':pct(v),bg=color(p),rgb=[1,3,5].map(i=>parseInt(bg.slice(i,i+2),16)),background=g.winner===p?\'rgb(\'+rgb.map(v=>Math.round(v*.54)).join(\',\')+\')\':\'rgb(\'+rgb.map(v=>Math.round(v*.12+255*.88)).join(\',\')+\')\';\n    body+=`<td title="${safe(db.parties[p])} · ${safe(g.label)} · ${v===null?\'Sin candidatura registrada\':pct(v)}" class="a30-colored" style="background-color:${background};color:${g.winner===p?\'#ffffff\':\'#17394d\'};font-weight:${g.winner===p?800:400};--a30-party:${bg};--a30-strength:${v===null?0:.12+.78*Math.min(1,v/65)}">${cell}${g.winner===p?\' <b>★</b>\':\'\'}</td>`;\n    row.push(v===null?\'\':v);\n   }body+=\'</tr>\';csv.push({anio:st.year,territorio:g.label,provincia:g.province,escala:st.level,...Object.fromEntries(domain.map((p,i)=>[db.parties[p],row[i+1]]))});\n  }\n  host.innerHTML=\'<table class="a30-heat-table"><thead><tr><th>Territorio</th>\'+domain.map(p=>`<th style="border-top:5px solid ${color(p)}">${safe(db.parties[p])}</th>`).join(\'\')+\'</tr></thead><tbody>\'+body+\'</tbody></table>\';\n  return csv;\n }\n function histories(){const eligible=(meta.years||YEARS).filter(y=>y>=st.from&&y<=st.to),count=new Map(),history=new Map();\n  for(const y of eligible){const nat=nationalShare(y,st.arena,st.party);if(nat===null)continue;\n   for(const g of indexGroups(y,st.arena,st.level)){const rec=metric(g,y,st.arena,st.party);if(rec.gap===null)continue;\n    if(!history.has(g.id))history.set(g.id,new Map());history.get(g.id).set(y,rec);\n    let s=count.get(g.id);if(!s){s={id:g.id,label:g.label,province:g.province,observed:0,above:0,leading:0,gaps:[]};count.set(g.id,s)}\n    s.observed++;if(rec.gap>0)s.above++;if(rec.first)s.leading++;s.gaps.push(rec.gap);\n   }\n  }\n  const ranked=[...count.values()].map(r=>({...r,mean:r.gaps.reduce((a,b)=>a+b,0)/r.gaps.length})).sort((a,b)=>b.above-a.above||b.leading-a.leading||b.mean-a.mean||a.label.localeCompare(b.label,\'es\'));\n  return {eligible,ranked,history};\n }\n function persistence(){const {eligible,ranked,history}=histories(),selected=st.display?ranked.slice(0,st.display):ranked;\n  let html=\'<table class="a30-persistence"><thead><tr><th>Territorio</th><th>Ventaja / observación</th>\'+eligible.map(y=>`<th>${y}</th>`).join(\'\')+\'</tr></thead><tbody>\',records=[];\n  for(const r of selected){html+=`<tr><th>${safe(r.label)}<small>${safe(r.province)}</small></th><td class="a30-ratio">${r.above}/${r.observed}<small>Primeros lugares: ${r.leading}</small></td>`;\n   for(const y of eligible){const d=history.get(r.id)?.get(y);if(!d){html+=\'<td class="a30-na" title="Sin resultado partidario documentado">—</td>\';continue}\n    const s=Math.min(1,Math.abs(d.gap)/25),col=d.gap>0?color(st.party):\'#c0cbd2\';\n    html+=`<td class="a30-persistent-cell" style="--a30-color:${col};--a30-persistent:${d.gap>0?.16+.70*s:.14+.30*s}" title="${safe(r.label)} · ${y} · ${pp(d.gap)} · ${d.first?\'Primer lugar\':\'Sin primer lugar\'}">${d.first?\'★ \':\'\'}${d.gap>0?\'+\':\'\'}${fmtDec.format(d.gap)}</td>`;\n    records.push({anio:y,arena:st.arena===0?\'Presidencia\':\'Diputaciones\',nivel:st.level,territorio:r.label,provincia:r.province,codigo:r.id,partido:db.parties[st.party],votos:d.votes,cuota_local:d.local,cuota_nacional:d.national,ventaja_pp:d.gap,primer_lugar:d.first,elecciones_con_ventaja:r.above,elecciones_observadas:r.observed});\n   }\n   html+=\'</tr>\';\n  }\n  html+=\'</tbody><tfoot><tr><th>Territorio</th><th>Elecciones</th>\'+eligible.map(y=>`<th>${y}</th>`).join(\'\')+\'</tr></tfoot></table>\';\n  uiGet(\'persistence\').innerHTML=html;return records;\n }\n  function historicalRows(){\n  const years=meta.years.filter(y=>y>=st.from&&y<=st.to),seen=new Map(),pairs=new Map();\n  for(const y of years)for(const g of indexGroups(y,st.arena,st.level)){\n   if(!seen.has(g.id))seen.set(g.id,new Map());seen.get(g.id).set(y,g);\n   if(g.tie||g.winner===null)continue;\n   const key=g.id+\'|\'+g.winner;\n   if(!pairs.has(key))pairs.set(key,{id:g.id,party:g.winner,label:g.label,province:g.province,years:[]});\n   const p=pairs.get(key);p.label=g.label;p.years.push(y);\n  }\n  const ranked=[...pairs.values()].map(p=>({...p,observed:seen.get(p.id).size,wins:p.years.length,ratio:p.years.length/seen.get(p.id).size})).sort((a,b)=>b.wins-a.wins||b.ratio-a.ratio||a.id.localeCompare(b.id)||a.party-b.party);\n  return {years,seen,ranked};\n }\n function historySummary(){const {years,seen,ranked}=historicalRows(),min=+(uiGet(\'hist-min\')?.value||3),rows=ranked.filter(p=>p.wins>=min&&(st.party===-1||p.party===st.party)),chosen=rows.slice(0,25),csv=[];\n  uiGet(\'kpis\').innerHTML=[[\'Partidos con algún primer lugar\',new Set(ranked.map(p=>p.party)).size],[\'Pares partido–territorio del filtro\',rows.length],[\'Elecciones del período\',years.length]].map(([label,n])=>`<div class="a30-kpi"><small>${safe(label)}</small><strong>${fmt(n)}</strong></div>`).join(\'\');\n  let html=\'<table class="a30-history"><thead><tr><th>Territorio · partido</th><th>Victorias / elecciones observadas</th>\'+years.map(()=>\'<th aria-hidden="true"></th>\').join(\'\')+\'</tr></thead><tbody>\';\n  for(const p of chosen){html+=`<tr><th>${safe(p.label)}<small style="border-left:4px solid ${color(p.party)};padding-left:7px">${safe(db.parties[p.party])}</small></th><td><strong>${p.wins}/${p.observed}</strong><small>${pct(100*p.ratio)}</small></td>`;\n   for(const y of years){const g=seen.get(p.id).get(y),win=g&&!g.tie&&g.winner===p.party;\n    html+=`<td style="background:${win?color(p.party):\'#f4f8fa\'};color:${win?\'#fff\':\'#526c7b\'};text-align:center" title="${safe(p.label)} · ${y} · ${g?g.tie?\'Empate\':safe(db.parties[g.winner]):\'Sin resultado territorial observado\'}">${!g?\'—\':g.tie?\'E\':win?\'●\':\'○\'}</td>`;\n    csv.push({anio:y,arena:st.arena===0?\'Presidencia\':\'Diputaciones\',nivel:st.level,codigo:p.id,territorio:p.label,provincia:p.province,partido:db.parties[p.party],primer_lugar_exclusivo:g?win:null,empate:g?.tie??null,partido_primero:g&&!g.tie?db.parties[g.winner]:null,victorias_periodo:p.wins,elecciones_territorio:p.observed,porcentaje_victorias:100*p.ratio});\n   }html+=\'</tr>\';\n  }\n  if(!chosen.length)html+=`<tr><td colspan="${years.length+2}">No hay pares con ese mínimo de victorias. Reduzca el mínimo o amplíe el período.</td></tr>`;\n  html+=\'</tbody><tfoot><tr><th>Año</th><th>n/N</th>\'+years.map(y=>\'<th>\'+y+\'</th>\').join(\'\')+\'</tr></tfoot></table>\';\n  uiGet(\'history\').innerHTML=html;\n  uiGet(\'history-note\').textContent=`${st.from}–${st.to} · ${st.arena===0?\'Presidencia · primera ronda\':\'Diputaciones\'} · ${smallLabel(st.level)} · ${chosen.length} de ${rows.length} pares. Orden: victorias acumuladas, proporción y código. N incluye todas las elecciones observadas del territorio, aunque el partido no compitiera.`;\n  return csv;\n }\n\n let exports={advantage:[],competition:[],persistence:[],profile:[],history:[]};\n function render(){if(!mounted?.isConnected)return;\n  buildOptions();if(st.party===null){uiGet(\'kpis\').textContent=\'Sin resultados partidarios para esta elección.\';return}\n  const rows=allTerritoryRecords();kpis(rows);\n  mounted.querySelectorAll(\'[data-a30-mode]\').forEach(b=>b.classList.toggle(\'selected\',b.dataset.a30Mode===st.mode));\n  mounted.querySelectorAll(\'[data-a30-panel]\').forEach(p=>p.hidden=p.dataset.a30Panel!==st.mode);\n  if(st.mode===\'advantage\')exports.advantage=drawAdvantage(rows);\n  if(st.mode===\'competition\')exports.competition=competition(rows);\n  if(st.mode===\'persistence\')exports.persistence=persistence();if(st.mode===\'history\')exports.history=historySummary();\n }\n function focus(territory){const year=st.year,lev=st.level,id=String(territory),name=indexGroups(year,st.arena,lev).find(g=>g.id===id)?.label||\'Territorio\';st.focusTerritory=id;\n  const rows=[];for(const y of meta.years){const g=indexGroups(y,st.arena,lev).find(g=>g.id===id);if(!g)continue;const r=metric(g,y,st.arena,st.party);if(r.local!==null)rows.push(r)}\n  exports.profile=rows.map(r=>({anio:r.year,territorio:r.territory,partido:db.parties[st.party],cuota_local:r.local,cuota_nacional:r.national,ventaja_pp:r.gap,primer_lugar:r.first,votos:r.votes}));\n  const panel=uiGet(\'profile\');panel.hidden=false;uiGet(\'profile-title\').textContent=`${name} · ${db.parties[st.party]}`;\n  const chart=uiGet(\'profile-plot\');if(window.Plotly)Plotly.react(chart,[{type:\'scatter\',mode:\'lines+markers\',name:\'Territorio\',x:rows.map(r=>r.year),y:rows.map(r=>r.local),line:{color:color(st.party),width:3},marker:{size:8},hovertemplate:\'%{x}: %{y:.2f}%<extra></extra>\'},{type:\'scatter\',mode:\'lines+markers\',name:\'Costa Rica\',x:rows.map(r=>r.year),y:rows.map(r=>r.national),line:{color:\'#758e9d\',width:2,dash:\'dash\'},hovertemplate:\'%{x}: %{y:.2f}%<extra></extra>\'}],{height:340,margin:{l:54,r:25,t:15,b:60},yaxis:{title:\'% de votos válidos\'},xaxis:{title:\'Elección\',tickmode:\'array\',tickvals:meta.years},paper_bgcolor:\'#ffffff\',plot_bgcolor:\'#ffffff\'},{displaylogo:false,responsive:true});\n  panel.scrollIntoView({behavior:\'smooth\',block:\'nearest\'});\n }\n function encodeCSV(records){if(!records.length)return \'\\ufeff\';const keys=Object.keys(records[0]);return \'\\ufeff\'+keys.join(\',\')+\'\\r\\n\'+records.map(r=>keys.map(k=>\'"\'+String(r[k]??\'\').replace(/"/g,\'""\')+\'"\').join(\',\')).join(\'\\r\\n\')}\n function download(name,content,type){const blob=new Blob([content],{type}),url=URL.createObjectURL(blob),a=document.createElement(\'a\');a.href=url;a.download=name;document.body.appendChild(a);a.click();setTimeout(()=>{URL.revokeObjectURL(url);a.remove()},1100)}\n function tableSVG(tab,title){const table=tab.querySelector(\'table\');if(!table)return \'<svg xmlns="http://www.w3.org/2000/svg"/>\';\n  const head=[...table.tHead.rows[0].cells].map(c=>c.textContent.trim()),rows=[...table.tBodies[0].rows].map(r=>[...r.cells].map(c=>c.textContent.trim().replace(/\\s+/g,\' \')));\n  const cols=head.length, widths=head.map((_,i)=>i===0?255:i===1&&cols>8?160:cols>15?86:170),w=widths.reduce((a,b)=>a+b,0)+20,rh=38,h=75+rh*(rows.length+1);\n  let xml=`<svg xmlns="http://www.w3.org/2000/svg" width="${w}" height="${h}" viewBox="0 0 ${w} ${h}"><rect width="100%" height="100%" fill="#fff"/><text x="15" y="30" fill="#1b4459" font-family="Arial" font-size="17" font-weight="700">${safe(title)}</text>`;\n  const x=[];let sum=10;for(const wid of widths){x.push(sum);sum+=wid}\n  head.forEach((t,i)=>{xml+=`<rect x="${x[i]}" y="48" width="${widths[i]-2}" height="${rh}" fill="#17475b"/><text x="${x[i]+4}" y="${68}" fill="#fff" font-size="10" font-family="Arial">${safe(t.slice(0,28))}</text>`});\n  rows.forEach((row,ri)=>row.forEach((cell,i)=>{const y=48+(ri+1)*rh,t=cell.textContent.trim().replace(/\\s+/g,\' \'),bg=getComputedStyle(cell).backgroundColor,fill=bg===\'rgba(0, 0, 0, 0)\'?(ri%2?\'#f0f6f8\':\'#fff\'):bg;xml+=`<rect x="${x[i]}" y="${y}" width="${widths[i]-2}" height="${rh-1}" fill="${fill}" stroke="#deebef" stroke-width="1"/><text x="${x[i]+4}" y="${y+21}" fill="#20475b" font-family="Arial" font-size="10">${safe(t.slice(0,i===0?31:17))}</text>`}));return xml+\'</svg>\';\n }\n function exportFigure(mode,format){const name=`Atlas_Bastiones_${st.arena===0?\'Presidencial\':\'Legislativo\'}_${st.level}_${st.year}_${mode}`;\n  if(format===\'csv\')return download(name+\'.csv\',encodeCSV(exports[mode]),\'text/csv;charset=utf-8\');if(mode===\'history\')return window.Atlas31TableImage(uiGet(\'history\').querySelector(\'table\'),\'Primeros lugares históricos · \'+uiGet(\'history-note\').textContent,format,name);\n  if(mode===\'advantage\'||mode===\'profile\'){const graph=uiGet(mode===\'advantage\'?\'advantage\':\'profile-plot\');if(window.Plotly?.downloadImage)return Plotly.downloadImage(graph,{format,filename:name,width:1700,height:mode===\'profile\'?700:Math.max(850,graph.scrollHeight),scale:format===\'png\'?2:1})}\n  const container=uiGet(mode),xml=tableSVG(container,mode===\'persistence\'?\'Persistencia territorial\':\'Matriz de competencia\');\n  if(format===\'svg\')return download(name+\'.svg\',xml,\'image/svg+xml\');\n  const url=URL.createObjectURL(new Blob([xml],{type:\'image/svg+xml\'})),img=new Image();img.onload=()=>{const can=document.createElement(\'canvas\');can.width=img.naturalWidth;can.height=img.naturalHeight;can.getContext(\'2d\').drawImage(img,0,0);can.toBlob(b=>{if(b)download(name+\'.png\',b,\'image/png\');URL.revokeObjectURL(url)},\'image/png\')};img.onerror=()=>URL.revokeObjectURL(url);img.src=url;\n }\n function listeners(){mounted.addEventListener(\'change\',event=>{const k=event.target.dataset.a30;if(!k)return;\n  if(k===\'year\'){st.year=+event.target.value;st.party=null}\n  if(k===\'arena\'){st.arena=+event.target.value;st.party=null}\n  if(k===\'level\'){st.level=event.target.value;st.party=null}\n  if(k===\'party\')st.party=+event.target.value;\n  if(k===\'all\')st.allParties=event.target.checked;\n  if(k===\'filter\')st.filter=event.target.value;\n  if(k===\'from\'||k===\'hfrom\')st.from=+event.target.value;\n  if(k===\'to\'||k===\'hto\')st.to=+event.target.value;\n  if(k===\'count\')st.display=+event.target.value;\n  if(st.from>st.to){if(k===\'from\'||k===\'hfrom\')st.to=st.from;else st.from=st.to}\n  render();\n });\n  mounted.addEventListener(\'click\',event=>{const tab=event.target.closest(\'[data-a30-mode]\');if(tab){st.mode=tab.dataset.a30Mode;if(st.mode===\'history\')st.party=-1;render();return}\n   const territory=event.target.closest(\'[data-a30-territory]\');if(territory){focus(territory.dataset.a30Territory);return}\n   const close=event.target.closest(\'[data-a30="close"]\');if(close){uiGet(\'profile\').hidden=true;return}\n   const exportButton=event.target.closest(\'[data-a30-export]\');if(exportButton){const [which,fmt]=exportButton.dataset.a30Export.split(\':\');exportFigure(which,fmt)}\n  });\n }\n function findHistoric(){const content=document.getElementById(\'content\');if(!content)return [];\n  const nodes=[...content.querySelectorAll(\'h2,h3,h4\')].filter(h=>/bastiones electorales|bastiones y concentracion/i.test(String(h.textContent).normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toLowerCase()));\n  return nodes.filter(h=>!h.closest(\'#\'+ROOT_ID)).map(h=>h.closest(\'section.a25-block,section.a25-card,.a25-block,.card,.panel,article\')||h.parentElement).filter(Boolean);\n }\n function reorder(){const content=document.getElementById(\'content\'),comp=content?.querySelector(\'#atlas-v29-composicion\');if(!content||!mounted)return;\n  for(const h of [...content.querySelectorAll(\'h2,h3,h4\')]){if(h.closest(\'#\'+ROOT_ID))continue;\n   if(/bastiones electorales|bastiones electorales y ventajas/i.test(h.textContent)){let node=h.parentElement;while(node&&node!==content&&!node.querySelector(\'select,table\'))node=node.parentElement;if(node===content)node=h.closest(\'section,article,.card,.panel\');if(node&&!node.contains(mounted)&&!node.contains(comp))node.remove();}\n  }\n  const leader=content.querySelector(\'#atlas-v31-leadership\');\n  if(comp&&leader&&leader.nextElementSibling!==comp)leader.after(comp);\n  if(comp){if(comp.nextElementSibling!==mounted)comp.after(mounted)}else if(!mounted.isConnected)content.append(mounted);\n  for(const header of [...content.querySelectorAll(\'h2,h3,h4\')]){if(/composición histórica.*liderazgos|liderazgos provinciales.*histórica/i.test(header.textContent)&&!header.closest(\'#\'+ROOT_ID)){const card=header.closest(\'.card,article\');if(card&&!card.contains(comp)&&!card.contains(mounted)&&mounted.nextElementSibling!==card)mounted.after(card)}}\n }\n function mount(){if(!prepare())return;const content=document.getElementById(\'content\');if(!content)return;\n  if(content.querySelector(\'#\'+ROOT_ID)){mounted=content.querySelector(\'#\'+ROOT_ID);reorder();return}\n  mounted=document.createElement(\'section\');mounted.id=ROOT_ID;mounted.innerHTML=ui();listeners();reorder();render();\n }\n function attach(){const original=(typeof views===\'object\')?views.partidos:null;if(typeof original===\'function\'){\n   views.partidos=function(...args){const out=original.apply(this,args);try{mount()}catch(e){console.error(\'Atlas territorial\',e)}return out};\n  }\n  if(typeof state!==\'undefined\'&&state.tab===\'partidos\')mount();\n  // Mantener el orden estable si otros selectores vuelven a dibujar las matrices.\n  document.addEventListener(\'change\',event=>{if(!event.target?.closest?.(\'#content\')||event.target.closest(\'#\'+ROOT_ID))return;\n   if(typeof state!==\'undefined\'&&state.tab===\'partidos\')setTimeout(()=>{if(mounted?.isConnected)reorder()},90);\n  },true);\n }\n if(document.readyState===\'loading\')document.addEventListener(\'DOMContentLoaded\',attach);else attach();\n})();\n'

_ATLAS_V31_CSS = '/* Presentación institucional del explorador de resultados. Se añade directamente al HTML generado. */\n.atlas-results{font-family:Inter,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;--a29-ink:#173e53;--a29-teal:#07848d;--a29-line:#d9e7ee;--a29-muted:#557589;max-width:none!important}\n.atlas-results .ar-shell{background:#fff;border:1px solid #d7e6ee;border-radius:20px;overflow:hidden;box-shadow:0 18px 45px rgba(12,64,82,.09)}\n.atlas-results .ar-top{background:radial-gradient(ellipse at 90% 0%,rgba(58,209,206,.22),transparent 50%),linear-gradient(125deg,#0d354b 0%,#0c6574 64%,#187b83 100%);padding:31px 33px}\n.atlas-results .ar-top h2{color:#fff;font-size:27px;letter-spacing:-.6px}\n.atlas-results .ar-top p{color:#ddf7f7;opacity:1;font-size:12px}\n.atlas-results .ar-top .ar-pill{color:#fff;border-color:#ffffff44}\n.atlas-results .ar-tabs[hidden],.atlas-results .ar-controls[hidden],.atlas-results .ar-kpis[hidden]{display:none!important}\n.atlas-results #atlas-v29-main{padding:0 22px 30px;background:linear-gradient(180deg,#f7fbfd,#fff 180px)}\n.atlas-results .a29-header{display:flex;justify-content:space-between;align-items:flex-start;gap:18px;padding:22px 6px 18px;flex-wrap:wrap}\n.atlas-results .a29-header h3{margin:7px 0 4px;font-size:22px;font-weight:820;letter-spacing:-.55px;color:#143f54}\n.atlas-results .a29-header p{font-size:12px;color:#55758a;margin:0}\n.atlas-results .a29-kicker{font-weight:850;letter-spacing:1.8px;color:#158792;font-size:10px}\n.atlas-results .a29-tag{border:1px solid #c4e1e4;background:#e7f7f5;border-radius:30px;color:#237c80;font-size:11px;font-weight:750;padding:9px 15px}\n.atlas-results .a29-toolbar{display:flex;gap:12px 11px;flex-wrap:wrap;padding:19px;border:1px solid #d6e6ec;background:#eef6f9;border-radius:14px}\n.atlas-results .a29-input{display:flex;flex-direction:column;flex:1 1 150px;gap:7px;max-width:330px;min-width:120px;color:#3b6173;font-size:11px;font-weight:750}\n.atlas-results .a29-input[hidden]{display:none}\n.atlas-results .a29-input select{border:1px solid #bed3df;border-radius:9px;padding:10px 11px;min-height:41px;font-size:12px;color:#224c62;background:white;font-weight:650;width:100%}\n.atlas-results .a29-input select:focus{outline:3px solid #188f9c33;border-color:#1b939d}\n.atlas-results .a29-breadcrumb{display:flex;gap:9px;align-items:center;flex-wrap:wrap;margin:16px 5px;color:#86a2ac;font-size:12px}\n.atlas-results .a29-breadcrumb button{padding:7px 12px;border-radius:20px;border:1px solid #c9e0e5;background:#fff;color:#20566d;font-weight:700}\n.atlas-results .a29-breadcrumb button:hover{background:#e7f6f6;color:#087985}\n.atlas-results .a29-kpis{display:grid;grid-template-columns:repeat(6,minmax(0,1fr));gap:11px;margin-bottom:22px}\n.atlas-results .a29-kpi{padding:16px 14px;background:white;border:1px solid #dae8ef;border-radius:12px;border-top:3px solid #2497a0;box-shadow:0 4px 15px #315e7110;min-width:0}\n.atlas-results .a29-kpi small{display:block;font-size:11px;color:#597b8b;margin-bottom:5px}\n.atlas-results .a29-kpi strong{display:block;font-size:21px;font-weight:820;letter-spacing:-.5px;color:#19495f;overflow-wrap:anywhere}\n.atlas-results .a29-visuals{display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1fr);gap:15px;align-items:start}\n.atlas-results .a29-map-panel,.atlas-results .a29-bar-panel,.atlas-results .a29-matrix-panel{border:1px solid #d7e6ec;background:#fff;border-radius:14px;min-width:0;overflow:hidden;box-shadow:0 5px 15px rgba(20,65,83,.035)}\n.atlas-results .a29-map-panel{height:auto;min-height:485px}\n.atlas-results .a29-map-panel .ar-panel-head{padding:19px 17px 8px}\n.atlas-results .a29-bar-panel{padding:0 0 10px;min-height:485px}\n.atlas-results .a29-bar-panel[hidden]{display:none}\n.atlas-results .a29-map-panel:has(+.a29-bar-panel[hidden]){grid-column:1/-1}\n.atlas-results .a29-bar-panel header,.atlas-results .a29-matrix-panel header{display:flex;justify-content:space-between;align-items:center;gap:14px;flex-wrap:wrap;padding:21px 20px 14px;border-bottom:1px solid #ecf1f4}\n.atlas-results .a29-bar-panel header h4,.atlas-results .a29-matrix-panel header h4{margin:0;font-size:16px;font-weight:800;color:#1a4a60}\n.atlas-results .a29-bar-panel .a29-bars{min-height:420px}\n.atlas-results .a29-export{display:flex;gap:7px}\n.atlas-results .a29-export button{border:1px solid #bdd8e0;background:#fafdfe;color:#276279;padding:9px 12px;border-radius:8px;font-size:11px;font-weight:800;cursor:pointer}\n.atlas-results .a29-export button:hover{background:#e7f7f7}\n.atlas-results .a29-matrix-panel{margin-top:16px}\n.atlas-results .a29-matrix-panel header p{font-size:11px;line-height:1.5;color:#577586;margin:7px 0 0}\n.atlas-results .a29-matrix-tools{display:flex;align-items:center;justify-content:space-between;gap:12px;padding:13px 18px;background:#f7fbfc;border-bottom:1px solid #e1edf1;font-size:12px;color:#577387}\n.atlas-results .a29-matrix-tools label{display:flex;align-items:center;gap:12px;color:#3c6071;font-weight:650}\n.atlas-results .a29-matrix-tools input{min-height:36px;padding:8px 12px;min-width:225px;border-radius:7px;border:1px solid #c5d9e2}\n.atlas-results .a29-table-box{max-height:670px;overflow:auto;scrollbar-width:thin;scrollbar-color:#a5c3cf #eff6f8}\n.atlas-results .a29-matrix{border-collapse:separate;border-spacing:0;width:max-content;min-width:100%;font-size:11px;line-height:1.45}\n.atlas-results .a29-matrix th,.atlas-results .a29-matrix td{padding:11px 10px;text-align:right;border:0;border-bottom:1px solid #e7eef2;white-space:nowrap}\n.atlas-results .a29-matrix thead th{position:sticky;top:0;z-index:3;background:#e9f3f6;color:#204c64;font-size:10px;font-weight:800;min-width:95px;vertical-align:bottom;border-bottom:2px solid #b5d5dd}\n.atlas-results .a29-matrix th.a29-party{border-top:5px solid var(--a29-color);background:color-mix(in srgb,var(--a29-color) 12%,white);color:#1b4255;max-width:152px;white-space:normal;text-align:center}\n.atlas-results .a29-matrix th.a29-party span{display:block;width:125px;line-height:1.25}\n.atlas-results .a29-matrix tr:nth-child(even) td{background:#f7fafc}\n.atlas-results .a29-matrix td:hover{background:#eaf8f7!important;color:#105e69}\n.atlas-results .a29-matrix .a29-geo{position:sticky;left:0;z-index:2;text-align:left;min-width:200px;max-width:255px;background:#f6fafc!important;box-shadow:1px 0 0 #cbdde5;white-space:normal}\n.atlas-results .a29-matrix thead .a29-geo{z-index:5}\n.atlas-results .a29-matrix .a29-geo strong{display:block;color:#204b60}\n.atlas-results .a29-matrix .a29-geo small{display:block;color:#618295;line-height:1.5;margin-top:4px}\n.atlas-results .a29-matrix .a29-geo button{display:block;border:1px solid #b7d9dc;border-radius:7px;background:#e9f7f7;color:#14777d;padding:6px 8px;margin-top:7px;font-weight:750;font-size:10px;cursor:pointer}\n.atlas-results .a29-valid{color:#115e69;font-weight:800;background:#eaf6f5}\n.atlas-results .a29-pagination{display:flex;align-items:center;justify-content:center;gap:12px;padding:14px;border-top:1px solid #e5edf1;color:#5d7b88;font-size:12px}\n.atlas-results .a29-pagination button{border:1px solid #c5dce4;background:white;border-radius:8px;padding:8px 14px;color:#23596b;font-weight:750}\n.atlas-results .a29-pagination button:disabled{opacity:.35;cursor:not-allowed}\n.atlas-results .a29-advanced{margin:12px 23px 20px;border:1px solid #d5e6eb;background:#fbfdff;border-radius:12px;padding:10px}\n.atlas-results .a29-advanced>summary{cursor:pointer;font-size:13px;font-weight:800;color:#1c5265;padding:11px 14px}\n.atlas-results .a29-advanced[open]{background:#fff}\n.atlas-results .a29-advanced .ar-integration{padding:15px;margin-top:12px}\n.atlas-results .ar-comment[hidden]{display:none}\n/* La ampliación anterior permanece en el HTML para auditoría, pero no representa un segundo explorador público. */\n#atlas-v28-visor,#atlas-v28-results,#atlas-v26-detail,#atlas-v25-territorial{display:none!important}\n@media(max-width:1150px){.atlas-results .a29-visuals{grid-template-columns:1fr}.atlas-results .a29-kpis{grid-template-columns:repeat(3,minmax(0,1fr))}.atlas-results .a29-map-panel:has(+.a29-bar-panel[hidden]){grid-column:auto}}\n@media(max-width:620px){.atlas-results #atlas-v29-main{padding:0 12px 20px}.atlas-results .a29-input{flex-basis:45%}.atlas-results .a29-kpis{grid-template-columns:repeat(2,minmax(0,1fr))}.atlas-results .a29-matrix .a29-geo{min-width:155px;max-width:155px}}\n\n#atlas-v29-composicion{border:1px solid #cfe3e9;border-radius:17px;background:#fff;margin:25px 0;padding:0;box-shadow:0 13px 29px rgba(23,67,92,.065);overflow:hidden}\n#atlas-v29-composicion>header{display:flex;align-items:center;justify-content:space-between;gap:20px;padding:23px 26px;background:linear-gradient(105deg,#e6f5f5,#f7fbfd);border-bottom:1px solid #cfe3e9}\n#atlas-v29-composicion>header h3{color:#1b4e63;font-size:20px;font-weight:830;margin:7px 0 5px;letter-spacing:-.4px}\n#atlas-v29-composicion>header p{color:#5c7989;font-size:12px;margin:4px 0}\n#atlas-v29-composicion .a29-kicker{color:#087e84;font-size:10px;letter-spacing:1.5px;font-weight:850}\n#atlas-v29-composicion .a29-compare-controls{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;padding:17px 22px;background:#f8fbfd;border-bottom:1px solid #e2edf1}\n#atlas-v29-composicion .a29-compare-controls label{display:flex;flex-direction:column;gap:6px;font-size:11px;font-weight:750;color:#44697c}\n#atlas-v29-composicion select{background:#fff;border:1px solid #c1d8e1;border-radius:9px;min-height:39px;padding:8px 11px;color:#21495a;font-weight:680;width:100%}\n#atlas-v29-composicion .a29-compare-stats{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;padding:18px 22px}\n#atlas-v29-composicion .a29-compare-stats>div{padding:12px 15px;border:1px solid #dae8ed;border-top:3px solid #168d98;border-radius:10px;background:#fbfdfd}\n#atlas-v29-composicion .a29-compare-stats span{display:block;font-size:11px;color:#5a7788}\n#atlas-v29-composicion .a29-compare-stats strong{display:block;font-size:24px;color:#1a4c61;margin-top:5px;font-weight:830}\n#atlas-v29-composicion .a29-compare-fig{min-height:330px;width:100%;padding:0 18px}\n#atlas-v29-composicion .a29-compare-footer{padding:16px 22px;border-top:1px solid #e3ecef;display:flex;align-items:center;justify-content:space-between;gap:10px;flex-wrap:wrap;font-size:11px;color:#597889}\n#atlas-v29-composicion .a29-compare-footer button{border:1px solid #bfd6df;background:#fff;padding:7px 12px;border-radius:9px;font-size:11px;font-weight:800;color:#205d6e;cursor:pointer}\n#atlas-v27-participacion .a27-rank-cell{border-top:3px solid var(--a27-territory-color,#1a8991)!important;background:color-mix(in srgb,var(--a27-territory-color,#1a8991) 14%,white)!important}\n#atlas-v27-participacion .a27-rank-cell strong{color:var(--a27-territory-color,#1d6570);font-size:13px}\n#atlas-v27-participacion .a27-matrix thead th{white-space:normal!important}\n#atlas-v27-participacion .a27-matrix tfoot th{position:sticky;bottom:0;background:#e2f1f5}\n@media(max-width:850px){#atlas-v29-composicion .a29-compare-controls{grid-template-columns:repeat(2,minmax(0,1fr))}}\n\n/* Bastiones · arquitectura editorial y analítica; no afecta gráficos provinciales originales */\n#atlas-v30-bastiones{width:100%;max-width:100%;min-width:0;box-sizing:border-box;clear:both;position:relative;margin:30px 0 36px;padding:0;border:1px solid #cfe4eb;border-radius:19px;background:#fff;box-shadow:0 12px 36px rgba(18,66,88,.07);overflow:hidden;color:#1d4155}\n#atlas-v30-bastiones *{box-sizing:border-box}\n#atlas-v30-bastiones .a30-hero{background:linear-gradient(112deg,#103e53 0%,#176879 65%,#1a7f83 100%);color:#fff;display:flex;gap:18px;justify-content:space-between;align-items:center;padding:28px 32px}\n#atlas-v30-bastiones .a30-hero h3{margin:7px 0 8px;font-size:clamp(21px,2.5vw,30px);line-height:1.17;letter-spacing:-.65px;color:#fff;font-weight:820}\n#atlas-v30-bastiones .a30-hero p{max-width:860px;margin:0;font-size:12px;line-height:1.63;color:#e2f6f7}\n#atlas-v30-bastiones .a30-overline{font-weight:830;letter-spacing:1.7px;font-size:10px;color:#a8efe5}\n#atlas-v30-bastiones .a30-hero-mark{font-size:19px;font-weight:850;letter-spacing:-.5px;white-space:nowrap;color:#bceee6;padding:15px 18px;border:1px solid rgba(255,255,255,.24);border-radius:10px;background:rgba(255,255,255,.09)}\n#atlas-v30-bastiones .a30-toolbar{display:flex;align-items:end;flex-wrap:wrap;gap:12px;background:#eff8f9;padding:19px 23px 17px;border-bottom:1px solid #d8e8ec}\n#atlas-v30-bastiones .a30-toolbar label:not(.a30-include),#atlas-v30-bastiones .a30-panel-top label{display:flex;flex-direction:column;gap:6px;min-width:145px;flex:1 1 156px;font-size:11px;color:#436776;font-weight:790}\n#atlas-v30-bastiones .a30-toolbar select,#atlas-v30-bastiones .a30-panel-top select{width:100%;min-height:40px;padding:9px 12px;background:#fff;color:#193f54;border:1px solid #bcd5df;border-radius:9px;font-weight:700;font-size:12px}\n#atlas-v30-bastiones .a30-toolbar .a30-include{display:flex;align-items:center;gap:7px;flex:0 0 auto;min-height:39px;color:#3f6472;font-size:11px;font-weight:740;white-space:nowrap}\n#atlas-v30-bastiones .a30-toolbar .a30-include input{width:17px;height:17px;accent-color:#0e8e9b}\n#atlas-v30-bastiones .a30-kpis{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:10px;padding:20px 24px;background:#fff}\n#atlas-v30-bastiones .a30-kpi{min-width:0;padding:14px 12px 12px;background:#f7fafc;border:1px solid #dce9ef;border-top:3px solid #1c8e96;border-radius:11px}\n#atlas-v30-bastiones .a30-kpi small{display:block;min-height:28px;color:#547589;font-weight:650;font-size:10.5px;line-height:1.35}\n#atlas-v30-bastiones .a30-kpi strong{display:block;color:#173e50;font-size:23px;font-weight:830;margin-top:4px;letter-spacing:-.35px}\n#atlas-v30-bastiones .a30-tabs{display:flex;flex-wrap:wrap;gap:7px;padding:0 23px 17px;border-bottom:1px solid #e4edef;background:#fff}\n#atlas-v30-bastiones .a30-tabs button{font-weight:750;font-size:12px;color:#346273;background:#f2f7f9;padding:11px 18px;border:1px solid #cee0e6;border-radius:9px;cursor:pointer}\n#atlas-v30-bastiones .a30-tabs button.selected{color:#fff;background:#14798b;border-color:#14798b;box-shadow:0 5px 12px rgba(20,121,139,.15)}\n#atlas-v30-bastiones .a30-panel{padding:18px 23px 10px;background:#fff}\n#atlas-v30-bastiones .a30-panel[hidden],#atlas-v30-bastiones .a30-profile[hidden]{display:none!important}\n#atlas-v30-bastiones .a30-panel-top{display:flex;justify-content:space-between;align-items:center;gap:15px;flex-wrap:wrap;padding:4px 5px 18px}\n#atlas-v30-bastiones .a30-panel-top h4{margin:2px 0 5px;font-size:18px;font-weight:820;color:#1b4b60}\n#atlas-v30-bastiones .a30-panel-top p{margin:0;line-height:1.53;font-size:12px;color:#5a7785}\n#atlas-v30-bastiones .a30-panel-top>label{max-width:210px;flex:1}\n#atlas-v30-bastiones .a30-period{display:flex;flex-wrap:wrap;gap:8px}\n#atlas-v30-bastiones .a30-period label{min-width:95px;flex:0 0 100px}\n#atlas-v30-bastiones .a30-figure{min-height:350px;padding:0 2px 20px;overflow:hidden}\n#atlas-v30-bastiones .a30-scrolldata{max-height:510px;overflow:auto;width:100%;border:1px solid #dbe8ed;border-radius:11px;scrollbar-width:thin;scrollbar-color:#a6c9d2 #edf4f6}\n#atlas-v30-bastiones .a30-competition-table{max-height:670px}\n#atlas-v30-bastiones .a30-persistence-table{max-height:660px}\n#atlas-v30-bastiones .a30-data-table{min-width:100%;width:100%;font-size:11px;border-collapse:separate;border-spacing:0}\n#atlas-v30-bastiones .a30-data-table thead th{position:sticky;top:0;z-index:2;background:#e2eef2;color:#345669;text-align:left;font-size:10.5px;font-weight:810;padding:12px 14px;white-space:nowrap}\n#atlas-v30-bastiones .a30-data-table tbody tr:nth-child(even){background:#f6fafb}\n#atlas-v30-bastiones .a30-data-table td{padding:10px 12px;border-bottom:1px solid #e2e9ee;white-space:nowrap;color:#385b6d}\n#atlas-v30-bastiones .a30-data-table td:first-child{font-weight:760;color:#20485b}\n#atlas-v30-bastiones .a30-data-table td i{display:inline-block;width:9px;height:9px;border-radius:3px;margin-right:9px}\n#atlas-v30-bastiones .a30-data-table td.a30-pos{color:#087b83;font-weight:830}\n#atlas-v30-bastiones .a30-data-table td.a30-neg{color:#9a6272;font-weight:770}\n#atlas-v30-bastiones .a30-data-table button{border:1px solid #bcdbe2;background:#edf8f9;color:#126d79;border-radius:7px;font-size:11px;font-weight:770;padding:6px 10px;cursor:pointer}\n#atlas-v30-bastiones .a30-heat-table,#atlas-v30-bastiones .a30-persistence{min-width:1100px;width:100%;border-collapse:separate;border-spacing:2px;font-size:11px}\n#atlas-v30-bastiones .a30-heat-table thead th,#atlas-v30-bastiones .a30-persistence thead th,#atlas-v30-bastiones .a30-persistence tfoot th{position:sticky;top:0;z-index:4;background:#e1eef2;color:#294e61;padding:10px 8px;font-weight:800;font-size:10px;min-width:82px;max-width:145px;word-break:break-word}\n#atlas-v30-bastiones .a30-persistence tfoot th{top:auto;bottom:0;background:#e1eef2}\n#atlas-v30-bastiones .a30-heat-table th:first-child,#atlas-v30-bastiones .a30-persistence th:first-child{position:sticky;left:0;z-index:5;min-width:215px;max-width:250px;background:#f2f7f9;text-align:left}\n#atlas-v30-bastiones .a30-heat-table tbody th,#atlas-v30-bastiones .a30-persistence tbody th{position:sticky;left:0;z-index:2;padding:9px;color:#235064;background:#f2f7fa;font-size:10.5px;text-align:left}\n#atlas-v30-bastiones .a30-persistence tbody th small{display:block;font-size:9.5px;color:#6c8796;margin-top:3px;font-weight:500}\n#atlas-v30-bastiones .a30-heat-table td{padding:9px 6px;min-width:85px;text-align:center;font-size:10.5px;font-weight:770}\n#atlas-v30-bastiones .a30-heat-table td.a30-colored{background:color-mix(in srgb,var(--a30-party) calc(var(--a30-strength)*100%),white);border-bottom:2px solid var(--a30-party);color:#173e50}\n#atlas-v30-bastiones .a30-heat-table td b{color:#175c65;font-size:10px}\n#atlas-v30-bastiones .a30-persistence td{padding:8px 5px;text-align:center;font-size:10px;white-space:nowrap;font-weight:760}\n#atlas-v30-bastiones .a30-persistence .a30-persistent-cell{background:color-mix(in srgb,var(--a30-color) calc(var(--a30-persistent)*100%),white);color:#1b495a}\n#atlas-v30-bastiones .a30-persistence td.a30-na{color:#9bacb6;background:#eef3f5}\n#atlas-v30-bastiones .a30-persistence .a30-ratio{min-width:105px;color:#176e80;font-weight:830;background:#e9f6f5}\n#atlas-v30-bastiones .a30-persistence .a30-ratio small{display:block;font-size:9px;color:#547484;font-weight:500}\n#atlas-v30-bastiones .a30-export{display:flex;align-items:center;justify-content:flex-end;flex-wrap:wrap;gap:7px;padding:16px 2px;border-bottom:1px solid #e3ecf0}\n#atlas-v30-bastiones .a30-export span{margin-right:auto;color:#668290;font-size:10.5px;max-width:700px;line-height:1.4}\n#atlas-v30-bastiones .a30-export button{font-weight:800;font-size:11px;padding:8px 13px;border:1px solid #bcd6df;background:#fff;color:#1b6475;border-radius:8px;cursor:pointer}\n#atlas-v30-bastiones .a30-profile{margin:6px 24px 20px;background:#f8fcfd;border:1px solid #d6e7ec;border-radius:12px;padding:18px}\n#atlas-v30-bastiones .a30-profile>header{display:flex;align-items:center;justify-content:space-between;gap:12px}\n#atlas-v30-bastiones .a30-profile h4{margin:5px 0 9px;font-size:17px;color:#1b4e62}\n#atlas-v30-bastiones .a30-profile>header button{background:#fff;border:1px solid #c6dce5;border-radius:8px;padding:8px 12px;color:#1e6071}\n#atlas-v30-bastiones .a30-profile-plot{width:100%;min-height:320px}\n#atlas-v30-bastiones .a30-method{padding:19px 25px 22px;background:#f0f7f8;color:#527182;font-size:11px;line-height:1.7}\n#atlas-v29-composicion{width:100%;clear:both;margin:24px 0!important}\n#atlas-v30-bastiones ~ #atlas-v29-composicion{margin-top:0}\n@media(max-width:1170px){#atlas-v30-bastiones .a30-kpis{grid-template-columns:repeat(3,minmax(0,1fr))}#atlas-v30-bastiones .a30-hero-mark{display:none}}\n@media(max-width:770px){#atlas-v30-bastiones .a30-kpis{grid-template-columns:repeat(2,minmax(0,1fr))}#atlas-v30-bastiones .a30-hero{padding:23px 18px}#atlas-v30-bastiones .a30-toolbar,#atlas-v30-bastiones .a30-panel{padding:15px}#atlas-v30-bastiones .a30-toolbar label:not(.a30-include){flex-basis:44%;min-width:135px}}\n#atlas-v30-bastiones{font-family:Inter,-apple-system,BlinkMacSystemFont,\'Segoe UI\',Arial,sans-serif}\n#atlas-v29-main .a29-pagination{display:none!important}\n#atlas-v29-main .a29-table-box{max-height:75vh;overflow:auto}\n#atlas-v30-bastiones,#atlas-v29-composicion{grid-column:1/-1!important}\n\n/* V31: explicit grouping and one continuous matrix. */\n#atlas-v29-main .a31-control-divider{flex-basis:100%;grid-column:1/-1;font-weight:800;color:#17394d;border-top:1px solid #cfe1e7;padding-top:14px;margin-top:8px}\n#atlas-v31-leadership{grid-column:1/-1;width:100%;clear:both;padding:22px;box-sizing:border-box}\n.a31-leader-table{border-collapse:separate;border-spacing:1px;width:max-content;min-width:100%;font:12px Arial,sans-serif;color:#17394d}\n.a31-leader-table th,.a31-leader-table td{padding:12px;min-width:100px;max-width:220px;text-align:center;line-height:1.4}\n.a31-leader-table th{background:#e8f2f5;position:sticky;left:0;z-index:2;text-align:left}\n.a31-leader-table thead th{position:sticky;top:0;z-index:3}.a31-leader-table tfoot th{position:sticky;bottom:0}\n.a31-leader-table caption{font-weight:700;text-align:left;padding:14px;background:white}\n#atlas-v29-main .a29-input[hidden],#atlas-v31-leadership label[hidden]{display:none!important}\n#atlas-v29-main .a29-toolbar{gap:14px;padding:22px;background:#f4f9fb}\n#atlas-v29-main .a29-breadcrumb{padding:15px 22px;background:white}\n#atlas-v29-main .a29-matrix tbody td{background-clip:padding-box}\n\n/* V32: original card grammar, provincial hierarchy, bottom year axis. */\n#atlas-v31-leadership{padding:0}.a32-head{display:flex;align-items:center;justify-content:space-between;gap:14px;padding:15px 22px}.a32-head h4{margin:0;color:#1b455b;font-size:17px}.a32-actions{display:flex;gap:6px;flex-wrap:wrap}.a32-actions button,#atlas-v31-leadership button{border:1px solid #bfd6df;background:#fff;color:#205d6e;padding:8px 13px;border-radius:8px;font:750 11px Inter,Arial,sans-serif;cursor:pointer}.a32-actions button:hover{background:#edf7f8;border-color:#168d98}.a32-scroll{overflow:auto;max-height:75vh;margin:0 18px 22px;border:1px solid #d5e5eb;border-radius:12px}.a32-method,[data-l="note"]{margin:12px 22px;color:#527486;font-size:12px;line-height:1.6}.a31-leader-table td{min-width:160px;max-width:190px;font-size:11px}.a31-leader-table small{display:block;font-size:10px;margin-top:6px;font-weight:500}.a31-leader-table .a32-group th{position:static;background:#e3f0f4;padding:11px;text-align:left;font-size:14px}.a31-leader-table thead th{height:5px;padding:0;border:0;min-height:0}.a31-leader-table thead th:first-child{font-size:11px;padding:8px}.a31-leader-table tfoot th{position:sticky;bottom:0;background:#e4f1f5;text-align:center}.a31-leader-table .a32-na{background:#edf2f4;color:#71828c}.a31-leader-table .a32-diff{color:#17394d;text-align:left}.a32-diff span{display:block;padding-left:6px;margin-top:6px;font-size:10px}.a32-agree{background:#edf6f4}.a32-different{background:#fff0d7}.a32-tie{background:#e9edf3}.a32-prov-composition{padding:12px 22px;border-top:1px solid #d7e6ec}.a32-prov-composition h4{font-size:17px;color:#1b455b}.a32-prov-composition p{color:#527486;font-size:12px}.a32-summary-table{width:100%;border-collapse:collapse;font-size:12px}.a32-summary-table th,.a32-summary-table td{padding:9px;border-bottom:1px solid #e1edf1;text-align:left}.a32-prov-composition details{overflow:auto;margin:14px 0}.a29-compare-fig{box-sizing:border-box}#atlas-v29-main [hidden]{display:none!important}\n#atlas-v27-participacion .a27-full-cell{min-width:115px;width:130px;height:62px;padding:9px 11px!important;vertical-align:middle;border-top:3px solid var(--a27-territory-color);background:color-mix(in srgb,var(--a27-territory-color) calc(var(--a27-alpha)*100%),white);text-align:left}.a27-full-cell strong{display:block;color:#17394d;font-size:13px;font-weight:650}.a27-full-cell small{display:block;color:#547382;font-size:10px;margin-top:5px}.a27-full-cell.a32-maximum{outline:2px solid var(--a27-territory-color);outline-offset:-2px}.a27-full-cell.a32-maximum strong{font-weight:900}.a27-prov-divider th{border-left:6px solid var(--a27-territory-color)}\n\n/* V33: navigation by province, full-width map and readable results. */\n#atlas-v29-main .a29-visuals{display:flex!important;flex-direction:column;gap:20px;width:100%;align-items:stretch}#atlas-v29-main [data-atlas29="map-host"],#atlas-v29-main .a29-bar-panel{width:100%;min-width:0;box-sizing:border-box}#atlas-v29-main .a29-map-panel{height:auto!important;max-height:none!important;overflow:visible!important}#atlas-v29-main [data-atlas29="map"]{height:650px!important;max-height:none!important;width:100%}#atlas-v29-main .a29-bar-panel{max-height:none!important;overflow:visible!important}#atlas-v29-main .a29-bars{height:auto!important;max-height:none!important;width:100%}.a33-near-navigation{border:1px solid #c9dfe7;border-radius:12px;padding:18px;background:#f3f9fb;box-sizing:border-box;position:relative}.a33-view-context{display:flex;justify-content:space-between;gap:12px;flex-wrap:wrap;color:#1b455b;margin-bottom:13px}.a33-view-context strong{font-size:17px}.a33-view-context span{font-size:12px;color:#527486}.a33-territory-actions,.a33-breakdowns{display:flex;gap:9px;flex-wrap:wrap}.a33-breakdowns{flex-basis:100%;padding-top:12px;border-top:1px solid #d6e5eb;margin-top:5px}.a33-territory-actions button{background:#fff;color:#20566b;border:1px solid #bcd6e1;border-radius:9px;padding:11px 15px;font-size:12px;font-weight:750;cursor:pointer}.a33-territory-actions button[aria-pressed="true"]{background:#14798b;color:#fff;border-color:#14798b}.a33-territory-actions button:disabled{opacity:.5;cursor:default}.a33-exact{padding:12px 20px;border-bottom:1px solid #e0edf2;font-size:12px;color:#326275}.a33-exact summary{cursor:pointer;font-weight:700}.a33-exact .a29-toolbar{margin-top:12px}.a33-province-divider th{position:static!important;text-align:left!important;padding:13px!important;background:#dcecf2!important;color:#17495f!important;font-size:14px!important;border-left:6px solid #168b96!important}.a33-canton-divider th{position:static!important;text-align:left!important;padding:10px 20px!important;background:#eef5f8!important;color:#315b70!important;font-size:12px!important}.a33-province-divider,.a33-canton-divider{scroll-margin-top:60px}\n#atlas-v31-leadership .a32-scroll{max-height:85vh;margin:8px 22px 30px}#atlas-v31-leadership .a31-leader-table td{min-width:190px;width:210px;max-width:230px;padding:20px 14px;font-size:12px;line-height:1.65}#atlas-v31-leadership [data-l="dominance"] td{min-width:220px;padding:24px 16px;line-height:1.8}#atlas-v31-leadership [data-l="differences"] td{min-width:240px;padding:22px 16px}#atlas-v31-leadership[data-scale="p"] [data-l="votes"] td{min-width:220px;padding:25px 18px;font-size:13px}#atlas-v31-leadership small{font-size:11px;margin-top:9px}.a32-head{padding-top:24px;padding-bottom:18px}.a33-runoff{margin:0 22px 15px;border:1px solid #d4e4eb;border-radius:9px;padding:12px 15px;font-size:12px;color:#3b6375}.a33-runoff summary{cursor:pointer;font-weight:750}.a33-runoff-toggle{display:flex;gap:8px;align-items:center;margin-top:14px}.a33-year-choices{display:flex;flex-wrap:wrap;gap:16px;margin-top:13px}.a33-year-choices label{display:flex;gap:6px;align-items:center}.a33-runoff[hidden],.a33-year-choices[hidden]{display:none!important}.a33-downloads{display:flex;gap:12px;align-items:center;flex-wrap:wrap}.a33-downloads>span{font-size:11px;font-weight:700;color:#527486}.a33-composition-panel{border-top:1px solid #d7e6ec;padding:0 0 18px}.a33-composition-panel>p{font-size:12px;color:#527486;margin:0 25px 15px}.a33-data{margin:15px 25px;font-size:12px;overflow:auto}#atlas-v29-composicion>header{flex-wrap:wrap}#atlas-v29-composicion .a29-compare-fig{padding:0 12px;overflow:visible;min-height:430px}#atlas-v31-leadership .a29-compare-controls{display:flex;flex-wrap:wrap;gap:15px;padding:20px 22px;background:#f4f9fb}#atlas-v31-leadership .a29-compare-controls label{display:flex;flex:1 1 210px;flex-direction:column;gap:8px;font-size:12px;font-weight:700}#atlas-v31-leadership .a29-compare-controls select{padding:11px;border:1px solid #bfd6df;border-radius:8px;background:white;color:#21485c}\n@media(max-width:700px){#atlas-v29-main [data-atlas29="map"]{height:650px!important}.a33-view-context{flex-direction:column}.a33-territory-actions button{flex:1 1 160px}#atlas-v29-composicion .a32-head{align-items:flex-start;flex-direction:column}.a33-downloads{gap:8px}}\n\n.a36-measure{display:flex;flex-direction:column;gap:6px;min-width:230px}.a36-measure>span{font-size:11px;font-weight:700}.a36-measure>div{display:flex;gap:0}.a36-measure button{border:1px solid #b7d4e0;background:#fff;color:#20566c;padding:9px 14px;font:inherit;cursor:pointer}.a36-measure button:first-child{border-radius:7px 0 0 7px}.a36-measure button:last-child{border-radius:0 7px 7px 0}.a36-measure button[aria-pressed="true"]{background:#126c7c;color:white;border-color:#126c7c;font-weight:700}.a36-measure small{font-size:10px;color:#536f80}.a36-measure button:focus-visible{outline:3px solid #b5d6ec;outline-offset:2px}\n#atlas-v31-leadership .a32-diff{min-width:135px;padding:12px 10px;line-height:1.5;border:1px solid #dce7eb;font-size:12px}#atlas-v31-leadership .a32-agree{background:#f0f6f7}#atlas-v31-leadership .a32-different{background:#fcf1df}#atlas-v31-leadership .a32-tie{background:#eeeaf5}#atlas-v31-leadership .a32-diff span{display:block;margin:5px 0;padding-left:7px}#atlas-v31-leadership .a32-diff strong{display:block;font-size:12px}.a30-history{border-collapse:separate;border-spacing:1px;min-width:1200px;width:100%;font-size:12px}.a30-history th,.a30-history td{padding:10px 8px;min-width:65px;background:#eef5f7}.a30-history tbody th{min-width:230px;text-align:left}.a30-history small{display:block;margin-top:6px;font-weight:400}.a30-history tfoot th{background:#deebf0}.a30-tabs{flex-wrap:wrap}\n'

_ATLAS32_COMPLETE_MATRIX = ' function completeMatrix(yset,a,l,m){\n  const items=l===\'p\'?provinces.map(p=>({key:p,name:p,province:p,sub:\'\'})):l===\'c\'?cantons.map(c=>({key:c.code,name:c.name,province:c.province,sub:\'\'})):districts.map(d=>({key:d.code,name:d.name,province:d.province,sub:d.canton}));\n  items.sort((x,z)=>provinces.indexOf(x.province)-provinces.indexOf(z.province)||x.name.localeCompare(z.name,\'es\'));\n  const extrema=new Map(yset.map(y=>[y,Math.max(...universe(y,a,l,m).map(r=>Number(r[m])))]));exportRows=[];let body=\'\',last=\'\';\n  for(const u of items){if(last!==u.province){last=u.province;body+=`<tr class="a27-prov-divider" style="--a27-territory-color:${palette[u.province]}"><th colspan="${yset.length+1}">${esc(last)}</th></tr>`}\n   body+=`<tr><th class="a27-freeze-col" title="${esc(u.sub)} · ${esc(u.province)}">${esc(u.name)}${u.sub?`<small>${esc(u.sub)}</small>`:\'\'}</th>`;\n   for(const y of yset){const r=dataMetric(y,a,l,u.key);const val=r?.[m];const exists=l===\'p\'||(l===\'c\'?V.canton_years[u.key]:V.district_years[u.key])?.includes(y);\n    const css=nvalid(val)?\'a27-intensity\':\'a27-missing\';const amt=nvalid(val)?Math.max(.12,Math.min(.96,Number(val)/(m.endsWith(\'_pct\')?100:Math.max(1,Math.max(...yset.map(q=>Number(dataMetric(q,a,l,u.key)?.[m]||0))))))):0;\n    exportRows.push({eleccion:y,escala:l,provincia:u.province,territorio:u.name,canton:u.sub,codigo:u.key,indicador:m,valor:nvalid(val)?val:null,estado:nvalid(val)?\'Observado\':exists?\'Sin dato\':\'No existía\'});\n    const maximum=nvalid(val)&&Number(val)===extrema.get(y),color=palette[u.province]||\'#54809a\';\n    body+=`<td class="${nvalid(val)?\'a27-full-cell\'+(maximum?\' a32-maximum\':\'\'):css}" style="--a27-territory-color:${color};--a27-alpha:${(.10+.22*amt).toFixed(3)}" title="${esc(u.name)} · ${esc(u.province)} · ${y} · ${nvalid(val)?shown(val,m):exists?\'Sin dato\':\'No existía en la elección\'}"><strong>${nvalid(val)?shown(val,m):exists?\'—\':\'×\'}</strong>${maximum?\'<small>◆ Máximo de la elección</small>\':\'\'}</td>`;\n   }body+=\'</tr>\';\n  }\n  if(l!==\'d\'){body+=`<tr class="a27-national"><th class="a27-freeze-col">Costa Rica</th>`+yset.map(y=>{const r=dataMetric(y,a,\'n\',\'CR\');exportRows.push({eleccion:y,escala:\'n\',provincia:\'Costa Rica\',territorio:\'Costa Rica\',indicador:m,valor:r?.[m]??null});return `<td>${shown(r?.[m],m)}</td>`}).join(\'\')+\'</tr>\'}\n  $(\'a27-p-table-region\').innerHTML=matrix(heading(\'Territorio\',yset.map(()=>\'\')),body,heading(\'Año\',yset));\n  $(\'a27-p-subtitle\').textContent=`Territorios agrupados por provincia; × significa que el territorio no figuraba en esa elección. El total nacional aparece al final cuando corresponde.`;\n }\n'

def _atlas36_participation_source(js):
    """Separar el indicador de su unidad, conservando los datos y las vistas existentes."""
    a=js.index('function participationExplorer(){');b=js.index('\nfunction matrixSVG',a)
    part=js[a:b]
    anchor=" const fields=["
    position=part.index(anchor)
    part=part[:position]+" dataRows.forEach(r=>{r.no_validos=r.blancos==null||r.nulos==null?null:Number(r.blancos)+Number(r.nulos)});\n"+part[position:]
    part=part.replace("['nulos','Votos nulos (cantidad)']", "['nulos','Votos nulos (cantidad)'],['no_validos','Votos no válidos (cantidad)']")
    control="${select('a27-p-indicator','Indicador',fields)}"
    replacement="""${select('a36-p-concept','Indicador',[['abstencion','Abstencionismo'],['participacion','Participación / emitidos'],['validos','Votos válidos'],['blancos','Votos en blanco'],['nulos','Votos nulos'],['no_validos','Votos no válidos']])}<div class="a36-measure" role="group" aria-label="Unidad del indicador"><span>Ver valores</span><div><button type="button" data-a36-unit="percent" aria-pressed="true">Porcentaje</button><button type="button" data-a36-unit="count" aria-pressed="false">Cantidad</button></div><small id="a36-p-denominator"></small></div>"""+control
    assert control in part;part=part.replace(control,replacement)
    anchor=" const nvalid=(v)=>"
    position=part.index(anchor)
    helper=""" let measureUnit='percent';
 const dimensions={abstencion:['abst_pct','abstenciones','Abstenciones / padrón × 100'],participacion:['part_pct','emitidos','Emitidos / padrón × 100'],validos:['valid_pct','validos','Válidos / emitidos × 100'],blancos:['blancos_pct','blancos','Blancos / emitidos × 100'],nulos:['nulos_pct','nulos','Nulos / emitidos × 100'],no_validos:['noval_pct','no_validos','(Blancos + nulos) / emitidos × 100']};
 $('a27-p-indicator').closest('label').hidden=true;
 function syncMeasure(){const def=dimensions[$('a36-p-concept').value];$('a27-p-indicator').value=def[measureUnit==='percent'?0:1];$('a36-p-denominator').textContent=measureUnit==='percent'?def[2]:'Cantidad absoluta · '+(def[1]==='abstenciones'?'personas que no votaron':'votos');sec.querySelectorAll('[data-a36-unit]').forEach(b=>b.setAttribute('aria-pressed',String(b.dataset.a36Unit===measureUnit)))}
 $('a36-p-concept').addEventListener('change',syncMeasure);
 sec.querySelectorAll('[data-a36-unit]').forEach(b=>b.addEventListener('click',()=>{measureUnit=b.dataset.a36Unit;syncMeasure();draw()}));
 syncMeasure();
"""
    part=part[:position]+helper+part[position:]
    return js[:a]+part+js[b:]


def _atlas37_consolidate_v27(js):
    """Unificar perfiles y mantener fórmulas y apellidos en sus consultas específicas."""
    start=js.index('function atlasPartyProfiles(){')
    end=js.index('function atlasPaperBallots(){',start)
    js=js[:start]+_ATLAS37_PROFILE_COMPLEMENT+js[end:]
    js=js.replace("formulas:()=>atlasNominalExplorer('formulas')", "formulas:()=>{}")
    js=js.replace("apellidos:()=>atlasNominalExplorer('apellidos')", "apellidos:()=>{}")
    start=js.index('function atlasNominalExplorer(section){')
    end=js.index('function atlasMethods()',start)
    nominal=js[start:end]
    nominal=nominal.replace('<button type="button" class="a26-person-option" data-nom="${i}">','<article class="a26-person-option" data-nom="${i}">')
    nominal=nominal.replace("${x.elected?'✓ Electa':'No electa'}</span></button>", "${x.elected?'✓ Electa':'No electa'}</span></article>")
    nominal=nominal.replace('<strong>${esc(x.name)}</strong>','<strong>${window.Atlas38Links?window.Atlas38Links.person({nombre:x.name}):esc(x.name)}</strong>')
    js=js[:start]+nominal+js[end:]
    return js

def _atlas32_source_renderers(html):
    pattern=r'(<script id="atlas-v27-js">)([\s\S]*?)(</script>)'
    match=re.search(pattern,html)
    if not match:raise RuntimeError('No se encontró el visor original de participación V27.')
    js=_atlas37_consolidate_v27(_atlas36_participation_source(match.group(2)))
    a=js.index(' function completeMatrix(');b=js.index(' function singleYear(',a)
    js=js[:a]+_ATLAS32_COMPLETE_MATRIX+js[b:]
    js=js.replace("matrix(heading('Puesto',yset),body,heading('Puesto',yset))", "matrix(heading('Puesto',yset.map(()=>'')),body,heading('Año',yset))")
    js=js.replace("const actual=(view==='matrix'&&l==='d')?'top10':view;","const actual=view;")
    # Export from the very same cells, including provincial headers, maxima and bottom years.
    js=js.replace("if(matrixMode)return downloadMatrix(matrixSVG($('a27-p-table-region'),$('a27-p-title').textContent),extension,file);","if(matrixMode)return window.Atlas31TableImage($('a27-p-table-region table'),$('a27-p-title').textContent,extension,file);")
    html=html[:match.start()]+match.group(1)+js+match.group(3)+html[match.end():]
    # Remove only the district restriction; the other compatibility improvements stay active.
    html=html.replace('complete.hidden=district;complete.disabled=district;', 'complete.hidden=false;complete.disabled=false;')
    html=html.replace("if(district&&mode.value==='matrix'){", "if(false){")
    return html

def _atlas34_prepare_source(html):
    """Finalize compatible renderers before the browser executes any script."""
    plot_pattern = re.compile(r"pending\.push\(Plotly\.newPlot\(id,traces,layout,([^\n]+?)\)\.then\(\(\)=>\{\$\(id\)\.atlasTitle=titleText;\$\(id\)\.atlasScope=opts\.subtitle\|\|suffix;if\(opts\.onClick\)\$\(id\)\.on\('plotly_click',opts\.onClick\)\}\)\);")
    def capture_chart(match):
        return "const chartNode=$(id);pending.push(Plotly.newPlot(chartNode,traces,layout,"+match.group(1)+").then(()=>{if(!chartNode.isConnected)return;chartNode.atlasTitle=titleText;chartNode.atlasScope=opts.subtitle||suffix;if(opts.onClick)chartNode.on('plotly_click',opts.onClick)}));"
    html, count = plot_pattern.subn(capture_chart, html)
    if count != 1 and 'const chartNode=$(id);pending.push(Plotly.newPlot(chartNode' not in html:
        raise RuntimeError('No se encontró el renderizador principal de gráficas; no se publica una aplicación sin comprobarlo.')
    runtime = re.search(r'(<script id="atlas-v28-runtime">)([\s\S]*?)(</script>)', html)
    if runtime:
        marker = '/* Compatibilidad puntual con las matrices de participación existentes en V27.'
        js = runtime.group(2)
        if marker in js:
            js = js[js.index(marker):]
        elif 'function attachBastions' in js:
            raise RuntimeError('No se pudo aislar el visor de bastiones obsoleto.')
        html = html[:runtime.start()] + runtime.group(1) + js + runtime.group(3) + html[runtime.end():]
    html=html.replace("note:!vals.length?'No existe un valor comparable", "onClick:opts.onClick,note:!vals.length?'No existe un valor comparable")
    table_anchor="$(id+'body').querySelectorAll('th').forEach(h=>h.onclick="
    table_link="if(state.tab!=='gobierno')window.Atlas38Links?.decorate($(id+'body'));"
    if table_link not in html:
        html=html.replace(table_anchor,table_link+table_anchor,1)
    render_anchor="  const renderPerson=r=>{"
    render_link="  const renderPerson=r=>{if(state.tab!=='gobierno'&&window.Atlas38Links)return window.Atlas38Links.person(r);"
    if render_link not in html:
        html=html.replace(render_anchor,render_link,1)
    chart_anchor="if(opts.onClick)chartNode.on('plotly_click',opts.onClick)"
    chart_link=chart_anchor+";if(state.tab!=='gobierno')window.Atlas38Links?.bindPlot(chartNode)"
    if chart_link not in html:html=html.replace(chart_anchor,chart_link,1)
    html = html.replace('Ocho bases de datos integradas y una novena base documental presidencial prevista para una etapa posterior.',
                        'Ocho bases integradas, con fuentes, cobertura y criterios de validación documentados.')
    html = re.sub(r'Versi[oó]n p[uú]blica\s*\d+(?:\.\d+)*', 'Edición pública 38', html, flags=re.I)
    html = re.sub(r'Versi[oó]n\s+\d+\.\d+(?:\.\d+)*', 'Edición pública 38', html, flags=re.I)
    html = re.sub(r'Atlas Electoral V\d+(?:[._]\d+)*', 'Atlas Electoral · Edición pública 38', html)
    html = re.sub(r'<title>[\s\S]*?</title>', '<title>Atlas Electoral de Costa Rica · Edición pública 38</title>', html, count=1)
    # Metadata is descriptive; table values and identifiers are left intact.
    blob = re.search(r'(<script id="atlas-data" type="application/json">)([\s\S]*?)(</script>)', html)
    if blob:
        data = json.loads(blob.group(2))
        meta = data.get('meta', {})
        for key in ['version', 'versión', 'version_publica']:
            if key in meta: meta[key] = '38.0'
        data_text = json.dumps(data, ensure_ascii=False, separators=(',', ':')).replace('</', '<\\/')
        html = html[:blob.start()] + blob.group(1) + data_text + blob.group(3) + html[blob.end():]
    html = re.sub(r'(?im)^[ \t]*//[^\n]*(?:\bV\d|versi[oó]n\s+\d)[^\n]*(?:\n|$)', '', html)
    def clean_revision_block(match):
        text=match.group(0)
        return '' if re.search(r'\bV\d|versi[oó]n\s+\d',text,re.I) else text
    html = re.sub(r'(?m)^[ \t]*/\*[\s\S]*?\*/', clean_revision_block, html)
    return html


def _atlas34_root_application(source, root):
    """Publish the complete application in the project root, with valid local links."""
    source, root = Path(source), Path(root)
    html = source.read_text(encoding='utf-8')
    html=re.sub(r'<script id="atlas-public-refinements">[\s\S]*?</script>','',html)
    html=html.replace('</body>','<script id="atlas-public-refinements">'+_ATLAS37_PUBLIC_REFINEMENT+'</script></body>',1)
    html = re.sub(r'<script id="atlas-public-edition">[\s\S]*?</script>', '', html)
    html = re.sub(r'<style id="atlas-public-design">[\s\S]*?</style>', '', html)
    html = html.replace('</head>', '<style id="atlas-public-design">'+_ATLAS34_DESIGN+'</style></head>', 1)
    html = html.replace('</body>', '<script id="atlas-public-edition">'+_ATLAS34_NAVIGATION+'</script></body>', 1)
    html = re.sub(r'(href=[\"\'])(?:\./)?Atlas_Bases_Electorales_Originales\.zip', r'\1salidas/Atlas_Bases_Electorales_Originales.zip', html)
    html = re.sub(r'([\"\'`])(?:\./)?datos/', r'\1salidas/datos/', html)
    html = html.replace('../recursos/', 'recursos/')
    html = re.sub(r'(href=[\"\'])\.\./(Base[^\"\']+\.xlsx)', r'\1\2', html)
    destination = root/'Atlas_Electoral_Costa_Rica_1953_2026.html'
    staged = None
    try:
        with NamedTemporaryFile('w', encoding='utf-8', dir=root, prefix='.atlas_public_', suffix='.tmp', delete=False) as stream:
            staged = Path(stream.name)
            stream.write(html)
        os.replace(staged, destination)
        staged = None
        if source.resolve() != destination.resolve(): source.unlink()
    finally:
        if staged is not None: staged.unlink(missing_ok=True)
    print('Aplicación pública:', destination)
    print('Datos derivados y archivos descargables:', root/'salidas'/'datos')
    return destination


def _atlas29_publish(path):
    path=Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'No se generó el HTML de la aplicación: {path}')
    html=_atlas34_prepare_source(_atlas32_source_renderers(path.read_text(encoding='utf-8')))
    if '<script id="atlas-v2317-js">' not in html:
        raise RuntimeError('Falta el mapa institucional de resultados de la sección 7; no se publica una versión incompleta.')
    if 'id="atlas-v28-dataset"' not in html:
        raise RuntimeError('Faltan los resultados territoriales V28 en el HTML. Ejecute las celdas previas del mismo notebook.')
    # Idempotencia: un solo módulo de resultados integrado y un solo estilo.
    html=re.sub(r'<script id="atlas-v29-results">[\s\S]*?</script>', '', html)
    html=re.sub(r'<script id="atlas-v29-analysis">[\s\S]*?</script>', '', html)
    html=re.sub(r'<style id="atlas-v29-theme">[\s\S]*?</style>', '', html)
    if html.count('</head>')!=1 or html.count('</body>')!=1:
        raise RuntimeError('El HTML está incompleto; se mantiene el archivo anterior sin cambios.')
    js1=_ATLAS_V31_RESULTS
    js2=_ATLAS_V31_ANALYSIS
    style=_ATLAS_V31_CSS
    for part in (js1,js2):
        if '</script>' in part.lower():raise RuntimeError('Un script contiene una etiqueta de cierre inesperada.')
    html=html.replace('</head>', '<style id="atlas-v29-theme">\n'+style+'\n</style>\n</head>',1)
    html=html.replace('</body>', '<script id="atlas-v29-results">\n'+js1+'\n</script>\n<script id="atlas-v29-analysis">\n'+js2+'\n</script>\n</body>',1)
    assert html.count('id="atlas-v29-results"')==1
    assert html.count('id="atlas-v29-analysis"')==1
    assert html.count('id="atlas-v29-theme"')==1
    publish=None
    try:
        with NamedTemporaryFile('w',encoding='utf-8',dir=path.parent,prefix='.atlas_v29_',suffix='.tmp',delete=False) as f:
            publish=Path(f.name)
            f.write(html)
        os.replace(publish,path)
        publish=None
    finally:
        if publish is not None:publish.unlink(missing_ok=True)
    print('Aplicación pública integrada:',path)
    print('Sección 7: mapa, barras, matriz territorial y exportación en un solo explorador.')
    print('Sección 8: liderazgos, composición temporal y análisis de bastiones con tres modalidades.')
    print('No se instalaron ni escribieron archivos JavaScript o CSS adicionales.')

_ATLAS29_OUTPUT=Path(OUTPUT_DIR)/'Atlas_Electoral_Costa_Rica_1953_2026.html'
_atlas29_publish(_ATLAS29_OUTPUT)

_ATLAS34_NAVIGATION = '/* Navegación interna y presentación de la edición pública. */\n(function(){\'use strict\';\n if(window.AtlasPublicEdition)return;window.AtlasPublicEdition=\'34.0\';\n const getContent=()=>document.getElementById(\'content\');let scheduled=false,signature=\'\',serial=0,navObserver=null,lastTargets=[];\n const normalize=s=>String(s||\'\').normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').toLowerCase();\n const related={participacion:[\'mapas\',\'competencia\'],competencia:[\'partidos\',\'dualidad\'],partidos:[\'competencia\',\'gobierno\',\'perfiles\'],gobierno:[\'dualidad\',\'representacion_voto\',\'partidos\'],dualidad:[\'gobierno\',\'competencia\'],representacion:[\'representacion_voto\',\'gobierno\'],representacion_voto:[\'representacion\',\'gobierno\'],candidaturas:[\'genero\',\'trayectorias\'],genero:[\'candidaturas\',\'trayectorias\'],trayectorias:[\'candidaturas\',\'formulas\'],formulas:[\'trayectorias\',\'apellidos\'],apellidos:[\'formulas\',\'trayectorias\'],papeletas:[\'datos\',\'metodo\'],cancelaciones:[\'partidos\',\'perfiles\'],perfiles:[\'partidos\',\'competencia\'],mapas:[\'participacion\',\'gobierno\'],laboratorio:[\'datos\',\'metodo\'],datos:[\'metodo\',\'panorama\'],panorama:[\'participacion\',\'competencia\'],metodo:[\'datos\']};\n const esc=s=>String(s??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\n function classifyNotice(content){const node=content.querySelector(\'.atlas-v2317-dimensiones\');if(!node||node.querySelector(\'details\'))return;const body=node.querySelector(\'p\');if(!body)return;const details=document.createElement(\'details\');details.className=\'a34-definition\';details.innerHTML=\'<summary>Cómo se clasifica la oferta partidaria</summary>\';details.append(body);node.replaceChildren(details)}\n function currentTab(){return typeof state!==\'undefined\'?state.tab:\'\'}\n function decorate(){const content=getContent();if(!content)return;classifyNotice(content);\n  const heads=[...content.querySelectorAll(\'h2,h3,h4\')].filter(h=>!h.closest(\'.a34-subnav,.a34-related,[hidden],.a30-profile[hidden],.a33-data,.reading-help,.atlas-v2317-dimensiones\'));\n  const targets=[],seen=new Set();for(const h of heads){const target=h.closest(\'#atlas-v31-leadership,#atlas-v29-composicion,#atlas-v30-bastiones,section.a25-block,article.card,.panel\')||h.closest(\'section\')||h.parentElement;\n   if(!target||target===content||seen.has(target)||target.closest(\'[hidden]\'))continue;const label=h.textContent.trim().replace(/\\s+/g,\' \');if(!label||label.length>150)continue;seen.add(target);if(!target.id)target.id=\'atlas-section-\'+currentTab()+\'-\'+(++serial);targets.push({target,label});}\n  const key=currentTab()+\'|\'+targets.map(t=>t.target.id+\':\'+t.label).join(\'|\');if(key===signature&&targets.length===lastTargets.length&&targets.every((t,i)=>t.target===lastTargets[i])&&(targets.length<2||content.querySelector(\'.a34-subnav\')))return;signature=key;lastTargets=targets.map(t=>t.target);\n  content.querySelector(\'.a34-subnav\')?.remove();content.querySelector(\'.a34-related\')?.remove();\n  if(targets.length>1){const nav=document.createElement(\'nav\');nav.className=\'a34-subnav\';nav.setAttribute(\'aria-label\',\'Temas de esta sección\');let links=targets;\n   if(currentTab()===\'partidos\'){\n    const offer=targets.find(t=>/oferta|cobertura|uniprovinc|multiprovinc|participa/.test(normalize(t.label))),leaders=targets.find(t=>t.target.id===\'atlas-v31-leadership\'),comp=targets.find(t=>t.target.id===\'atlas-v29-composicion\'),bastions=targets.find(t=>t.target.id===\'atlas-v30-bastiones\');\n    links=[offer&&{...offer,label:\'Oferta y cobertura\'},leaders&&{...leaders,label:\'Liderazgos y diferencias\'},comp&&{...comp,label:\'Composición territorial\'},bastions&&{...bastions,label:\'Bastiones\'}].filter(Boolean);\n    const selected=new Set(links.map(t=>t.target));const extra=targets.find(t=>!selected.has(t.target));if(extra)links.push({...extra,label:\'Otras vistas\'});\n   }\n   nav.innerHTML=\'<span class="a34-nav-label">En esta sección</span><div class="a34-topic-links">\'+links.slice(0,10).map(t=>`<button type="button" data-a34-anchor="${esc(t.target.id)}" title="${esc(t.label)}">${esc(t.label)}</button>`).join(\'\')+\'</div><label class="a34-chart-jump"><span>Ir a una gráfica</span><select><option value="">Seleccionar…</option>\'+targets.map(t=>`<option value="${esc(t.target.id)}">${esc(t.label)}</option>`).join(\'\')+\'</select></label>\';\n   const head=content.querySelector(\'.section-head\');if(head)head.after(nav);else content.prepend(nav);\n   const go=id=>{const target=document.getElementById(id);if(target&&content.contains(target)){target.scrollIntoView({behavior:matchMedia(\'(prefers-reduced-motion: reduce)\').matches?\'auto\':\'smooth\',block:\'start\'});target.setAttribute(\'tabindex\',\'-1\');target.focus({preventScroll:true})}};\n   nav.addEventListener(\'click\',e=>{const b=e.target.closest(\'[data-a34-anchor]\');if(b)go(b.dataset.a34Anchor)});nav.querySelector(\'select\').addEventListener(\'change\',e=>{if(e.target.value)go(e.target.value)});\n  }\n  const names=typeof tabs!==\'undefined\'?new Map(tabs):new Map(),dest=(related[currentTab()]||[]).filter(key=>typeof views!==\'undefined\'&&typeof views[key]===\'function\');\n  if(dest.length){const aside=document.createElement(\'nav\');aside.className=\'a34-related\';aside.setAttribute(\'aria-label\',\'Secciones relacionadas\');aside.innerHTML=\'<span>Continuar explorando</span>\'+dest.map(key=>`<button type="button" data-a34-related="${esc(key)}">${esc(names.get(key)||key)}</button>`).join(\'\');aside.addEventListener(\'click\',e=>{const b=e.target.closest(\'[data-a34-related]\');if(b&&typeof gotoTab===\'function\')gotoTab(b.dataset.a34Related)});content.append(aside)}\n }\n function schedule(){if(scheduled)return;scheduled=true;requestAnimationFrame(()=>{scheduled=false;decorate()})}\n if(typeof render===\'function\'){const original=render;render=async function(...args){const result=await original.apply(this,args);schedule();return result}}\n function start(){const content=getContent();if(!content)return;navObserver=new MutationObserver(schedule);navObserver.observe(content,{childList:true});schedule();const filters=document.querySelector(\'.filters\');if(filters&&typeof ResizeObserver===\'function\'){const observer=new ResizeObserver(()=>{const sticky=getComputedStyle(filters).position===\'sticky\';document.documentElement.style.setProperty(\'--atlas-sticky-controls\',sticky?Math.ceil(filters.getBoundingClientRect().height)+\'px\':\'0px\')});observer.observe(filters)}}\n if(document.readyState===\'loading\')document.addEventListener(\'DOMContentLoaded\',start);else start();\n})();\n'
_ATLAS34_DESIGN = ':root{--a34-ink:#17394d;--a34-muted:#567282;--a34-line:#d4e3eb;--a34-teal:#127e89;--atlas-sticky-controls:0px}\nbody{background:#f2f7fa;color:var(--a34-ink);-webkit-font-smoothing:antialiased}button,select,input{font-family:inherit}button:focus-visible,select:focus-visible,input:focus-visible,summary:focus-visible{outline:3px solid #dfad48;outline-offset:3px}button{transition:background-color .15s,border-color .15s,box-shadow .15s}button:disabled{cursor:default;opacity:.55}.masthead{background:linear-gradient(115deg,#fff 10%,#eaf6f8);border-bottom:1px solid var(--a34-line)}.masthead h1{letter-spacing:-1px;line-height:1.15}.filters{background:#fbfdff;border-bottom:1px solid var(--a34-line)}.filters label{gap:7px}.filters select,.filters input,.controls select{border-radius:8px;min-height:38px}main{max-width:1800px;margin-left:auto;margin-right:auto}.section-head{padding-bottom:20px;margin-bottom:22px;border-bottom:1px solid var(--a34-line)}.section-head h2{letter-spacing:-.55px;line-height:1.2}.card{border:1px solid var(--a34-line);border-radius:14px;background:#fff;box-shadow:0 4px 18px rgba(24,62,80,.035);padding:22px}.card-head{gap:18px;align-items:flex-start}.card-head h3{line-height:1.4;letter-spacing:-.2px}.chart-note,.mini{line-height:1.65}.exports,.a25-actions,.a29-export{gap:7px;flex-wrap:wrap}.exports button,.a25-actions button,.a29-export button{border:1px solid #bed6df;border-radius:8px;background:#fff;color:#20566b;font-size:11px;font-weight:750;padding:8px 12px;cursor:pointer}.exports button:hover,.a25-actions button:hover,.a29-export button:hover{background:#edf7f9;border-color:#12848f}.grid{gap:22px}.kpis{gap:15px}.kpi{border:1px solid var(--a34-line);border-top:3px solid #198d99;border-radius:12px}.kpi strong{font-variant-numeric:tabular-nums}.a34-subnav{position:sticky;top:var(--atlas-sticky-controls);z-index:15;display:flex;align-items:center;gap:14px;flex-wrap:wrap;background:rgba(255,255,255,.98);border:1px solid var(--a34-line);border-radius:12px;padding:12px 15px;margin:0 0 26px;box-shadow:0 5px 18px rgba(25,65,83,.06)}.a34-nav-label{font-size:10px;font-weight:850;letter-spacing:.9px;text-transform:uppercase;color:#527486;white-space:nowrap}.a34-topic-links{display:flex;gap:7px;overflow-x:auto;flex:1;min-width:180px;padding:2px}.a34-topic-links button,.a34-related button{border:1px solid #cee0e8;border-radius:8px;background:#f3f9fb;color:#245b70;padding:9px 12px;font-size:11px;font-weight:750;white-space:nowrap;cursor:pointer}.a34-topic-links button:hover,.a34-related button:hover{background:#e3f2f5;border-color:#178a95}.a34-chart-jump{display:flex;align-items:center;gap:8px;font-size:10px;color:#527486;max-width:310px}.a34-chart-jump select{min-width:120px;max-width:240px;background:white;border:1px solid #c7dce5;border-radius:7px;padding:8px;color:#214b60;font-size:11px}.a34-related{clear:both;display:flex;align-items:center;flex-wrap:wrap;gap:10px;border-top:1px solid var(--a34-line);padding:22px 0 8px;margin-top:32px}.a34-related>span{font-size:12px;color:#527486;margin-right:6px}.a34-definition summary{cursor:pointer;font-weight:750;font-size:12px;color:#245c70}.a34-definition p{font-size:12px;line-height:1.8;margin:13px 0 0}.atlas-v2317-dimensiones{padding:13px 17px!important;background:#f2f8fa!important;border:1px solid #d6e6ed!important;border-left:3px solid #198d99!important;box-shadow:none!important}main section[id],main article[id]{scroll-margin-top:calc(var(--atlas-sticky-controls) + 90px)}#atlas-v30-bastiones{margin-top:30px}#atlas-v30-bastiones .a30-hero p{color:#e1f1f5!important}#atlas-v29-main .a29-input[hidden]{display:none!important}.footerline{line-height:1.8;border-top:1px solid var(--a34-line);font-size:11px;color:#567282}\n@media(max-width:850px){.a34-subnav{position:relative;top:auto}.a34-chart-jump{width:100%;max-width:none}.a34-chart-jump select{flex:1;max-width:none}.card{padding:16px}.a34-related>span{flex-basis:100%}}\n@media(prefers-reduced-motion:reduce){*,*::before,*::after{scroll-behavior:auto!important;transition:none!important}}\n'
_ATLAS34_DESIGN += '\n#atlas-v27-participacion .a27-controls>label[hidden]{display:none!important}\n'
_ATLAS34_DESIGN += '\n/* La navegación temática conserva espacio propio en el flujo de lectura. */\n.a34-subnav{position:static!important;top:auto!important;z-index:auto!important;box-sizing:border-box;width:100%;margin:18px 0 28px!important;padding:15px 17px;clear:both;flex-wrap:wrap;min-height:65px}.a34-topic-links{flex:1 1 680px;max-width:100%;min-width:0;padding:4px 0 8px}.a34-chart-jump{flex:1 1 260px;max-width:360px;min-width:220px}.a34-chart-jump select{flex:1;min-width:0}.section-head+.a34-subnav{margin-top:0!important}.a34-nav-label{flex:0 0 auto}.a34-subnav+.kpis{margin-top:0}\n'
_ATLAS34_DESIGN += '.a37-workspace{margin:20px 0 28px;padding-bottom:20px}.a37-toolbar{display:flex;gap:14px;flex-wrap:wrap;align-items:end;padding:16px 20px;background:#f2f8fa;border-block:1px solid #dce9ee}.a37-toolbar label{display:flex;flex-direction:column;gap:7px;min-width:170px;flex:1 1 180px;font-size:12px;font-weight:700;color:#31586b}.a37-toolbar select,.a37-toolbar input,.a37-map-context select{width:100%;box-sizing:border-box;padding:10px 12px;border:1px solid #bbd3df;border-radius:7px;background:white;color:#21485b;font:inherit}.a37-toolbar [hidden],.a37-workspace [hidden],.a37-map-context [hidden]{display:none!important}.a37-workspace button,.a37-map-context button{border:1px solid #b8d4e0;border-radius:7px;background:white;color:#1d5b70;padding:8px 12px;cursor:pointer;font:inherit;font-size:12px;font-weight:700}.a37-workspace button:hover,.a37-map-context button:hover{background:#eaf4f7}.a37-workspace button:disabled{opacity:.4;cursor:default}.a37-workspace button:focus-visible,.a37-workspace select:focus-visible,.a37-workspace input:focus-visible,.a37-map-context button:focus-visible{outline:3px solid #93c6d5;outline-offset:2px}.a37-charthead{display:flex;align-items:center;justify-content:space-between;gap:20px;padding:18px 20px}.a37-charthead h4{margin:0;font-size:16px}.a37-scroll{overflow:auto;max-height:640px;margin:0 20px 14px}.a37-table{border-collapse:collapse;width:100%;font-size:12px}.a37-table th,.a37-table td{padding:11px 14px;border-bottom:1px solid #e1ebf0;text-align:left;vertical-align:top}.a37-table thead th{background:#eaf3f6;position:sticky;top:0;z-index:1}.a37-table tbody tr:nth-child(even){background:#f8fbfc}.a37-method{margin:18px 20px;padding:12px 15px;border-left:3px solid #278b94;background:#f1f8f9;font-size:12px;line-height:1.7}.a37-detail{margin:22px 20px;border-top:1px solid #d9e8ee;padding-top:12px}.a37-person{border:1px solid #d9e8ee;padding:15px;border-radius:8px;background:#fbfdfe}.a37-person h4{margin:0 0 8px}.a37-detail [data37=person-list]{display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:12px}.a37-records{margin:20px}.a37-records summary{color:#126d7a;font-weight:700;cursor:pointer}.a37-records p{font-size:12px;line-height:1.7}.a37-formulas{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:16px;padding:20px}.a37-ticket{border:1px solid #d5e6ed;border-radius:9px;overflow:hidden;background:#fff}.a37-ticket header{padding:14px 16px;background:#edf6f8;display:flex;align-items:center;justify-content:space-between;gap:12px}.a37-ticket h4{font-size:14px;margin:0}.a37-ticket header span{font-size:11px;color:#2d6875}.a37-ticket>div{padding:12px 16px;border-top:1px solid #edf2f5}.a37-ticket small,.a37-ticket strong{display:block}.a37-ticket small{font-size:11px;color:#607d8c;margin-bottom:5px}.a37-ticket strong{font-size:13px}.a37-map-context{display:flex;flex-wrap:wrap;align-items:center;gap:15px;border:1px solid #d3e7ef;border-radius:9px;background:#f4fafc;margin:18px 0;padding:16px 20px}.a37-map-context label{display:flex;align-items:center;gap:12px;font-weight:700;font-size:12px}.a37-map-context select{width:110px}.a37-map-context p{margin:0;flex:1 1 280px;font-size:12px}.a37-map-context>span{font-size:12px;color:#537587}#atlas37-profile-territory .a26-chart-title{padding:15px 20px}#atlas37-profile-territory th,#atlas37-profile-territory td{padding:12px 15px}.a37-workspace .a26-metric-cards{padding:16px 20px}.a37-workspace .a25-header{padding:20px}.a37-workspace .a25-actions button{font-size:11px}body[data-atlas-tab*="Perfiles"] .a26-profile-swatch{display:none}@media(max-width:700px){.a37-toolbar{padding:12px}.a37-toolbar label{min-width:140px}.a37-charthead{flex-wrap:wrap}.a37-formulas{grid-template-columns:1fr;padding:12px}.a37-detail{margin:15px 10px}}\n'
_ATLAS34_DESIGN += '.a38-link{border:0!important;background:transparent!important;color:#086c7b!important;padding:0!important;font:inherit!important;font-weight:700!important;text-decoration:underline;text-decoration-color:#a4d1d7;text-underline-offset:3px;cursor:pointer}.a38-link:hover{color:#104b62!important;text-decoration-color:currentColor}.a38-link:focus-visible{outline:3px solid #a5d0df;outline-offset:4px;border-radius:2px}.a38-context-links{display:flex;flex-wrap:wrap;gap:11px 18px;margin:16px 0;padding:16px 20px;border:1px solid #d8e8ee;background:#f4fafb;border-radius:9px}.a38-context-links button{font-size:12px!important}.a38-head{display:flex;justify-content:space-between;align-items:start;gap:20px;flex-wrap:wrap;padding:20px}.a38-head h3{font-size:18px;margin:0 0 8px;color:#21485b}.a38-head p{font-size:12px;line-height:1.7;margin:0;color:#557386}.a38-card{margin:22px 0;overflow:visible}.a38-card [data-a38-plot]{min-height:300px;width:100%}.a38-map-controls{display:grid;grid-template-columns:repeat(auto-fit,minmax(185px,1fr));gap:16px;background:#f1f8fa;border:1px solid #d7e7ee;border-radius:10px;padding:20px;margin:18px 0}.a38-map-controls label{display:flex;flex-direction:column;gap:8px;font-size:12px;font-weight:700;color:#345d70}.a38-map-controls select{padding:10px 12px;background:white;border:1px solid #bcd5e1;border-radius:7px;color:#244c60;max-width:100%;min-width:0}.a38-map-controls [hidden],.a38-party-analysis [hidden],.a38-multivariables[hidden],.a38-reference[hidden]{display:none!important}.a38-province-focus{border:1px solid #cce3eb;border-radius:10px;padding-bottom:15px;background:#f8fcfd;margin:20px 0}.a38-province-tabs,.a38-subtabs{display:flex;gap:8px;flex-wrap:wrap}.a38-province-tabs button,.a38-subtabs button{border:1px solid #bdd8e2;border-radius:7px;background:#fff;padding:9px 13px;font-weight:700;font-size:12px;color:#285d72;cursor:pointer}.a38-province-tabs button[aria-pressed=true],.a38-subtabs button[aria-pressed=true]{background:#126e80;color:white;border-color:#126e80}.a38-province-focus .a26-metric-cards{padding:0 20px}.a38-province-focus .a38-context-links{margin:15px 20px}.a38-party-analysis{margin:20px 0 30px;border:1px solid #d3e6ed;border-radius:12px;padding:0 18px 20px;background:#fafdfe}.a38-party-analysis>.a38-context-links{margin:0 0 20px}.a38-subtabs{padding-bottom:12px;border-bottom:1px solid #d9e9ef}.a38-person-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(250px,1fr));gap:12px;margin:15px 0;max-height:850px;overflow:auto}.a38-result{font-size:11px;border-radius:20px;background:#edf3f6;color:#526b7a;padding:5px 9px;display:inline-block;margin:6px 0 12px}.a38-result.won{background:#dff1eb;color:#176954}.a38-reference{padding:16px 20px;border-inline-start:4px solid #17818e;margin:16px 20px;background:#edf7f8;border-radius:7px}.a38-reference p{font-size:12px;margin:8px 0}.a38-reference small{font-size:11px;color:#537487}.a38-check{align-self:center;flex-direction:row!important;align-items:center}.a38-check input{width:auto!important}.a38-multivariables{padding:15px 20px;background:#f7fbfc;border-bottom:1px solid #dcebf0}.a38-multivariables>strong{display:block;font-size:12px;margin-bottom:12px}.a38-multivariables>div{display:flex;flex-wrap:wrap;gap:9px 18px;max-height:220px;overflow:auto}.a38-multivariables label{display:flex;gap:7px;font-size:12px;align-items:center}.a38-loadings{margin:15px 20px;padding:15px;border:1px solid #d6e7ed;border-radius:8px}.a38-loadings summary{font-weight:700;color:#176d7a;cursor:pointer;margin-bottom:15px}.a38-person-grid .a37-person h4{line-height:1.6}.a37-person h4 .a38-link{font-size:14px!important}.a37-person small .a38-link{font-size:11px!important}@media(max-width:700px){.a38-head{padding:15px}.a38-context-links{padding:12px}.a38-map-controls{padding:14px;grid-template-columns:1fr 1fr}.a38-person-grid{grid-template-columns:1fr}.a38-party-analysis{padding:0 10px 12px}.a38-head .a25-actions{margin-left:auto}}\n'
PUBLIC_APPLICATION = _atlas34_root_application(_ATLAS29_OUTPUT, PROJECT_DIR)


Aplicación pública integrada: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/Atlas_Electoral_Costa_Rica_1953_2026.html
Sección 7: mapa, barras, matriz territorial y exportación en un solo explorador.
Sección 8: liderazgos, composición temporal y análisis de bastiones con tres modalidades.
No se instalaron ni escribieron archivos JavaScript o CSS adicionales.
Aplicación pública: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/Atlas_Electoral_Costa_Rica_1953_2026.html
Datos derivados y archivos descargables: /Users/juani/Documents/GitHub/atlas-electoral-costa-rica/salidas/datos
